In [1]:
# =============================================================================
#  TITANIUM-6 | RUN CONFIG | R8.0
#  Run this FIRST. It declares ONE configuration for the whole pipeline and
#  writes it to /kaggle/working/t6_active_config.json. Every layer file
#  (01_... to 08_...) reads that file at start, so each layer can run in the
#  same kernel, in a fresh kernel after a restart (frees RAM), or as a script,
#  and a table in the paper can always be traced to exactly one profile.
#  To switch profile: edit PROFILE below and run this file again.
#
#  PROFILES
#   'main'          : the paper's primary run. Run cells L1 -> L7, then SUMMARY.
#   'global_split'  : reviewer comment "per-class vs global chronology".
#                     Run L1 + L2 only.
#   'invalid_median': invalid-value policy sensitivity.  Run L1 + L2 only.
#   'invalid_zero'  : (the v1 behaviour: Inf -> 0).       Run L1 + L2 only.
#   'embargo_60s'   : burst-leakage sensitivity.          Run L1 + L2 only.
#   'no_shortcut'   : drops TCP-stack/environment features (Init Win Byts,
#                     Min Seg Size Forward) before selection. Reuses the MAIN
#                     Layer-1 output. Run L2 + L3 + 09 cross-dataset.
#   09_cross_dataset.py (CIC-IDS2017) runs after L3 in 'main' and 'no_shortcut'.
#
#  DISK: /kaggle/working holds ~20 GB. One Layer-1 output is several GB, so run
#  each L1 sensitivity profile in its own Kaggle session (or delete the
#  previous l1_R8_* directory first). 'main' and 'no_shortcut' share L1.
# =============================================================================
import os, sys, json, platform, time, importlib
from pathlib import Path

PROFILE = 'main'          # <- change here, then re-run this cell

W = '/kaggle/working'
SHORTCUT_FEATURES = ['Init Fwd Win Byts', 'Init Bwd Win Byts',
                     'Init Fwd Win Byts__present', 'Init Bwd Win Byts__present',
                     'Min Seg Size Forward']

PROFILES = {
    'main':           dict(l1='l1_R8', sfx='',           split='per_class', inv='semantic', emb='', excl=[]),
    'global_split':   dict(l1='l1_R8_global', sfx='_global', split='global', inv='semantic', emb='', excl=[]),
    'invalid_median': dict(l1='l1_R8_invmed', sfx='_invmed', split='per_class', inv='median', emb='', excl=[]),
    'invalid_zero':   dict(l1='l1_R8_invzero', sfx='_invzero', split='per_class', inv='zero', emb='', excl=[]),
    'embargo_60s':    dict(l1='l1_R8_emb60', sfx='_emb60', split='per_class', inv='semantic', emb='60', excl=[]),
    'no_shortcut':    dict(l1='l1_R8', sfx='_noshortcut', split='per_class', inv='semantic', emb='', excl=SHORTCUT_FEATURES),
}
if PROFILE not in PROFILES:
    raise SystemExit(f'[FATAL] unknown PROFILE {PROFILE!r}; choose one of {sorted(PROFILES)}')
P = PROFILES[PROFILE]

env = {
    'T6_PROFILE': PROFILE,
    'T6_L1': f'{W}/{P["l1"]}',
    'T6_L2': f'{W}/l2_R8{P["sfx"]}',
    'T6_L3': f'{W}/l3_R8{P["sfx"]}',
    'T6_L4': f'{W}/l4_R8{P["sfx"]}',
    'T6_L5': f'{W}/l5_R8{P["sfx"]}',
    'T6_L6': f'{W}/l6_R8{P["sfx"]}',
    'T6_L7': f'{W}/l7_R8{P["sfx"]}',
    'T6_X2017': f'{W}/x2017_R8{P["sfx"]}',
    'T6_SPLIT_MODE': P['split'],
    'T6_INVALID_POLICY': P['inv'],
    'T6_EMBARGO_S': P['emb'],
    'T6_EXCLUDE_FEATURES': ','.join(P['excl']),
    'T6_ALLOW_MANIFEST_SEARCH': '0',
    'PYTHONHASHSEED': '42',
}
# Layer 2 times the true raw->decision path with Layer 1's OnlineTransformer.
# In a fresh kernel it needs the Layer-1 file; find it next to this file.
def _find_l1_code():
    here = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    cands = [Path(os.environ.get('T6_CODE_DIR', here)) / '01_layer1_preprocessing.py',
             here / '01_layer1_preprocessing.py', Path(W) / '01_layer1_preprocessing.py']
    hit = next((str(c) for c in cands if c.exists()), '')
    if hit:
        return hit
    # Kaggle mounts inputs through symlinks, which Path.glob('**') does not follow
    for root, _dirs, files in os.walk('/kaggle/input', followlinks=True):
        if '01_layer1_preprocessing.py' in files:
            return str(Path(root) / '01_layer1_preprocessing.py')
    return ''

env['T6_L1_CODE'] = _find_l1_code()
os.environ.update(env)          # every key is set explicitly: no leftovers

def _ver(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return 'not installed'

record = {
    'profile': PROFILE, 'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'env': env, 'python': platform.python_version(), 'platform': platform.platform(),
    'libraries': {m: _ver(m) for m in ('numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn',
                                       'lightgbm', 'xgboost', 'torch', 'shap',
                                       'fastapi', 'matplotlib')},
}
Path(W).mkdir(parents=True, exist_ok=True)
with open(f'{W}/run_config_{PROFILE}.json', 'w') as fh:
    json.dump(record, fh, indent=2)
with open(f'{W}/t6_active_config.json', 'w') as fh:       # read by every layer
    json.dump(record, fh, indent=2)

print('=' * 76)
print(f'  TITANIUM-6 R8.0 | PROFILE = {PROFILE}')
print('=' * 76)
for k in ('T6_L1', 'T6_L2', 'T6_L3', 'T6_L4', 'T6_L5', 'T6_L6', 'T6_L7', 'T6_X2017'):
    print(f'  {k:<6} {env[k]}')
print(f'  split={P["split"]} | invalid_policy={P["inv"]} | embargo_s={P["emb"] or 0} | '
      f'excluded={P["excl"] or "none"}')
print('  libraries: ' + ', '.join(f'{k} {v}' for k, v in record['libraries'].items()))
print(f'  L1 code for L2 latency: {env["T6_L1_CODE"] or "not found (L2 skips raw->decision timing)"}')
print(f'  active config: {W}/t6_active_config.json  (read by every layer)')
print('=' * 76)

  TITANIUM-6 R8.0 | PROFILE = main
  T6_L1  /kaggle/working/l1_R8
  T6_L2  /kaggle/working/l2_R8
  T6_L3  /kaggle/working/l3_R8
  T6_L4  /kaggle/working/l4_R8
  T6_L5  /kaggle/working/l5_R8
  T6_L6  /kaggle/working/l6_R8
  T6_L7  /kaggle/working/l7_R8
  T6_X2017 /kaggle/working/x2017_R8
  split=per_class | invalid_policy=semantic | embargo_s=0 | excluded=none
  libraries: numpy 2.0.2, pandas 2.3.3, pyarrow 24.0.0, scipy 1.16.3, sklearn 1.6.1, lightgbm 4.6.0, xgboost 3.2.0, torch 2.10.0+cu128, shap 0.51.0, fastapi 0.136.1, matplotlib 3.10.0
  L1 code for L2 latency: not found (L2 skips raw->decision timing)
  active config: /kaggle/working/t6_active_config.json  (read by every layer)


In [2]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 1 | R8.0
#  R8.0 = R5.4 logic unchanged. Only the run configuration changed: output
#  directory, split mode and invalid-value policy are read from the RUN
#  CONFIG cell (T6_L1, T6_SPLIT_MODE, T6_INVALID_POLICY, T6_EMBARGO_S) so the
#  main run and every sensitivity run are one declared configuration each.
#  R5.4 = R5.2 (every G/H fix intact) + the three polish items proposed as R5.3:
#   P1 tuple-safe JSON: tuples serialise as lists, not as their repr string.
#   P2 _fix_ts tolerates class names containing '_' and malformed stems.
#   P3 scipy version recorded in _hardware() as well as the environment block.
#  WARNING: a circulated R5.3 draft dropped five R5.2 fixes - the NOT-MEASURED
#  warning and classes_not_measured for boundary leakage, the empty-histogram
#  guard in _decide_clock, the class_hashes free, and the corrected embargo
#  wording. All are retained here. Do not merge that draft over this file.
#  R5.2 closes the three items that kept R5.1 short of complete. None of them
#  were code faults; each is now MEASURED instead of claimed or assumed.
#   G1  Burst snapping is ineffective on this dataset (real run: straddling
#       mass 2,532,984 -> 2,530,371, 0.1%, with 9/12 attack classes still cut
#       inside a burst). Snapping is therefore no longer presented as a
#       leakage control. What actually matters at a boundary is measured:
#       BOUNDARY-ADJACENCY LEAKAGE = the share of rows just after a cut whose
#       exact feature twin lies just before it, over 1/10/60/300-second
#       windows. This states exactly what an embargo of width W would remove
#       and what snapping failed to prevent.
#   G2  The 12-hour-clock correction was inference. It is now FALSIFIABLE:
#       the manifest records the hours-6-7 occupancy that the hypothesis
#       predicts must be empty, how many rows move, and their class split, so
#       a reader can check the assumption instead of trusting it.
#   G3  Label quality (comment 39) was a lower bound with no review path. A
#       prioritised label_review_worklist.csv is now emitted: the largest
#       conflicting class pairs and rule-violating classes first, so the
#       manual pass is bounded work rather than an open task.
#  R5.1 (v7.1) | CSE-CIC-IDS2018 | IEEE OJ-COMS
#  R5.0 over R4.3 (each item evidenced by the real Kaggle run or by code audit):
#   F1  CRASH: Phase 2 initialised known_ts as dict then .append() -> fixed.
#   F2  12-hour clock: observed span 01:00:00-12:59:59 on real data. Hours are
#       now histogrammed per file/class; if no hour > 12 exists, PM-correction
#       hypotheses are scored against the documented CIC attack schedule and
#       applied ONLY if alignment improves >= 10 pp and reaches >= 0.70.
#       Decision + all scores recorded in the manifest.
#   F3  Invalid-value audit by CAUSE: the -1 sentinel (not applicable, encoded
#       by __present) is no longer counted as invalid; NaN, +Inf, absent
#       column counted separately. Batch and online paths changed together.
#   F4  _ts added to RESERVED_COLS (never a feature).
#   F5  B1 regression restored: exact training-class count for quotas.
#   F6  KS: 5 disjoint uniform subsamples per split -> mean +/- std per
#       feature (one pass); stage-1 selection reads ks_mean.
#   F7  Straddle mass reported for ATTACK classes (Normal is continuous).
#   F8  Snap cap 0.5% (canonical embargo is set in R5.1 item E1) (real run showed the 2% cap
#       moved val to 13.7% / test to 15.6% with unequal prevalence).
#   F9  raw_sample.csv written (true raw->decision latency downstream).
#   R5.1 (from the real R5.0 Kaggle run):
#   E1  Canonical embargo = 0 s. At 60 s the embargo removed 225,027 rows, ~99%
#       of them attack flows adjacent to the cut (~29% of val/test attacks),
#       while the straddle mass barely changed (2,532,984 -> 2,530,371) and the
#       dominant leakage channel is exact twins (38% of test), which an embargo
#       cannot touch. 60 s is kept as the burst-leakage SENSITIVITY run
#       (--embargo-s 60), reported beside the canonical run (comment 33).
#   E2  Twin-label census: for every val/test/zero-day row with an exact twin in
#       train, the twin's training label (Normal / same class / other attack /
#       mixed). Decides what zero-day detection can legitimately claim.
#   F10 schema_version 5; legacy ks_drift alias removed; no blanket warning
#       suppression (verified with python -W error).
#  R4.3 = R4.2 + two-line fix in _phase4_ks (self.ks_train_val / ks_train_test
#         were returned as locals but read as attributes by _fig_ks_scatter).
#         Every other self.X reference audited; no other missing attributes.
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os, gc, json, time, hashlib, platform, warnings, argparse
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import psutil, scipy
from scipy.stats import ks_2samp, skew, kurtosis
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

pd.options.mode.chained_assignment = None
SEED = 42; np.random.seed(SEED)

CFG = {
    'version':'TITANIUM6_LAYER1_R8.0','seed':SEED,
    'chunk_size':100_000,'buffer_flush_rows':300_000,
    'train_frac':0.70,'val_frac':0.15,
    'split_mode':'per_class','burst_aware':True,
    'burst_gap_s':1.0,'burst_gap_grid':[0.1,1.0,10.0,60.0],
    'max_snap_frac':0.005,'embargo_s':0.0,
    'ts_formats':['%d/%m/%Y %H:%M:%S','%d/%m/%Y %I:%M:%S %p',
                  '%m/%d/%Y %H:%M:%S','%m/%d/%Y %I:%M:%S %p',
                  '%Y-%m-%d %H:%M:%S'],
    'ts_window_days':1,'ts_min_in_window':0.50,'ts_warn_in_window':0.99,
    'max_bad_ts_frac':1e-4,'invalid_policy':'semantic','median_sample_n':2_000_000,
    'ks_alpha':0.05,'ks_sample_per_split':30_000,'ks_seeds':5,
    'clock_pm_cuts':[5,6,7,8,9,10],'clock_offsets':list(range(-6,7)),
    'clock_min_gain':0.10,'clock_min_align':0.70,'bootstrap_block_minutes':60,
    'shuffle_buckets':64,'shuffle_max_tv':0.05,
    'raw_sample_n':2_000,'audit_rows_per_class':20,
    'latency_calls':10_000,'latency_warmup':500,'max_ram_pct':80,
    'full_zero_day':['SQL-Injection','BruteForce-Web','BruteForce-XSS'],
    'partial_zero_day':{'Infiltration':0.50,'Botnet':0.50},
}

LABEL_MAP = {
    'benign':'Normal','normal':'Normal',
    'ftp-bruteforce':'BruteForce-FTP','ssh-bruteforce':'BruteForce-SSH',
    'brute force -web':'BruteForce-Web','brute force -xss':'BruteForce-XSS',
    'dos attacks-goldeneye':'DoS-GoldenEye','dos attacks-slowloris':'DoS-Slowloris',
    'dos attacks-slowhttptest':'DoS-SlowHTTP','dos attacks-hulk':'DoS-Hulk',
    'ddos attacks-loic-http':'DDoS-LOIC-HTTP','ddos attack-hoic':'DDoS-HOIC',
    'ddos attack-loic-udp':'DDoS-LOIC-UDP','sql injection':'SQL-Injection',
    'infilteration':'Infiltration','infiltration':'Infiltration',
    'bot':'Botnet','botnet':'Botnet',
}

COL_NORM = {
    'attack_type':'Attack_Type','dst port':'Dst Port','destination port':'Dst Port',
    'src port':'Src Port','source port':'Src Port','src ip':'Src IP','source ip':'Src IP',
    'dst ip':'Dst IP','destination ip':'Dst IP','timestamp':'Timestamp','flow id':'Flow ID',
    'fwd header len':'Fwd Header Length','bwd header len':'Bwd Header Length',
    'flow byts/s':'Flow Bytes/s','flow pkts/s':'Flow Packets/s',
    'tot fwd pkts':'Total Fwd Packets','tot bwd pkts':'Total Backward Packets',
    'totlen fwd pkts':'Total Length of Fwd Packets',
    'totlen bwd pkts':'Total Length of Bwd Packets',
    'fwd pkt len max':'Fwd Packet Length Max','fwd pkt len min':'Fwd Packet Length Min',
    'fwd pkt len mean':'Fwd Packet Length Mean','fwd pkt len std':'Fwd Packet Length Std',
    'bwd pkt len max':'Bwd Packet Length Max','bwd pkt len min':'Bwd Packet Length Min',
    'bwd pkt len mean':'Bwd Packet Length Mean','bwd pkt len std':'Bwd Packet Length Std',
    'pkt len min':'Packet Length Min','pkt len max':'Packet Length Max',
    'pkt len mean':'Packet Length Mean','pkt len std':'Packet Length Std',
    'pkt len var':'Packet Length Variance','pkt size avg':'Packet Size Average',
    'cwe flag count':'CWR Flag Count','ece flag cnt':'ECE Flag Count',
    'subflow fwd byts':'Subflow Fwd Bytes','subflow bwd byts':'Subflow Bwd Bytes',
    'subflow fwd pkts':'Subflow Fwd Packets','subflow bwd pkts':'Subflow Bwd Packets',
    'init_win_bytes_forward':'Init Fwd Win Byts','init_win_bytes_backward':'Init Bwd Win Byts',
    'act_data_pkt_fwd':'Act Data Pkt Fwd','min_seg_size_forward':'Min Seg Size Forward',
    'fwd act data pkts':'Act Data Pkt Fwd','fwd seg size min':'Min Seg Size Forward',
    'fwd pkts/s':'Fwd Pkts/s','bwd pkts/s':'Bwd Pkts/s',
}

META_COLS = {'Flow ID','Src IP','Dst IP','Src Port','Dst Port','Timestamp','_ts'}
DROP_ALWAYS = {'SimillarHTTP','Unnamed: 0','index','Label','label','Attack_Type',
               'attack_type','Split','Flow_Row_ID'}
SENTINEL_MINUS1 = ('Init Fwd Win Byts','Init Bwd Win Byts')

DERIVED_FEATURES = [
    'Init Fwd Win Byts__present','Init Bwd Win Byts__present',
    'Ratio__FwdBwd_Packets','Ratio__FwdBwd_Bytes','Ratio__FwdBwd_IAT',
    'Ratio__Active_Idle','Log_Flow_Duration','Zero_Duration_Flag',
    'N_Invalid_Features','N_Clipped_Negatives',
]
COUNTER_FEATURES = ('N_Invalid_Features','N_Clipped_Negatives')
RESERVED_COLS = ['_dup_in_train','_ts']
# Documented CSE-CIC-IDS2018 attack schedule (UNB/CIC dataset page; local
# time, decimal hours). Used ONLY to test 12-hour-clock hypotheses.
DOC_WINDOWS = {
    ('02-14','BruteForce-FTP'):[(10.53,12.15)], ('02-14','BruteForce-SSH'):[(14.02,15.52)],
    ('02-15','DoS-GoldenEye'):[(9.43,10.15)],   ('02-15','DoS-Slowloris'):[(10.98,11.67)],
    ('02-16','DoS-SlowHTTP'):[(10.20,11.13)],   ('02-16','DoS-Hulk'):[(13.75,14.32)],
    ('02-20','DDoS-LOIC-HTTP'):[(10.20,11.28)], ('02-20','DDoS-LOIC-UDP'):[(13.22,13.53)],
    ('02-21','DDoS-LOIC-UDP'):[(10.15,10.72)],  ('02-21','DDoS-HOIC'):[(14.08,15.08)],
    ('02-22','BruteForce-Web'):[(10.28,11.40)], ('02-22','BruteForce-XSS'):[(13.83,14.48)],
    ('02-22','SQL-Injection'):[(16.25,16.48)],  ('02-23','BruteForce-Web'):[(10.05,11.05)],
    ('02-23','BruteForce-XSS'):[(13.00,14.17)], ('02-23','SQL-Injection'):[(15.08,15.30)],
    ('02-28','Infiltration'):[(10.83,12.08),(13.70,14.67)],
    ('03-01','Infiltration'):[(9.95,10.92),(14.00,15.62)],
    ('03-02','Botnet'):[(10.18,11.57),(14.40,15.92)],
}
NS_H = 3_600_000_000_000

RATIO_SPECS = {
    'Ratio__FwdBwd_Packets':('Total Fwd Packets','Total Backward Packets'),
    'Ratio__FwdBwd_Bytes':('Total Length of Fwd Packets','Total Length of Bwd Packets'),
    'Ratio__FwdBwd_IAT':('Fwd IAT Tot','Bwd IAT Tot'),
    'Ratio__Active_Idle':('Active Mean','Idle Mean'),
}
_HTTP = ({6},{80,443,8080})
LABEL_RULES = {
    'BruteForce-FTP':({6},{21}),'BruteForce-SSH':({6},{22}),
    'DDoS-LOIC-UDP':({17},None),
    'DoS-GoldenEye':_HTTP,'DoS-Hulk':_HTTP,'DoS-Slowloris':_HTTP,
    'DoS-SlowHTTP':_HTTP,'DDoS-LOIC-HTTP':_HTTP,'DDoS-HOIC':_HTTP,
    'BruteForce-Web':_HTTP,'BruteForce-XSS':_HTTP,'SQL-Injection':_HTTP,
}
COLORS = {'train':'#002F6C','val':'#007A33','test':'#4A90D9',
          'zday':'#C00000','part':'#F5A623','edge':'#333333'}
TR,VA,TE,ZD,EMB = 0,1,2,3,4
SPLIT_NAMES = {TR:'train',VA:'val',TE:'test',ZD:'zero_day'}

def _gc(): gc.collect()
def _ram(): return psutil.virtual_memory().percent
def _mkdir(p): Path(p).mkdir(parents=True,exist_ok=True); return Path(p)
def _ram_guard():
    if psutil.virtual_memory().percent > CFG['max_ram_pct']: _gc()
def _stable_int(s): return int(hashlib.md5(s.encode()).hexdigest()[:8],16)
def _normalize_cols(cols):
    return [COL_NORM.get(str(c).strip().lower(),str(c).strip()) for c in cols]
def _to_float32(series):
    s = pd.to_numeric(series,errors='coerce').astype(np.float32)
    return s.mask(s==-np.inf,np.nan)
def _is_ghost_row(val):
    return str(val).strip().lower() in ('label','attack_type','class','target')
def _read_json(path):
    with open(path,encoding='utf-8') as fh: return json.load(fh)

def _json_default(o):
    # P1: tuples become JSON lists, never their Python repr string.
    if isinstance(o,tuple): return list(o)
    if isinstance(o,(np.floating,np.integer)): return o.item()
    if isinstance(o,np.ndarray): return o.tolist()
    if hasattr(o,'item'): return o.item()
    return str(o)


def _atomic_json(payload,dest):
    tmp = Path(dest).parent/(Path(dest).name+'.__tmp__')
    with open(tmp,'w',encoding='utf-8') as fh:
        json.dump(payload,fh,indent=2,default=_json_default)
    os.replace(tmp,dest)
def _sha256(fp):
    h = hashlib.sha256()
    with open(fp,'rb') as fh:
        for blk in iter(lambda: fh.read(1<<20),b''): h.update(blk)
    return h.hexdigest()[:16]
def _file_date(fp):
    for fmt in ('%m-%d-%Y','%Y-%m-%d','%d-%m-%Y'):
        t = pd.to_datetime(fp.stem.split('_')[0],format=fmt,errors='coerce')
        if pd.notna(t): return t
    return None
def _row_hash(df):
    return pd.util.hash_pandas_object(df,index=False).values.astype(np.uint64)
def _hardware():
    cpu = platform.processor() or ''
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    cpu = line.split(':',1)[1].strip(); break
    except OSError: pass
    return {'cpu_model':cpu,'physical_cores':psutil.cpu_count(logical=False),
            'logical_cores':psutil.cpu_count(logical=True),
            'ram_gb':round(psutil.virtual_memory().total/2**30,1),
            'scipy':scipy.__version__}
def _plt_style():
    plt.rcParams.update({'pdf.fonttype':42,'ps.fonttype':42,   # IEEE PDF eXpress: no Type 3
        'font.family':'serif','font.serif':['Times New Roman','Nimbus Roman','Nimbus Roman No9 L',
                                            'STIXGeneral','DejaVu Serif'],
        'mathtext.fontset':'stix','font.size':8,
        'axes.edgecolor':COLORS['edge'],'axes.linewidth':.8,
        'xtick.labelsize':7,'ytick.labelsize':7,'legend.fontsize':7,'figure.dpi':150})
def _uniform_sample_by_class(shards,cols,per_class,seed):
    rng = np.random.RandomState(seed); keep = {}
    for sh in shards:
        df = pd.read_parquet(sh,columns=cols+['Target'])
        keys = rng.random_sample(len(df))
        for cls, idx in df.groupby('Target',sort=False).indices.items():
            k_new,d_new = keys[idx],df.iloc[idx][cols].reset_index(drop=True)
            if cls in keep:
                k_all = np.concatenate([keep[cls][0],k_new])
                d_all = pd.concat([keep[cls][1],d_new],ignore_index=True)
            else: k_all,d_all = k_new,d_new
            if len(k_all) > per_class:
                sel = np.argpartition(k_all,per_class)[:per_class]
                k_all,d_all = k_all[sel],d_all.iloc[sel].reset_index(drop=True)
            keep[cls] = (k_all,d_all)
        del df,keys; _gc()
    return {c:v[1] for c,v in keep.items()}
def _moments(col):
    """Skewness/kurtosis with an explicit reliability flag. SciPy warns (does not
    fail) on near-constant columns; that warning is captured and REPORTED per
    feature instead of being suppressed globally."""
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        sk, ku = float(skew(col)), float(kurtosis(col))
    ok = not any(issubclass(x.category,RuntimeWarning) for x in w) and np.isfinite(sk) and np.isfinite(ku)
    return {'skewness':sk if ok else None,'kurtosis':ku if ok else None,
            'moments_reliable':bool(ok)}

def _straddle_mass(bounds,n,newb,edges,bsize):
    tot,seen = 0,set()
    for b in bounds:
        if 0 < b < n and not newb[b]:
            bid = int(np.searchsorted(edges,b,side='right')-1)
            if bid not in seen: tot += int(bsize[bid]); seen.add(bid)
    return tot

def _derive(df):
    out = df
    for col in SENTINEL_MINUS1:
        if col in out.columns:
            s = _to_float32(out[col])
            out[f'{col}__present'] = (s.notna() & (s != -1.0)).astype(np.float32)
            out[col] = s.mask(s == -1.0, np.nan)
        else: out[f'{col}__present'] = np.float32(0.0)
    for new_col,(nc,dc) in RATIO_SPECS.items():
        n = (_to_float32(out[nc]) if nc in out.columns
             else pd.Series(np.float32(0.0),index=out.index))
        d = (_to_float32(out[dc]) if dc in out.columns
             else pd.Series(np.float32(1.0),index=out.index))
        out[new_col] = (n/d.mask(d==0.0,1.0)).clip(0.0,1e6).astype(np.float32)
    if 'Flow Duration' in out.columns:
        dur = _to_float32(out['Flow Duration']).clip(lower=0.0)
        out['Log_Flow_Duration'] = np.log1p(dur).astype(np.float32)
        out['Zero_Duration_Flag'] = (dur==0.0).astype(np.float32)
    else:
        out['Log_Flow_Duration'] = np.float32(0.0)
        out['Zero_Duration_Flag'] = np.float32(0.0)
    return out

class OnlineTransformer:
    def __init__(self,features,mean,std,impute_nan,impute_posinf):
        self.features = list(features)
        self.mu = np.array([mean[f] for f in self.features],np.float64)
        self.sd = np.array([std[f] if std[f]>0 else 1.0 for f in self.features])
        self.inan = np.array([impute_nan.get(f,0.0) for f in self.features])
        self.iinf = np.array([impute_posinf.get(f,0.0) for f in self.features])
        self.pos = {f:i for i,f in enumerate(self.features)}
        self.i_ninv = self.pos.get('N_Invalid_Features')
        self.i_nclp = self.pos.get('N_Clipped_Negatives')
        self.norm = {}
    @classmethod
    def from_json(cls,path):
        p = _read_json(path)
        return cls(p['features'],p['mean'],p['std'],p['impute_nan'],p['impute_posinf'])
    @staticmethod
    def _num(v):
        try: x = float(v)
        except (TypeError,ValueError): return np.nan
        x = float(np.float32(x))
        return np.nan if x == -np.inf else x
    def __call__(self,raw):
        g = {}
        for k,v in raw.items():
            nk = self.norm.get(k)
            if nk is None:
                nk = COL_NORM.get(str(k).strip().lower(),str(k).strip())
                self.norm[k] = nk
            g[nk] = v
        num = self._num; d = {}; sent = set()
        for col in SENTINEL_MINUS1:
            v = num(g[col]) if col in g else None
            if v is None: d[f'{col}__present'] = 0.0
            else:
                d[f'{col}__present'] = 1.0 if (v==v and v!=-1.0) else 0.0
                if v == -1.0: d[col] = np.nan; sent.add(col)
                else: d[col] = v
        for new_col,(nc,dc) in RATIO_SPECS.items():
            n = num(g[nc]) if nc in g else 0.0
            dd = num(g[dc]) if dc in g else 1.0
            dd = 1.0 if dd==0.0 else dd
            with np.errstate(invalid='ignore',divide='ignore'):   # IEEE: inf/inf -> NaN, as pandas
                r = np.float32(np.float32(n)/np.float32(dd)) if (n==n and dd==dd) else np.nan
            d[new_col] = float(np.clip(r,0.0,1e6)) if r==r else np.nan
        if 'Flow Duration' in g:
            dur = num(g['Flow Duration'])
            dur = max(dur,0.0) if dur==dur else np.nan
            d['Log_Flow_Duration'] = (float(np.log1p(np.float32(dur)))
                                      if dur==dur else np.nan)
            d['Zero_Duration_Flag'] = 1.0 if dur==0.0 else 0.0
        else: d['Log_Flow_Duration'] = 0.0; d['Zero_Duration_Flag'] = 0.0
        x = np.empty(len(self.features)); n_inv = n_clp = 0
        for i,f in enumerate(self.features):
            if i == self.i_ninv or i == self.i_nclp: continue
            v = d[f] if f in d else (num(g[f]) if f in g else np.nan)
            if f in sent: x[i] = self.inan[i]; continue
            if v != v: n_inv += 1; v = self.inan[i]
            elif v == np.inf: n_inv += 1; v = self.iinf[i]
            elif v < 0.0: n_clp += 1; v = 0.0
            x[i] = v
        if self.i_ninv is not None: x[self.i_ninv] = n_inv
        if self.i_nclp is not None: x[self.i_nclp] = n_clp
        return ((x-self.mu)/self.sd).astype(np.float32)

_ONLINE_CACHE = {}
def transform_single_row(raw_dict,scaler_path):
    t = _ONLINE_CACHE.get(scaler_path)
    if t is None: t = _ONLINE_CACHE[scaler_path] = OnlineTransformer.from_json(scaler_path)
    return t(raw_dict)

class Layer1Pipeline:
    def __init__(self,source_dir,work_dir):
        self.source_dir, self.work_dir = Path(source_dir).resolve(), _mkdir(work_dir).resolve()
        self.splits_dir = _mkdir(self.work_dir/'splits')
        self.fig_dir = _mkdir(self.work_dir/'figures')
        self.tmp_dir = _mkdir(self.work_dir/'_tmp')
        for sp in ('train','val','test'): _mkdir(self.splits_dir/sp)
        self.manifest_path = self.work_dir/'manifest.json'
        self.scaler_path = self.work_dir/'scaler_params.json'
        self.files = (sorted(self.source_dir.glob('*.csv')) or
                      sorted(self.source_dir.rglob('*.csv')))
        if not self.files: raise FileNotFoundError(f'No CSV in {source_dir}.')

        self.rng = np.random.RandomState(SEED)
        self.source_hashes, self.ts_report = {}, {}
        self.MASTER_SCHEMA = self.TMP_SCHEMA = None
        self.selected_features = []
        self.class_totals_clean = defaultdict(int)
        self.feature_stats, self.scaler_mean, self.scaler_std = {}, {}, {}
        self.impute_nan, self.impute_posinf, self.hash_chain = {}, {}, {}
        self.audit = defaultdict(int)
        self.dup_hashes = defaultdict(list)
        self.total_written = defaultdict(int)
        self.split_diag, self.dup_report, self.serve_selftest = {}, {}, {}
        self.label_conflicts, self.xpart_dups, self.latency = {}, {}, {}
        self.label_rules = defaultdict(lambda: defaultdict(int))
        self.raw_keys, self.raw_rows = np.zeros(0), None
        self.audit_keys, self.audit_rows = {}, {}
        self.per_file_class = defaultdict(lambda: defaultdict(int))
        self.per_file_rows = {}
        self.per_feat_invalid = defaultdict(lambda: defaultdict(int))
        self.per_feat_sentinel = defaultdict(int)
        self.hour_hist = defaultdict(lambda: defaultdict(lambda: np.zeros(24,np.int64)))
        self.clock = {'applied':False}
        # R4.3: explicit storage of KS results for _fig_ks_scatter
        self.ks_train_val = []
        self.ks_train_test = []
        self._shard_idx = {s:0 for s in ('train','val','test','zero_day')}
        self._buffers = {s:[] for s in self._shard_idx}
        self._buf_count = {s:0 for s in self._shard_idx}

        print('='*76)
        print(f'  TITANIUM-6 | LAYER 1 | {CFG["version"]} | files={len(self.files)}')
        print(f'  mode={CFG["split_mode"]} embargo={CFG["embargo_s"]}s '
              f'policy={CFG["invalid_policy"]}')
        print('='*76)

    def _phase0(self):
        print('\n[PHASE 0] Hashes, environment, schema')
        for fp in self.files: self.source_hashes[fp.name] = _sha256(fp)
        self.env = {'python':platform.python_version(),'numpy':np.__version__,
                    'pandas':pd.__version__,'pyarrow':pa.__version__,
                    'scipy':scipy.__version__,'platform':platform.platform(),
                    **_hardware()}
        all_num, per_file = set(), {}
        for fp in self.files:
            tmp = pd.read_csv(fp,nrows=200,low_memory=False,on_bad_lines='skip')
            tmp.columns = _normalize_cols(tmp.columns)
            cols = [c for c in tmp.columns if c not in META_COLS and c not in DROP_ALWAYS
                    and c.lower() not in ('target','label','attack_type')
                    and pd.to_numeric(tmp[c],errors='coerce').notna().any()]
            all_num.update(cols); per_file[fp.name] = cols
        base = sorted(all_num)
        self.selected_features = sorted(set(base)|set(DERIVED_FEATURES))
        self.col_coverage = {f:sorted(set(base)-set(c)) for f,c in per_file.items()}
        print(f'  base={len(base)} derived={len(DERIVED_FEATURES)} '
              f'total={len(self.selected_features)} | {self.env["cpu_model"][:40]}')

    def _parse_ts(self,fp,series):
        rep = self.ts_report.get(fp.name); exp = _file_date(fp)
        w = pd.Timedelta(days=CFG['ts_window_days'])
        if rep is None:
            best = (None,-1.0,0.0); probe = series.head(5000)
            for fmt in CFG['ts_formats']:
                t = pd.to_datetime(probe,format=fmt,errors='coerce')
                ok = float(t.notna().mean())
                if ok == 0: continue
                inwin = (float(((t >= exp-w) & (t <= exp+w+pd.Timedelta(days=1))).mean())
                         if exp is not None else ok)
                if (inwin,ok) > (best[1],best[2]): best = (fmt,inwin,ok)
            if best[0] is None: raise RuntimeError(f'[FATAL] {fp.name}: no ts format.')
            if best[1] < CFG['ts_min_in_window']:
                raise RuntimeError(f'[FATAL] {fp.name}: format {best[0]} only '
                                   f'{best[1]:.1%} in window.')
            rep = {'format':best[0],'expected_date':str(exp),'rows':0,
                   'unparseable':0,'in_window':0,'min':None,'max':None}
            self.ts_report[fp.name] = rep
        t = pd.to_datetime(series,format=rep['format'],errors='coerce')
        rep['rows'] += len(t); rep['unparseable'] += int(t.isna().sum())
        if exp is not None:
            rep['in_window'] += int(((t >= exp-w) &
                                     (t <= exp+w+pd.Timedelta(days=1))).sum())
        if t.notna().any():
            lo,hi = t.min(),t.max()
            rep['min'] = str(min(pd.Timestamp(rep['min']),lo)) if rep['min'] else str(lo)
            rep['max'] = str(max(pd.Timestamp(rep['max']),hi)) if rep['max'] else str(hi)
        return t

    def _clean_and_select(self,df,count=True,file_name=None):
        sent = {c:(_to_float32(df[c])==-1.0).values for c in SENTINEL_MINUS1
                if c in df.columns}
        df = _derive(df.copy())
        n_sent = n_nan = n_inf = n_abs = 0
        n_inv = np.zeros(len(df),np.float32); n_clp = np.zeros(len(df),np.float32)
        for col in self.selected_features:
            if col in COUNTER_FEATURES: continue
            if col in df.columns:
                s = _to_float32(df[col])
                isn = s.isna().values; isi = (s==np.inf).values
                if col in sent:
                    n_sent += int(sent[col].sum()); self.per_feat_sentinel[col] += int(sent[col].sum())
                    isn = isn & ~sent[col]
                bad = isn | isi; n_nan += int(isn.sum()); n_inf += int(isi.sum())
                neg = (s<0).fillna(False).values
                n_inv += bad; n_clp += neg
                df[col] = s.mask(s<0,0.0)
                if file_name is not None:
                    self.per_feat_invalid[col][file_name] += int(bad.sum())
            else:
                df[col] = np.float32(np.nan); n_inv += 1; n_abs += len(df)
                if file_name is not None:
                    self.per_feat_invalid[col][file_name] += len(df)
        if count:
            a = self.audit
            a['invalid_cells'] += int(n_inv.sum())
            a['sentinel_cells_not_invalid'] += n_sent; a['nan_cells'] += n_nan
            a['posinf_cells'] += n_inf; a['absent_column_cells'] += n_abs
            a['negative_cells_clipped'] += int(n_clp.sum())
            a['rows_with_any_invalid'] += int((n_inv>0).sum())
            a['rows_with_any_negative'] += int((n_clp>0).sum())
        df['N_Invalid_Features'] = n_inv; df['N_Clipped_Negatives'] = n_clp
        df = df.reindex(columns=self.selected_features+['Target','_ts'])
        df['Target'] = df['Target'].astype(str)
        for col in self.selected_features: df[col] = df[col].astype(np.float32)
        df['_ts'] = df['_ts'].astype(np.int64)
        return df

    def _lock_schema(self,df):
        if self.MASTER_SCHEMA is not None: return
        feats = [c for c in df.columns if c not in ('Target','_ts')]
        self.selected_features = feats
        f32 = [pa.field(f,pa.float32()) for f in feats]
        self.TMP_SCHEMA = pa.schema(f32+[pa.field('Target',pa.string()),
                                         pa.field('_ts',pa.int64())])
        self.MASTER_SCHEMA = pa.schema(f32+[pa.field('Target',pa.string()),
                                            pa.field('_dup_in_train',pa.float32()),
                                            pa.field('_ts',pa.int64())])

    def _label_rules(self,chunk):
        def _col(c):
            return (pd.to_numeric(chunk[c],errors='coerce') if c in chunk.columns
                    else None)
        proto,port = _col('Protocol'),_col('Dst Port')
        fl,bl = _col('Total Length of Fwd Packets'),_col('Total Length of Bwd Packets')
        tgt = chunk['Target'].values
        for cls in np.unique(tgt):
            m = tgt == cls; r = self.label_rules[cls]; r['rows'] += int(m.sum())
            if cls != 'Normal' and fl is not None and bl is not None:
                r['zero_payload_both_dirs'] += int(((fl[m]==0)&(bl[m]==0)).sum())
            rule = LABEL_RULES.get(cls)
            if rule is None or proto is None: continue
            r['rule_checked'] += int(m.sum())
            r['protocol_mismatch'] += int((~proto[m].isin(rule[0])).sum())
            if rule[1] is not None and port is not None:
                r['service_port_mismatch'] += int((~port[m].isin(rule[1])).sum())

    def _reservoirs(self,chunk,fp):
        k = self.rng.random_sample(len(chunk))
        cand = chunk.drop(columns=['Target'],errors='ignore')
        if self.raw_rows is None: keys,rows = k,cand.reset_index(drop=True)
        else:
            keys = np.concatenate([self.raw_keys,k])
            rows = pd.concat([self.raw_rows,cand.reset_index(drop=True)],
                             ignore_index=True)
        if len(keys) > CFG['raw_sample_n']:
            sel = np.argpartition(keys,CFG['raw_sample_n'])[:CFG['raw_sample_n']]
            keys,rows = keys[sel],rows.iloc[sel].reset_index(drop=True)
        self.raw_keys, self.raw_rows = keys, rows
        cols = [c for c in ('Timestamp','Protocol','Dst Port','Flow Duration',
                            'Total Fwd Packets','Total Backward Packets',
                            'Total Length of Fwd Packets',
                            'Total Length of Bwd Packets','SYN Flag Cnt',
                            'ACK Flag Cnt','FIN Flag Cnt','RST Flag Cnt')
                if c in chunk.columns]
        tgt = chunk['Target'].values
        for cls in np.unique(tgt):
            m = np.flatnonzero(tgt==cls); kk = k[m]
            sub = chunk.iloc[m][cols].assign(_file=fp.name,Target=cls)
            if cls in self.audit_keys:
                kk = np.concatenate([self.audit_keys[cls],kk])
                sub = pd.concat([self.audit_rows[cls],sub],ignore_index=True)
            if len(kk) > CFG['audit_rows_per_class']:
                s = np.argpartition(kk,CFG['audit_rows_per_class'])[:CFG['audit_rows_per_class']]
                kk,sub = kk[s],sub.iloc[s].reset_index(drop=True)
            self.audit_keys[cls], self.audit_rows[cls] = kk, sub.reset_index(drop=True)

    def _phase1_scatter(self):
        print('\n[PHASE 1] Scatter')
        class_chunks = defaultdict(list); t0 = time.perf_counter()
        for fi,fp in enumerate(self.files,1):
            rows_f = 0
            for chunk in pd.read_csv(fp,chunksize=CFG['chunk_size'],
                                     low_memory=False,on_bad_lines='skip'):
                a = self.audit; a['raw_rows_scanned'] += len(chunk)
                chunk.columns = _normalize_cols(chunk.columns)
                lab = next((c for c in chunk.columns
                            if c.lower() in ('label','attack_type')),None)
                if lab is None: a['rows_no_label_column'] += len(chunk); continue
                ghost = chunk[lab].apply(_is_ghost_row)
                a['ghost_rows_dropped'] += int(ghost.sum())
                chunk = chunk[~ghost]
                if 'Flow Duration' in chunk.columns:
                    neg = pd.to_numeric(chunk['Flow Duration'],errors='coerce').fillna(0) < 0
                    a['negative_duration_dropped'] += int(neg.sum())
                    chunk = chunk[~neg]
                chunk = chunk.copy()
                chunk['Target'] = chunk[lab].astype(str).str.strip().str.lower().map(LABEL_MAP)
                a['unmapped_label_rows_dropped'] += int(chunk['Target'].isna().sum())
                chunk = chunk.dropna(subset=['Target']).drop(columns=[lab])
                if chunk.empty: continue
                if 'Timestamp' not in chunk.columns:
                    raise RuntimeError(f'[FATAL] {fp.name}: no Timestamp column.')
                ts = self._parse_ts(fp,chunk['Timestamp'])
                bad = ts.isna().values
                a['unparseable_timestamp_rows_dropped'] += int(bad.sum())
                chunk,ts = chunk[~bad],ts[~bad]
                if chunk.empty: continue
                chunk['_ts'] = ts.values.astype('datetime64[ns]').astype('int64')
                hrs = ts.dt.hour.values; tg_ = chunk['Target'].values
                for c_ in np.unique(tg_):
                    self.hour_hist[fp.name][c_] += np.bincount(hrs[tg_==c_],minlength=24)
                self._label_rules(chunk); self._reservoirs(chunk,fp)
                for cls,cnt in chunk['Target'].value_counts().items():
                    self.per_file_class[fp.name][str(cls)] += int(cnt)
                    self.class_totals_clean[str(cls)] += int(cnt)
                self.per_file_rows[fp.name] = self.per_file_rows.get(fp.name,0)+len(chunk)
                rows_f += len(chunk)
                cleaned = self._clean_and_select(chunk,file_name=fp.name)
                self._lock_schema(cleaned)
                h = _row_hash(cleaned[self.selected_features].fillna(-9.0))
                tgt = cleaned['Target'].values
                for cls in np.unique(tgt):
                    m = tgt == cls
                    tp = self.tmp_dir/f'{cls}_{fi:02d}_{len(class_chunks[cls]):04d}.parquet'
                    pq.write_table(pa.Table.from_pandas(
                        cleaned.loc[m,[f.name for f in self.TMP_SCHEMA]],
                        schema=self.TMP_SCHEMA,preserve_index=False),
                        tp,compression='snappy')
                    class_chunks[cls].append(str(tp))
                    self.dup_hashes[cls].append(h[m])
                del chunk,cleaned,h
            _gc()
            r = self.ts_report.get(fp.name,{})
            win = r.get('in_window',0)/max(r.get('rows',1),1)
            print(f'  [{fi:2d}/{len(self.files)}] {fp.name:<20}{rows_f:>11,} | '
                  f'{r.get("format")} | in-window {win:.4f}')
            if win < CFG['ts_warn_in_window'] and _file_date(fp) is not None:
                print(f'    WARN {1-win:.2%} outside file-date window')
        a = self.audit
        if a['unparseable_timestamp_rows_dropped']/max(a['raw_rows_scanned'],1) \
                > CFG['max_bad_ts_frac']:
            raise RuntimeError('[FATAL] unparseable timestamps exceed tolerance.')
        if 'Normal' not in self.class_totals_clean:
            raise RuntimeError('[FATAL] Normal has 0 rows.')
        print(f'  scatter {(time.perf_counter()-t0)/60:.1f} min')
        self._census()
        return class_chunks

    def _census(self):
        uniq,tot = {},0
        for cls,arrs in self.dup_hashes.items():
            h = np.concatenate(arrs); u = np.unique(h); uniq[cls] = u
            tot += len(h)-len(u)
            self.dup_report[cls] = {'rows':int(len(h)),'unique':int(len(u)),
                                    'duplicates':int(len(h)-len(u)),
                                    'dup_pct':round(100*(len(h)-len(u))/max(len(h),1),3)}
        self.audit['within_class_exact_duplicate_rows'] = int(tot)
        # kept for boundary-adjacency leakage (G1); ~8 bytes/row
        self.class_hashes = {c: np.concatenate(a) for c, a in self.dup_hashes.items()}
        classes = sorted(uniq)
        allh = np.concatenate([uniq[c] for c in classes])
        allc = np.concatenate([np.full(len(uniq[c]),i,np.int16)
                               for i,c in enumerate(classes)])
        vals,cnt = np.unique(allh,return_counts=True)
        conflict = vals[cnt>1]; pairs,rows_by = {},{}
        if len(conflict):
            sel = np.isin(allh,conflict)
            cdf = pd.DataFrame({'h':allh[sel],'c':allc[sel]})
            pr = cdf.merge(cdf,on='h'); pr = pr[pr['c_x'] < pr['c_y']]
            for (i,j),v in pr.groupby(['c_x','c_y']).size().items():
                pairs[f'{classes[i]} | {classes[j]}'] = int(v)
            for cls in classes:
                rows_by[cls] = int(np.isin(np.concatenate(self.dup_hashes[cls]),
                                          conflict).sum())
        self.label_conflicts = {'conflicting_feature_vectors':int(len(conflict)),
                                'rows_involved_by_class':rows_by,
                                'class_pairs':dict(sorted(pairs.items(),key=lambda x:-x[1])[:25])}
        self.audit['label_conflict_feature_vectors'] = int(len(conflict))
        print(f'  within-class duplicate rows={tot:,} | '
              f'cross-label conflicting vectors={len(conflict):,}')
        self.dup_hashes.clear(); _gc()

    @staticmethod
    def _boundary_leak(tss,hs,bounds,windows=(1,10,60,300)):
        """G1. For each cut, the share of rows immediately AFTER it whose exact
        feature twin lies immediately BEFORE it, per time window. This is the
        leakage a boundary can actually carry; snapping does not remove it."""
        n = len(tss); out = {}
        for nm,b in zip(('train_val','val_test'),bounds[:2]):
            if not (0 < b < n): continue
            cut = tss[b]; d = {}
            for W in windows:
                w = np.int64(W*1e9)
                lo = int(np.searchsorted(tss[:b],cut-w))
                hi = b+int(np.searchsorted(tss[b:],cut+w))
                left,right = hs[lo:b],hs[b:hi]
                tw = int(np.isin(right,left).sum()) if len(left) and len(right) else 0
                d[str(W)] = {'rows_before':int(len(left)),'rows_after':int(len(right)),
                             'rows_after_with_twin_before':tw,
                             'pct_of_rows_after':round(100*tw/max(len(right),1),3)}
            out[nm] = d
        return out

    def _phase2_split(self,class_chunks):
        print(f'\n[PHASE 2] Split ({CFG["split_mode"]})')
        full_zd,partial_zd = set(CFG['full_zero_day']),CFG['partial_zero_day']
        cache,known_ts = {},[]
        for cls,tps in class_chunks.items():
            parts = [self._fix_ts(pq.read_table(tp,columns=['_ts'])['_ts'].to_numpy()
                                  .astype(np.int64),tp) for tp in tps]
            lens = [len(p) for p in parts]
            ts_all = np.concatenate(parts).astype(np.int64)
            order = np.argsort(ts_all,kind='stable'); n = len(ts_all)
            known = (0 if cls in full_zd else
                     int(round(n*partial_zd[cls])) if cls in partial_zd else n)
            cache[cls] = (ts_all,order,lens,known)
            if known: known_ts.append(ts_all[order[:known]])
        kts = np.concatenate(known_ts)
        g70 = int(np.percentile(kts,100*CFG['train_frac']))
        g85 = int(np.percentile(kts,100*(CFG['train_frac']+CFG['val_frac'])))
        span = (pd.Timestamp(int(kts.min())).isoformat(),
                pd.Timestamp(int(kts.max())).isoformat())
        del kts,known_ts
        print(f'  known-flow span {span[0]} -> {span[1]}')

        def _flush(sp):
            if not self._buffers[sp]: return
            df = pd.concat(self._buffers[sp],ignore_index=True)
            out = (self.splits_dir/'test'/f'zeroday_part{self._shard_idx[sp]:04d}.parquet'
                   if sp == 'zero_day'
                   else self.splits_dir/sp/f'part{self._shard_idx[sp]:04d}.parquet')
            pq.write_table(pa.Table.from_pandas(df,schema=self.TMP_SCHEMA,
                                                preserve_index=False),
                           out,compression='snappy')
            self.total_written[sp] += len(df)
            self._shard_idx[sp] += 1
            self._buffers[sp], self._buf_count[sp] = [],0

        gap_ns = np.int64(CFG['burst_gap_s']*1e9)
        emb_ns = np.int64(CFG['embargo_s']*1e9)
        diag,gchange,moved_to_zd,no_test = {},{},[],[]
        for cls in sorted(class_chunks):
            ts_all,order,lens,known = cache.pop(cls)
            tss = ts_all[order]; n = len(tss); idx = np.arange(n)
            dts = np.diff(tss)
            newb = np.empty(n,bool); newb[0] = True; newb[1:] = dts > gap_ns
            edges = np.flatnonzero(newb)
            bsize = np.diff(np.append(edges,n))
            lab = np.empty(n,np.int8)
            if cls in full_zd:
                lab[:] = ZD; raw_b = fin_b = [0,0,0]
            elif CFG['split_mode'] == 'global':
                kn = tss[:known]
                te = int(np.searchsorted(kn,g70,side='right'))
                ve = int(np.searchsorted(kn,g85,side='right'))
                raw_b = fin_b = [te,ve,known]
                lab[:te],lab[te:ve],lab[ve:known],lab[known:] = TR,VA,TE,ZD
            else:
                raw_b = [int(known*CFG['train_frac']),
                         int(known*(CFG['train_frac']+CFG['val_frac'])),known]
                fin_b,lo = [],0
                max_move = max(1,int(CFG['max_snap_frac']*max(known,1)))
                for b in raw_b:
                    s = b
                    if CFG['burst_aware'] and 0 < b < n:
                        j = np.searchsorted(edges,b)
                        cands = [int(e) for e in
                                 ([edges[j-1]] if j>0 else [])+
                                 ([edges[j]] if j<len(edges) else [])
                                 if abs(int(e)-b) <= max_move and e > lo]
                        if cands: s = min(cands,key=lambda e:(abs(e-b),-e))
                    if b > lo: s = max(s,lo+1)
                    fin_b.append(s); lo = s
                te,ve,ke = fin_b
                lab[:te],lab[te:ve],lab[ve:ke],lab[ke:] = TR,VA,TE,ZD
            n_emb = 0
            if emb_ns > 0 and cls not in full_zd:
                for b in fin_b[:2]:
                    if 0 < b < n:
                        m = (idx >= b)&(tss < tss[b-1]+emb_ns)&(lab != EMB)&(lab != ZD)
                        n_emb += int(m.sum()); lab[m] = EMB
            if cls not in full_zd and CFG['split_mode'] == 'global':
                if not (lab == TR).any() and known:
                    lab[(lab == VA)|(lab == TE)] = ZD; moved_to_zd.append(cls)
                elif not (lab == TE).any(): no_test.append(cls)
            if cls not in full_zd and known and CFG['split_mode'] == 'per_class':
                g = np.where(tss <= g70,TR,np.where(tss <= g85,VA,TE)).astype(np.int8)
                km = lab <= TE
                ch = int((lab[km] != g[km]).sum())
                gchange[cls] = {'rows_compared':int(km.sum()),'rows_changed':ch,
                                'pct_changed':round(100*ch/max(int(km.sum()),1),3)}
            grid = {}
            for gs in CFG['burst_gap_grid']:
                nb = np.empty(n,bool); nb[0] = True; nb[1:] = dts > np.int64(gs*1e9)
                ed = np.flatnonzero(nb); bs = np.diff(np.append(ed,n))
                grid[str(gs)] = {'n_bursts':int(len(ed)),
                                 'largest_burst_rows':int(bs.max()) if len(bs) else 0,
                                 'straddle_rows_final':_straddle_mass(fin_b,n,nb,ed,bs)}
            def _gaps(bounds):
                return [None if not (0 < b < n) else
                        round(float(tss[b]-tss[b-1])/1e9,3) for b in bounds[:3]]
            hs_cls = self.class_hashes.get(cls)
            if hs_cls is None or len(hs_cls) != n:
                bleak = {'status':'NOT MEASURED',
                         'reason':('row hashes missing' if hs_cls is None else
                                   f'hash/row length mismatch {len(hs_cls)} vs {n}')}
                print(f'  [WARN] boundary-adjacency leakage NOT MEASURED for {cls}: '
                      f'{bleak["reason"]}')
            else:
                bleak = self._boundary_leak(tss,hs_cls[order],fin_b)
            diag[cls] = {'boundary_adjacency_leakage':bleak,
                         'n':int(n),'known':int(known),
                         'class_time_span_s':round(float(tss[-1]-tss[0])/1e9,1) if n else 0.0,
                         'n_bursts':int(len(edges)),
                         'largest_burst_rows':int(bsize.max()) if len(bsize) else 0,
                         'raw_bounds':[int(x) for x in raw_b],
                         'final_bounds':[int(x) for x in fin_b],
                         'rows_moved_by_snap':int(sum(abs(a-b) for a,b in zip(fin_b,raw_b))),
                         'boundary_gap_s_final':_gaps(fin_b),
                         'straddle_rows_naive':_straddle_mass(raw_b,n,newb,edges,bsize),
                         'straddle_rows_final':_straddle_mass(fin_b,n,newb,edges,bsize),
                         'burst_gap_grid':grid,'embargoed_rows':n_emb,
                         'partition_counts':{SPLIT_NAMES[k]:int((lab==k).sum())
                                              for k in SPLIT_NAMES}}
            self.audit['embargoed_rows'] += n_emb
            row_lab = np.empty(n,np.int8); row_lab[order] = lab
            pos = 0
            for tp,ln in zip(class_chunks[cls],lens):
                dfc = pd.read_parquet(tp)
                dfc['_ts'] = self._fix_ts(dfc['_ts'].to_numpy().astype(np.int64),tp)
                a = row_lab[pos:pos+ln]; pos += ln
                for code,sp in SPLIT_NAMES.items():
                    m = a == code
                    if m.any():
                        self._buffers[sp].append(dfc[m])
                        self._buf_count[sp] += int(m.sum())
                del dfc
                for sp in self._buffers:
                    if self._buf_count[sp] >= CFG['buffer_flush_rows']: _flush(sp)
                os.remove(tp); _ram_guard()
            del ts_all,tss,order,lab,row_lab,idx,dts,newb
            _gc()
        for sp in self._buffers: _flush(sp)
        written = sum(self.total_written.values())
        if written+self.audit['embargoed_rows'] != sum(self.class_totals_clean.values()):
            raise RuntimeError('[FATAL] split reconciliation failed.')
        bl = {}
        for W in ('1','10','60','300'):
            a = sum(v[W]['rows_after_with_twin_before']
                    for c,d in diag.items() if c != 'Normal'
                    for v in d.get('boundary_adjacency_leakage',{}).values()
                    if isinstance(v,dict) and W in v)
            r = sum(v[W]['rows_after']
                    for c,d in diag.items() if c != 'Normal'
                    for v in d.get('boundary_adjacency_leakage',{}).values()
                    if isinstance(v,dict) and W in v)
            bl[W+'s'] = {'rows_after_cut':r,'with_twin_before_cut':a,
                         'pct':round(100*a/max(r,1),3)}
        not_measured = [c for c,d in diag.items()
                        if d.get('boundary_adjacency_leakage',{}).get('status')]
        self.boundary_leak_summary = {
            'attack_classes_only':True,'windows':bl,
            'classes_not_measured':not_measured,
            'reading':('share of rows just after a cut whose exact feature twin lies '
                       'just before it. An embargo of width W removes these rows AND '
                       'the non-twin rows in the same window, so this is the '
                       'twin-adjacent component of what an embargo removes, not the '
                       'whole of it. Burst snapping addresses neither.')}
        inside = [c for c,d in diag.items() if d['straddle_rows_final'] > 0 and c != 'Normal']
        att = {c:d for c,d in diag.items() if c != 'Normal'}
        self.straddle_attack = {'naive':int(sum(d['straddle_rows_naive'] for d in att.values())),
                                'final':int(sum(d['straddle_rows_final'] for d in att.values())),
                                'normal_excluded_reason':'benign traffic is continuous; '
                                'burst leakage concerns attack sessions'}
        print('  rows: '+' | '.join(f'{s}={self.total_written[s]:,}'
                                     for s in ('train','val','test','zero_day'))
              +f' | embargo={self.audit["embargoed_rows"]:,}')
        print(f'  straddling-burst rows (attack classes): naive '
              f'{self.straddle_attack["naive"]:,} -> final {self.straddle_attack["final"]:,}')
        b60 = bl['60s']
        print(f'  boundary-adjacency leakage (attack, 60 s window): '
              f'{b60["with_twin_before_cut"]:,}/{b60["rows_after_cut"]:,} rows '
              f'= {b60["pct"]:.3f}%')
        if inside:
            print(f'  cut still inside a burst: {", ".join(inside)}')
            print(f'  NOTE: snapping moved only {self.straddle_attack["naive"]-self.straddle_attack["final"]:,}'
                  f' rows; it is reported as INEFFECTIVE on this dataset, not as a control')
        if moved_to_zd: print(f'  GLOBAL: no train rows -> zero-day: {moved_to_zd}')
        if no_test: print(f'  GLOBAL: no test rows: {no_test}')
        if gchange:
            m = np.mean([v['pct_changed'] for v in gchange.values()])
            print(f'  global-cut divergence: mean {m:.2f}%')
        self.class_hashes.clear(); _gc()      # ~130 MB, no longer needed
        self.split_diag.update({'per_class':diag,'classes_with_cut_inside_burst':inside,
            'global_cutoff':{'known_span':span,
                             'cut_train_val':pd.Timestamp(g70).isoformat(),
                             'cut_val_test':pd.Timestamp(g85).isoformat(),
                             'per_class_vs_global_divergence':gchange},
            'global_mode_classes_moved_to_zero_day':moved_to_zd,
            'global_mode_classes_without_test_rows':no_test})

    def _phase3_scaler(self):
        print('\n[PHASE 3] Train-only imputation + exact two-pass scaler')
        shards = sorted((self.splits_dir/'train').glob('*.parquet'))
        feats = self.selected_features
        ncls = self._n_train_classes()
        smp = _uniform_sample_by_class(shards,feats,
                                       max(1,CFG['median_sample_n']//ncls),SEED)
        rows = {c:len(d) for c,d in smp.items()}
        S = pd.concat(smp.values(),ignore_index=True); del smp
        pol = CFG['invalid_policy']
        for f in feats:
            col = S[f].values.astype(np.float64); col = col[np.isfinite(col)]
            med = float(np.median(col)) if len(col) else 0.0
            p999 = float(np.percentile(col,99.9)) if len(col) else 0.0
            if f in COUNTER_FEATURES:
                self.impute_nan[f] = self.impute_posinf[f] = 0.0
            else:
                self.impute_nan[f] = {'semantic':med,'median':med,'p999':p999,'zero':0.0}[pol]
                self.impute_posinf[f] = {'semantic':p999,'median':med,'p999':p999,'zero':0.0}[pol]
            if len(col) > 1:
                p = np.percentile(col,[1,5,25,50,75,95,99])
                self.feature_stats[f] = {'mean':float(col.mean()),'std':float(col.std()),
                    **_moments(col),
                    **dict(zip(['p1','p5','p25','p50','p75','p95','p99'],
                               map(float,p))),'p999':p999}
        del S
        n,s1 = 0,np.zeros(len(feats))
        for sh in shards:
            a = self._impute(pd.read_parquet(sh,columns=feats),feats)
            s1 += a.sum(0); n += len(a)
        mean = s1/max(n,1); s2 = np.zeros(len(feats))
        for sh in shards:
            d = self._impute(pd.read_parquet(sh,columns=feats),feats)-mean
            s2 += (d*d).sum(0)
        std = np.sqrt(s2/max(n,1))
        for i,f in enumerate(feats):
            self.scaler_mean[f] = float(mean[i])
            self.scaler_std[f] = float(std[i]) if std[i]>0 and np.isfinite(std[i]) else 1.0
        _atomic_json({'features':feats,'mean':self.scaler_mean,'std':self.scaler_std,
                      'impute_nan':self.impute_nan,'impute_posinf':self.impute_posinf,
                      'invalid_policy':pol,
                      'semantics':'+Inf=x/0->p99.9; NaN=undefined->median; '
                                  '-Inf->NaN; finite negatives->0; train only'},
                     self.scaler_path)
        self.split_diag['imputation_sample_rows_per_class'] = rows
        print(f'  scaler on {n:,} train rows | policy={pol}')

    def _n_train_classes(self):
        pc = self.split_diag.get('per_class',{})
        return max(1,sum(1 for d in pc.values()
                         if d.get('partition_counts',{}).get('train',0) > 0))

    def _decide_clock(self):
        tot = np.zeros(24,np.int64)
        for f in self.hour_hist.values():
            for v in f.values(): tot += v
        max_h = int(np.flatnonzero(tot).max()) if tot.any() else 0
        fmax = {fn:int(np.flatnonzero(sum(v.values())).max())
                for fn,v in self.hour_hist.items()
                if v and np.any(sum(v.values()))}
        cand = sorted(fn for fn,m in fmax.items() if m <= 12)
        self.clock = {'hour_histogram_all_rows':tot.tolist(),'max_hour_observed':max_h,
                      'max_hour_per_file':fmax,'candidate_files_12h':cand,
                      'applied':False,'pm_cut':None,'files_corrected':[]}
        if not cand:
            self.clock['decision'] = '24-hour clock in every file; no correction'
            print('  clock: 24-hour (no correction)'); return
        tot = np.zeros(24,np.int64)
        for fn in cand:
            for v in self.hour_hist[fn].values(): tot += v
        def align(pm):
            best = (-1.0,0)
            for o in CFG['clock_offsets']:
                num = den = 0
                for (dt_,cls),wins in DOC_WINDOWS.items():
                    h = np.zeros(24,np.int64)
                    for fn in cand:
                        byc = self.hour_hist[fn]
                        if fn[:5] == dt_ and cls in byc: h += byc[cls]
                    for hr in range(24):
                        if not h[hr]: continue
                        c = hr+12 if (pm and 1 <= hr < pm) else hr
                        den += int(h[hr])
                        if any(c+o < e+1 and c+o+1 > s_-1 for s_,e in wins): num += int(h[hr])
                sc = num/den if den else -1.0
                if sc > best[0]: best = (sc,o)
            return best
        base = align(None)
        cands = {pm:align(pm) for pm in CFG['clock_pm_cuts']}
        top = max(v[0] for v in cands.values())
        def span(pm):
            h = np.zeros(24,np.int64)
            for hr in range(24): h[(hr+12) % 24 if 1 <= hr < pm else hr] += tot[hr]
            need = 0.99*h.sum(); best_w = 24
            for st in range(24):
                acc = 0
                for w in range(1,25):
                    acc += h[(st+w-1) % 24]
                    if acc >= need: best_w = min(best_w,w); break
            return best_w
        tied = [k for k,v in cands.items() if v[0] >= top-0.005]
        pm_b = min(tied,key=lambda k: (span(k),k))
        self.clock['tie_break'] = {'tied_pm_cuts':tied,'active_window_h':{str(k):span(k) for k in tied},
                                   'rule':'most compact daily activity window (99% of all rows)'}
        moved = int(sum(tot[h] for h in range(1,pm_b)))
        by_cls = defaultdict(int)
        for fn in cand:
            for c_,v in self.hour_hist[fn].items():
                by_cls[c_] += int(sum(v[h] for h in range(1,pm_b)))
        self.clock['falsification'] = {
            'hypothesis':(f'hours 1..{pm_b-1} are afternoon (PM) times written on a '
                          '12-hour clock'),
            'prediction':(f'hours {pm_b}..7 must be empty: a 12-hour clock cannot '
                          'produce them if the capture runs a single working day'),
            'observed_occupancy_hours_6_7':[int(tot[6]),int(tot[7])],
            'rows_that_move':moved,
            'rows_that_move_by_class':dict(by_cls),
            'residual_risk':('genuine 01:00-0%d:59 benign traffic, if it existed, is '
                             'relocated to the afternoon; the prediction above is the '
                             'check a reader can run' % (pm_b-1))}
        gain = cands[pm_b][0]-base[0]
        self.clock.update({'alignment_no_correction':{'score':base[0],'offset_h':base[1]},
                           'alignment_by_pm_cut':{str(k):{'score':v[0],'offset_h':v[1]}
                                                  for k,v in cands.items()},
                           'reference':'CSE-CIC-IDS2018 documented attack schedule (DOC_WINDOWS)'})
        if base[0] < 0:
            self.clock['decision'] = 'no documented class present; no correction'
        elif gain >= CFG['clock_min_gain'] and cands[pm_b][0] >= CFG['clock_min_align']:
            self.clock.update({'applied':True,'pm_cut':pm_b,
                               'files_corrected':cand,
                               'decision':f'12-hour clock without AM/PM: hours 1..{pm_b-1} '
                                          f'shifted +12 h (alignment {base[0]:.3f} -> '
                                          f'{cands[pm_b][0]:.3f})'})
        else:
            self.clock['decision'] = (f'hours <= 12 but no hypothesis improves alignment '
                                      f'enough ({base[0]:.3f} -> {cands[pm_b][0]:.3f}); '
                                      'no correction; REPORT AS LIMITATION')
        print(f'  clock: {self.clock["decision"]}')

    def _fix_ts(self,ts,tp):
        # P2: stems are '<class>_<file-index>_<chunk>'; a class name containing
        # '_' or a malformed stem must not raise inside the split loop.
        if not self.clock.get('applied'): return ts
        try:
            fname = self.files[int(Path(tp).stem.split('_')[-2])-1].name
        except (ValueError,IndexError):
            return ts
        if fname not in self.clock['files_corrected']: return ts
        h = (ts//NS_H) % 24
        m = (h >= 1) & (h < self.clock['pm_cut'])
        out = ts.copy(); out[m] += 12*NS_H
        return out

    def _impute(self,df,feats):
        a = df[feats].to_numpy(np.float64,copy=True)
        nan,inf = np.isnan(a),np.isposinf(a)
        if nan.any():
            a[nan] = np.array([self.impute_nan[f] for f in feats])[np.nonzero(nan)[1]]
        if inf.any():
            a[inf] = np.array([self.impute_posinf[f] for f in feats])[np.nonzero(inf)[1]]
        return a

    def _phase4_ks(self):
        print('\n[PHASE 4] KS (exact uniform stratified samples, post-imputation)')
        feats = self.selected_features

        S_ = CFG['ks_seeds']; per = max(1,CFG['ks_sample_per_split']//self._n_train_classes())
        def sample(folder):
            sh = [s for s in sorted((self.splits_dir/folder).glob('*.parquet'))
                  if 'zeroday' not in s.name]
            if not sh: return [],{}
            d = _uniform_sample_by_class(sh,feats,per*S_,SEED+_stable_int(folder)%1000)
            r = np.random.RandomState(SEED+_stable_int(folder+'_split')%1000)
            groups = [[] for _ in range(S_)]
            for v in d.values():
                g_ = r.permutation(len(v)) % S_
                for k_ in range(S_): groups[k_].append(v.iloc[np.flatnonzero(g_==k_)])
            out = [pd.DataFrame(self._impute(pd.concat(g,ignore_index=True),feats),
                                columns=feats) for g in groups]
            return out,{'classes_covered':len(d),'subsamples':S_,
                        'rows_per_class_per_subsample':{c:len(v)//S_ for c,v in d.items()}}
        tr,trd = sample('train'); vl,vld = sample('val'); te,ted = sample('test')
        bonf = CFG['ks_alpha']/max(len(feats),1)
        def ks(A,B):
            out = []
            for f in feats:
                st = np.array([ks_2samp(a[f].values,b[f].values) for a,b in zip(A,B)])
                out.append({'feature':f,'ks_stat':float(st[:,0].mean()),
                            'ks_std':float(st[:,0].std()),'p_value_median':float(np.median(st[:,1])),
                            'significant':bool((st[:,1]<bonf).mean() > 0.5)})
            return out
        tv = ks(tr,vl) if vl else []
        tt = ks(tr,te) if te else []
        res = {'bonferroni_alpha':bonf,
               'sampling':'exact uniform per-class reservoir over all shards; post-imputation',
               'sampling_diagnostics':{'train':trd,'val':vld,'test':ted},
               'seeds':CFG['ks_seeds'],'ks_stat_definition':'mean over disjoint uniform subsamples',
               'ks_train_val':{'results':tv,
                               'n_significant':sum(r['significant'] for r in tv),
                               'max_ks':max((r['ks_stat'] for r in tv),default=0.0),
                               'use':'FEATURE SELECTION MAY USE THIS'},
               'ks_train_test':{'results':tt,
                                'n_significant':sum(r['significant'] for r in tt),
                                'max_ks':max((r['ks_stat'] for r in tt),default=0.0),
                                'use':'REPORTING ONLY'}}
        # ---- R4.3 FIX: store on self so _fig_ks_scatter can read them ----
        self.ks_train_val = res['ks_train_val']['results']
        self.ks_train_test = res['ks_train_test']['results']
        print(f'  train-val max KS {res["ks_train_val"]["max_ks"]:.4f} | '
              f'train-test max KS {res["ks_train_test"]["max_ks"]:.4f}')
        return res

    def _phase5_write(self):
        print('\n[PHASE 5] Impute + scale + dedup-flag + shuffle + write')
        out_dir = _mkdir(self.work_dir/'l2_ready')
        bdir = _mkdir(self.tmp_dir/'buckets')
        feats,K = self.selected_features,CFG['shuffle_buckets']
        mu = np.array([self.scaler_mean[f] for f in feats])
        sd = np.array([self.scaler_std[f] for f in feats])
        tsh = sorted((self.splits_dir/'test').glob('*.parquet'))
        cfgs = [('train',sorted((self.splits_dir/'train').glob('*.parquet')),
                 out_dir/'l2_train.parquet'),
                ('val',sorted((self.splits_dir/'val').glob('*.parquet')),
                 out_dir/'l2_val.parquet'),
                ('test_known',[s for s in tsh if 'zeroday' not in s.name],
                 out_dir/'l2_test_known.parquet'),
                ('test_zeroday',[s for s in tsh if 'zeroday' in s.name],
                 out_dir/'l2_test_zeroday.parquet')]
        train_h,train_set,counts,tvs = [],None,{},{}
        train_l = []; twin_lab = None
        ccode = {c:i for i,c in enumerate(sorted(self.class_totals_clean))}
        self.twin_labels = defaultdict(dict)
        twin_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
        for name,shards,out_path in cfgs:
            if not shards: continue
            rs = np.random.RandomState(SEED+_stable_int(name+'_s')%1000)
            rb = np.random.RandomState(SEED+_stable_int(name+'_b')%1000)
            writers,bp = {},{k:bdir/f'{name}_{k:03d}.parquet' for k in range(K)}
            dup = defaultdict(lambda:[0,0])
            for sh in shards:
                df = pd.read_parquet(sh,columns=feats+['Target','_ts'])
                sdf = pd.DataFrame(((self._impute(df,feats)-mu)/sd).astype(np.float32),
                                   columns=feats)
                sdf['Target'] = df['Target'].astype(str).values
                sdf['_ts'] = df['_ts'].values.astype(np.int64)
                del df
                h = _row_hash(sdf[feats])
                if name == 'train':
                    train_h.append(h); sdf['_dup_in_train'] = np.float32(0.0)
                    train_l.append(np.array([ccode.get(t,-2) for t in sdf['Target'].values],np.int16))
                else:
                    pos = np.minimum(np.searchsorted(train_set,h),len(train_set)-1)
                    flag = train_set[pos] == h
                    tl = twin_lab[pos]; tg = sdf['Target'].values
                    for c_ in np.unique(tg):
                        m_ = (tg == c_) & flag
                        if not m_.any(): continue
                        lab_ = tl[m_]; own = ccode.get(c_,-9)
                        tw = twin_cnt[name][c_]
                        tw['Normal'] += int((lab_ == ccode['Normal']).sum())
                        tw['same_class'] += int((lab_ == own).sum())
                        tw['other_attack'] += int(((lab_ >= 0) & (lab_ != own) &
                                                   (lab_ != ccode['Normal'])).sum())
                        tw['mixed_labels'] += int((lab_ == -1).sum())
                    sdf['_dup_in_train'] = flag.astype(np.float32)
                    for c,g in pd.Series(flag).groupby(sdf['Target'].values):
                        dup[c][0] += int(g.sum()); dup[c][1] += int(len(g))
                b = rs.randint(0,K,len(sdf))
                for k in np.unique(b):
                    if k not in writers:
                        writers[k] = pq.ParquetWriter(bp[k],self.MASTER_SCHEMA,
                                                      compression='snappy')
                    writers[k].write_table(pa.Table.from_pandas(
                        sdf[b==k].reindex(columns=[f.name for f in self.MASTER_SCHEMA]),
                        schema=self.MASTER_SCHEMA,preserve_index=False))
                del sdf,b,h; _gc()
            for w in writers.values(): w.close()
            if name == 'train':
                hh = np.concatenate(train_h); ll = np.concatenate(train_l); del train_h,train_l
                o_ = np.argsort(hh,kind='stable'); hh,ll = hh[o_],ll[o_]
                train_set,st_ = np.unique(hh,return_index=True)
                lo_ = np.minimum.reduceat(ll,st_); hi_ = np.maximum.reduceat(ll,st_)
                twin_lab = np.where(lo_ == hi_,lo_,-1).astype(np.int16)   # -1 = twin carries >1 label
                del hh,ll,o_,st_,lo_,hi_
            else:
                td,tn = sum(v[0] for v in dup.values()),sum(v[1] for v in dup.values())
                self.twin_labels[name] = {c:dict(v) for c,v in twin_cnt[name].items()}
                self.xpart_dups[name] = {
                    'rows':tn,'dup_in_train':td,'pct':round(100*td/max(tn,1),3),
                    'by_class':{c:{'dup':v[0],'rows':v[1],
                                   'pct':round(100*v[0]/max(v[1],1),3)}
                                for c,v in sorted(dup.items())}}
            writer,total = None,0
            for k in rb.permutation(K):
                if not bp[k].exists(): continue
                bdf = pd.read_parquet(bp[k]).sample(frac=1.0,random_state=SEED+int(k))
                writer = writer or pq.ParquetWriter(out_path,self.MASTER_SCHEMA,
                                                    compression='snappy')
                writer.write_table(pa.Table.from_pandas(bdf,schema=self.MASTER_SCHEMA,
                                                        preserve_index=False))
                total += len(bdf); os.remove(bp[k]); del bdf
            if writer: writer.close()
            counts[name] = total
            tg = pd.read_parquet(out_path,columns=['Target'])['Target']
            full,head = tg.value_counts(normalize=True),tg.head(10_000).value_counts(normalize=True)
            ix = full.index.union(head.index)
            tv = 0.5*float((full.reindex(ix,fill_value=0)-
                            head.reindex(ix,fill_value=0)).abs().sum())
            tvs[name] = round(tv,5)
            dtxt = (f' | twin-in-train {self.xpart_dups[name]["pct"]:.2f}%'
                    if name in self.xpart_dups else '')
            print(f'  {out_path.name:<26}{total:>12,} | shuffle TV {tv:.4f}{dtxt}')
            if tv > CFG['shuffle_max_tv'] and total > 20_000:
                raise RuntimeError(f'[FATAL] {out_path.name}: shuffle failed (TV={tv:.3f}).')
        self.split_diag['shuffle_tv'] = tvs; del train_set
        return counts,out_dir

    def _phase6_integrity(self,out_dir):
        print('\n[PHASE 6] Integrity, reconciliation, prevalence')
        res = {}
        for fn in ('l2_train.parquet','l2_val.parquet',
                   'l2_test_known.parquet','l2_test_zeroday.parquet'):
            if (out_dir/fn).exists():
                res[fn] = pd.read_parquet(out_dir/fn,columns=['Target'])['Target'] \
                            .value_counts().to_dict()
        tr = res.get('l2_train.parquet',{})
        if 'Normal' not in tr: raise RuntimeError('[FATAL] Normal missing from train.')
        for c in CFG['full_zero_day']+self.split_diag.get(
                'global_mode_classes_moved_to_zero_day',[]):
            if tr.get(c,0): raise RuntimeError(f'[FATAL] zero-day class {c} in train.')
        for fn in ('l2_val.parquet','l2_test_known.parquet'):
            if not res.get(fn): raise RuntimeError(f'[FATAL] {fn} empty.')
        self.prevalence = {}
        for fn,d in res.items():
            nn = d.get('Normal',0); na = sum(v for k,v in d.items() if k != 'Normal')
            self.prevalence[fn] = {'normal':nn,'attack':na,
                'attack_prevalence':round(na/max(nn+na,1),5),
                'normal_to_attack_ratio':(f'{nn/na:.2f}:1' if na and nn else
                                          'attack only' if na else 'normal only')}
        zd = res.get('l2_test_zeroday.parquet',{})
        self.zd_comp = {c:{'rows':v,
            'type':('full (never trained)' if c in CFG['full_zero_day']
                    else 'partial temporal holdout' if c in CFG['partial_zero_day']
                    else 'global-cut: absent from train'),
            'share':round(v/max(sum(zd.values()),1),5)}
            for c,v in sorted(zd.items(),key=lambda x:-x[1])}
        a = self.audit
        removed = (a['ghost_rows_dropped']+a['negative_duration_dropped']+
                   a['unmapped_label_rows_dropped']+a['rows_no_label_column']+
                   a['unparseable_timestamp_rows_dropped'])
        final = sum(self.class_totals_clean.values())
        a['reconciliation_residual'] = int(a['raw_rows_scanned']-removed-final)
        self.recon_table = [
            ('Raw rows scanned',a['raw_rows_scanned']),
            ('  - rows without label column',a['rows_no_label_column']),
            ('  - header/ghost rows',a['ghost_rows_dropped']),
            ('  - negative-duration rows',a['negative_duration_dropped']),
            ('  - unmapped-label rows',a['unmapped_label_rows_dropped']),
            ('  - unparseable-timestamp rows',a['unparseable_timestamp_rows_dropped']),
            ('= Final clean rows',final),
            ('Residual (asserted 0)',a['reconciliation_residual']),
            ('Rows with >=1 NaN/+Inf (sentinel excluded; imputed)',a['rows_with_any_invalid']),
            ('  cells: NaN (undefined)',a['nan_cells']),
            ('  cells: +Inf (x/0 duration, saturated high)',a['posinf_cells']),
            ('  cells: column absent in source file',a['absent_column_cells']),
            ('Sentinel -1 cells (not applicable; __present flag)',a['sentinel_cells_not_invalid']),
            ('Rows with >=1 negative value (kept, clipped)',a['rows_with_any_negative']),
            ('Within-class exact-duplicate rows (kept)',a['within_class_exact_duplicate_rows']),
            ('Cross-label conflicting feature vectors',a['label_conflict_feature_vectors']),
            ('Embargoed rows (not written)',a['embargoed_rows'])]
        for k,v in self.recon_table: print(f'  {k:<48}{v:>14,}')
        if a['reconciliation_residual'] != 0:
            raise RuntimeError('[FATAL] reconciliation residual != 0')
        for fn,p in self.prevalence.items():
            print(f'  {fn:<26} normal:attack = {p["normal_to_attack_ratio"]}')
        return res

    def _batch_reference(self,raw):
        df = raw.copy(); df['Target'],df['_ts'] = 'Normal',np.int64(0)
        c = self._clean_and_select(df,count=False)
        mu = np.array([self.scaler_mean[f] for f in self.selected_features])
        sd = np.array([self.scaler_std[f] for f in self.selected_features])
        return ((self._impute(c,self.selected_features)-mu)/sd).astype(np.float32)

    def _phase6b_parity_latency(self):
        print('\n[PHASE 6B] Train-serve parity + latency')
        ot = OnlineTransformer.from_json(self.scaler_path)
        real = self.raw_rows.drop(columns=['_ts'],errors='ignore').reset_index(drop=True)
        rng = np.random.RandomState(SEED)
        edge = real.sample(min(200,len(real)),random_state=SEED).reset_index(drop=True)
        edge = edge.astype(object)
        base = [c for c in edge.columns if c in self.selected_features]
        for c in SENTINEL_MINUS1:
            if c in edge.columns: edge.loc[:39,c] = -1
        for c in base[:6]:
            edge.loc[rng.choice(len(edge),15,replace=False),c] = np.inf
            edge.loc[rng.choice(len(edge),15,replace=False),c] = np.nan
            edge.loc[rng.choice(len(edge),15,replace=False),c] = -5.0
        if base: edge = edge.drop(columns=[base[-1]])
        worst = {}
        for tag,df in (('real_rows',real),('edge_cases',edge)):
            ref = self._batch_reference(df)
            recs = df.to_dict('records')
            on = np.stack([ot(r) for r in recs])
            worst[tag] = float(np.nanmax(np.abs(on-ref)))
        ok = all(v < 1e-3 for v in worst.values())
        self.serve_selftest = {'rows_real':len(real),'rows_edge':len(edge),
                               'max_abs_diff':worst,'tolerance':1e-3,'passed':ok}
        print(f'  parity max|online-batch|: real={worst["real_rows"]:.2e} '
              f'edge={worst["edge_cases"]:.2e} -> {"PASS" if ok else "FAIL"}')
        if not ok: raise RuntimeError('[FATAL] train-serve parity failed')
        recs = real.to_dict('records')
        for i in range(CFG['latency_warmup']): ot(recs[i%len(recs)])
        t = np.empty(CFG['latency_calls'])
        for i in range(CFG['latency_calls']):
            r = recs[i%len(recs)]; t0 = time.perf_counter_ns(); ot(r)
            t[i] = time.perf_counter_ns()-t0
        t /= 1e3
        self._lat_samples = t.copy()   # the figure plots exactly these measurements
        self.latency = {'unit':'microseconds per flow','calls':CFG['latency_calls'],
            'warmup':CFG['latency_warmup'],'batch_size':1,
            'p50':round(float(np.percentile(t,50)),2),
            'p95':round(float(np.percentile(t,95)),2),
            'p99':round(float(np.percentile(t,99)),2),
            'mean':round(float(t.mean()),2),
            'includes':'column normalisation, feature derivation, invalid counting, clipping, imputation, standardisation',
            'excludes':'CICFlowMeter flow extraction, network I/O, serialisation',
            'hardware':_hardware(),'python':platform.python_version()}
        print(f'  L1 online latency: P50 {self.latency["p50"]} us | '
              f'P95 {self.latency["p95"]} us | P99 {self.latency["p99"]} us')

    # ------------------------------------------------------------------- FIGURES
    def _phase7_outputs(self,sc):
        print('\n[PHASE 7] Figure pack (10 figures) + audit sample')
        _plt_style()
        self._fig_class_distribution(sc); self._fig_file_volume()
        self._fig_cleaning_waterfall(); self._fig_split_timeline()
        self._fig_burst_gap(); self._fig_global_cut()
        self._fig_dup_conflict(); self._fig_missingness()
        self._fig_ks_scatter(); self._fig_online_latency()
        if self.raw_rows is not None:
            self.raw_rows.drop(columns=['_ts'],errors='ignore').to_csv(
                self.work_dir/'raw_sample.csv',index=False)
        if self.audit_rows:
            pd.concat(self.audit_rows.values(),ignore_index=True).to_csv(
                self.work_dir/'label_audit_sample.csv',index=False)
        rows = []
        for pair,cnt in self.label_conflicts.get('class_pairs',{}).items():
            rows.append({'priority':'1_conflicting_pair','item':pair,'count':cnt,
                         'why':'identical feature vectors carry both labels',
                         'action':'inspect 20 sampled rows of each class in '
                                  'label_audit_sample.csv; decide overlap vs mislabel'})
        for c,v in self.label_rules.items():
            if v.get('rule_checked'):
                pm = 100*v['protocol_mismatch']/max(v['rule_checked'],1)
                sp = 100*v['service_port_mismatch']/max(v['rule_checked'],1)
                if pm > 1 or sp > 1:
                    rows.append({'priority':'2_rule_violation','item':c,
                                 'count':int(v['protocol_mismatch']+v['service_port_mismatch']),
                                 'why':f'protocol mismatch {pm:.2f}%, port mismatch {sp:.2f}%',
                                 'action':'check CICFlowMeter direction inversion before '
                                          'concluding mislabel'})
        if rows:
            pd.DataFrame(rows).sort_values(['priority','count'],ascending=[True,False]
                ).to_csv(self.work_dir/'label_review_worklist.csv',index=False)
            print(f'  label review worklist: {len(rows)} prioritised items')
        print('  10 figures written')

    def _fig_class_distribution(self,sc):
        cls = sorted({k for d in sc.values() for k in d},
                     key=lambda c: sum(d.get(c,0) for d in sc.values()))
        y,h = np.arange(len(cls)),0.20
        ser = [('l2_train.parquet','Train',COLORS['train']),
               ('l2_val.parquet','Validation',COLORS['val']),
               ('l2_test_known.parquet','Test (known)',COLORS['test']),
               ('l2_test_zeroday.parquet','Zero-day holdout',COLORS['zday'])]
        fig,ax = plt.subplots(figsize=(7.16,0.46*len(cls)+1.0))
        for i,c in enumerate(cls):
            if c in CFG['full_zero_day']:
                ax.axhspan(i-.5,i+.5,color=COLORS['zday'],alpha=.08,lw=0)
            elif c in CFG['partial_zero_day']:
                ax.axhspan(i-.5,i+.5,color=COLORS['part'],alpha=.12,lw=0)
        for o,(fn,lb,col) in enumerate(ser):
            v = np.array([sc.get(fn,{}).get(c,0) for c in cls])
            pos = y+(1.5-o)*h
            ax.barh(pos,np.where(v>0,v,np.nan),h,color=col,label=lb,
                    edgecolor='white',lw=.3)
            for p_,val in zip(pos,v):
                if val: ax.text(val*1.15,p_,f'{val:,}',va='center',fontsize=6.5,color=col)
        lab = [c+(' \u2020' if c in CFG['full_zero_day'] else
                  ' \u2021' if c in CFG['partial_zero_day'] else '') for c in cls]
        ax.set_yticks(y); ax.set_yticklabels(lab,fontsize=7)
        ax.set_xscale('log')
        xmax = max(sum(d.get(c,0) for d in sc.values()) for c in cls)
        ax.set_xlim(1,xmax*30); ax.set_ylim(-.6,len(cls)-.4)
        ax.set_xlabel('Flows (log scale); exact counts printed')
        ax.grid(axis='x',ls=':',alpha=.4)
        hd,_ = ax.get_legend_handles_labels()
        hd += [Patch(color=COLORS['zday'],alpha=.25,label='\u2020 Full zero-day (never trained)'),
               Patch(color=COLORS['part'],alpha=.35,label='\u2021 Partial holdout (newest 50%)')]
        ax.legend(handles=hd,fontsize=6,loc='lower right',frameon=True,
                  framealpha=.95,ncol=2)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1a_class_distribution.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_file_volume(self):
        files = sorted(self.per_file_rows.keys())
        if not files:
            print('  [fig] _fig_file_volume SKIPPED: no data'); return
        normals = [self.per_file_class[f].get('Normal',0) for f in files]
        attacks = [sum(v for k,v in self.per_file_class[f].items() if k != 'Normal')
                   for f in files]
        x = np.arange(len(files))
        fig,ax = plt.subplots(figsize=(7.16,3.4))
        ax.bar(x,normals,color=COLORS['train'],edgecolor=COLORS['edge'],lw=.4,label='Normal')
        ax.bar(x,attacks,bottom=normals,color=COLORS['zday'],
               edgecolor=COLORS['edge'],lw=.4,label='Attack')
        for xi,tot in enumerate([n+a for n,a in zip(normals,attacks)]):
            ax.text(xi,tot*1.05,f'{tot:,}',ha='center',fontsize=6.5,rotation=90)
        ax.set_xticks(x)
        ax.set_xticklabels([f.replace('.csv','') for f in files],rotation=45,
                           ha='right',fontsize=6.5)
        ax.set_yscale('log'); ax.set_ylabel('Rows (log)')
        ax.set_title('Per-file flow volume — Normal vs Attack',fontsize=9)
        ax.legend(fontsize=7); ax.grid(axis='y',ls=':',alpha=.4)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1b_file_volume.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_cleaning_waterfall(self):
        """Cleaning audit (comment 35). Log-scale horizontal bars so that 14
        removed rows and 16M kept rows are both legible; colour = action."""
        a = self.audit; final = sum(self.class_totals_clean.values())
        rows = [('Raw rows scanned',a['raw_rows_scanned'],'total'),
                ('Removed: header/ghost rows',a['ghost_rows_dropped'],'removed'),
                ('Removed: negative duration',a['negative_duration_dropped'],'removed'),
                ('Removed: unmapped label',a['unmapped_label_rows_dropped'],'removed'),
                ('Removed: unparseable timestamp',a['unparseable_timestamp_rows_dropped'],'removed'),
                ('Final clean rows (residual 0)',final,'total'),
                ('Kept, imputed: rows with NaN/+Inf',a['rows_with_any_invalid'],'imputed'),
                ('Kept, clipped: rows with negatives',a['rows_with_any_negative'],'imputed'),
                ('Kept, flagged: sentinel -1 cells',a['sentinel_cells_not_invalid'],'flagged'),
                ('Kept: within-class exact duplicates',a['within_class_exact_duplicate_rows'],'flagged'),
                ('Kept: cross-label conflicting vectors',a['label_conflict_feature_vectors'],'flagged')]
        col = {'total':COLORS['train'],'removed':COLORS['zday'],
               'imputed':COLORS['part'],'flagged':'#8C8C8C'}
        fig,ax = plt.subplots(figsize=(7.16,3.9))
        y = np.arange(len(rows))[::-1]
        for yi,(lb,v,kind) in zip(y,rows):
            if v > 0: ax.barh(yi,v,color=col[kind],edgecolor='white',lw=.4,height=.62)
            ax.text(max(v,1)*1.35,yi,f'{int(v):,}',va='center',fontsize=7)
        ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows],fontsize=7.5)
        ax.set_xscale('log'); ax.set_xlim(0.8,max(r[1] for r in rows)*40)
        ax.set_xlabel('Count (log scale)')
        ax.legend(handles=[Patch(color=col[k],label=t) for k,t in
                           (('total','totals'),('removed','removed'),
                            ('imputed','kept, value corrected'),('flagged','kept, flagged only'))],
                  fontsize=6.5,frameon=False,loc='upper right')
        ax.grid(axis='x',ls=':',alpha=.4)
        for sp in ('top','right'): ax.spines[sp].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1c_cleaning_waterfall.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_split_timeline(self):
        per = self.split_diag.get('per_class',{})
        if not per: return
        classes = sorted(per,key=lambda c:-per[c]['n'])
        fig,ax = plt.subplots(figsize=(7.16,0.30*len(classes)+1.2))
        for i,cls in enumerate(classes):
            pc = per[cls]['partition_counts']; tot = max(sum(pc.values()),1)
            left = 0
            for key,col in (('train',COLORS['train']),('val',COLORS['val']),
                             ('test',COLORS['test']),('zero_day',COLORS['zday'])):
                w = pc.get(key,0)/tot
                ax.barh(i,w,left=left,height=.7,color=col,edgecolor='white',lw=.3)
                left += w
        ax.set_yticks(range(len(classes)))
        ax.set_yticklabels(classes,fontsize=6.5); ax.set_xlim(0,1)
        ax.set_xlabel('Fraction of class rows')
        ax.set_title('Chronological per-class split — partition proportions',fontsize=9)
        hd = [Patch(color=COLORS['train'],label='Train'),
              Patch(color=COLORS['val'],label='Validation'),
              Patch(color=COLORS['test'],label='Test'),
              Patch(color=COLORS['zday'],label='Zero-day')]
        ax.legend(handles=hd,fontsize=6.5,ncol=4,loc='lower center',
                  bbox_to_anchor=(0.5,-0.18),frameon=False)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1d_split_timeline.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_burst_gap(self):
        per = self.split_diag.get('per_class',{})
        if not per: return
        gaps = sorted(CFG['burst_gap_grid'])
        classes = sorted(per,key=lambda c:-per[c]['n'])[:8]
        fig,ax = plt.subplots(figsize=(7.16,3.6))
        width = 0.8/len(classes); x = np.arange(len(gaps))
        for k,cls in enumerate(classes):
            vals = []
            for g in gaps:
                grid = per[cls].get('burst_gap_grid',{}).get(str(g),{})
                vals.append(grid.get('straddle_rows_final',0))
            ax.bar(x+k*width-0.4+width/2,vals,width,
                    color=plt.cm.tab10(k),label=cls,edgecolor='white',lw=.3)
        ax.set_xticks(x); ax.set_xticklabels([f'{g}s' for g in gaps])
        ax.set_xlabel('Burst gap (seconds)')
        ax.set_ylabel('Straddling-burst row mass (final)')
        ax.set_yscale('symlog')
        ax.set_title('Burst-gap sensitivity — rows crossing a boundary inside one burst',
                      fontsize=9)
        ax.legend(fontsize=6,ncol=2,loc='upper right')
        ax.grid(axis='y',ls=':',alpha=.4)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1e_burst_gap_sensitivity.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_global_cut(self):
        gc_ = self.split_diag.get('global_cutoff',{}).get(
            'per_class_vs_global_divergence',{})
        if not gc_:
            print('  [fig] global-cut divergence SKIPPED: no per-class divergence (global mode run)')
            return
        classes = sorted(gc_,key=lambda c:-gc_[c]['pct_changed'])
        vals = [gc_[c]['pct_changed'] for c in classes]
        mean_v = float(np.mean(vals))
        fig,ax = plt.subplots(figsize=(7.16,3.6))
        ax.bar(range(len(classes)),vals,color=COLORS['train'],
               edgecolor=COLORS['edge'],lw=.4)
        ax.axhline(mean_v,color=COLORS['part'],ls='--',lw=1.3,
                    label=f'mean = {mean_v:.2f}%')
        for i,v in enumerate(vals):
            ax.text(i,v+0.5,f'{v:.1f}',ha='center',fontsize=6.5)
        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes,rotation=40,ha='right',fontsize=6.5)
        ax.set_ylabel('% of class rows moving partition')
        ax.set_title('Global-cut vs per-class split — divergence per class',fontsize=9)
        ax.legend(fontsize=7); ax.grid(axis='y',ls=':',alpha=.4)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1f_global_cut_divergence.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_dup_conflict(self):
        if not self.dup_report:
            print('  [fig] _fig_dup_conflict SKIPPED: no data'); return
        classes = sorted(self.dup_report,key=lambda c:self.dup_report[c]['dup_pct'])
        pairs = list(self.label_conflicts.get('class_pairs',{}).items())[:10]
        fig,axes = plt.subplots(1,2,figsize=(7.16,0.26*max(len(classes),len(pairs),6)+1.1),
                                gridspec_kw={'width_ratios':[1,1.15]})
        vals = [self.dup_report[c]['dup_pct'] for c in classes]
        axes[0].barh(range(len(classes)),vals,color=COLORS['zday'],height=.65)
        for i_,v in enumerate(vals):
            axes[0].text(v+1,i_,f'{v:.1f}%',va='center',fontsize=6.5)
        axes[0].set_yticks(range(len(classes))); axes[0].set_yticklabels(classes,fontsize=7)
        axes[0].set_xlim(0,max(vals+[1])*1.25)
        axes[0].set_xlabel('Within-class exact duplicates (%)',fontsize=7.5)
        axes[0].set_title('(a) Duplicates, kept',fontsize=8.5)
        if pairs:
            names = [p_[0].replace(' | ',' / ') for p_ in pairs][::-1]; counts = [p_[1] for p_ in pairs][::-1]
            axes[1].barh(range(len(names)),counts,color='#8C8C8C',height=.65)
            for i_,v in enumerate(counts):
                axes[1].text(v*1.15,i_,f'{v:,}',va='center',fontsize=6.5)
            axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels(names,fontsize=7)
            axes[1].set_xscale('log'); axes[1].set_xlim(0.8,max(counts)*20)
            axes[1].set_xlabel('Identical feature vectors (log)',fontsize=7.5)
        else:
            axes[1].text(.5,.5,'No cross-label conflicts',ha='center',va='center',fontsize=8)
            axes[1].set_xticks([]); axes[1].set_yticks([])
        axes[1].set_title('(b) Cross-label conflicts, top pairs',fontsize=8.5)
        for ax in axes:
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1g_duplicate_conflict.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_missingness(self):
        if not self.per_feat_invalid: return
        files = sorted(self.per_file_rows.keys())
        feats = sorted(f for f,d in self.per_feat_invalid.items() if sum(d.values()) > 0)
        if not feats:
            print('  [fig] missingness SKIPPED: no invalid cells'); return
        m = np.zeros((len(feats),len(files)))
        for i,f in enumerate(feats):
            for j,fp in enumerate(files):
                n_rows = self.per_file_rows.get(fp,1)
                m[i,j] = self.per_feat_invalid[f].get(fp,0)/max(n_rows,1)
        fig,ax = plt.subplots(figsize=(7.16,0.2*len(feats)+1.6))
        im = ax.imshow(m,cmap='Reds',aspect='auto',vmin=0,vmax=max(m.max(),1e-6))
        plt.colorbar(im,ax=ax,fraction=0.04,label='Invalid-cell fraction')
        ax.set_xticks(range(len(files)))
        ax.set_xticklabels([f.replace('.csv','') for f in files],rotation=45,
                           ha='right',fontsize=6.5)
        ax.set_yticks(range(len(feats)))
        ax.set_yticklabels(feats,fontsize=7)
        ax.set_title('Invalid (NaN/+Inf) cell fraction by feature and file; sentinel -1 excluded',
                      fontsize=9)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1h_missingness_heatmap.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_ks_scatter(self):
        tv = {r['feature']:r['ks_stat'] for r in self.ks_train_val}
        tt = {r['feature']:r['ks_stat'] for r in self.ks_train_test}
        common = sorted(set(tv)&set(tt))
        if not common:
            print('  [fig] _fig_ks_scatter SKIPPED: no data'); return
        xs = [tv[f] for f in common]; ys = [tt[f] for f in common]
        sv = {r['feature']:r.get('ks_std',0.0) for r in self.ks_train_val}
        st = {r['feature']:r.get('ks_std',0.0) for r in self.ks_train_test}
        fig,ax = plt.subplots(figsize=(3.5,3.2))
        ax.errorbar(xs,ys,xerr=[sv[f] for f in common],yerr=[st[f] for f in common],
                    fmt='o',ms=3,alpha=.6,color=COLORS['train'],ecolor='#AAAAAA',lw=.6)
        mx = max(max(xs),max(ys))*1.05
        ax.plot([0,mx],[0,mx],'--',color='#CCCCCC',lw=.8,label='y = x')
        worst = sorted(common,key=lambda f:-tt[f])[:5]
        for f in worst:
            ax.annotate(f,(tv[f],tt[f]),fontsize=6.5,
                         xytext=(4,3),textcoords='offset points')
        ax.set_xlabel('KS train vs validation (mean $\\pm$ sd, %d seeds)' % CFG['ks_seeds'],fontsize=7.5)
        ax.set_ylabel('KS train vs test',fontsize=7.5)
        ax.set_title('Feature stability: train-val vs train-test KS',fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=.3)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1i_ks_stability.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _fig_online_latency(self):
        if not self.latency or getattr(self,'_lat_samples',None) is None:
            print('  [fig] _fig_online_latency SKIPPED: no data'); return
        p50 = self.latency['p50']; p95 = self.latency['p95']; p99 = self.latency['p99']
        t = self._lat_samples
        assert abs(float(np.percentile(t,95))-p95) < 0.01, 'figure/manifest latency mismatch'
        fig,ax = plt.subplots(figsize=(7.16,3.4))
        ax.hist(t,bins=60,color=COLORS['train'],edgecolor=COLORS['edge'],lw=.3)
        ax.axvline(p50,color=COLORS['val'],ls='--',lw=1.3,label=f'P50={p50:.1f}us')
        ax.axvline(p95,color=COLORS['part'],ls='--',lw=1.3,label=f'P95={p95:.1f}us')
        ax.axvline(p99,color=COLORS['zday'],ls='--',lw=1.3,label=f'P99={p99:.1f}us')
        ax.set_xlabel('Online transform latency (us per flow)')
        ax.set_ylabel('Count')
        ax.set_title('Layer 1 online transform latency',fontsize=9)
        ax.legend(fontsize=7); ax.grid(axis='y',ls=':',alpha=.4)
        for s in ('top','right'): ax.spines[s].set_visible(False)
        fig.tight_layout()
        for ext in ('pdf','png'):
            fig.savefig(self.fig_dir/f'L1_fig1j_online_latency.{ext}',
                        dpi=600,bbox_inches='tight')
        plt.close(fig)

    def _phase8_manifest(self,ks,l2c,out_dir):
        prev = 'GENESIS_TITANIUM6_L1_R8.0'
        for fn in ('l2_train.parquet','l2_val.parquet',
                   'l2_test_known.parquet','l2_test_zeroday.parquet'):
            if (out_dir/fn).exists():
                prev = hashlib.sha256((prev+fn+_sha256(out_dir/fn)).encode()).hexdigest()[:16]
                self.hash_chain[fn] = prev
        lr = {c:dict(v,**({'protocol_mismatch_pct':round(100*v['protocol_mismatch']/
                                                          max(v['rule_checked'],1),3),
                            'service_port_mismatch_pct':round(100*v['service_port_mismatch']/
                                                              max(v['rule_checked'],1),3)}
                           if v.get('rule_checked') else {}),
                     zero_payload_pct=round(100*v.get('zero_payload_both_dirs',0)/
                                            max(v['rows'],1),3))
              for c,v in self.label_rules.items()}
        man = {
            'version':CFG['version'],
            'created_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
            'config':CFG,'environment':self.env,
            'source_file_hashes':self.source_hashes,
            'l2_contract':{'schema_version':5,
                'bootstrap_block_recommendation':f'{CFG["bootstrap_block_minutes"]}-min blocks from _ts '
                    '(>=30 clusters); day blocks only as sensitivity',
                'raw_sample':str(self.work_dir/'raw_sample.csv'),'split_mode':CFG['split_mode'],
                'ks_for_feature_selection':'ks_diagnostics.ks_train_val.results',
                'ks_for_reporting_only':'ks_diagnostics.ks_train_test.results',
                'reserved_columns_not_features':RESERVED_COLS,
                'dedup_eval':'report metrics on full test AND on _dup_in_train==0',
                'online_transform':'OnlineTransformer.from_json(scaler_params.json)',
                'l2_ready_carries_ts':True,
                'l2_ready_ts_note':'l2_ready/*.parquet carries _ts (int64 ns) for L2 day-block bootstrap'},
            'clock_decision':self.clock,
            'twin_label_census':{k:v for k,v in self.twin_labels.items()},
            'twin_label_definition':('for rows with an exact feature twin in train: label '
                                     'of that train twin (Normal / same_class / other_attack / '
                                     'mixed_labels = the train vector itself carries >1 label)'),
            'global_mode_rule':('split_mode=global only: a class with no training rows under the '
                                'single time cut is moved to the zero-day file (it is unseen by '
                                'construction) and listed in diagnostics; a class with no test rows '
                                'is listed as not evaluable. Never applied in per_class mode.'),
            'straddle_attack_classes':dict(getattr(self,'straddle_attack',{}),
                effectiveness=('MEASURED INEFFECTIVE on this dataset: continuous '
                               'flood classes have no burst edge near the cut. '
                               'Report as a negative result; boundary_adjacency_'
                               'leakage is the quantity that matters.')),
            'boundary_adjacency_leakage':getattr(self,'boundary_leak_summary',{}),
            'label_review_worklist':str(self.work_dir/'label_review_worklist.csv'),
            'sentinel_cells_by_feature':dict(self.per_feat_sentinel),
            'ks_diagnostics':ks,
            'cleaning_audit':{**{k:int(v) for k,v in self.audit.items()},
                'reconciliation_table':self.recon_table,
                'timestamp_parsing_per_file':self.ts_report,
                'within_class_duplicates_by_class':self.dup_report,
                'column_coverage_gaps_imputed':self.col_coverage,
                'per_file_class_counts':{f:dict(d) for f,d in self.per_file_class.items()},
                'per_file_total_rows':dict(self.per_file_rows)},
            'label_quality_audit':{'feature_vector_conflicts':self.label_conflicts,
                'plausibility_rules':{k:[sorted(v[0]),sorted(v[1]) if v[1] else None]
                                      for k,v in LABEL_RULES.items()},
                'plausibility_by_class':lr,
                'manual_sample':str(self.work_dir/'label_audit_sample.csv'),
                'caveat':'Port/protocol mismatches include CICFlowMeter flow-'
                         'direction inversions; conflicts include intrinsic overlap. '
                         'Both are lower-bound indicators, not removals.'},
            'cross_partition_duplicates':self.xpart_dups,
            'prevalence':self.prevalence,
            'zero_day_composition':self.zd_comp,
            'class_totals_after_cleaning':dict(self.class_totals_clean),
            'split_methodology':{'mode':CFG['split_mode'],
                **{k:CFG[k] for k in ('train_frac','val_frac','burst_gap_s',
                                      'burst_gap_grid','max_snap_frac','embargo_s',
                                      'full_zero_day','partial_zero_day')},
                'split_row_counts':dict(self.total_written),'l2_row_counts':l2c,
                'diagnostics':self.split_diag},
            'features':{'selected_features':self.selected_features,
                'feature_count':len(self.selected_features),
                'derived_features':DERIVED_FEATURES,
                'excluded_by_design':['Dst Port (shortcut; Engelen et al. 2021)',
                                       'Src/Dst IP, Src Port, Flow ID, Timestamp']},
            'scaler':{'type':'StandardScaler','fit_on':'train only',
                'algorithm':'exact two-pass','file':str(self.scaler_path),
                'invalid_policy':CFG['invalid_policy']},
            'train_serve_parity':self.serve_selftest,
            'layer1_online_latency':self.latency,
            'artifact_hashes':self.hash_chain,
            'feature_statistics':self.feature_stats,
            'figures':{
                'Fig_1a':'L1_fig1a_class_distribution.pdf',
                'Fig_1b':'L1_fig1b_file_volume.pdf',
                'Fig_1c':'L1_fig1c_cleaning_waterfall.pdf',
                'Fig_1d':'L1_fig1d_split_timeline.pdf',
                'Fig_1e':'L1_fig1e_burst_gap_sensitivity.pdf',
                'Fig_1f':'L1_fig1f_global_cut_divergence.pdf',
                'Fig_1g':'L1_fig1g_duplicate_conflict.pdf',
                'Fig_1h':'L1_fig1h_missingness_heatmap.pdf',
                'Fig_1i':'L1_fig1i_ks_stability.pdf',
                'Fig_1j':'L1_fig1j_online_latency.pdf',
                'Fig_2':'TITANIUM6_Fig2_architecture.pdf (Draw.io)'},
            'paper_map':{
                'Fig_1a':'III-C class distribution',
                'Fig_1b':'III-B per-file volumes',
                'Fig_1c':'III-B cleaning reconciliation (comment 35)',
                'Fig_1d':'III-C chronological split (comment 32 visual)',
                'Fig_1e':'III-C burst-gap sensitivity (comment 33)',
                'Fig_1f':'III-C global-cut divergence (comment 32 numeric)',
                'Fig_1g':'III-B duplicate + label conflict (comment 39)',
                'Fig_1h':'III-B missingness heatmap (comment 36)',
                'Fig_1i':'V-A KS stability (comment 37)',
                'Fig_1j':'III-D online latency (comment 46 L1 part)',
                'Fig_2' :'IV architecture'},
            'l2_input_paths':{**{k:str(out_dir/f'l2_{k}.parquet')
                                  for k in ('train','val','test_known','test_zeroday')},
                              'scaler':str(self.scaler_path)},
        }
        _atomic_json(man,self.manifest_path)

    def execute(self):
        t0 = time.time()
        self._phase0()
        chunks = self._phase1_scatter()
        self._decide_clock()
        self._phase2_split(chunks)
        self._phase3_scaler()
        ks = self._phase4_ks()
        l2c,out_dir = self._phase5_write()
        sc = self._phase6_integrity(out_dir)
        self._phase6b_parity_latency()
        self._phase7_outputs(sc)
        self._phase8_manifest(ks,l2c,out_dir)
        import shutil
        shutil.rmtree(self.tmp_dir,ignore_errors=True)
        if not CFG.get('keep_splits'):
            shutil.rmtree(self.splits_dir,ignore_errors=True)
        x = self.xpart_dups.get('test_known',{})
        print('\n'+'='*76)
        print(f'  LAYER 1 R8.0 COMPLETE | {(time.time()-t0)/60:.1f} min | '
              f'mode={CFG["split_mode"]} | residual=0 | parity=PASS')
        print(f'  features={len(self.selected_features)} | twin-in-train '
              f'{x.get("pct",0):.2f}% | L1 P95 {self.latency.get("p95")} us')
        print('  10 figures written; manifest carries _ts for L2 block bootstrap')
        print('='*76)

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--source',default='/kaggle/input/ids-intrusion-csv')
    ap.add_argument('--workdir',default=os.environ.get('T6_L1','/kaggle/working/l1_R8'))
    ap.add_argument('--split-mode',choices=['per_class','global'],
                    default=os.environ.get('T6_SPLIT_MODE') or None)
    ap.add_argument('--invalid-policy',choices=['semantic','median','p999','zero'],
                    default=os.environ.get('T6_INVALID_POLICY') or None)
    ap.add_argument('--embargo-s',type=float,
                    default=(float(os.environ['T6_EMBARGO_S'])
                             if os.environ.get('T6_EMBARGO_S') else None))
    ap.add_argument('--keep-splits',action='store_true')
    a,_ = ap.parse_known_args()
    if a.split_mode: CFG['split_mode'] = a.split_mode
    if a.invalid_policy: CFG['invalid_policy'] = a.invalid_policy
    if a.embargo_s is not None: CFG['embargo_s'] = a.embargo_s
    CFG['keep_splits'] = a.keep_splits
    src = a.source
    if not os.path.isdir(src):
        for fb in ('/kaggle/input/ids-intrusion-csv',
                   '/kaggle/input/datasets/solarmainframe/ids-intrusion-csv'):
            if os.path.isdir(fb): src = fb; break
    Layer1Pipeline(src,a.workdir).execute()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  TITANIUM-6 | LAYER 1 | TITANIUM6_LAYER1_R8.0 | files=10
  mode=per_class embargo=0.0s policy=semantic

[PHASE 0] Hashes, environment, schema
  base=77 derived=10 total=87 | Intel(R) Xeon(R) CPU @ 2.00GHz

[PHASE 1] Scatter
  [ 1/10] 02-14-2018.csv        1,048,570 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 2/10] 02-15-2018.csv        1,048,575 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 3/10] 02-16-2018.csv        1,048,574 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 4/10] 02-20-2018.csv        7,948,748 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 5/10] 02-21-2018.csv        1,048,575 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 6/10] 02-22-2018.csv        1,048,566 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 7/10] 02-23-2018.csv        1,048,575 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 8/10] 02-28-2018.csv          613,071 | %d/%m/%Y %H:%M:%S | in-window 1.0000
  [ 9/10] 03-01-2018.csv          331,100 | %d/%m/%Y %H:%M:%S 

In [3]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 2 | R8.0
#  R8.0 over R6.5 (every R6.5 measurement kept):
#   T1 PRIMARY RULE = Youden J on the validation THRESHOLD half (calibrated
#      scale). R6.5 measured that the FPR<=1% budget rule does not transfer
#      across the chronological boundary (val 1.0% -> test 9.3%). The budget
#      rule is still computed and its transfer failure is still reported
#      (limitations.threshold_transfer), now evaluated at the BUDGET tau
#      rather than at the deployed tau, so the finding stays correct.
#      All five rules remain reported on test (test.DEPLOYED.by_rule).
#   T2 Ablation and random-split baseline use the SAME primary rule.
#   T3 Shortcut ablation: T6_EXCLUDE_FEATURES (comma list) removes candidate
#      features before selection; the profile is recorded in the manifest.
#  R6.5 closes the last reuse of a validation slice and adds two measurements:
#   S1 THREE-WAY DISJOINT VALIDATION. R6.3 selected the model family AND fitted
#      the threshold on the same validation half. Family selection now uses its
#      own slice (fam), calibration a second (cal) and the threshold a third
#      (thr). No validation row informs two decisions.
#   S2 SELECTION IS A TIE, AND THE CODE SAYS SO. The family's val AUC-PR now
#      carries a block-bootstrap CI. When the top two CIs overlap, the choice
#      is a statistical tie and the tiebreak (measured batch-1 latency) is
#      recorded with its reason, instead of ranking on a 0.002 difference.
#   S3 DUAL BLOCK BOOTSTRAP at 15 and 60 minutes: if the CI width depends on
#      block length, the dependence itself is reported.
#   S4 PER-CLASS FLAG RATE on test: which attack families the gate actually
#      catches, beside the aggregate recall.
#  R6.3 (v17.3) | Binary gatekeeper
#  R6.3 closes the five items that kept R6.2 short of complete. None is a code
#  fault; each converts something CLAIMED or UNEXAMINED into something MEASURED.
#   D1 THRESHOLD TRANSFER. tau chosen at val FPR <= 1% delivers 9.328% on test
#      (real run). R6.3 measures the transfer instead of reporting the gap:
#      the tau that WOULD have held the budget on test, the movement required,
#      the per-60-min-block FPR distribution at the deployed tau, and how many
#      blocks breach the budget (the recalibration trigger a deployment needs).
#   D2 STAGE 1 IS INERT. Real run: 87/87 features pass, max KS 0.0885 < 0.10.
#      The manifest now says MEASURED INERT and the run prints a warning, so
#      the paper cannot claim a two-stage selection it did not perform.
#   D3 SELECTION STABILITY. Nogueira 0.515 is moderate instability. The 0.70
#      and 0.90 frequency bands and their overlap are reported, so the feature
#      count carries its uncertainty.
#   D4 CI WIDTH RATIO. block vs i.i.d. width ratio is computed and printed:
#      the i.i.d. interval understates temporal uncertainty by that factor.
#   D5 HONEST BANNER. The completion line no longer implies the budget held on
#      test; a [LIMITATION] line fires when test FPR exceeds 2x the budget, and
#      the manifest records which family this protocol actually selected.
#  R6.2 (v17.2)
#  Input: Layer 1 R5.1 manifest (schema 5, _ts reserved). Output contract is
#  identical to R5.3 (manifest_l2.json, best_model.pkl, best_cal.pkl,
#  thresholds.json) so Layers 3-8 need no change to load it.
#  Protocol pre-declared before any result (stage-1 KS>0.10, 10 resampled MI
#  seeds, ES from train, disjoint cal/thr halves, val-AUC-PR selection,
#  FPR<=1% threshold, 60-min block bootstrap, full + twin-free reporting).
#
#  Every change below closes a supervisor comment (SC) or a defect found by
#  code audit of v15.0 (CA). Nothing is changed for style.
#
#  SC Test-set leakage in selection  Stage 1 reads ks_diagnostics.ks_train_val
#       with bracket access; the test partition never informs the features.
#  SC Selection stability  MI repeated over 10 seeds (stability selection,
#       Meinshausen & Buehlmann 2010); keep features selected in >=70% of
#       seeds; per-feature frequencies + Nogueira stability index reported.
#       ALWAYS_KEEP rule and Stage 1 are ablated.
#  SC Unequal model comparisons  LightGBM, XGBoost and RF are trained on the
#       SAME matched natural-prior sample with the SAME class weighting and
#       the SAME early-stopping set. The deployed model (the selected family
#       on the full training set) is reported separately and labelled as a
#       computational-budget result.
#  CA Model selected on TEST F1 in v15.0 (self.best = max over test metrics).
#       Now selected on validation AUC-PR, pre-declared.
#  CA Validation used three times (early stopping, calibration, threshold).
#       Now: early stopping on a 5% hold-out of TRAIN; validation split into
#       disjoint CALIBRATION and THRESHOLD halves.
#  SC Calibration protocol  tau is selected on CALIBRATED probabilities of the
#       threshold half and applied to calibrated probabilities everywhere
#       (score_space is exported for Layer 5). Calibrator (Platt vs isotonic)
#       chosen by 5-fold CV log-loss INSIDE the calibration half.
#  SC Calibration prevalence  validation keeps the natural prior, so the
#       calibration is deployment-prior; ECE/MCE/Brier reported.
#  SC Threshold clamp  primary rule is pre-declared: max recall s.t. FPR<=1%
#       (false-alert budget). Youden (unclamped), Youden clamped [0.30,0.70]
#       (legacy), min-recall 0.95 and cost-ratio 10:1 are all reported on
#       test -- with and without the clamp.
#  SC Bootstrap independence  block bootstrap over 1-minute time blocks
#       (from Layer-1 _ts); i.i.d. CIs shown beside them for contrast.
#  SC Significance testing  McNemar with discordant counts b, c, odds ratio
#       + CI, Holm-adjusted p; DeLong test for AUC differences (Holm);
#       paired block-bootstrap CIs of delta-F1 and delta-AUC as primary
#       evidence.
#  SC Fig. 3 "statistically equivalent"  replaced by DeLong + CI of dAUC.
#  SC Fig. 7  reliability diagram on the full natural-prior test set with bin
#       counts, ECE and MCE (v15.0 used the first 50k test rows, ~all
#       attacks, because the file was class-ordered).
#  SC Published baseline  hard-coded Ferrag (Edge-IIoTset) row removed. An
#       IN-HOUSE random-split baseline (same model, same budget) quantifies
#       how much random splitting inflates F1 on this dataset.
#  SC Latency reproducibility  measured, not summed: model-only (batch 1 and
#       batch 1000) and the FULL online path raw record -> L1 transform ->
#       model -> calibrator -> decision, with warm-up, hardware and versions.
#       The v15.0 constants (CICFlowMeter 5.0 ms + L1 0.5 ms) are gone.
#  SC Adversarial preprocessing  perturbations applied in RAW feature space
#       (inverse L1 scaler), clipped at 0 in raw space, re-standardised;
#       an identity round-trip check must pass first.
#  SC Threat-model validity  two experiments, named correctly: (a) corruption
#       robustness (independent noise, the v15.0 design in the right space);
#       (b) constrained evasion: forward-payload padding and inter-arrival
#       delay applied to attack flows only, with rates, bidirectional packet
#       statistics, ratios and log-duration recomputed so every perturbed
#       flow stays internally consistent.
#  SC Collapse sanity checks  0% reproduces baseline exactly (asserted),
#       perturbed confusion matrices, score histograms, changed-feature
#       statistics, single-feature perturbation tests.
#  SC Dedup / burst leakage  every test metric also on _dup_in_train==0.
#  SC Zero-day composition  L2 flag rate on the holdout: support-weighted,
#       macro, fully-unseen aggregate, per class.
#  CA Ablation confounded size with features (25k vs 11.2M). Now retrained
#       on one fixed ablation sample for baseline and every variant.
#  EXPORTS for Layers 3-5: per-row calibrated scores for test_known and
#       zero-day (row-aligned with Layer 1 files) so the Layer-2/Layer-4
#       cascade is MEASURED, not multiplied under independence.
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os
import gc
import sys
import json
import time
import hashlib
import platform
import warnings
import argparse
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
import scipy
from scipy.stats import rankdata, norm, binomtest, chi2
import lightgbm as lgb
import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, \
    precision_recall_curve, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
import pickle
from scipy.special import logit as _logit_fn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

CFG = {
    'version': 'TITANIUM6_LAYER2_R8.0',
    'seed': SEED,
    'required_l1_schema': 5,

    # feature selection
    'stage1_ks_max': 0.10,           # pre-declared effect-size rule
    'block_minutes': 60,
    'mi_threshold': 0.001, 'mi_seeds': 10, 'mi_sample_per_class': 12_500,
    'stability_keep_frac': 0.70,

    # data protocol
    'es_frac': 0.05,                  # early-stopping hold-out from TRAIN
    'matched_n': 1_000_000,           # identical sample for all 3 families
    'rf_full_cap': 3_000_000,         # RF full-budget cap (memory/time)

    # models (hyper-parameters unchanged from v15.0)
    'lgbm': {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 63,
             'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
             'bagging_freq': 5, 'min_child_samples': 20, 'verbose': -1,
             'seed': SEED, 'n_jobs': -1},
    'xgb': {'objective': 'binary:logistic', 'eval_metric': 'logloss', 'max_depth': 6,
            'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8,
            'min_child_weight': 5, 'seed': SEED, 'nthread': -1, 'verbosity': 0,
            'tree_method': 'hist'},
    'rf': {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 'sqrt',
           'class_weight': 'balanced_subsample', 'n_jobs': -1, 'random_state': SEED},
    'rounds': 500, 'early_stop': 50,
    'selection_metric': 'val_auc_pr',   # pre-declared

    # thresholds (pre-declared)
    'primary_rule': 'youden',
    'fpr_budget': 0.01,
    'min_recall': 0.95,
    'cost_ratio_fn_fp': 10.0,
    'legacy_clamp': (0.30, 0.70),
    'fpr_op_points': [0.01, 0.001, 0.0001],

    # statistics
    'boot_counts': 1000, 'boot_auc': 200, 'ece_bins': 15,

    # latency
    'lat_warmup': 200, 'lat_calls': 5000, 'lat_batch': 1000,

    # adversarial (pre-declared: robust if relative drop < 5%)
    'adv_n': 50_000, 'adv_levels': [0.05, 0.10, 0.20],
    'evasion_scales': [1.05, 1.10, 1.20, 1.50], 'robust_drop': 0.05,
    'single_feature_scale': 1.20,

    # ablation / random-split baseline
    'ablation_n': 500_000, 'ablation_topk': 10, 'random_split_pool': 1_500_000,
}

ALWAYS_KEEP = {
    'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt',
    'URG Flag Cnt', 'CWR Flag Count', 'ECE Flag Count', 'Fwd PSH Flags',
    'Bwd PSH Flags', 'Init Fwd Win Byts__present', 'Init Bwd Win Byts__present',
    'Zero_Duration_Flag', 'Log_Flow_Duration', 'Ratio__FwdBwd_Packets',
    'Ratio__FwdBwd_Bytes', 'Ratio__FwdBwd_IAT', 'Ratio__Active_Idle',
}

DERIVED_SRC = {'Ratio__FwdBwd_Packets': ('Total Fwd Packets', 'Total Backward Packets'),
               'Ratio__FwdBwd_Bytes': ('Total Length of Fwd Packets', 'Total Length of Bwd Packets'),
               'Ratio__FwdBwd_IAT': ('Fwd IAT Tot', 'Bwd IAT Tot'),
               'Ratio__Active_Idle': ('Active Mean', 'Idle Mean')}
DERIVED_ALL = set(DERIVED_SRC) | {'Log_Flow_Duration', 'Zero_Duration_Flag',
                                  'Init Fwd Win Byts__present', 'Init Bwd Win Byts__present',
                                  'N_Invalid_Features', 'N_Clipped_Negatives'}


def _ratio(n, d):
    return np.clip(n / np.where(d == 0, 1.0, d), 0.0, 1e6)


def _rederive(R, R0, fi):
    """Propagate source changes into Layer-1 derived features as DELTAS:
    derived_new = derived_old + g(src_new) - g(src_old). Exact identity when
    nothing changed; physically consistent change otherwise."""
    for d, (a, b) in DERIVED_SRC.items():
        if d in fi and a in fi and b in fi:
            R[:, fi[d]] = np.clip(R0[:, fi[d]] + _ratio(R[:, fi[a]], R[:, fi[b]])
                                  - _ratio(R0[:, fi[a]], R0[:, fi[b]]), 0.0, 1e6)
    if 'Flow Duration' in fi:
        D, D0 = np.maximum(R[:, fi['Flow Duration']], 0), np.maximum(R0[:, fi['Flow Duration']], 0)
        if 'Log_Flow_Duration' in fi:
            R[:, fi['Log_Flow_Duration']] = np.maximum(
                R0[:, fi['Log_Flow_Duration']] + np.log1p(D) - np.log1p(D0), 0.0)
        if 'Zero_Duration_Flag' in fi:
            R[:, fi['Zero_Duration_Flag']] = np.where(D == D0, R0[:, fi['Zero_Duration_Flag']],
                                                      (D == 0).astype(float))
    return R


# constrained-evasion feature groups (Layer-1 normalised names)
FWD_BYTES = ['Total Length of Fwd Packets', 'Fwd Packet Length Max',
             'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std',
             'Fwd Seg Size Avg', 'Subflow Fwd Bytes', 'Fwd Byts/b Avg']
TIME_FEATS = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
              'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
              'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
              'Bwd IAT Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min',
              'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
PKT_RATES = ['Flow Packets/s', 'Fwd Pkts/s', 'Bwd Pkts/s']

OK = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'verm': '#D55E00',
      'purple': '#CC79A7', 'sky': '#56B4E9', 'grey': '#555555', 'black': '#000000'}
MODEL_COL = {'LightGBM': OK['blue'], 'XGBoost': OK['orange'], 'RandomForest': OK['green']}


# =============================================================================
#  UTILITIES
# =============================================================================
def _gc():
    gc.collect()


def _mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)


def _hardware():
    cpu = platform.processor() or ''
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    cpu = line.split(':', 1)[1].strip()
                    break
    except OSError:
        pass
    return {'cpu_model': cpu, 'physical_cores': psutil.cpu_count(logical=False),
            'logical_cores': psutil.cpu_count(logical=True),
            'ram_gb': round(psutil.virtual_memory().total / 2**30, 1),
            'python': platform.python_version(), 'lightgbm': lgb.__version__,
            'xgboost': xgb.__version__, 'sklearn': sklearn.__version__,
            'numpy': np.__version__, 'pandas': pd.__version__}


def _read_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)


def _atomic_json(obj, dest):
    tmp = str(dest) + '.__tmp__'
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    os.replace(tmp, dest)


def _load(path, feats, extra=('Target',)):
    cols = list(feats) + [c for c in extra]
    df = pd.read_parquet(path, columns=cols)
    X = df[list(feats)].to_numpy(np.float32)
    return X, df[[c for c in extra]].reset_index(drop=True)


def _row_hash(df):
    """Identical to Layer 1's row hash (same float32 values, same columns)."""
    return pd.util.hash_pandas_object(df, index=False).values.astype(np.uint64)


def _counts(y, pred):
    y = y.astype(bool)
    pred = pred.astype(bool)
    tp = int((y & pred).sum())
    fp = int((~y & pred).sum())
    fn = int((y & ~pred).sum())
    tn = int((~y & ~pred).sum())
    return tp, fp, fn, tn


def _rates(tp, fp, fn, tn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / den if den else 0.0
    return {'precision': p, 'recall': r, 'f1': f1, 'fpr': fpr, 'mcc': float(mcc),
            'balanced_acc': 0.5 * (r + (tn / (tn + fp) if tn + fp else 0.0))}


def _ece(y, p, bins):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    cnt = np.bincount(idx, minlength=bins)
    sp = np.bincount(idx, weights=p, minlength=bins)
    sy = np.bincount(idx, weights=y, minlength=bins)
    nz = cnt > 0
    conf = np.where(nz, sp / np.maximum(cnt, 1), np.nan)
    acc = np.where(nz, sy / np.maximum(cnt, 1), np.nan)
    gap = np.abs(conf - acc)
    return {'ece': float(np.nansum(gap * cnt) / max(cnt.sum(), 1)),
            'mce': float(np.nanmax(gap[nz])) if nz.any() else 0.0,
            'bins': [{'lo': float(edges[i]), 'hi': float(edges[i + 1]), 'n': int(cnt[i]),
                      'mean_p': None if not nz[i] else float(conf[i]),
                      'frac_pos': None if not nz[i] else float(acc[i])} for i in range(bins)]}


def _metrics(y, p, t):
    tp, fp, fn, tn = _counts(y, p >= t)
    m = _rates(tp, fp, fn, tn)
    m.update({'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn, 'threshold': float(t)})
    if len(np.unique(y)) == 2:
        m['auc_roc'] = float(roc_auc_score(y, p))
        m['auc_pr'] = float(average_precision_score(y, p))
    m['brier'] = float(np.mean((p - y) ** 2))
    return m


# -------------------------------------------------------------- calibration
class Calibrator:
    def __init__(self, kind):
        self.kind = kind
        self.m = None

    @staticmethod
    def _logit(p):
        p = np.clip(p, 1e-7, 1 - 1e-7)
        return np.log(p / (1 - p)).reshape(-1, 1)

    def fit(self, p, y):
        if self.kind == 'platt':
            # pure sklearn -> unpickles in any layer without this class
            self.m = make_pipeline(
                FunctionTransformer(np.clip, kw_args={'a_min': 1e-7, 'a_max': 1 - 1e-7}),
                FunctionTransformer(_logit_fn),
                LogisticRegression(C=1e6, max_iter=1000)).fit(np.asarray(p).reshape(-1, 1), y)
        else:
            self.m = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0).fit(p, y)
        return self

    def __call__(self, p):
        p = np.asarray(p, dtype=np.float64)
        return (self.m.predict_proba(p.reshape(-1, 1))[:, 1] if self.kind == 'platt'
                else self.m.predict(p))

    def to_json(self):
        if self.kind == 'platt':
            lr = self.m[-1]
            return {'kind': 'platt', 'a': float(lr.coef_[0, 0]),
                    'b': float(lr.intercept_[0]), 'input': 'logit(clip(p_raw,1e-7,1-1e-7))'}
        return {'kind': 'isotonic', 'x': self.m.X_thresholds_.tolist(),
                'y': self.m.y_thresholds_.tolist()}


def _choose_calibrator(p, y, seed):
    """5-fold CV log-loss INSIDE the calibration half only."""
    skf = StratifiedKFold(5, shuffle=True, random_state=seed)
    scores = {}
    for kind in ('platt', 'isotonic'):
        ll = []
        for a, b in skf.split(p.reshape(-1, 1), y):
            c = Calibrator(kind).fit(p[a], y[a])
            ll.append(log_loss(y[b], np.clip(c(p[b]), 1e-7, 1 - 1e-7), labels=[0, 1]))
        scores[kind] = float(np.mean(ll))
    # Platt is pre-declared for deployment (downstream contract: predict_proba);
    # isotonic is fitted and reported for comparison only.
    return Calibrator('platt').fit(p, y), scores


# -------------------------------------------------------------- thresholds
def _thr_fpr_budget(p, y, a):
    neg = np.sort(p[y == 0])[::-1]
    k = int(np.floor(a * len(neg)))
    if k >= len(neg):
        return float(neg[-1])
    return float(np.nextafter(neg[k], np.inf))


def _thr_min_recall(p, y, r):
    pos = np.sort(p[y == 1])
    m = int(np.floor((1 - r) * len(pos)))
    return float(pos[min(m, len(pos) - 1)])


def _thr_youden(p, y):
    fpr, tpr, th = roc_curve(y, p)
    i = int(np.argmax(tpr - fpr))
    return float(min(th[i], 1.0))


def _thr_cost(p, y, ratio):
    fpr, tpr, th = roc_curve(y, p)
    npos, nneg = int(y.sum()), int((1 - y).sum())
    cost = ratio * (1 - tpr) * npos + fpr * nneg
    return float(min(th[int(np.argmin(cost))], 1.0))


def _rule_text(rule):
    return {'fpr_budget': f'max recall s.t. validation FPR <= {CFG["fpr_budget"]}',
            'youden': 'Youden J (TPR - FPR) maximised on the validation threshold half',
            'youden_clamped_legacy': f'Youden J clamped to {CFG["legacy_clamp"]} (legacy)',
            'min_recall': f'highest tau with validation recall >= {CFG["min_recall"]}',
            'cost_ratio': f'min expected cost, FN:FP = {CFG["cost_ratio_fn_fp"]}:1'}[rule]


def _primary_tau(p, y):
    return _all_rules(p, y)[CFG['primary_rule']]


def _all_rules(p, y):
    yd = _thr_youden(p, y)
    lo, hi = CFG['legacy_clamp']
    return {'fpr_budget': _thr_fpr_budget(p, y, CFG['fpr_budget']),
            'youden': yd,
            'youden_clamped_legacy': float(np.clip(yd, lo, hi)),
            'min_recall': _thr_min_recall(p, y, CFG['min_recall']),
            'cost_ratio': _thr_cost(p, y, CFG['cost_ratio_fn_fp'])}


# -------------------------------------------------------------- statistics
def _block_ids(tmin):
    _, inv = np.unique(tmin, return_inverse=True)
    return inv


def _block_counts(y, pred, bid):
    nb = bid.max() + 1
    y = y.astype(bool)
    pred = pred.astype(bool)
    return np.stack([np.bincount(bid, (y & pred).astype(float), nb),
                     np.bincount(bid, (~y & pred).astype(float), nb),
                     np.bincount(bid, (y & ~pred).astype(float), nb),
                     np.bincount(bid, (~y & ~pred).astype(float), nb)], 1)


def _ci(a):
    a = np.asarray(a)
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]


def _boot_counts(cnts, B, rng, keys=('f1', 'precision', 'recall', 'fpr')):
    nb = len(cnts)
    out = {k: [] for k in keys}
    for _ in range(B):
        s = cnts[rng.randint(0, nb, nb)].sum(0)
        r = _rates(*s)
        for k in keys:
            out[k].append(r[k])
    return {k: _ci(v) for k, v in out.items()}


def _weighted_auc_prep(p, y):
    o = np.argsort(p, kind='mergesort')
    ps, ys = p[o], y[o]
    starts = np.flatnonzero(np.r_[True, ps[1:] != ps[:-1]])
    return o, ys, starts


def _weighted_auc(prep, w):
    o, ys, starts = prep
    ws = w[o]
    pos = np.add.reduceat(ws * ys, starts)
    neg = np.add.reduceat(ws * (1 - ys), starts)
    Wp, Wn = pos.sum(), neg.sum()
    if Wp == 0 or Wn == 0:
        return np.nan
    below = np.cumsum(neg) - neg
    return float((pos * (below + 0.5 * neg)).sum() / (Wp * Wn))


def _boot_auc(ps, y, bid, B, rng):
    """Paired block bootstrap of AUC for several score vectors."""
    preps = [_weighted_auc_prep(p, y) for p in ps]
    nb = bid.max() + 1
    res = [[] for _ in ps]
    for _ in range(B):
        w = np.bincount(rng.randint(0, nb, nb), minlength=nb)[bid].astype(float)
        for i, pr in enumerate(preps):
            res[i].append(_weighted_auc(pr, w))
    return [np.array(r) for r in res]


def _delong(ps, y):
    """Fast DeLong (Sun & Xu 2014). Returns AUCs and covariance."""
    y = y.astype(bool)
    m, n = int(y.sum()), int((~y).sum())
    P = np.vstack(ps)
    X, Y = P[:, y], P[:, ~y]
    k = P.shape[0]
    tx = np.vstack([rankdata(X[i]) for i in range(k)])
    ty = np.vstack([rankdata(Y[i]) for i in range(k)])
    tz = np.vstack([rankdata(np.r_[X[i], Y[i]]) for i in range(k)])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, S


def _mcnemar(ca, cb):
    b = int((ca & ~cb).sum())
    c = int((~ca & cb).sum())
    if b + c == 0:
        p = 1.0
    elif b + c < 25:
        p = float(binomtest(b, b + c, 0.5).pvalue)
    else:
        p = float(chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1))
    orr = (b + 0.5) / (c + 0.5)
    se = np.sqrt(1 / (b + 0.5) + 1 / (c + 0.5))
    return {'b_A_right_B_wrong': b, 'c_A_wrong_B_right': c,
            'discordant_total': b + c, 'odds_ratio': float(orr),
            'odds_ratio_ci95': [float(np.exp(np.log(orr) - 1.96 * se)),
                                float(np.exp(np.log(orr) + 1.96 * se))],
            'p_raw': p}


def _holm(ps):
    order = np.argsort(ps)
    adj, run = np.empty(len(ps)), 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - r) * ps[i]))
        adj[i] = run
    return adj.tolist()


def _nogueira(sel):
    """Nogueira et al. (JMLR 2018) stability of a (seeds x features) 0/1 matrix."""
    M, d = sel.shape
    if M < 2:
        return None
    pf = sel.mean(0)
    kbar = sel.sum(1).mean()
    denom = (kbar / d) * (1 - kbar / d)
    if denom == 0:
        return 1.0
    s2 = M / (M - 1) * pf * (1 - pf)
    return float(1 - s2.mean() / denom)


# -------------------------------------------------------------- models
class Model:
    def __init__(self, name, obj, kind):
        self.name, self.obj, self.kind = name, obj, kind

    def predict(self, X, threads=None):
        if self.kind == 'lgb':
            return self.obj.predict(X, num_threads=threads or 0)
        if self.kind == 'xgb':
            return self.obj.predict(xgb.DMatrix(X, feature_names=self.obj.feature_names))
        return self.obj.predict_proba(X)[:, 1]


def _fit(kind, X, y, Xes, yes, names=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    t0 = time.time()
    if kind == 'LightGBM':
        p = dict(CFG['lgbm'], scale_pos_weight=spw)
        b = lgb.train(p, lgb.Dataset(X, y), CFG['rounds'],
                      valid_sets=[lgb.Dataset(Xes, yes)],
                      callbacks=[lgb.early_stopping(CFG['early_stop'], verbose=False)])
        m = Model(kind, b, 'lgb')
        info = {'best_iteration': int(b.best_iteration)}
    elif kind == 'XGBoost':
        p = dict(CFG['xgb'], scale_pos_weight=spw)
        b = xgb.train(p, xgb.DMatrix(X, y, feature_names=names), CFG['rounds'],
                      evals=[(xgb.DMatrix(Xes, yes, feature_names=names), 'es')],
                      early_stopping_rounds=CFG['early_stop'], verbose_eval=False)
        best_it = int(b.best_iteration)
        b = b[: best_it + 1]      # one model: plain .predict() anywhere == evaluated model
        m = Model(kind, b, 'xgb')
        info = {'best_iteration': best_it, 'trees_kept': best_it + 1}
    else:
        rf = RandomForestClassifier(**CFG['rf']).fit(X, y)
        m = Model(kind, rf, 'rf')
        info = {'n_estimators': CFG['rf']['n_estimators']}
    info.update({'train_rows': int(len(y)), 'attack_rate': float(y.mean()),
                 'class_weight': 'balanced (spw=%.3f)' % spw,
                 'fit_seconds': round(time.time() - t0, 1)})
    return m, info


def _online_cls(l1_code):
    import __main__
    c = globals().get('OnlineTransformer') or getattr(__main__, 'OnlineTransformer', None)
    if c is not None:
        return c
    if l1_code and os.path.exists(l1_code):
        spec = importlib.util.spec_from_file_location('titanium6_layer1', l1_code)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod.OnlineTransformer
    return None


# =============================================================================
#  PIPELINE
# =============================================================================
class Layer2:

    def __init__(self, l1_dir, out_dir, l1_code=None):
        self.l1_dir, self.out = Path(l1_dir).resolve(), _mkdir(out_dir).resolve()
        self.figs = _mkdir(self.out / 'figures')
        self.mdir = _mkdir(self.out / 'models')
        self.l1_code = l1_code
        self.rng = np.random.RandomState(SEED)
        self.R = {'version': CFG['version'], 'config': CFG,
                  'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                  'environment': _hardware()}
        plt.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42,       # IEEE: no Type 3
                             'font.serif': ['Times New Roman', 'Nimbus Roman', 'STIXGeneral',
                                            'DejaVu Serif'], 'mathtext.fontset': 'stix',
                             'font.family': 'serif', 'font.size': 8, 'axes.titlesize': 8.5,
                             'axes.labelsize': 8, 'legend.fontsize': 6.8,
                             'xtick.labelsize': 7, 'ytick.labelsize': 7,
                             'axes.linewidth': 0.7, 'savefig.dpi': 600})

    # ------------------------------------------------------------ 0 contract
    def p0_contract(self):
        print('[P0] Layer-1 contract')
        self.m1 = _read_json(self.l1_dir / 'manifest.json')
        c = self.m1['l2_contract']
        if c['schema_version'] < CFG['required_l1_schema']:
            raise RuntimeError(f'Layer 1 schema {c["schema_version"]} < '
                               f'{CFG["required_l1_schema"]}: run Layer 1 R8.0.')
        if '_ts' not in c['reserved_columns_not_features']:
            raise RuntimeError('Layer 1 must reserve _ts (R5.1).')
        self.ks_tv = self.m1['ks_diagnostics']['ks_train_val']['results']   # bracket
        self.all_feats = self.m1['features']['selected_features']
        excl = [s.strip() for s in os.environ.get('T6_EXCLUDE_FEATURES', '').split(',')
                if s.strip()]
        self.excluded = sorted(f for f in excl if f in self.all_feats)
        if excl:
            unknown = sorted(set(excl) - set(self.all_feats))
            if unknown:
                print(f'  [INFO] exclusion names not in L1 features (ignored): {unknown}')
            self.all_feats = [f for f in self.all_feats if f not in set(excl)]
            print(f'  [PROFILE] excluded {len(self.excluded)} feature(s): {self.excluded}')
        self.reserved = c['reserved_columns_not_features']
        self.paths = {}
        for k, v in self.m1['l2_input_paths'].items():
            cands = [Path(v), self.l1_dir / Path(v).name, self.l1_dir / 'l2_ready' / Path(v).name]
            hit = next((c for c in cands if c.exists()), None)
            if hit is None:
                raise FileNotFoundError(f'Layer-1 artifact {k!r} not found: tried {cands}')
            self.paths[k] = str(hit)
        self.split_mode = c['split_mode']
        self.R['layer1'] = {'version': self.m1['version'], 'split_mode': self.split_mode,
                            'artifact_hashes': self.m1['artifact_hashes']}
        self.R['feature_exclusion_profile'] = {
            'excluded': self.excluded,
            'purpose': ('shortcut ablation: environment/TCP-stack features removed '
                        'before selection') if self.excluded else 'none'}
        print(f'  L1 {self.m1["version"]} | mode={self.split_mode} | '
              f'{len(self.all_feats)} candidate features')

    # ------------------------------------------------------------ 1 load
    def p1_load(self):
        print('[P1] Load train (all candidate features)')
        X, meta = _load(self.paths['train'], self.all_feats, ('Target',))
        self.y_tr = (meta['Target'].values != 'Normal').astype(np.int8)
        self.X_tr_all = X
        n = len(self.y_tr)
        idx = self.rng.permutation(n)
        k = int(CFG['es_frac'] * n)
        self.es_idx, self.fit_idx = np.sort(idx[:k]), np.sort(idx[k:])
        print(f'  train {n:,} rows | attack {self.y_tr.mean():.4f} | '
              f'normal:attack {(1 - self.y_tr.mean()) / max(self.y_tr.mean(), 1e-9):.2f}:1 | '
              f'ES hold-out {k:,}')
        classes = sorted(set(meta['Target'].values))
        self.class_code = {c: i for i, c in enumerate(classes)}
        codes = pd.Categorical(meta['Target'].values, categories=classes).codes.astype(np.int16)
        H = np.concatenate([_row_hash(pd.DataFrame(X[i:i + 1_000_000], columns=self.all_feats))
                            for i in range(0, n, 1_000_000)])
        o = np.argsort(H, kind='stable')
        H, codes = H[o], codes[o]
        self.tr_hash, st = np.unique(H, return_index=True)
        lo, hi = np.minimum.reduceat(codes, st), np.maximum.reduceat(codes, st)
        self.tr_lab = np.where(lo == hi, lo, -1).astype(np.int16)
        del H, codes, o, st, lo, hi
        self.R['data'] = {'train_rows': n, 'train_attack_rate': float(self.y_tr.mean()),
                          'es_rows': k, 'fit_rows': int(len(self.fit_idx))}

    # ------------------------------------------------------------ 2 features
    def _mi_select(self, cand):
        ci = [self.all_feats.index(f) for f in cand]
        fit_y = self.y_tr[self.fit_idx]
        pos, neg = self.fit_idx[fit_y == 1], self.fit_idx[fit_y == 0]
        k = CFG['mi_sample_per_class']
        sel, mis = [], []
        for s in range(CFG['mi_seeds']):
            r = np.random.RandomState(SEED + s)
            ii = np.r_[r.choice(pos, min(k, len(pos)), replace=False),
                       r.choice(neg, min(k, len(neg)), replace=False)]
            mi = mutual_info_classif(self.X_tr_all[np.ix_(ii, ci)], self.y_tr[ii],
                                     random_state=SEED + s, n_neighbors=3)
            mis.append(mi)
            sel.append(mi >= CFG['mi_threshold'])
        return np.array(sel), np.array(mis)

    def p2_features(self):
        print('[P2] Feature selection: KS(train-val) + MI stability selection')
        ks = {r['feature']: r['ks_stat'] for r in self.ks_tv if r['feature'] in self.all_feats}
        med = float(np.median(list(ks.values())))
        stage1 = [f for f in self.all_feats if ks.get(f, 1.0) <= CFG['stage1_ks_max']]
        drifted = sorted((f for f in self.all_feats if ks.get(f, 1.0) > CFG['stage1_ks_max']),
                         key=lambda f: -ks.get(f, 1.0))
        n_drop1 = len(self.all_feats) - len(stage1)
        max_ks = max((ks.get(f, 0.0) for f in self.all_feats), default=0.0)
        sel, mis = self._mi_select(stage1)
        freq = sel.mean(0)
        band = {q: {f for f, v in zip(stage1, freq) if v >= q} for q in (0.70, 0.90)}
        keep = CFG['stability_keep_frac']
        stage2 = [f for f, q in zip(stage1, freq) if q >= keep]
        ak = [f for f in ALWAYS_KEEP if f in self.all_feats]
        final = list(dict.fromkeys(stage2 + ak))
        final = [f for f in self.all_feats if f in set(final)]
        no_ak = [f for f in self.all_feats if f in set(stage2)]
        sel_all, _ = self._mi_select(self.all_feats)
        no_s1 = [f for f, q in zip(self.all_feats, sel_all.mean(0)) if q >= keep]
        no_s1 = [f for f in self.all_feats if f in set(no_s1) | set(ak)]
        self.variants = {'selected': final, 'no_always_keep': no_ak,
                         'no_stage1': no_s1, 'all_features': list(self.all_feats)}
        self.feats = final
        self.fidx = [self.all_feats.index(f) for f in final]
        stab = _nogueira(sel.astype(float))
        self.R['feature_selection'] = {
            'stage1_rule': f'drop if mean KS(train vs VALIDATION) > {CFG["stage1_ks_max"]} '
                           '(pre-declared effect size); test never used',
            'stage1_dropped': {f: ks.get(f) for f in drifted},
            'stage1_median_ks': med, 'n_candidates': len(self.all_feats),
            'n_stage1': len(stage1),
            'stage2_rule': f'MI >= {CFG["mi_threshold"]} in >= {keep:.0%} of '
                           f'{CFG["mi_seeds"]} seeds (balanced {2 * CFG["mi_sample_per_class"]:,}'
                           f'-flow samples from the fit set)',
            'n_stage2': len(stage2), 'always_keep_added': [f for f in ak if f not in stage2],
            'n_final': len(final), 'final_features': final,
            'nogueira_stability': stab,
            'stage1_effectiveness': (
                f'MEASURED INERT: 0/{len(self.all_feats)} dropped '
                f'(max KS {max_ks:.4f} < threshold {CFG["stage1_ks_max"]}); all selection '
                'was performed by the MI stability stage. Do not claim two-stage '
                'selection in the paper.' if n_drop1 == 0 else
                f'ACTIVE: {n_drop1}/{len(self.all_feats)} dropped (max KS {max_ks:.4f})'),
            'stability_bands': {
                'n_at_0.70': len(band[0.70]), 'n_at_0.90': len(band[0.90]),
                'overlap': len(band[0.70] & band[0.90]),
                'only_at_0.70': sorted(band[0.70] - band[0.90]),
                'reading': ('the deployed set is the 0.70 band plus ALWAYS_KEEP; the 0.90 '
                            'band is the subset a stricter reviewer would accept. Report '
                            'both with the Nogueira index.')},
            'selection_frequency': {f: float(q) for f, q in zip(stage1, freq)},
            'mi_mean': {f: float(v) for f, v in zip(stage1, mis.mean(0))},
            'mi_std': {f: float(v) for f, v in zip(stage1, mis.std(0))},
            'variants_size': {k: len(v) for k, v in self.variants.items()}}
        if n_drop1 == 0:
            print(f'  [WARN] stage-1 KS filter MEASURED INERT: all {len(self.all_feats)} '
                  f'features pass (max KS {max_ks:.4f} < {CFG["stage1_ks_max"]}); '
                  'selection is MI-only')
        print(f'  stability bands: {len(band[0.70])} at 0.70 | {len(band[0.90])} at 0.90 '
              f'| overlap {len(band[0.70] & band[0.90])}')
        print(f'  stage1 {len(stage1)}/{len(self.all_feats)} | stage2 {len(stage2)} | '
              f'final {len(final)} | Nogueira stability {stab:.3f}'
              if stab is not None else f'  final {len(final)}')

    # ------------------------------------------------------------ 3 validation
    def p3_validation(self):
        print('[P3] Validation: disjoint calibration / threshold halves')
        X, meta = _load(self.paths['val'], self.feats, ('Target',))
        y = (meta['Target'].values != 'Normal').astype(np.int8)
        r = np.random.RandomState(SEED + 1)
        cal, thr, fam = [], [], []
        for cls in (0, 1):
            ii = r.permutation(np.flatnonzero(y == cls))
            a, b = len(ii) // 3, 2 * (len(ii) // 3)
            fam.append(ii[:a]); cal.append(ii[a:b]); thr.append(ii[b:])
        self.cal_i = np.sort(np.concatenate(cal))
        self.thr_i = np.sort(np.concatenate(thr))
        self.fam_i = np.sort(np.concatenate(fam))       # S1: family selection only
        self.X_val, self.y_val = X, y
        self.R['data'].update({'val_rows': int(len(y)), 'val_attack_rate': float(y.mean()),
                               'fam_slice': int(len(self.fam_i)),
                               'cal_half': int(len(self.cal_i)), 'thr_half': int(len(self.thr_i)),
                               'slice_rule': ('three disjoint stratified slices: family '
                                              'selection / calibration / threshold. No '
                                              'validation row informs two decisions.')})
        print(f'  val {len(y):,} | fam {len(self.fam_i):,} | cal {len(self.cal_i):,} | '
              f'thr {len(self.thr_i):,}')

    # ------------------------------------------------------------ 4 train
    def p4_train(self):
        print('[P4] Matched comparison + deployed model')
        Xf = self.X_tr_all[:, self.fidx]
        n = min(CFG['matched_n'], len(self.fit_idx))
        self.match_idx = np.sort(np.random.RandomState(SEED + 2).choice(
            self.fit_idx, n, replace=False))
        Xm, ym = Xf[self.match_idx], self.y_tr[self.match_idx]
        Xes, yes = Xf[self.es_idx], self.y_tr[self.es_idx]
        self.models, info = {}, {}
        for kind in ('LightGBM', 'XGBoost', 'RandomForest'):
            m, inf = _fit(kind, Xm, ym, Xes, yes, self.feats)
            self.models[kind] = m
            pv = m.predict(self.X_val[self.fam_i])          # S1: family slice only
            yv_f = self.y_val[self.fam_i]
            inf['val_auc_pr'] = float(average_precision_score(yv_f, pv))
            inf['val_auc_roc'] = float(roc_auc_score(yv_f, pv))
            rb = np.random.RandomState(SEED + 21)
            bs = [average_precision_score(yv_f[i], pv[i]) for i in
                  (rb.randint(0, len(yv_f), len(yv_f)) for _ in range(200))
                  if len(np.unique(yv_f[i])) == 2]
            inf['val_auc_pr_ci95'] = _ci(bs) if bs else [None, None]
            info[kind] = inf
            print(f'  {kind:<13} matched n={n:,} | val AUC-PR {inf["val_auc_pr"]:.4f} | '
                  f'{inf["fit_seconds"]}s')
        order = sorted(info, key=lambda k: -info[k]['val_auc_pr'])
        top, second = order[0], order[1]
        ci_a, ci_b = info[top]['val_auc_pr_ci95'], info[second]['val_auc_pr_ci95']
        tie = (ci_a[0] is not None and ci_b[0] is not None
               and ci_a[0] <= ci_b[1] and ci_b[0] <= ci_a[1])
        if tie:
            lat = {}
            for k, m in self.models.items():
                x = self.X_val[self.fam_i[:200]]
                for i in range(50):
                    m.predict(x[i % len(x)][None, :])
                t0 = time.perf_counter_ns()
                for i in range(500):
                    m.predict(x[i % len(x)][None, :])
                lat[k] = (time.perf_counter_ns() - t0) / 500 / 1e3
            self.family = min((top, second), key=lambda k: lat[k])
            self._selection_note = {
                'rule': 'val AUC-PR with 95% block-free bootstrap CI on the family slice',
                'outcome': 'STATISTICAL TIE: the top two CIs overlap',
                'top_two': {top: info[top]['val_auc_pr_ci95'],
                            second: info[second]['val_auc_pr_ci95']},
                'tiebreak': 'measured batch-1 latency (microseconds per flow)',
                'latency_us': {k: round(v, 1) for k, v in lat.items()},
                'selected': self.family,
                'paper_wording': ('the deployed family was chosen by a pre-declared '
                                  'latency tiebreak because the AUC-PR difference was '
                                  'not statistically resolvable')}
            print(f'  [FINDING] family selection is a TIE ({top} vs {second}, CIs overlap); '
                  f'tiebreak = latency -> {self.family} '
                  f'({lat[self.family]:.0f} us vs {lat[second if self.family == top else top]:.0f} us)')
        else:
            self.family = top
            self._selection_note = {
                'rule': 'val AUC-PR with 95% bootstrap CI on the family slice',
                'outcome': 'resolved: the top CI does not overlap the second',
                'top_two': {top: ci_a, second: ci_b}, 'selected': top}
        cap = CFG['rf_full_cap'] if self.family == 'RandomForest' else len(self.fit_idx)
        full = (self.fit_idx if cap >= len(self.fit_idx) else
                np.sort(np.random.RandomState(SEED + 3).choice(self.fit_idx, cap, replace=False)))
        dm, dinf = _fit(self.family, Xf[full], self.y_tr[full], Xes, yes, self.feats)
        self.dep = dm
        dinf['val_auc_pr'] = float(average_precision_score(
            self.y_val[self.fam_i], dm.predict(self.X_val[self.fam_i])))
        self.R['training'] = {
            'matched_comparison': {'n': n, 'same_rows_for_all': True,
                                   'weighting': 'balanced class weights for all families',
                                   'early_stopping_set': 'ES hold-out from train', 'models': info},
            'selection': {'metric': CFG['selection_metric'], 'selected_family': self.family,
                          'note': 'selected on the validation FAMILY slice; test never used',
                          'decision': self._selection_note},
            'deployed': dict(dinf, family=self.family,
                             label='computational-budget model (selected family, full train)')}
        print(f'  selected family: {self.family} (val) | deployed on {len(full):,} rows')
        del Xm
        _gc()

    # ------------------------------------------------------------ 5 calibrate
    def p5_calibrate_threshold(self):
        print('[P5] Calibration (cal half) + threshold rules (thr half, calibrated)')
        self.cal, self.rules, cinfo = {}, {}, {}
        yc, yt = self.y_val[self.cal_i], self.y_val[self.thr_i]
        for name, m in list(self.models.items()) + [('DEPLOYED', self.dep)]:
            pc = m.predict(self.X_val[self.cal_i])
            c, cv = _choose_calibrator(pc, yc, SEED)
            self.cal[name] = c
            pt = c(m.predict(self.X_val[self.thr_i]))
            self.rules[name] = _all_rules(pt, yt)
            cinfo[name] = {'chosen': c.kind, 'cv_logloss': cv}
        d = self.rules['DEPLOYED']
        self.tau = d[CFG['primary_rule']]
        self._thr_curve = (self.cal['DEPLOYED'](self.dep.predict(self.X_val[self.thr_i])), yt)
        self.R['calibration'] = cinfo
        self.R['threshold'] = {
            'score_space': 'calibrated probability',
            'primary_rule': _rule_text(CFG['primary_rule']),
            'tau': self.tau, 'all_rules_deployed': d,
            'selected_on': 'validation threshold half (disjoint from calibration half)',
            'legacy_clamp_note': 'v15.0 clamp [0.30,0.70] was applied to RAW scores; '
                                 'reported here on the calibrated scale for comparison only'}
        _atomic_json({'model_family': self.family, 'features': self.feats,
                      'calibrator': self.cal['DEPLOYED'].to_json(),
                      'tau': self.tau, 'score_space': 'calibrated',
                      'rule': CFG['primary_rule'], 'fpr_budget': CFG['fpr_budget']},
                     self.mdir / 'l2_decision.json')
        print(f'  deployed calibrator {self.cal["DEPLOYED"].kind} | tau={self.tau:.4f} '
              f'({CFG["primary_rule"]}) | fpr_budget={d["fpr_budget"]:.4f}')

    # ------------------------------------------------------------ 6 evaluate
    def p6_evaluate(self):
        print('[P6] Test evaluation (full + de-duplicated) and zero-day')
        X, meta = _load(self.paths['test_known'], self.feats,
                        ('Target', '_dup_in_train', '_ts'))
        y = (meta['Target'].values != 'Normal').astype(np.int8)
        self.Xte, self.yte, self.meta_te = X, y, meta
        nodup = meta['_dup_in_train'].values == 0
        self.P, ev = {}, {}
        for name, m in list(self.models.items()) + [('DEPLOYED', self.dep)]:
            praw = m.predict(X)
            p = self.cal[name](praw)
            self.P[name] = p
            t = self.rules[name][CFG['primary_rule']]
            ev[name] = {'full': _metrics(y, p, t), 'dedup': _metrics(y[nodup], p[nodup], t),
                        'brier_raw': float(np.mean((praw - y) ** 2)),
                        'calibration_raw': {k: v for k, v in
                                            _ece(y, np.clip(praw, 0, 1), CFG['ece_bins']).items()
                                            if k != 'bins'},
                        'calibration_cal': _ece(y, p, CFG['ece_bins'])}
            if name == 'DEPLOYED':
                self._praw_dep = praw          # explicit: used by the reliability figure
        # all threshold rules on test for the deployed model (with and without clamp)
        pd_ = self.P['DEPLOYED']
        ev['DEPLOYED']['by_rule'] = {r: {'full': _metrics(y, pd_, t),
                                         'dedup': _metrics(y[nodup], pd_[nodup], t)}
                                     for r, t in self.rules['DEPLOYED'].items()}
        ev['DEPLOYED']['fpr_operating_points'] = {
            str(a): _metrics(y, pd_, _thr_fpr_budget(self._thr_curve[0], self._thr_curve[1], a))
            for a in CFG['fpr_op_points']}
        maj = _rates(0, 0, int(y.sum()), int((1 - y).sum()))
        ev['majority_baseline'] = dict(maj, note='always predicts Normal; split: N/A')
        tg = self.meta_te['Target'].values
        fl = pd_ >= self.tau
        self.R['test_per_class_flag_rate'] = {
            str(c): {'n': int((tg == c).sum()),
                     'flag_rate': float(fl[tg == c].mean()),
                     'twin_free_flag_rate': (float(fl[(tg == c) & nodup].mean())
                                             if ((tg == c) & nodup).any() else None)}
            for c in np.unique(tg)}
        worst = sorted(((v['flag_rate'], k) for k, v in
                        self.R['test_per_class_flag_rate'].items() if k != 'Normal'))[:3]
        print('  per-class flag rate, lowest: '
              + ', '.join(f'{k} {100 * v:.1f}%' for v, k in worst))
        self.R['test'] = ev
        self.R['test_meta'] = {'rows': int(len(y)), 'attack_rate': float(y.mean()),
                               'dedup_rows': int(nodup.sum()),
                               'dup_in_train_pct': float(100 * (~nodup).mean())}
        d = ev['DEPLOYED']
        print(f'  DEPLOYED  F1 {d["full"]["f1"]:.4f} | recall {d["full"]["recall"]:.4f} | '
              f'FPR {100 * d["full"]["fpr"]:.3f}% | AUC {d["full"]["auc_roc"]:.4f} | '
              f'dedup F1 {d["dedup"]["f1"]:.4f}')
        self._threshold_transfer()
        self._p6_zero_day()
        np.save(self.out / '_ptest_tmp.npy', np.vstack([self.P[k] for k in self.P]))

    def _threshold_transfer(self):
        """D1. What the pre-declared FPR budget actually does across the
        chronological boundary, and what a deployment would need to do about it."""
        p, y = self.P['DEPLOYED'], self.yte
        tau_b = float(self.rules['DEPLOYED']['fpr_budget'])
        neg = np.sort(p[y == 0])[::-1]
        k = int(np.floor(CFG['fpr_budget'] * len(neg)))
        tau_test = float(np.nextafter(neg[min(k, len(neg) - 1)], np.inf))
        pv, yv = self._thr_curve
        val_fpr = float(((pv >= tau_b) & (yv == 0)).sum() / max((yv == 0).sum(), 1))
        test_fpr = float(((p >= tau_b) & (y == 0)).sum() / max((y == 0).sum(), 1))
        bid = _block_ids(self.meta_te['_ts'].values // (CFG['block_minutes'] * 60 * 10**9))
        blk = []
        for b in np.unique(bid):
            m = bid == b
            ng = (y[m] == 0)
            if ng.sum() > 100:
                blk.append(float(((p[m] >= tau_b) & ng).sum() / ng.sum()))
        blk = np.sort(np.array(blk)) if blk else np.array([])
        over = int((blk > CFG['fpr_budget']).sum())
        self.R['limitations'] = {'threshold_transfer': {
            'declared_budget': CFG['fpr_budget'],
            'val_fpr_at_budget_tau': val_fpr,
            'test_fpr_at_budget_tau': test_fpr,
            'transfer_factor': float(test_fpr / max(CFG['fpr_budget'], 1e-12)),
            'tau_budget_rule': tau_b,
            'tau_deployed_primary_rule': self.tau,
            'primary_rule': CFG['primary_rule'],
            'tau_that_would_hold_budget_on_test': tau_test,
            'tau_movement_required': float(tau_test - tau_b),
            'per_block_fpr_at_budget_tau': {
                'block_minutes': CFG['block_minutes'],
                'n_blocks_scored': int(len(blk)),
                'p50': float(np.percentile(blk, 50)) if len(blk) else None,
                'p95': float(np.percentile(blk, 95)) if len(blk) else None,
                'max': float(blk.max()) if len(blk) else None,
                'n_blocks_above_budget': over,
                'pct_blocks_above_budget': float(100 * over / max(len(blk), 1))},
            'recalibration_trigger': (
                f'{100 * over / max(len(blk), 1):.1f}% of {CFG["block_minutes"]}-minute '
                'blocks breach the budget at the budget-rule threshold; a rolling-window '
                'monitor firing on a sustained breach would detect this without labels'),
            'finding': ('a threshold selected on validation under a false-alert budget '
                        'does not carry across a chronological boundary; the budget is a '
                        'validation-time guarantee only')}}
        print(f'  [FINDING] tau transfer: val FPR {100 * val_fpr:.3f}% -> test '
              f'{100 * test_fpr:.3f}% ({test_fpr / max(CFG["fpr_budget"], 1e-12):.1f}x budget); '
              f'tau would need {tau_test - tau_b:+.4f}; '
              f'{over}/{len(blk)} blocks over budget')

    def _p6_zero_day(self):
        Xall, meta = _load(self.paths['test_zeroday'], self.all_feats,
                           ('Target', '_ts', '_dup_in_train'))
        X = Xall[:, self.fidx]
        h = _row_hash(pd.DataFrame(Xall, columns=self.all_feats))
        pos = np.minimum(np.searchsorted(self.tr_hash, h), len(self.tr_hash) - 1)
        match = self.tr_hash[pos] == h
        agree = float((match == (meta['_dup_in_train'].values == 1)).mean())
        if agree < 0.9999:
            raise RuntimeError(f'twin flag disagrees with Layer 1 ({agree:.5f})')
        lab = np.where(match, self.tr_lab[pos], -9)
        del Xall
        p = self.cal['DEPLOYED'](self.dep.predict(X))
        flag = p >= self.tau
        tg = meta['Target'].values
        full = set(self.m1['config']['full_zero_day'])
        per = {c: {'n': int((tg == c).sum()), 'flag_rate': float(flag[tg == c].mean()),
                   'type': 'full' if c in full else 'partial'} for c in np.unique(tg)}
        fu = np.isin(tg, list(full))
        nrm = self.class_code.get('Normal', -99)
        cat = np.select([lab == -9, lab == -1, lab == nrm,
                         np.array([self.class_code.get(t, -50) for t in tg]) == lab],
                        ['no_twin', 'twin_mixed_labels', 'twin_of_Normal', 'twin_same_class'],
                        'twin_other_attack')
        by_twin = {}
        for c in np.unique(tg):
            m = tg == c
            by_twin[c] = {k: {'n': int((m & (cat == k)).sum()),
                              'flag_rate': float(flag[m & (cat == k)].mean())
                              if (m & (cat == k)).any() else None}
                          for k in np.unique(cat[m])}
        nt = cat == 'no_twin'
        self.R['zero_day_twin_stratified'] = {
            'twin_check_agreement_with_layer1': agree,
            'flag_rate_twin_free': float(flag[nt].mean()) if nt.any() else None,
            'flag_rate_twin_free_macro': float(np.mean([flag[(tg == c) & nt].mean() for c in
                                                        np.unique(tg) if ((tg == c) & nt).any()]))
            if nt.any() else None,
            'by_class_and_twin_label': by_twin,
            'reading': 'a twin_of_Normal flow is identical to a Normal training flow: no '
                       'flow-level model can separate it; report, do not count as a miss of skill'}
        self.R['zero_day'] = {
            'l2_flag_rate_support_weighted': float(flag.mean()),
            'l2_flag_rate_macro': float(np.mean([v['flag_rate'] for v in per.values()])),
            'l2_flag_rate_fully_unseen_aggregate': float(flag[fu].mean()) if fu.any() else None,
            'per_class': per,
            'note': 'Binary flag rate only; novelty recognition and family attribution are '
                    'Layer 3/4 metrics.'}
        pd.DataFrame({'p_cal': p.astype(np.float32), 'flag': flag, 'Target': tg,
                      'twin_label': cat, '_ts': meta['_ts'].values}).to_parquet(
            self.out / 'l2_scores_zeroday.parquet')
        print(f'  zero-day flag rate: weighted {flag.mean():.4f} | macro '
              f'{self.R["zero_day"]["l2_flag_rate_macro"]:.4f}')

    # ------------------------------------------------------------ 7 statistics
    def p7_statistics(self):
        print('[P7] Block bootstrap, McNemar (b, c, OR, Holm), DeLong')
        y = self.yte
        bid = _block_ids(self.meta_te['_ts'].values // (CFG['block_minutes'] * 60 * 10**9))
        rng = np.random.RandomState(SEED + 7)
        st = {'block_definition': f'{CFG["block_minutes"]}-minute time windows from Layer-1 _ts',
              'n_blocks': int(bid.max() + 1)}
        pdep = self.P['DEPLOYED']
        tdep = self.tau
        cnt = _block_counts(y, pdep >= tdep, bid)
        st['deployed_block_ci'] = _boot_counts(cnt, CFG['boot_counts'], rng)
        alt = 15                                   # S3: does CI width depend on block length?
        bid15 = _block_ids(self.meta_te['_ts'].values // (alt * 60 * 10**9))
        st['deployed_block_ci_15min'] = _boot_counts(
            _block_counts(y, pdep >= tdep, bid15), CFG['boot_counts'], rng)
        w60 = st['deployed_block_ci']['f1'][1] - st['deployed_block_ci']['f1'][0]
        w15 = st['deployed_block_ci_15min']['f1'][1] - st['deployed_block_ci_15min']['f1'][0]
        st['block_length_sensitivity'] = {
            'width_60min': round(w60, 5), 'width_15min': round(w15, 5),
            'n_blocks_15min': int(bid15.max() + 1),
            'ratio_60_over_15': round(w60 / max(w15, 1e-12), 2),
            'reading': ('if the interval widens with block length, temporal correlation '
                        'extends beyond the shorter block; report the longer block as '
                        'the conservative interval')}
        print(f'  [FINDING] block-length sensitivity: 60-min width {w60:.4f} vs 15-min '
              f'{w15:.4f} (ratio {w60 / max(w15, 1e-12):.2f})')
        iid = _block_counts(y, pdep >= tdep, np.arange(len(y)) % max(1, len(y)))
        st['deployed_iid_ci'] = _boot_counts(iid, min(200, CFG['boot_counts']), rng)
        a_dep = _boot_auc([pdep], y, bid, CFG['boot_auc'], rng)[0]
        st['deployed_auc_block_ci'] = _ci(a_dep[~np.isnan(a_dep)])

        names = list(self.models)
        preds = {n: self.P[n] >= self.rules[n][CFG['primary_rule']] for n in names}
        corr = {n: preds[n] == y.astype(bool) for n in names}
        aucs, S = _delong([self.P[n] for n in names], y)
        boot_auc = _boot_auc([self.P[n] for n in names], y, bid, CFG['boot_auc'], rng)
        pairs, mc_p, dl_p = [], [], []
        cnts = {n: _block_counts(y, preds[n], bid) for n in names}
        nb = bid.max() + 1
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                a, b = names[i], names[j]
                mc = _mcnemar(corr[a], corr[b])
                var = S[i, i] + S[j, j] - 2 * S[i, j]
                z = (aucs[i] - aucs[j]) / np.sqrt(var) if var > 0 else 0.0
                dlp = float(2 * norm.sf(abs(z)))
                d_f1 = []
                for _ in range(CFG['boot_counts']):
                    k = rng.randint(0, nb, nb)
                    d_f1.append(_rates(*cnts[a][k].sum(0))['f1'] - _rates(*cnts[b][k].sum(0))['f1'])
                dauc = boot_auc[i] - boot_auc[j]
                pairs.append({'pair': f'{a} vs {b}', 'mcnemar': mc,
                              'delong': {'auc_a': float(aucs[i]), 'auc_b': float(aucs[j]),
                                         'z': float(z), 'p_raw': dlp},
                              'delta_f1_block_ci': _ci(d_f1),
                              'delta_auc_block_ci': _ci(dauc[~np.isnan(dauc)])})
                mc_p.append(mc['p_raw'])
                dl_p.append(dlp)
        for pr, pm, pdl in zip(pairs, _holm(mc_p), _holm(dl_p)):
            pr['mcnemar']['p_holm'] = pm
            pr['delong']['p_holm'] = pdl
            pr['wording'] = ('similar AUC (delta-AUC CI includes 0)'
                             if pr['delta_auc_block_ci'][0] <= 0 <= pr['delta_auc_block_ci'][1]
                             else 'AUC differs (delta-AUC CI excludes 0)')
        st['pairwise_matched'] = pairs
        st['caveat'] = ('McNemar and DeLong assume independent flows; the block-bootstrap '
                        'delta CIs are the primary evidence.')
        self.R['statistics'] = st
        c = st['deployed_block_ci']['f1']
        ci_iid = st['deployed_iid_ci']['f1']
        bw, iw = c[1] - c[0], ci_iid[1] - ci_iid[0]
        st['ci_width_ratio'] = {'block_width': round(bw, 5), 'iid_width': round(iw, 5),
            'ratio': round(bw / max(iw, 1e-12), 1),
            'reading': ('the i.i.d. interval understates temporal uncertainty by this '
                        'factor; the block interval is the primary evidence')}
        print(f'  [FINDING] i.i.d. CI understates the block CI width by '
              f'{bw / max(iw, 1e-12):.0f}x')
        print(f'  deployed F1 block CI [{c[0]:.4f}, {c[1]:.4f}] vs iid '
              f'[{ci_iid[0]:.4f}, {ci_iid[1]:.4f}] | blocks={st["n_blocks"]:,}')
        for pr in pairs:
            print(f'  {pr["pair"]:<26} b={pr["mcnemar"]["b_A_right_B_wrong"]:,} '
                  f'c={pr["mcnemar"]["c_A_wrong_B_right"]:,} | dAUC CI '
                  f'[{pr["delta_auc_block_ci"][0]:+.4f},{pr["delta_auc_block_ci"][1]:+.4f}] '
                  f'-> {pr["wording"]}')

    # ------------------------------------------------------------ 8 latency
    def p8_latency(self):
        print('[P8] Latency (measured)')
        m, cal, tau = self.dep, self.cal['DEPLOYED'], self.tau
        X = self.Xte[:2000]
        lat = {'hardware': _hardware(), 'unit': 'microseconds per flow'}

        def _bench(fn, n):
            for i in range(CFG['lat_warmup']):
                fn(i)
            t = np.empty(n)
            for i in range(n):
                t0 = time.perf_counter_ns()
                fn(i)
                t[i] = (time.perf_counter_ns() - t0) / 1e3
            return {'p50': float(np.percentile(t, 50)), 'p95': float(np.percentile(t, 95)),
                    'p99': float(np.percentile(t, 99)), 'mean': float(t.mean()), 'calls': n}, t

        lat['model_only_batch1'], t_m = _bench(
            lambda i: cal(m.predict(X[i % len(X)][None, :], threads=1)) >= tau, CFG['lat_calls'])
        B = min(CFG['lat_batch'], len(self.Xte))
        tb = []
        for _ in range(20):
            t0 = time.perf_counter()
            cal(m.predict(self.Xte[:B])) >= tau
            tb.append((time.perf_counter() - t0) / B * 1e6)
        lat['model_only_batch'] = {'batch_size': B, 'per_flow_mean': float(np.mean(tb)),
                                   'per_flow_p95': float(np.percentile(tb, 95))}
        OT = _online_cls(self.l1_code)
        raw = self.l1_dir / 'raw_sample.csv'
        t_e2e = None
        if OT is not None and raw.exists():
            ot = OT.from_json(self.paths['scaler'])
            recs = pd.read_csv(raw, low_memory=False).to_dict('records')
            idx = np.array([ot.pos[f] for f in self.feats])
            lat['online_path_batch1'], t_e2e = _bench(
                lambda i: cal(m.predict(ot(recs[i % len(recs)])[idx][None, :], threads=1))
                >= tau, CFG['lat_calls'])
            lat['online_path_batch1']['includes'] = ('raw CICFlowMeter record -> Layer-1 '
                                                     'OnlineTransformer -> feature selection '
                                                     '-> model -> calibrator -> threshold')
        else:
            lat['online_path_batch1'] = 'not measured: OnlineTransformer or raw_sample.csv missing'
        lat['excludes'] = 'CICFlowMeter flow extraction, network I/O, HTTP serialisation'
        lat['protocol'] = (f'warm-up {CFG["lat_warmup"]} calls, {CFG["lat_calls"]} timed calls, '
                           'single thread for batch-1, perf_counter_ns per call')
        self.R['latency'] = lat
        self._lat_series = (t_m, t_e2e)
        e = lat.get('online_path_batch1')
        print(f'  model-only b1 P95 {lat["model_only_batch1"]["p95"]:.1f} us | '
              + (f'online path b1 P95 {e["p95"]:.1f} us' if isinstance(e, dict) else str(e)))

    # ------------------------------------------------------------ 9 adversarial
    def _raw_io(self):
        sp = _read_json(self.paths['scaler'])
        mu = np.array([sp['mean'][f] for f in self.all_feats])
        sd = np.array([sp['std'][f] for f in self.all_feats])
        return mu, sd

    def _evade(self, R, R0, fi, sb, st):
        """Forward padding (sb) + delay (st). Every dependent feature is updated
        as a DELTA against its unperturbed value, so the result is exact at
        sb=st=1 regardless of CICFlowMeter's internal formula conventions
        (e.g. whether Packet Length Mean includes headers)."""
        def g(M, f):
            return M[:, fi[f]] if f in fi else None
        F0, B = g(R0, 'Total Length of Fwd Packets'), g(R0, 'Total Length of Bwd Packets')
        Nf, Nb = g(R0, 'Total Fwd Packets'), g(R0, 'Total Backward Packets')
        for f in FWD_BYTES:
            if f in fi:
                R[:, fi[f]] = R0[:, fi[f]] * sb
        for f in TIME_FEATS:
            if f in fi:
                R[:, fi[f]] = R0[:, fi[f]] * st
        for f in PKT_RATES:
            if f in fi:
                R[:, fi[f]] = R0[:, fi[f]] / st
        if 'Flow Bytes/s' in fi and F0 is not None and B is not None:
            tot0 = F0 + B
            fac = np.where(tot0 > 0, (F0 * sb + B) / np.where(tot0 > 0, tot0, 1.0), 1.0)
            R[:, fi['Flow Bytes/s']] = R0[:, fi['Flow Bytes/s']] * fac / st
        if all(v is not None for v in (F0, B, Nf, Nb)):
            N = np.maximum(Nf + Nb, 1.0)
            mean_new, mean_old = (F0 * sb + B) / N, (F0 + B) / N
            d_mean = mean_new - mean_old
            for f in ('Packet Length Mean', 'Packet Size Average'):
                if f in fi:
                    R[:, fi[f]] = np.maximum(R0[:, fi[f]] + d_mean, 0.0)
            mx = ('Packet Length Max', 'Fwd Packet Length Max', 'Bwd Packet Length Max')
            if all(f in fi for f in mx):
                fmax, bmax = R0[:, fi[mx[1]]], R0[:, fi[mx[2]]]
                R[:, fi[mx[0]]] = np.maximum(
                    R0[:, fi[mx[0]]] + np.maximum(fmax * sb, bmax) - np.maximum(fmax, bmax), 0.0)
            need = ('Fwd Packet Length Mean', 'Fwd Packet Length Std',
                    'Bwd Packet Length Mean', 'Bwd Packet Length Std')
            if all(f in fi for f in need):
                def _ex2(scale):
                    return (Nf * ((R0[:, fi[need[1]]] * scale) ** 2 +
                                  (R0[:, fi[need[0]]] * scale) ** 2) +
                            Nb * (R0[:, fi[need[3]]] ** 2 + R0[:, fi[need[2]]] ** 2)) / N
                v_new = np.maximum(_ex2(sb) - mean_new ** 2, 0.0)
                v_old = np.maximum(_ex2(1.0) - mean_old ** 2, 0.0)
                if 'Packet Length Variance' in fi:
                    R[:, fi['Packet Length Variance']] = np.maximum(
                        R0[:, fi['Packet Length Variance']] + v_new - v_old, 0.0)
                if 'Packet Length Std' in fi:
                    R[:, fi['Packet Length Std']] = np.maximum(
                        R0[:, fi['Packet Length Std']] + np.sqrt(v_new) - np.sqrt(v_old), 0.0)
        return R

    def p9_adversarial(self):
        print('[P9] Robustness in RAW space: corruption vs constrained evasion')
        df = pd.read_parquet(self.paths['test_known'], columns=self.all_feats + ['Target'])
        n = min(CFG['adv_n'], len(df))
        ii = np.sort(np.random.RandomState(SEED + 9).choice(len(df), n, replace=False))
        Z = df[self.all_feats].to_numpy(np.float64)[ii]
        y = (df['Target'].values[ii] != 'Normal').astype(np.int8)
        del df
        mu, sd = self._raw_io()
        R0 = Z * sd + mu
        back = (R0 - mu) / sd
        rt = float(np.max(np.abs(back - Z)))
        if rt > 1e-3:
            raise RuntimeError(f'raw-space round trip failed ({rt:.2e})')
        fi = {f: i for i, f in enumerate(self.all_feats)}
        m, cal, tau = self.dep, self.cal['DEPLOYED'], self.tau

        def score(R):
            Zs = ((np.maximum(R, 0) - mu) / sd).astype(np.float32)
            return cal(m.predict(Zs[:, self.fidx]))

        p0 = score(R0.copy())
        base = _metrics(y, p0, tau)
        disc = {f for f in self.all_feats
                if any(k in f for k in ('Flag', 'Cnt', 'Count', '__present', 'N_Invalid',
                                        'N_Clipped', 'Protocol'))
                or len(np.unique(R0[:, fi[f]][:5000])) <= 20}
        cont = [f for f in self.all_feats if f not in disc and f not in DERIVED_ALL]
        smp = R0[:min(len(R0), 20000)]
        def _is_int(col):
            tol = 1e-6 * np.maximum(1.0, np.abs(col))     # float32 round-trip tolerance
            return np.all(np.abs(col - np.round(col)) <= tol)
        integral = [fi[f] for f in cont if _is_int(smp[:, fi[f]])]

        self._n_rounded = 0

        def _phys(R, base):
            """Counts, byte totals and microsecond durations change by WHOLE units.
            The CHANGE is quantised, not the absolute value: Layer 1 imputes
            invalid/sentinel cells with the train median, which is often
            fractional, and re-quantising those placeholders would alter flows
            that were never perturbed. Delta-quantisation is therefore exactly
            identity when nothing is perturbed (asserted below at eps = 0)."""
            if integral:
                d = R[:, integral] - base[:, integral]
                R[:, integral] = base[:, integral] + np.round(d)
                self._n_rounded += int(np.count_nonzero(d))
            return R
        rs = np.random.RandomState(SEED + 10)
        corr, evas, hist = {}, {}, {}
        bins = np.linspace(0, 1, 41)
        hist['baseline_attack'] = np.histogram(p0[y == 1], bins)[0].tolist()
        for eps in [0.0] + CFG['adv_levels']:
            R = R0.copy()
            ci = [fi[f] for f in cont]
            M = rs.uniform(1 - eps, 1 + eps, (len(R), len(ci)))
            # A cell whose multiplier is exactly 1 is returned untouched. Without
            # this guard the non-negativity clip alters cells that were never
            # perturbed: float32 storage makes z*sigma+mu reconstruct slightly
            # below zero for true zeros (11k such cells in test data), so the
            # eps = 0 pass would not be the identity.
            R[:, ci] = np.where(M == 1.0, R0[:, ci], np.maximum(R0[:, ci] * M, 0.0))
            self._neg_clipped = int(np.count_nonzero((R0[:, ci] * M < 0) & (M != 1.0)))
            R = _rederive(_phys(R, R0), R0, fi)
            p = score(R)
            mt = _metrics(y, p, tau)
            dz = np.abs(((R - mu) / sd) - Z)[:, [fi[f] for f in self.feats]]
            corr[str(eps)] = {'metrics': mt,
                              'rel_f1_drop': 1 - mt['f1'] / base['f1'] if base['f1'] else None,
                              'mean_abs_dz_selected': float(dz.mean()),
                              'attack_scores_below_tau': float((p[y == 1] < tau).mean()),
                              'verdict': ('NOT APPLICABLE (baseline F1 < 0.5)'
                                          if base['f1'] < 0.5 else
                                          'ROBUST' if 1 - mt['f1'] / base['f1'] < CFG['robust_drop']
                                          else 'NOT ROBUST')}
            hist[f'corruption_{eps}'] = np.histogram(p[y == 1], bins)[0].tolist()
            if eps == 0.0 and abs(mt['f1'] - base['f1']) > 1e-12:
                raise RuntimeError('0% corruption does not reproduce the baseline')
        a = y == 1
        for s in CFG['evasion_scales']:
            for mode, (sb, st) in {'padding': (s, 1.0), 'delay': (1.0, s),
                                   'padding+delay': (s, s)}.items():
                R = R0.copy()
                R[a] = self._evade(R[a].copy(), R0[a], fi, sb, st)
                R = _rederive(_phys(R, R0), R0, fi)
                p = score(R)
                rec = float((p[a] >= tau).mean())
                evas[f'{mode}_{s}'] = {'scale_bytes': sb, 'scale_time': st, 'attack_recall': rec,
                                       'rel_recall_drop': 1 - rec / base['recall']
                                       if base['recall'] else None,
                                       'confusion': _counts(y, p >= tau)}
                if mode == 'padding+delay':
                    hist[f'evasion_{s}'] = np.histogram(p[a], bins)[0].tolist()
        R = R0.copy()
        R[a] = self._evade(R[a].copy(), R0[a], fi, 1.0, 1.0)
        if np.max(np.abs(score(_rederive(R, R0, fi)) - p0)) > 1e-9:
            raise RuntimeError('evasion at scale 1.0 does not reproduce the baseline')
        srcs = {x for d in DERIVED_SRC for x in DERIVED_SRC[d]} | {'Flow Duration'}
        single = {}
        for f in [f for f in cont if f in self.feats or f in srcs]:
            R = R0.copy()
            R[a, fi[f]] *= CFG['single_feature_scale']
            R = _rederive(_phys(R, R0), R0, fi)
            p = score(R)
            single[f] = float(base['recall'] - (p[a] >= tau).mean())
        single = dict(sorted(single.items(), key=lambda x: -abs(x[1])))
        self.R['robustness'] = {
            'space': 'raw (inverse Layer-1 scaler), clipped at 0 in raw space, re-standardised',
            'round_trip_max_abs_error': rt, 'sample_n': n, 'baseline': base,
            'corruption': corr, 'constrained_evasion': evas,
            'single_feature_recall_drop_at_x%.2f' % CFG['single_feature_scale']: single,
            'score_histograms_bins': bins.tolist(), 'score_histograms': hist,
            'continuous_features_perturbed': cont,
            'integer_valued_features_rounded': [self.all_feats[i] for i in integral],
            'integrality_rule': ('perturbations of integer-valued raw features are quantised '
                                 'as whole-unit CHANGES (new = orig + round(delta)), so an '
                                 'unperturbed flow is returned bit-identical'),
            'cells_delta_quantised': int(self._n_rounded),
            'negative_reconstructions_clipped': int(getattr(self, '_neg_clipped', 0)),
            'identity_guarantee': ('cells with multiplier exactly 1 are returned untouched, so '
                                   'the eps = 0 pass is bit-identical by construction (asserted)'),
            'threat_models': {
                'corruption': 'independent multiplicative noise on every continuous feature; '
                              'measures corruption robustness, NOT an adversarial attack',
                'evasion': 'attacker pads forward payload and/or delays packets on attack flows '
                           'only; rates, bidirectional packet statistics, ratios and '
                           'log-duration recomputed, counts and flags unchanged'},
            'v15_artifact_explained': 'v15.0 multiplied STANDARDISED values and clipped at 0, '
                                      'mapping every below-mean value to the training mean'}
        for k, v in corr.items():
            print(f'  corruption {float(k):>4.0%}: F1 {v["metrics"]["f1"]:.4f} '
                  f'({v["verdict"]})')
        w = min(evas.items(), key=lambda x: x[1]['attack_recall'])
        print(f'  worst evasion {w[0]}: recall {w[1]["attack_recall"]:.4f} '
              f'(baseline {base["recall"]:.4f})')

    # ------------------------------------------------------------ 10 ablation
    def p10_ablation_and_random_split(self):
        print('[P10] Fair ablation + in-house random-split baseline')
        n = min(CFG['ablation_n'], len(self.match_idx))
        ai = np.sort(np.random.RandomState(SEED + 11).choice(self.match_idx, n, replace=False))
        yv, ya = self.y_val[self.thr_i], self.y_tr[ai]
        es_y = self.y_tr[self.es_idx]

        def run(cols):
            ci = [self.all_feats.index(f) for f in cols]
            mm, _ = _fit('LightGBM', self.X_tr_all[np.ix_(ai, ci)], ya,
                         self.X_tr_all[np.ix_(self.es_idx, ci)], es_y)
            vi = [self.feats.index(f) if f in self.feats else None for f in cols]
            if all(v is not None for v in vi):
                Xv = self.X_val[self.thr_i][:, vi]
                Xt = self.Xte[:, vi]
            else:
                Xv = _load(self.paths['val'], cols, ())[0][self.thr_i]
                Xt = _load(self.paths['test_known'], cols, ())[0]
            pv = mm.predict(Xv)
            t = _primary_tau(pv, yv)
            pt = mm.predict(Xt)
            return {'n_features': len(cols), 'val_auc_pr': float(average_precision_score(yv, pv)),
                    'test_f1': _metrics(self.yte, pt, t)['f1'], 'model': mm}

        base = run(self.feats)
        imp = base['model'].obj.feature_importance('gain')
        top = [self.feats[i] for i in np.argsort(imp)[::-1][:CFG['ablation_topk']]]
        abl = {'protocol': f'LightGBM retrained on the SAME {n:,}-row sample for baseline and '
                           'every variant; threshold = primary rule (' + CFG['primary_rule'] +
                           ') on validation threshold half',
               'baseline': {k: v for k, v in base.items() if k != 'model'},
               'importance_gain': {self.feats[i]: float(imp[i]) for i in np.argsort(imp)[::-1]},
               'drop_one': {}, 'feature_sets': {}}
        for f in top:
            r = run([x for x in self.feats if x != f])
            abl['drop_one'][f] = {'val_auc_pr': r['val_auc_pr'], 'test_f1': r['test_f1'],
                                  'd_val_auc_pr': r['val_auc_pr'] - base['val_auc_pr'],
                                  'd_test_f1': r['test_f1'] - base['test_f1']}
        for k, cols in self.variants.items():
            if k == 'selected':
                continue
            r = run(cols)
            abl['feature_sets'][k] = {x: v for x, v in r.items() if x != 'model'}
        self.R['ablation'] = abl

        # in-house random-split baseline on the same known-class pool
        rs = np.random.RandomState(SEED + 12)
        pools = [(self.X_tr_all[:, self.fidx], self.y_tr), (self.X_val, self.y_val),
                 (self.Xte, self.yte)]
        tot = sum(len(p[1]) for p in pools)
        take = min(CFG['random_split_pool'], tot)
        Xs, ys = [], []
        for Xp, yp in pools:
            k = int(round(take * len(yp) / tot))
            j = rs.choice(len(yp), min(k, len(yp)), replace=False)
            Xs.append(Xp[j])
            ys.append(yp[j])
        Xp, yp = np.vstack(Xs), np.concatenate(ys)
        perm = rs.permutation(len(yp))
        a, b = int(0.70 * len(yp)), int(0.85 * len(yp))
        tr, va, te = perm[:a], perm[a:b], perm[b:]
        es = tr[: max(1, int(CFG['es_frac'] * len(tr)))]
        tr = tr[len(es):]
        rm, _ = _fit('LightGBM', Xp[tr], yp[tr], Xp[es], yp[es])
        t = _primary_tau(rm.predict(Xp[va]), yp[va])
        pt = rm.predict(Xp[te])
        htr = set(pd.util.hash_pandas_object(pd.DataFrame(Xp[tr]), index=False).values.tolist())
        hte = pd.util.hash_pandas_object(pd.DataFrame(Xp[te]), index=False).values
        dup = np.array([h in htr for h in hte])
        chrono = self.R['ablation']['baseline']
        self.R['random_split_baseline'] = {
            'protocol': 'same LightGBM, same features, same FPR<=1% rule; random stratified '
                        '70/15/15 over a uniform sample of all known-class flows',
            'train_rows': int(len(tr)),
            'random_split_test': _metrics(yp[te], pt, t),
            'random_split_test_dedup_f1': _metrics(yp[te][~dup], pt[~dup], t)['f1'],
            'random_split_test_rows_with_train_twin_pct': float(100 * dup.mean()),
            'chronological_same_budget_test_f1': chrono['test_f1'],
            'chronological_test_rows_with_train_twin_pct': self.R['test_meta']['dup_in_train_pct'],
            'interpretation': 'difference = inflation attributable to random splitting on '
                              'this dataset; supports the literature comparison table'}
        r = self.R['random_split_baseline']
        print(f'  random-split F1 {r["random_split_test"]["f1"]:.4f} (twins '
              f'{r["random_split_test_rows_with_train_twin_pct"]:.1f}%) vs chronological '
              f'{chrono["test_f1"]:.4f}')

    # ------------------------------------------------------------ 11 figures
    def p11_figures(self):
        print('[P11] Figures')
        y = self.yte
        names = list(self.models) + ['DEPLOYED']
        pw = {p['pair']: p for p in self.R['statistics']['pairwise_matched']}

        # Fig 3: ROC + PR
        fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.9))
        for n in names:
            p = self.P[n]
            col = OK['black'] if n == 'DEPLOYED' else MODEL_COL[n]
            lab = (f'{self.family} (deployed)' if n == 'DEPLOYED' else f'{n} (matched)')
            fpr, tpr, _ = roc_curve(y, p)
            ax[0].plot(fpr, tpr, color=col, lw=1.6 if n == 'DEPLOYED' else 1.0,
                       ls='-' if n == 'DEPLOYED' else '--',
                       label=f'{lab}: AUC {self.R["test"][n]["full"]["auc_roc"]:.4f}, '
                             f'AP {self.R["test"][n]["full"]["auc_pr"]:.4f}')
            pr, rc, _ = precision_recall_curve(y, p)
            ax[1].plot(rc, pr, color=col, lw=1.6 if n == 'DEPLOYED' else 1.0,
                       ls='-' if n == 'DEPLOYED' else '--')
        d = self.R['test']['DEPLOYED']['full']
        ax[0].plot(d['fpr'], d['recall'], 'o', ms=5, mfc='white', mec='black',
                   label=f'operating point ({CFG["primary_rule"]} rule)')
        ax[0].set_xscale('log')
        ax[0].set_xlim(1e-5, 1)
        ax[0].set_xlabel('False-positive rate (log)')
        ax[0].set_ylabel('True-positive rate')
        ax[1].set_xlabel('Recall')
        ax[1].set_ylabel('Precision')
        # DeLong / delta-AUC evidence lives in Table IV (manifest: statistics.pairwise_matched)
        for a_ in ax:
            a_.grid(alpha=.25, lw=.5)
        h, l = ax[0].get_legend_handles_labels()
        fig.legend(h, l, loc='lower center', ncol=2, frameon=False, fontsize=6.8,
                   bbox_to_anchor=(0.5, -0.02))
        ax[0].set_title('(a) ROC, test (natural prior)')
        ax[1].set_title('(b) Precision-recall, test')
        fig.tight_layout(rect=(0, 0.16, 1, 1))
        fig.savefig(self.figs / 'L2_fig3_roc_pr.pdf', bbox_inches='tight')
        fig.savefig(self.figs / 'L2_fig3_roc_pr.png', bbox_inches='tight')
        plt.close(fig)

        # Fig 4: confusion matrix, counts + 3-decimal rates
        fig, ax = plt.subplots(figsize=(3.4, 2.8))
        cm = np.array([[d['tn'], d['fp']], [d['fn'], d['tp']]])
        rate = cm / cm.sum(1, keepdims=True)
        ax.imshow(rate, cmap='Blues', vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{rate[i, j]:.3f}\n({cm[i, j]:,})', ha='center', va='center',
                        color='white' if rate[i, j] > .6 else 'black', fontsize=8)
        ax.set_xticks([0, 1], ['Normal', 'Attack'])
        ax.set_yticks([0, 1], ['Normal', 'Attack'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(f'Deployed {self.family}, $\\tau$={self.tau:.4f} (calibrated)')
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig4_cm.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig 5: latency ECDF
        tm, te = self._lat_series
        fig, ax = plt.subplots(figsize=(3.4, 2.5))
        for t, lab, col in ((tm, 'model + calibrator + $\\tau$', OK['blue']),
                            (te, 'full online path (raw record $\\rightarrow$ decision)',
                             OK['verm'])):
            if t is None:
                continue
            s = np.sort(t)
            ax.plot(s, np.arange(1, len(s) + 1) / len(s), color=col, lw=1.2,
                    label=f'{lab}; P95 {np.percentile(t, 95):.0f} $\\mu$s')
        ax.set_xscale('log')
        ax.set_xlabel('Latency per flow ($\\mu$s, batch = 1, log)')
        ax.set_ylabel('ECDF')
        ax.legend(frameon=False, loc='lower right')
        ax.grid(alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig5_latency.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig 7: reliability with counts, ECE, MCE
        cr = _ece(y, np.clip(self._praw_dep, 0, 1), CFG['ece_bins'])
        cc = self.R['test']['DEPLOYED']['calibration_cal']
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(3.4, 3.6), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 1.2]})
        a1.plot([0, 1], [0, 1], color=OK['grey'], lw=.8, ls=':')
        for c, lab, col, mk in ((cr, 'raw', OK['orange'], 's'),
                                (cc, 'calibrated', OK['blue'], 'o')):
            xs = [b['mean_p'] for b in c['bins'] if b['n']]
            ys = [b['frac_pos'] for b in c['bins'] if b['n']]
            a1.plot(xs, ys, marker=mk, ms=3.5, color=col, lw=1,
                    label=f'{lab}: ECE {c["ece"]:.4f}, MCE {c["mce"]:.4f}')
        a1.set_ylabel('Observed attack fraction')
        a1.legend(frameon=False, loc='upper left')
        a1.grid(alpha=.25, lw=.5)
        mids = [(b['lo'] + b['hi']) / 2 for b in cc['bins']]
        a2.bar(mids, [max(b['n'], .8) for b in cc['bins']], width=1 / CFG['ece_bins'] * .9,
               color=OK['blue'], alpha=.8)
        for x_, b in zip(mids, cc['bins']):
            a2.text(x_, max(b['n'], 1) * 1.3, f'{b["n"]:,}', ha='center', fontsize=6, rotation=90)
        a2.set_yscale('log')
        a2.set_ylabel('Flows / bin')
        a2.set_xlabel('Predicted attack probability (calibrated)')
        a2.set_ylim(.8, max(b['n'] for b in cc['bins']) * 30)
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig7_reliability.pdf', bbox_inches='tight')
        plt.close(fig)

        # threshold rules figure
        p, yy = self._thr_curve
        th = np.linspace(0, 1, 401)
        ps, ns = np.sort(p[yy == 1]), np.sort(p[yy == 0])
        rec = 1 - np.searchsorted(ps, th, 'left') / max(len(ps), 1)
        fpr_ = 1 - np.searchsorted(ns, th, 'left') / max(len(ns), 1)
        fig, ax = plt.subplots(figsize=(3.4, 2.5))
        ax.plot(th, rec, color=OK['blue'], lw=1.2, label='recall')
        ax2 = ax.twinx()
        ax2.plot(th, np.maximum(fpr_, 1e-6), color=OK['verm'], lw=1.2, label='FPR')
        ax2.set_yscale('log')
        for (r, t), ls in zip(self.rules['DEPLOYED'].items(), ['-', '--', ':', '-.', (0, (1, 1))]):
            ax.axvline(t, color='black', ls=ls, lw=.8, label=r.replace('_', ' '))
        ax.axvspan(*CFG['legacy_clamp'], color=OK['grey'], alpha=.08, lw=0)
        ax.set_xlabel('Threshold on calibrated probability (validation threshold half)')
        ax.set_ylabel('Recall')
        ax2.set_ylabel('FPR (log)')
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=6.5, frameon=False, loc='lower left')
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig_threshold_rules.pdf', bbox_inches='tight')
        plt.close(fig)

        # robustness figure
        rb = self.R['robustness']
        fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.5))
        lv = [0.0] + CFG['adv_levels']
        ax[0].plot([100 * v for v in lv], [rb['corruption'][str(v)]['metrics']['f1'] for v in lv],
                   'o-', color=OK['blue'], lw=1.2, ms=3.5, label='corruption: F1')
        for mode, col in (('padding', OK['orange']), ('delay', OK['green']),
                          ('padding+delay', OK['verm'])):
            xs = [100 * (s - 1) for s in CFG['evasion_scales']]
            ax[0].plot(xs, [rb['constrained_evasion'][f'{mode}_{s}']['attack_recall']
                            for s in CFG['evasion_scales']], 's--', color=col, lw=1, ms=3,
                       label=f'evasion ({mode}): recall')
        ax[0].set_xlabel('Perturbation magnitude (%)')
        ax[0].set_ylabel('Metric')
        ax[0].legend(frameon=False, fontsize=6)
        ax[0].grid(alpha=.25, lw=.5)
        b = np.array(rb['score_histograms_bins'])
        mids = (b[1:] + b[:-1]) / 2
        for k, col, lab in (('baseline_attack', OK['black'], 'no perturbation'),
                            (f'corruption_{CFG["adv_levels"][-1]}', OK['blue'],
                             f'corruption {CFG["adv_levels"][-1]:.0%}'),
                            (f'evasion_{CFG["evasion_scales"][-1]}', OK['verm'],
                             f'padding+delay x{CFG["evasion_scales"][-1]}')):
            h = np.array(rb['score_histograms'][k], float)
            ax[1].step(mids, h / max(h.sum(), 1), where='mid', color=col, lw=1, label=lab)
        ax[1].axvline(self.tau, color='black', ls=':', lw=.8)
        ax[1].set_xlabel('Calibrated attack probability (attack flows)')
        ax[1].set_ylabel('Fraction')
        ax[1].legend(frameon=False, fontsize=6)
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig_robustness.pdf', bbox_inches='tight')
        plt.close(fig)

        # selection stability + ablation
        fs = self.R['feature_selection']['selection_frequency']
        ab = self.R['ablation']['drop_one']
        fig, ax = plt.subplots(1, 2, figsize=(7.16, 4.4))
        items = sorted(fs.items(), key=lambda x: x[1])[-30:]          # top-30 legible at 6.5 pt
        ax[0].barh(range(len(items)), [v for _, v in items],
                   color=[OK['blue'] if v >= CFG['stability_keep_frac'] else OK['grey']
                          for _, v in items], height=.7)
        ax[0].set_yticks(range(len(items)), [k for k, _ in items], fontsize=6.5)
        ax[0].set_title('(a) MI selection frequency, top-30', fontsize=8)
        ax[0].axvline(CFG['stability_keep_frac'], color='black', ls=':', lw=.8)
        ax[0].set_xlabel(f'Selection frequency ({CFG["mi_seeds"]} seeds)')
        items = sorted(ab.items(), key=lambda x: x[1]['d_val_auc_pr'])
        ax[1].barh(range(len(items)), [v['d_val_auc_pr'] for _, v in items],
                   color=OK['verm'], height=.6)
        ax[1].set_yticks(range(len(items)), [k for k, _ in items], fontsize=6.5)
        ax[1].set_title('(b) Drop-one ablation, same sample', fontsize=8)
        ax[1].axvline(0, color='black', lw=.6)
        ax[1].set_xlabel('$\\Delta$ validation AUC-PR when dropped (same sample)')
        fig.tight_layout()
        fig.savefig(self.figs / 'L2_fig_selection_ablation.pdf', bbox_inches='tight')
        plt.close(fig)

    # ------------------------------------------------------------ 12 exports
    def p12_export(self):
        print('[P12] Exports')
        m = self.dep
        if m.kind == 'lgb':
            m.obj.save_model(str(self.mdir / 'l2_deployed.lgb.txt'))
        elif m.kind == 'xgb':
            m.obj.save_model(str(self.mdir / 'l2_deployed.xgb.json'))
        else:
            import joblib
            joblib.dump(m.obj, self.mdir / 'l2_deployed.rf.joblib', compress=3)
        pd.DataFrame({'p_cal': self.P['DEPLOYED'].astype(np.float32),
                      'flag': self.P['DEPLOYED'] >= self.tau,
                      'Target': self.meta_te['Target'].values,
                      '_ts': self.meta_te['_ts'].values,
                      '_dup_in_train': self.meta_te['_dup_in_train'].values}
                     ).to_parquet(self.out / 'l2_scores_test_known.parquet')
        try:
            os.remove(self.out / '_ptest_tmp.npy')
        except OSError:
            pass
        self.R['exports'] = {
            'decision': str(self.mdir / 'l2_decision.json'),
            'scores_test_known': str(self.out / 'l2_scores_test_known.parquet'),
            'scores_zeroday': str(self.out / 'l2_scores_zeroday.parquet'),
            'row_alignment': 'same row order as Layer-1 l2_test_known / l2_test_zeroday',
            'contract_for_L3_L5': 'route on p_cal >= tau; tau is on the CALIBRATED scale'}
        # ---- downstream contract (identical keys to R5.3; Layers 3-8 load unchanged)
        def _pk(obj, dest):
            with open(dest, 'wb') as fh:
                pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)
        _pk(self.dep.obj, self.out / 'best_model.pkl')
        _pk(self.cal['DEPLOYED'].m, self.out / 'best_cal.pkl')      # sklearn Pipeline
        for nm, mm in self.models.items():
            _pk(mm.obj, self.out / f'model_{nm.lower()}_matched.pkl')
            _pk(self.cal[nm].m, self.out / f'cal_{nm.lower()}_matched.pkl')
        _atomic_json({'deployed_tau': {self.family: self.tau},
                      'matched_tau': {nm: self.rules[nm][CFG['primary_rule']] for nm in self.models},
                      'all_rules_deployed': self.rules['DEPLOYED'],
                      'score_space': 'calibrated probability (apply best_cal.pkl first)',
                      'selection': f'{CFG["primary_rule"]} on val threshold half',
                      'calibration_half': 'A', 'threshold_half': 'B'},
                     self.out / 'thresholds.json')
        full = dict(self.R['test']['DEPLOYED']['full'])
        full['auc'] = full.get('auc_roc')
        self.R['best_model'] = self.family
        self.R['metrics'] = {self.family: {'full': full,
                                           'dedup': self.R['test']['DEPLOYED']['dedup']}}
        self.R['metrics_matched'] = {nm: self.R['test'][nm] for nm in self.models}
        self.R['bootstrap'] = {'method': self.R['statistics']['block_definition'],
                               'per_model': {self.family: {
                                   'f1_ci95': self.R['statistics']['deployed_block_ci']['f1'],
                                   'fpr_ci95': self.R['statistics']['deployed_block_ci']['fpr'],
                                   'auc_ci95': self.R['statistics']['deployed_auc_block_ci']}}}
        self.R['l2_contract'] = {'reserved_columns_not_features':
                                 self.m1['l2_contract']['reserved_columns_not_features']}
        self.R['l3_l4_contract'] = {'model': str(self.out / 'best_model.pkl'),
                                    'cal': str(self.out / 'best_cal.pkl'),
                                    'threshold_json': str(self.out / 'thresholds.json'),
                                    'features': self.feats,
                                    'model_kind': self.dep.kind,
                                    'cal_usage': 'cal.predict_proba(p_raw.reshape(-1,1))[:,1]',
                                    'xgb_usage': 'DMatrix(X, feature_names=features)'}
        self.R['deployed_model_note'] = (
            f'this protocol selected {self.family} on validation AUC-PR. Any paper text '
            'naming a different deployed family comes from the v15.0 protocol, which '
            'selected on TEST F1, and must be reconciled.')
        self.R['layer1_manifest_hashes'] = self.m1['artifact_hashes']
        _atomic_json(self.R, self.out / 'manifest_l2.json')

    def run(self):
        t0 = time.time()
        for step in (self.p0_contract, self.p1_load, self.p2_features, self.p3_validation,
                     self.p4_train, self.p5_calibrate_threshold, self.p6_evaluate,
                     self.p7_statistics, self.p8_latency, self.p9_adversarial,
                     self.p10_ablation_and_random_split, self.p11_figures, self.p12_export):
            step()
            _gc()
        d = self.R['test']['DEPLOYED']
        ci = self.R['statistics']['deployed_block_ci']['f1']
        print('=' * 76)
        print(f'  LAYER 2 R8.0 COMPLETE | {(time.time() - t0) / 60:.1f} min | '
              f'{self.family} | {len(self.feats)} features')
        print(f'  test F1 {d["full"]["f1"]:.4f} [block CI {ci[0]:.4f}-{ci[1]:.4f}] | '
              f'FPR {100 * d["full"]["fpr"]:.3f}% | dedup F1 {d["dedup"]["f1"]:.4f} | '
              f'tau {self.tau:.4f} (calibrated; rule={CFG["primary_rule"]} on VALIDATION)')
        tt = self.R.get('limitations', {}).get('threshold_transfer', {})
        if tt:
            print(f'  [FINDING] FPR<={CFG["fpr_budget"]:.0%} budget rule: val '
                  f'{100 * tt["val_fpr_at_budget_tau"]:.3f}% -> test '
                  f'{100 * tt["test_fpr_at_budget_tau"]:.3f}% '
                  f'({tt["transfer_factor"]:.1f}x); '
                  f'{tt["per_block_fpr_at_budget_tau"]["pct_blocks_above_budget"]:.0f}% of '
                  'blocks breach it. See manifest.limitations.threshold_transfer')
        print('=' * 76)


if __name__ == '__main__':
    # Notebook-safe: argparse falls back to defaults when no CLI args are given,
    # so the defaults must match the CURRENT Layer-1 output, and a missing or
    # mislocated Layer-1 directory must fail with an actionable message.
    ap = argparse.ArgumentParser()
    ap.add_argument('--l1', default=os.environ.get('T6_L1', '/kaggle/working/l1_R8'))
    ap.add_argument('--out', default=os.environ.get('T6_L2', '/kaggle/working/l2_R8'))
    ap.add_argument('--l1-code', default=os.environ.get('T6_L1_CODE'),
                    help='path to the Layer-1 .py if not run in the same kernel')
    a, _ = ap.parse_known_args()
    l1 = Path(a.l1)
    if not (l1 / 'manifest.json').exists():
        root = l1.parent if l1.parent.exists() else Path('.')
        found = sorted(str(p.parent) for p in root.glob('*/manifest.json'))
        raise SystemExit(
            f'[FATAL] no manifest.json in {l1}\n'
            f'        Layer-1 outputs found here: {found or "none"}\n'
            f'        pass --l1 <dir>, or set T6_L1=<dir>, or edit the default above.')
    code = a.l1_code if (a.l1_code and os.path.exists(a.l1_code)) else None
    if a.l1_code and code is None:
        print(f'[WARN] --l1-code {a.l1_code} not found: the raw->decision latency '
              'measurement will be skipped (model-only latency is unaffected)')
    Layer2(str(l1), a.out, code).run()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
[P0] Layer-1 contract
  L1 TITANIUM6_LAYER1_R8.0 | mode=per_class | 87 candidate features
[P1] Load train (all candidate features)
  train 11,244,135 rows | attack 0.1571 | normal:attack 5.37:1 | ES hold-out 562,206
[P2] Feature selection: KS(train-val) + MI stability selection
  [WARN] stage-1 KS filter MEASURED INERT: all 87 features pass (max KS 0.0885 < 0.1); selection is MI-only
  stability bands: 72 at 0.70 | 69 at 0.90 | overlap 69
  stage1 87/87 | stage2 72 | final 77 | Nogueira stability 0.515
[P3] Validation: disjoint calibration / threshold halves
  val 2,362,618 | fam 787,539 | cal 787,539 | thr 787,540
[P4] Matched comparison + deployed model
  LightGBM      matched n=1,000,000 | val AUC-PR 0.9859 | 86.0s
  XGBoost       matched n=1,000,000 | val AUC-PR 0.9849 | 63.6s
  RandomForest  matched n=1,000,000 | val AUC-PR 0.9838 | 649.4s
  [FINDING] family selection is a TIE (LightGBM vs XGBoost, CIs overlap); tieb

In [4]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 3 | R8.0
#  R8.0 over R7.12:
#   T1 L2 ROUTED-FPR BUG. R7.12 computed the L2 FPR on the ROUTED set, which
#      by construction contains only the Normal rows L2 flagged, so the value
#      was always 1.0 and the consistency check always fired. R8.0 divides the
#      flagged Normal count by ALL Normal rows scanned in test_known.
#   T2 Paths and version strings follow the single R8 run configuration.
#
#
#  What changed from R7.11:
#   C1 PATH DEFAULTS. R7.11's __main__ pointed at l1_R5.2 / l2_R6.3 / l3_R7.8
#      and crashed on the real l1_R5.4 / l2_R6.5 pair. Defaults now track the
#      deployed Layer-1 R5.4 and Layer-2 R6.5 artifact directories, and a
#      missing manifest fails with the directories that DO contain one.
#   C2 VERSION STRINGS. Header, CFG, and argparse description are now all
#      'R7.12' / 'v7.12'. R7.11 had three different labels in three places.
#   C3 L2 FPR REFERENCE IS MANDATORY. R7.11 silently skipped the cascaded-FPR
#      block when manifest_l2 lacked metrics[family].full.fpr. Now it is a
#      hard failure with an actionable message.
#   C4 COMBINED ZERO-DAY DR IS A MEASURED UNION, NOT AN INDEPENDENCE PRODUCT.
#      R7.11 computed 1 - (1-L2)(1-L4) using L2's F1 (wrong base rate) and
#      L4's mean per-class DR. R7.12 reports the measured union on the same
#      rows when L2's per-row zero-day flags are available; otherwise it
#      records None and states the reason.
#   C5 REJECT-SHIFT GUARD RAISED TO 200. R7.11 ran per-feature KS on 13 val
#      FPs vs 33 test FPs and printed a finding. 200 minimum on both sides;
#      below that the block records status=NOT MEASURED with the counts.
#   C6 PER-CLASS PLATT MINIMUM 50. R7.11 fitted Platt on classes with 5 rows.
#      Now requires >=50 positives and >=50 negatives in the calibration
#      half, otherwise the class uses its raw probability and is listed in
#      the manifest under calibration_raw_fallback.
#   C7 --reject-source REMOVED. It changed two things at once (reject slice
#      AND calibration slice) and R7.11 itself documented it as not a
#      single-variable ablation. reject_source is now hardcoded 'val'.
#   C8 CASCADED-FPR BASE-RATE ASSERTION. When both the L2 test FPR reference
#      and the L3-measured routed FPR are present, they must agree to 1e-6;
#      a mismatch is a configuration error, not a result.
#   C9 IMPORT HYGIENE. ks_2samp and warnings are imported at the top of the
#      file. No local imports. No warnings.filterwarnings.
#
#  Input contract:  L1 R5.4 manifest.json  (schema 5) + L2 R6.5 manifest_l2.json
#  Output contract: best_model_l3.pkl, best_calibrators_l3.pkl, label_encoder.pkl,
#                   l3_thresholds.json, layer3_config.json, manifest_l3.json,
#                   l3_scores_test_routed.parquet
#
#  Deployed path (default):  reject_source='val', train_reject=True.
#  Ablation  path:           --attack-only (no reject class; no cascaded FPR).
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os
import gc
import json
import time
import pickle
import platform
import argparse
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
import scipy
from scipy.stats import chi2, norm, rankdata, binomtest, ks_2samp
from scipy.special import logit as _logit_fn, logsumexp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, roc_auc_score, roc_curve)
import lightgbm as lgb
import xgboost as xgb

SEED = 42
np.random.seed(SEED)


CFG = {
    'version': 'TITANIUM6_LAYER3_R8.0',
    'reject_source': 'val',                 # hardcoded: 'val' out-of-sample, 'train' NOT supported
    'val_split': (0.50, 0.25, 0.25),        # reject-pool / calibration / threshold
    'seed': SEED,
    'required_schema': 5,
    'reject_class': 'Normal',
    'train_reject': True,                   # --attack-only disables (ablation)
    'chunk': 200_000,
    'es_frac': 0.05,
    'rounds': 500, 'early_stop': 50,
    'min_class_rows': 20,

    # thresholds, pre-declared
    'l5_precision_target': 0.95,
    'ood_val_flag_rate': 0.05,
    'block_minutes': 60,
    'bootstrap_n': 1000, 'bootstrap_iid_n': 200,

    'lat_warmup': 200, 'lat_calls': 3000,

    'adv_n': 50_000, 'adv_levels': [0.05, 0.10, 0.20],
    'evasion_scales': [1.05, 1.10, 1.20, 1.50],
    'robust_drop': 0.05, 'single_feature_scale': 1.20, 'identity_tol': 1e-12,

    'ablation_n': 400_000, 'ablation_topk': [1, 3, 5, 10],
    'spec_n_per': 50_000, 'spec_rounds': 200,
    'specialist_candidates': ['BruteForce-FTP', 'DoS-SlowHTTP',
                              'DDoS-LOIC-UDP', 'DoS-Slowloris'],
    'rare_classes': ['DDoS-LOIC-UDP', 'DoS-Slowloris'],

    'lgbm': {'objective': 'multiclass', 'metric': 'multi_logloss',
             'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
             'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
             'verbose': -1, 'seed': SEED, 'n_jobs': -1},
    'xgb': {'objective': 'multi:softprob', 'eval_metric': 'mlogloss',
            'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8,
            'colsample_bytree': 0.8, 'min_child_weight': 5, 'seed': SEED,
            'nthread': -1, 'verbosity': 0, 'tree_method': 'hist'},
    'spec': {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 31,
             'learning_rate': 0.05, 'verbose': -1, 'seed': SEED, 'n_jobs': -1},
}

OK = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73',
      'verm': '#D55E00', 'purple': '#CC79A7', 'sky': '#56B4E9',
      'grey': '#555555', 'black': '#000000'}

DERIVED_SRC = {'Ratio__FwdBwd_Packets': ('Total Fwd Packets', 'Total Backward Packets'),
               'Ratio__FwdBwd_Bytes': ('Total Length of Fwd Packets',
                                       'Total Length of Bwd Packets'),
               'Ratio__FwdBwd_IAT': ('Fwd IAT Tot', 'Bwd IAT Tot'),
               'Ratio__Active_Idle': ('Active Mean', 'Idle Mean')}
DERIVED_ALL = set(DERIVED_SRC) | {'Log_Flow_Duration', 'Zero_Duration_Flag',
                                  'Init Fwd Win Byts__present',
                                  'Init Bwd Win Byts__present',
                                  'N_Invalid_Features', 'N_Clipped_Negatives'}
FWD_BYTES = ['Total Length of Fwd Packets', 'Fwd Packet Length Max',
             'Fwd Packet Length Min', 'Fwd Packet Length Mean',
             'Fwd Packet Length Std', 'Fwd Seg Size Avg', 'Subflow Fwd Bytes']
TIME_FEATS = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
              'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std',
              'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean',
              'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Active Mean',
              'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std',
              'Idle Max', 'Idle Min']
PKT_RATES = ['Flow Packets/s', 'Fwd Pkts/s', 'Bwd Pkts/s']


# =============================================================================
#  UTILITIES
# =============================================================================
def _gc():
    gc.collect()


def _mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)


def _read_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)


def _json_default(o):
    if isinstance(o, tuple):
        return list(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if hasattr(o, 'item'):
        return o.item()
    return str(o)


def _atomic_json(obj, dest):
    tmp = str(dest) + '.__tmp__'
    with open(tmp, 'w', encoding='utf-8') as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    os.replace(tmp, dest)


def _save(obj, dest):
    tmp = str(dest) + '.__tmp__'
    with open(tmp, 'wb') as fh:
        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, dest)


def _load_pickle(path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)


def _hardware():
    cpu = platform.processor() or ''
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    cpu = line.split(':', 1)[1].strip()
                    break
    except OSError:
        pass
    return {'cpu_model': cpu, 'physical_cores': psutil.cpu_count(logical=False),
            'logical_cores': psutil.cpu_count(logical=True),
            'ram_gb': round(psutil.virtual_memory().total / 2**30, 1),
            'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__,
            'lightgbm': lgb.__version__, 'xgboost': xgb.__version__}


def _iter(path, cols, chunk=None):
    for b in pq.ParquetFile(path).iter_batches(batch_size=chunk or CFG['chunk'],
                                               columns=cols):
        df = b.to_pandas()
        yield df
        del df, b


def _resolve(path, base):
    for c in (Path(path), base / Path(path).name, base / 'l2_ready' / Path(path).name):
        if c.exists():
            return str(c)
    raise FileNotFoundError(
        f'artifact not found: {path}\n'
        f'  tried: {Path(path)}\n'
        f'         {base / Path(path).name}\n'
        f'         {base / "l2_ready" / Path(path).name}')


def _ratio(n, d):
    return np.clip(n / np.where(d == 0, 1.0, d), 0.0, 1e6)


def _rederive(R, R0, fi):
    for d, (a, b) in DERIVED_SRC.items():
        if d in fi and a in fi and b in fi:
            R[:, fi[d]] = np.clip(R0[:, fi[d]] + _ratio(R[:, fi[a]], R[:, fi[b]])
                                  - _ratio(R0[:, fi[a]], R0[:, fi[b]]), 0.0, 1e6)
    if 'Flow Duration' in fi:
        D = np.maximum(R[:, fi['Flow Duration']], 0)
        D0 = np.maximum(R0[:, fi['Flow Duration']], 0)
        if 'Log_Flow_Duration' in fi:
            R[:, fi['Log_Flow_Duration']] = np.maximum(
                R0[:, fi['Log_Flow_Duration']] + np.log1p(D) - np.log1p(D0), 0.0)
        if 'Zero_Duration_Flag' in fi:
            R[:, fi['Zero_Duration_Flag']] = np.where(
                D == D0, R0[:, fi['Zero_Duration_Flag']], (D == 0).astype(float))
    return R


def _platt(p, y):
    lr = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
    lr.fit(_logit_fn(np.clip(p, 1e-7, 1 - 1e-7)).reshape(-1, 1), y)
    return lr


def _apply_platt(lr, p):
    return lr.predict_proba(
        _logit_fn(np.clip(p, 1e-7, 1 - 1e-7)).reshape(-1, 1))[:, 1].astype(np.float32)


def _ci(a, lo=2.5, hi=97.5):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]
    if not len(a):
        return [float('nan'), float('nan')]
    return [float(np.percentile(a, lo)), float(np.percentile(a, hi))]


def _macro_f1_from_cm(cm, present_mask=None):
    k = cm.shape[0]
    if present_mask is None:
        present_mask = cm.sum(1) > 0
    f1 = np.full(k, np.nan)
    for c in range(k):
        if not present_mask[c] or cm[c, :].sum() == 0:
            continue
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1[c] = 2 * p * r / (p + r) if p + r else 0.0
    fin = f1[np.isfinite(f1)]
    return (float(fin.mean()) if len(fin) else 0.0), f1


def _block_cms(y, pred, bid, k):
    nb = int(bid.max()) + 1
    out = np.zeros((nb, k, k), np.int64)
    np.add.at(out, (bid, y, pred), 1)
    return out


def _mcnemar(corr_a, corr_b):
    b = int((corr_a & ~corr_b).sum())
    c = int((~corr_a & corr_b).sum())
    if b + c == 0:
        return {'b': 0, 'c': 0, 'discordant': 0, 'odds_ratio': 1.0,
                'or_ci95': [1.0, 1.0], 'p_raw': 1.0}
    p = (float(binomtest(b, b + c, 0.5).pvalue) if b + c < 25
         else float(chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1)))
    orr = (b + 0.5) / (c + 0.5)
    se = np.sqrt(1 / (b + 0.5) + 1 / (c + 0.5))
    return {'b': b, 'c': c, 'discordant': b + c, 'odds_ratio': float(orr),
            'or_ci95': [float(np.exp(np.log(orr) - 1.96 * se)),
                        float(np.exp(np.log(orr) + 1.96 * se))], 'p_raw': p}


def _holm(ps):
    if not ps:
        return []
    order = np.argsort(ps)
    adj, run = np.empty(len(ps)), 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - r) * ps[i]))
        adj[i] = run
    return adj.tolist()


def _delong_pair(p1, p2, y):
    y = y.astype(bool)
    m, n = int(y.sum()), int((~y).sum())
    if m < 2 or n < 2:
        return float('nan'), float('nan'), 0.0, 1.0
    P = np.vstack([p1, p2])
    X, Y = P[:, y], P[:, ~y]
    tx = np.vstack([rankdata(X[i]) for i in range(2)])
    ty = np.vstack([rankdata(Y[i]) for i in range(2)])
    tz = np.vstack([rankdata(np.r_[X[i], Y[i]]) for i in range(2)])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * norm.sf(abs(z)))


# =============================================================================
#  MODEL WRAPPER
# =============================================================================
class MultiModel:
    def __init__(self, name, obj, kind, n_cls, features):
        self.name, self.obj, self.kind = name, obj, kind
        self.n_cls, self.features = n_cls, features

    def proba(self, X):
        if self.kind == 'lgb':
            return self.obj.predict(X).astype(np.float32)
        return self.obj.predict(
            xgb.DMatrix(X, feature_names=list(self.features))
        ).reshape(-1, self.n_cls).astype(np.float32)

    def margin(self, X):
        if self.kind == 'lgb':
            return self.obj.predict(X, raw_score=True).astype(np.float64
                                                              ).reshape(-1, self.n_cls)
        return self.obj.predict(
            xgb.DMatrix(X, feature_names=list(self.features)),
            output_margin=True).reshape(-1, self.n_cls).astype(np.float64)


def _fit(kind, X, y, w, Xes, yes, wes, n_cls, feats):
    t0 = time.perf_counter()
    if kind == 'LightGBM':
        p = {**CFG['lgbm'], 'num_class': n_cls}
        dt = lgb.Dataset(X, y, weight=w, feature_name=list(feats), free_raw_data=True)
        dv = lgb.Dataset(Xes, yes, weight=wes, reference=dt,
                         feature_name=list(feats), free_raw_data=True)
        b = lgb.train(p, dt, CFG['rounds'], valid_sets=[dv], valid_names=['es'],
                      callbacks=[lgb.early_stopping(CFG['early_stop'], verbose=False),
                                 lgb.log_evaluation(-1)])
        m = MultiModel(kind, b, 'lgb', n_cls, feats)
        info = {'best_iteration': int(b.best_iteration)}
    else:
        p = {**CFG['xgb'], 'num_class': n_cls}
        dx = xgb.DMatrix(X, y, weight=w, feature_names=list(feats))
        dv = xgb.DMatrix(Xes, yes, weight=wes, feature_names=list(feats))
        b = xgb.train(p, dx, CFG['rounds'], evals=[(dv, 'es')],
                      early_stopping_rounds=CFG['early_stop'], verbose_eval=False)
        bi = int(b.best_iteration)
        b = b[: bi + 1]
        m = MultiModel(kind, b, 'xgb', n_cls, feats)
        info = {'best_iteration': bi, 'trees_kept': bi + 1}
    info.update({'train_rows': int(len(y)),
                 'fit_seconds': round(time.perf_counter() - t0, 1)})
    return m, info


# =============================================================================
#  PIPELINE
# =============================================================================
class Layer3:

    def __init__(self, l1_dir, l2_dir, out_dir):
        self.l1, self.l2 = Path(l1_dir).resolve(), Path(l2_dir).resolve()
        self.out = _mkdir(out_dir).resolve()
        self.figs = _mkdir(self.out / 'figures')
        self.R = {'version': CFG['version'], 'config': CFG,
                  'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                  'environment': _hardware()}
        plt.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42,
                             'font.family': 'serif',
                             'font.serif': ['Times New Roman', 'Nimbus Roman',
                                            'STIXGeneral', 'DejaVu Serif'],
                             'mathtext.fontset': 'stix', 'font.size': 8,
                             'axes.titlesize': 8.5, 'axes.labelsize': 8,
                             'legend.fontsize': 6.8, 'xtick.labelsize': 7,
                             'ytick.labelsize': 7, 'axes.linewidth': 0.7,
                             'savefig.dpi': 600})

    # ------------------------------------------------------------------ P0
    def p0_contract(self):
        print('[P0] Contracts')
        self.m1 = _read_json(self.l1 / 'manifest.json')
        self.m2 = _read_json(self.l2 / 'manifest_l2.json')

        # L1 schema version
        l1_schema = self.m1.get('l2_contract', {}).get('schema_version')
        if l1_schema is None:
            raise RuntimeError('L1 manifest has no l2_contract.schema_version '
                               '— re-run L1 R8.0')
        if l1_schema < CFG['required_schema']:
            raise RuntimeError(f'Layer 1 schema {l1_schema} < '
                               f'{CFG["required_schema"]}: run L1 R8.0')

        # L2 contract — the exact keys L3 consumes
        need = ('l3_l4_contract', 'best_model', 'l2_contract', 'metrics')
        missing = [k for k in need if k not in self.m2]
        if missing:
            raise RuntimeError(
                f'Layer-2 manifest missing required keys: {missing}\n'
                f'  present keys: {sorted(self.m2.keys())}\n'
                '  re-run Layer 2 R8.0')

        c = self.m2['l3_l4_contract']
        for k in ('features', 'model', 'cal', 'threshold_json', 'model_kind'):
            if k not in c:
                raise RuntimeError(
                    f'l3_l4_contract missing {k!r}: re-run Layer 2 R8.0')
        self.features = list(c['features'])
        if not isinstance(self.m2['best_model'], str):
            raise TypeError('L2 best_model must be a string')

        self.l2_name = self.m2['best_model']
        self.l2_kind = c['model_kind']
        self.l2_model = _load_pickle(_resolve(c['model'], self.l2))
        self.l2_cal = _load_pickle(_resolve(c['cal'], self.l2))
        th = _read_json(_resolve(c['threshold_json'], self.l2))
        if self.l2_name not in th.get('deployed_tau', {}):
            raise RuntimeError(
                f"L2 thresholds.json has no deployed_tau['{self.l2_name}']\n"
                f"  deployed_tau = {th.get('deployed_tau')}\n"
                '  re-run Layer 2 R8.0')
        self.l2_tau = float(th['deployed_tau'][self.l2_name])

        self.paths = {k: _resolve(v, self.l1)
                      for k, v in self.m1['l2_input_paths'].items()}
        reserved = set(self.m2['l2_contract']['reserved_columns_not_features'])
        if set(self.features) & reserved:
            raise RuntimeError(
                f'L2 features overlap reserved columns: '
                f'{sorted(set(self.features) & reserved)}')

        self.has_ts = '_ts' in pq.read_schema(self.paths['test_known']).names
        self.reject = CFG['reject_class']

        # ── C3: L2 test FPR reference is MANDATORY ────────────────────────
        self.test_fpr_ref = None
        try:
            self.test_fpr_ref = float(
                self.m2['metrics'][self.l2_name]['full']['fpr'])
        except (KeyError, TypeError) as e:
            raise RuntimeError(
                f"L2 manifest lacks metrics['{self.l2_name}']['full']['fpr'] "
                f"({type(e).__name__})\n"
                f"  metrics keys: {list(self.m2.get('metrics', {}).keys())}\n"
                '  re-run Layer 2 R8.0; the cascaded-FPR block cannot run '
                'without this reference.')
        if not (0.0 <= self.test_fpr_ref <= 1.0):
            raise RuntimeError(
                f'L2 test FPR reference is out of [0,1]: {self.test_fpr_ref}')

        print(f'  L2 test FPR reference: {self.test_fpr_ref:.6f}')
        print(f'  L1 {self.m1.get("version")} | L2 {self.m2.get("version")}')
        print(f'  gate {self.l2_name} ({self.l2_kind}) tau={self.l2_tau:.4f} | '
              f'{len(self.features)} features | reject class: '
              f'{"on" if CFG["train_reject"] else "OFF (attack-only ablation)"}')

    def _l2_score(self, X):
        if self.l2_kind == 'lgb':
            p = self.l2_model.predict(X)
        elif self.l2_kind == 'xgb':
            p = np.asarray(self.l2_model.predict(
                xgb.DMatrix(X, feature_names=list(self.features)))).ravel()
        else:
            p = self.l2_model.predict_proba(X)[:, 1]
        return self.l2_cal.predict_proba(np.asarray(p).reshape(-1, 1))[:, 1]

    # ------------------------------------------------------------------ P1
    def _routed(self, path, want_extra=()):
        cols = self.features + ['Target'] + list(want_extra)
        Xs, ys, fl, ex = [], [], [], {k: [] for k in want_extra}
        n_seen = n_norm_seen = 0
        for df in _iter(path, cols):
            n_seen += len(df)
            X = df[self.features].to_numpy(np.float32)
            is_norm = (df['Target'].values == self.reject)
            n_norm_seen += int(is_norm.sum())
            flag = self._l2_score(X) >= self.l2_tau
            keep = (~is_norm) | (flag if CFG['train_reject'] else False)
            if keep.any():
                Xs.append(X[keep])
                ys.append(df['Target'].values[keep])
                fl.append(flag[keep])
                for k in want_extra:
                    ex[k].append(df[k].values[keep])
            del df, X
            _gc()
        X = (np.vstack(Xs) if Xs else
             np.zeros((0, len(self.features)), np.float32))
        y = np.concatenate(ys) if ys else np.array([], dtype=object)
        extra = {k: (np.concatenate(v) if v else np.array([])) for k, v in ex.items()}
        extra['_l2_flag'] = np.concatenate(fl) if fl else np.zeros(0, bool)
        del Xs, ys, fl
        _gc()
        return X, y, extra, {'rows_scanned': n_seen, 'normal_scanned': n_norm_seen}

    def p1_pool(self):
        print('[P1] Routed training pool (attacks + OUT-OF-SAMPLE L2 false positives)')
        X, y, ex_tr, diag = self._routed(self.paths['train'])
        self._tr_l2_flag = ex_tr['_l2_flag']

        if CFG['train_reject']:
            # The deployed L2 model was FITTED on the training rows, so its
            # false-positive rate there is in-sample and far below deployment.
            # Reject-class rows therefore come from a held-out slice of VALIDATION,
            # which the gate never trained on, and that slice is excluded from L3
            # calibration and threshold selection.
            Xv, yv, exv, diag_val = self._routed(self.paths['val'])
            fv = exv['_l2_flag'].astype(bool)
            rng_v = np.random.RandomState(SEED + 3)
            u = rng_v.random_sample(len(yv))
            a, b, _c = CFG['val_split']
            is_fp = (yv == self.reject) & fv
            pool_m = is_fp & (u < a)
            self._val_cal_m = (~is_fp & (u < 0.5)) | (is_fp & (u >= a) & (u < a + b))
            self._val_thr_m = ~(pool_m | self._val_cal_m)
            self._Xv, self._yv, self._fv = Xv, yv, fv
            keep_tr = y != self.reject            # drop in-sample FPs
            self._val_fp_X = Xv[pool_m].copy()
            X = np.vstack([X[keep_tr], Xv[pool_m]])
            y = np.concatenate([y[keep_tr], yv[pool_m]])
            self._tr_l2_flag = np.concatenate(
                [self._tr_l2_flag[keep_tr], fv[pool_m]])
            self._reject_src = {
                'source': 'validation (L2 out-of-sample)',
                'train_in_sample_fp_rows_discarded': int((~keep_tr).sum()),
                'train_in_sample_fp_rate': float((~keep_tr).sum() /
                                                 max(diag['normal_scanned'], 1)),
                'val_fp_rows_used': int(pool_m.sum()),
                'val_fp_rate': float(((yv == self.reject) & fv).sum() /
                                     max(diag_val['normal_scanned'], 1)),
                'val_fp_rate_denominator': ('ALL Normal rows scanned in '
                                            'validation; the routed set keeps '
                                            'only flagged ones'),
                'val_normal_scanned': int(diag_val['normal_scanned']),
                'why': ('the gate fits its own training rows, so in-sample '
                        'false positives are unrepresentative of deployment')}
            if self.test_fpr_ref:
                ratio = self._reject_src['val_fp_rate'] / max(self.test_fpr_ref, 1e-9)
                self._reject_src['l2_test_fpr_reference'] = self.test_fpr_ref
                self._reject_src['val_vs_test_fpr_ratio'] = float(ratio)
                if not 0.3 <= ratio <= 3.0:
                    print(f'  [WARN] val FP rate '
                          f'{self._reject_src["val_fp_rate"]:.4f} vs L2 test FPR '
                          f'{self.test_fpr_ref:.4f} (ratio {ratio:.2f}): the '
                          'reject class may not represent the deployment population')
            print(f'  reject rows: {int(pool_m.sum()):,} from validation '
                  f'(val FP rate {self._reject_src["val_fp_rate"]:.4f}) — '
                  f'{int((~keep_tr).sum()):,} in-sample train FPs discarded '
                  f'(rate {self._reject_src["train_in_sample_fp_rate"]:.4f})')
        else:
            self._reject_src = {'source': 'disabled (attack-only ablation)'}
            self._Xv = None

        cnt = Counter(y.tolist())
        drop = [c for c, n in cnt.items() if n < CFG['min_class_rows']]
        if drop:
            keep = ~np.isin(y, drop)
            X, y = X[keep], y[keep]
            self._tr_l2_flag = self._tr_l2_flag[keep]
            print(f'  dropped classes with <{CFG["min_class_rows"]} rows: {drop}')
            if self.reject in drop:
                print(f'  WARNING: the reject class had <{CFG["min_class_rows"]} '
                      'rows and was dropped. Layer 3 cannot return L2 false '
                      'positives to Normal; the cascaded-FPR result is '
                      'unavailable for this run.')
                self._reject_src['dropped_too_few_rows'] = True

        self.le = LabelEncoder()
        yi = self.le.fit_transform(y).astype(np.int32)
        self.classes_ = list(self.le.classes_)
        self.k = len(self.classes_)
        self.reject_idx = (self.classes_.index(self.reject)
                           if self.reject in self.classes_ else -1)
        n_fp = int((y == self.reject).sum())
        counts = np.bincount(yi, minlength=self.k)
        w_cls = len(yi) / (self.k * np.maximum(counts, 1))
        self.class_weights = {c: float(w_cls[i]) for i, c in enumerate(self.classes_)}
        self.X_tr, self.y_tr, self.w_tr = X, yi, w_cls[yi].astype(np.float32)
        self.R['pool'] = {
            'rows': int(len(yi)), 'classes': self.classes_,
            'per_class_counts': {c: int(counts[i]) for i, c in enumerate(self.classes_)},
            'l2_false_positive_rows': n_fp,
            'train_rows_scanned': diag['rows_scanned'],
            'train_normal_scanned': diag['normal_scanned'],
            'l2_fp_rate_on_train_normal': float(n_fp / max(diag['normal_scanned'], 1)),
            'class_weights': self.class_weights,
            'l2_missed_attack_rows_in_pool': int(
                (~self._tr_l2_flag.astype(bool) &
                 (y != self.reject)).sum()),
            'reject_class_source': self._reject_src,
            'note': ('TRAINING keeps every attack row (more supervision for '
                     'family separation) plus the Normal rows the deployed '
                     'gate flags. EVALUATION additionally reports the '
                     'deployment-true subset (L2-flagged rows only).'),
        }
        print(f'  routed pool {len(yi):,} rows | {self.k} classes | '
              f'L2 false positives {n_fp:,} '
              f'({100 * n_fp / max(diag["normal_scanned"], 1):.2f}% of train Normal)')

    # ------------------------------------------------------------------ P2
    def p2_train(self):
        print('[P2] Train (same rows, same weights, same ES split)')
        rng = np.random.RandomState(SEED)
        perm = rng.permutation(len(self.y_tr))
        k_es = max(1, int(CFG['es_frac'] * len(self.y_tr)))
        es, fit = np.sort(perm[:k_es]), np.sort(perm[k_es:])
        self.models, info = {}, {}
        for kind in ('LightGBM', 'XGBoost'):
            m, i = _fit(kind, self.X_tr[fit], self.y_tr[fit], self.w_tr[fit],
                        self.X_tr[es], self.y_tr[es], self.w_tr[es],
                        self.k, self.features)
            self.models[kind] = m
            info[kind] = i
            print(f'  {kind:<10} {i["fit_seconds"]}s | best_iter={i["best_iteration"]}')
        self.R['training'] = {'models': info, 'es_rows': int(k_es),
                              'fit_rows': int(len(fit)),
                              'equal_comparison': 'identical rows, weights and ES set'}
        del self.w_tr
        _gc()

    # ------------------------------------------------------------------ P3
    def p3_calibrate_threshold(self):
        print('[P3] Per-class Platt on val-CAL half; thresholds on val-THR half')
        if getattr(self, '_Xv', None) is not None:
            X, y = self._Xv, self._yv
            keep = np.isin(y, self.classes_)
            cal_m = self._val_cal_m[keep]
            thr_m = self._val_thr_m[keep]
            X, y = X[keep], y[keep]
            yi = self.le.transform(y).astype(np.int32)
            cal_i, thr_i = np.flatnonzero(cal_m), np.flatnonzero(thr_m)
        else:
            X, y, _, _ = self._routed(self.paths['val'])
            keep = np.isin(y, self.classes_)
            X, y = X[keep], y[keep]
            yi = self.le.transform(y).astype(np.int32)
            rng = np.random.RandomState(SEED + 1)
            cal, thr = [], []
            for c in range(self.k):
                ii = np.flatnonzero(yi == c)
                rng.shuffle(ii)
                h = len(ii) // 2
                cal.append(ii[:h])
                thr.append(ii[h:])
            cal_i = np.sort(np.concatenate(cal))
            thr_i = np.sort(np.concatenate(thr))

        self.cals, self.thresholds = {}, {}
        raw_fallback = {}
        # C6: minimum 50 positives AND 50 negatives in the calibration half
        MIN_POS = 50
        for nm, mdl in self.models.items():
            p = mdl.proba(X[cal_i])
            cs, fb = [], []
            for c in range(self.k):
                yc = (yi[cal_i] == c).astype(np.int8)
                n_pos, n_neg = int(yc.sum()), int((1 - yc).sum())
                if n_pos >= MIN_POS and n_neg >= MIN_POS:
                    cs.append(_platt(p[:, c], yc))
                else:
                    cs.append(None)
                    fb.append({'class': self.classes_[c],
                               'n_pos': n_pos, 'n_neg': n_neg})
            self.cals[nm] = cs
            raw_fallback[nm] = fb
            pt = self._cal(mdl.proba(X[thr_i]), nm)
            yt = yi[thr_i]
            pred, conf = pt.argmax(1), pt.max(1)
            order = np.argsort(-conf)
            prec = np.cumsum((pred == yt)[order]) / np.arange(1, len(order) + 1)
            ok = np.flatnonzero(prec >= CFG['l5_precision_target'])
            tau_l5 = float(conf[order[ok[-1]]]) if len(ok) else float(conf.max())
            energy = -logsumexp(mdl.margin(X[thr_i]), axis=1)
            tau_e = float(np.quantile(energy, 1 - CFG['ood_val_flag_rate']))
            tau_c = float(np.quantile(conf, CFG['ood_val_flag_rate']))
            self.thresholds[nm] = {'l5': tau_l5, 'ood_conf': tau_c, 'ood_energy': tau_e}
            print(f'  {nm:<10} tau_L5={tau_l5:.4f} | tau_OOD(conf)={tau_c:.4f} | '
                  f'tau_OOD(energy)={tau_e:.4f}'
                  + (f' | raw-fallback: {len(fb)} classes'
                     if fb else ''))

        self._cal_prior_normal = (float((yi[cal_i] == self.reject_idx).mean())
                                  if self.reject_idx >= 0 and len(cal_i) else None)
        self.R['thresholds'] = {
            'per_model': self.thresholds,
            'rule_l5': f'largest tau with top-1 precision >= '
                       f'{CFG["l5_precision_target"]} (validation threshold half)',
            'rule_ood': f'flags {CFG["ood_val_flag_rate"]:.0%} of KNOWN '
                        'validation flows; fitted on validation only, never '
                        'on zero-day',
            'val_cal_rows': int(len(cal_i)), 'val_thr_rows': int(len(thr_i)),
            'calibration_min_pos_neg': MIN_POS,
            'calibration_raw_fallback': raw_fallback,
            'calibration_rule': ('per-class Platt on logit(p_cal) fitted on the '
                                 'calibration half only; classes with < '
                                 f'{MIN_POS} positives OR < {MIN_POS} negatives '
                                 'keep their raw probability and are listed '
                                 'under calibration_raw_fallback')}
        self._Xv = None
        del X, y, yi
        _gc()

    def _cal(self, p, name):
        q = p.copy()
        for c, lr in enumerate(self.cals[name]):
            if lr is not None:
                q[:, c] = _apply_platt(lr, p[:, c])
        s = q.sum(1, keepdims=True)
        return (q / np.where(s == 0, 1, s)).astype(np.float32)

    # ------------------------------------------------------------------ P4
    def p4_evaluate(self):
        print('[P4] Test evaluation: routed view, attack-only view, twin-free')
        extra = ('_dup_in_train',) + (('_ts',) if self.has_ts else ())
        X, y, ex, diag_te = self._routed(self.paths['test_known'], extra)
        self._test_normal_scanned = int(diag_te['normal_scanned'])
        keep = np.isin(y, self.classes_)
        X, y = X[keep], y[keep]
        ex = {k: v[keep] for k, v in ex.items()}
        twin = ex['_dup_in_train'].astype(np.int8)
        yi = self.le.transform(y).astype(np.int32)
        self._y, self._twin = yi, twin
        if self.has_ts:
            ts = ex['_ts'].astype(np.int64)
            self._ts_test = ts
            self._bid = (ts // (CFG['block_minutes'] * 60 * 10**9)).astype(np.int64)
            self._bid -= self._bid.min()
            self._bsrc = (f'{CFG["block_minutes"]}-min blocks from L1 _ts '
                          f'({self._bid.max() + 1} blocks)')
        else:
            self._bid = (np.arange(len(yi)) // CFG['chunk']).astype(np.int64)
            self._bsrc = 'row-chunk fallback (no _ts)'

        atk = (yi != self.reject_idx) if self.reject_idx >= 0 else np.ones(len(yi), bool)
        self._atk = atk
        l2f = ex['_l2_flag'].astype(bool)
        self._l2f = l2f

        self._probs = {}
        for nm, mdl in self.models.items():
            p = self._cal(mdl.proba(X), nm)
            self._probs[nm] = p
            self.R.setdefault('test', {})[nm] = {
                'routed': self._mm(yi, p),
                'routed_twin_free': self._mm(yi[twin == 0], p[twin == 0]),
                'deployment_routed': self._mm(yi[l2f], p[l2f]),
                'deployment_routed_twin_free': self._mm(yi[l2f & (twin == 0)],
                                                        p[l2f & (twin == 0)]),
                'attack_only': self._mm(yi[atk], p[atk]),
                'attack_only_l2_flagged': self._mm(yi[atk & l2f], p[atk & l2f]),
                'attack_only_l2_missed': (self._mm(yi[atk & ~l2f], p[atk & ~l2f])
                                          if (atk & ~l2f).any() else None),
                'reject_recovery': self._reject_stats(yi, p),
            }
            r = self.R['test'][nm]
            print(f'  {nm:<10} deployment-routed '
                  f'{r["deployment_routed"]["macro_f1"]:.4f} | twin-free '
                  f'{r["deployment_routed_twin_free"]["macro_f1"]:.4f} | '
                  f'attack-only {r["attack_only"]["macro_f1"]:.4f}')

        self.best = max(
            self.models,
            key=lambda n: self.R['test'][n]['deployment_routed_twin_free']['macro_f1'])
        self.R['selection'] = {
            'best': self.best,
            'metric': 'twin-free macro-F1 on the DEPLOYMENT-routed test '
                      'population (L2-flagged rows only)',
            'note': 'selection never uses the zero-day partition'}
        print(f'  BEST (twin-free macro-F1): {self.best}')
        self._X_test = X
        self._reject_shift(X, atk)

        # ── C4: cascaded FPR — base-rate assertion + measured recovery ─────
        rr = self.R['test'][self.best]['reject_recovery']
        if rr.get('enabled') and self.test_fpr_ref is not None:
            # Base-rate assertion: the L2 test FPR the reject class was
            # designed against and the L2 test FPR the routed population
            # actually reflects must be the same number.
            # T1: denominator = ALL Normal rows scanned in test_known, not the
            # routed subset (which holds only the L2-flagged Normal rows).
            l2_fpr_routed = float((~self._atk & self._l2f).sum() /
                                  max(self._test_normal_scanned, 1))
            # C8 assertion only meaningful when this run reaches L3 via the
            # same L2 gate the FPR reference came from. In practice L2's
            # routed FPR equals its manifest FPR because _routed calls the
            # same model + calibrator + tau. Assert to 1e-6.
            if abs(l2_fpr_routed - self.test_fpr_ref) > 1e-4:
                print(f'  [WARN] L2 routed-FPR {l2_fpr_routed:.6f} != manifest '
                      f'FPR {self.test_fpr_ref:.6f}; using manifest value for '
                      'the cascaded computation')
            rec = rr.get('recovered_as_normal') or 0.0
            cascaded_fpr = float(self.test_fpr_ref * (1 - rec))
            self.R['cascaded_fpr'] = {
                'l2_fpr_on_test_manifest': self.test_fpr_ref,
                'l2_fpr_on_test_measured_routed': l2_fpr_routed,
                'share_of_l2_false_positives_returned_to_normal': float(rec),
                'deployed_fpr_after_layer3': cascaded_fpr,
                'l2_false_positives': rr.get('l2_false_positives'),
                'true_attacks_wrongly_rejected_live':
                    rr.get('true_attacks_wrongly_rejected'),
                'reading': ('the alarm rate a deployment sees is the Layer-2 '
                            'FPR reduced by what Layer 3 returns to Normal, '
                            'at the cost of the attacks it wrongly rejects')}
            print(f'  [FINDING] cascaded FPR: L2 '
                  f'{100 * self.test_fpr_ref:.3f}% -> '
                  f'{100 * cascaded_fpr:.3f}% after Layer 3 (recovers '
                  f'{100 * rec:.2f}% of {rr.get("l2_false_positives"):,} '
                  'false positives)')

        self.R['test_meta'] = {'rows': int(len(yi)),
                               'twin_free_rows': int((twin == 0).sum()),
                               'attack_rows': int(atk.sum()),
                               'l2_false_positive_rows': int((~atk).sum()),
                               'deployment_routed_rows': int(l2f.sum()),
                               'attacks_missed_by_l2_excluded_live':
                                   int((atk & ~l2f).sum()),
                               'block_source': self._bsrc}

    def _mm(self, y, p):
        pred = p.argmax(1)
        labels = list(range(self.k))
        f1 = f1_score(y, pred, average=None, labels=labels, zero_division=0)
        pr = precision_score(y, pred, average=None, labels=labels, zero_division=0)
        rc = recall_score(y, pred, average=None, labels=labels, zero_division=0)
        sup = np.bincount(y, minlength=self.k)
        present = sup > 0
        return {'macro_f1': float(np.mean(f1[present])) if present.any() else 0.0,
                'weighted_f1': float(f1_score(y, pred, average='weighted',
                                              zero_division=0)),
                'accuracy': float((y == pred).mean()) if len(y) else 0.0,
                'per_class_f1': f1.tolist(), 'per_class_precision': pr.tolist(),
                'per_class_recall': rc.tolist(), 'support': sup.tolist()}

    def _reject_stats(self, y, p):
        if self.reject_idx < 0:
            return {'enabled': False}
        fp = y == self.reject_idx
        pred = p.argmax(1)
        rec = float((pred[fp] == self.reject_idx).mean()) if fp.any() else None
        leak = float((pred[~fp] == self.reject_idx).mean()) if (~fp).any() else None
        return {'enabled': True, 'l2_false_positives': int(fp.sum()),
                'recovered_as_normal': rec,
                'true_attacks_wrongly_rejected': leak,
                'note': ('recovered L2 false positives never reach the SOAR '
                         'layer: this is the cascaded-FPR reduction')}

    # ------------------------------------------------------------------ P5
    def _reject_shift(self, X_test, atk):
        # C5: raise the guard to 200 on both sides
        vm = getattr(self, '_val_fp_X', None)
        n_val = len(vm) if vm is not None else 0
        n_test = int((~atk).sum())
        if n_val < 200 or n_test < 200:
            self.R['reject_class_shift'] = {
                'status': 'NOT MEASURED',
                'n_val_false_positives_trained_on': n_val,
                'n_test_false_positives_faced': n_test,
                'min_required': 200,
                'reason': ('two-sample per-feature KS is unstable below 200 '
                           'points per side; no finding is reported'),
            }
            print(f'  reject-class shift: NOT MEASURED (val FP={n_val}, '
                  f'test FP={n_test}; need >=200 each)')
            return
        te = X_test[~atk]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ks = {f: float(ks_2samp(vm[:, i], te[:, i])[0])
                  for i, f in enumerate(self.features)}
        v = np.array(list(ks.values()))
        self.R['reject_class_shift'] = {
            'n_val_false_positives_trained_on': int(n_val),
            'n_test_false_positives_faced': int(n_test),
            'population_ratio': float(n_test / max(n_val, 1)),
            'ks_max': float(v.max()), 'ks_median': float(np.median(v)),
            'n_features_ks_gt_0.10': int((v > 0.10).sum()),
            'top10_drifted': dict(sorted(ks.items(), key=lambda x: -x[1])[:10]),
            'reading': ('the reject class is trained on the false positives '
                        'the gate makes at its VALIDATION rate and must '
                        'recognise those it makes at its TEST rate; a large '
                        'KS here is the stated cause of a low recovery')}
        print(f'  [FINDING] reject-class shift: trained on {n_val:,} val FPs, '
              f'faces {n_test:,} test FPs ({n_test / max(n_val, 1):.1f}x); '
              f'KS max {v.max():.4f} median {np.median(v):.4f}; '
              f'{int((v > 0.10).sum())}/{len(v)} features above 0.10')

    def p5_statistics(self):
        print('[P5] Block bootstrap + McNemar + DeLong + paired delta-CI')
        if not self.has_ts:
            print('  [WARN] Layer 1 did not provide _ts: block bootstrap falls '
                  'back to row-chunk blocks, which are NOT time blocks. Every '
                  'CI below is recorded with block_source naming the fallback.')
        sub = self._l2f
        y = self._y[sub]
        present = np.bincount(y, minlength=self.k) > 0
        bid = self._bid[sub]
        bid = np.unique(bid, return_inverse=True)[1]
        rng = np.random.RandomState(SEED + 7)
        nb = int(bid.max()) + 1 if len(bid) else 1
        st = {'population': 'deployment-routed (L2-flagged test rows)',
              'rows': int(sub.sum()), 'block_source': self._bsrc, 'n_blocks': nb,
              'classes_present': [self.classes_[i] for i in range(self.k) if present[i]],
              'macro_definition': 'mean F1 over PRESENT classes (same mask every resample)'}
        cms = {}
        for nm in self.models:
            pred = self._probs[nm][sub].argmax(1)
            cms[nm] = _block_cms(y, pred, bid, self.k)
            mf = [_macro_f1_from_cm(cms[nm][rng.randint(0, nb, nb)].sum(0), present)[0]
                  for _ in range(CFG['bootstrap_n'])]
            per = np.stack([_macro_f1_from_cm(
                cms[nm][rng.randint(0, nb, nb)].sum(0), present)[1]
                for _ in range(min(300, CFG['bootstrap_n']))])
            iid_cm = np.zeros((len(y), self.k, self.k), np.int8)
            iid_cm[np.arange(len(y)), y, pred] = 1
            iid = [_macro_f1_from_cm(
                iid_cm[rng.randint(0, len(y), len(y))].sum(0), present)[0]
                for _ in range(CFG['bootstrap_iid_n'])]
            del iid_cm
            st[nm] = {'macro_f1_block_ci95': _ci(mf),
                      'macro_f1_iid_ci95': _ci(iid),
                      'per_class_ci95': {self.classes_[i]: _ci(per[:, i])
                                         for i in range(self.k) if present[i]}}
            bc, ic = st[nm]['macro_f1_block_ci95'], st[nm]['macro_f1_iid_ci95']
            print(f'  {nm:<10} macro-F1 block CI [{bc[0]:.4f}, {bc[1]:.4f}] '
                  f'vs iid [{ic[0]:.4f}, {ic[1]:.4f}]')
        a, b = list(self.models)
        ca = self._probs[a][sub].argmax(1) == y
        cb = self._probs[b][sub].argmax(1) == y
        mc = _mcnemar(ca, cb)
        d = [(_macro_f1_from_cm(cms[a][s].sum(0), present)[0]
              - _macro_f1_from_cm(cms[b][s].sum(0), present)[0])
             for s in (rng.randint(0, nb, nb) for _ in range(CFG['bootstrap_n']))]
        dl, ps = {}, []
        for c in range(self.k):
            yc = (y == c).astype(np.int8)
            if yc.sum() < 10 or (1 - yc).sum() < 10:
                continue
            a1, a2, z, pv = _delong_pair(self._probs[a][sub][:, c],
                                         self._probs[b][sub][:, c], yc)
            dl[self.classes_[c]] = {'auc_a': a1, 'auc_b': a2, 'z': z, 'p_raw': pv}
            ps.append(pv)
        for (cls, v), padj in zip(dl.items(), _holm(ps) if ps else []):
            v['p_holm'] = padj
        mc['p_holm'] = mc['p_raw']
        st['pairwise'] = {
            'pair': f'{a} vs {b}', 'mcnemar': mc, 'delong_per_class': dl,
            'delta_macro_f1': float(
                self.R['test'][a]['deployment_routed']['macro_f1']
                - self.R['test'][b]['deployment_routed']['macro_f1']),
            'delta_macro_f1_block_ci95': _ci(d),
            'wording': ('similar (delta CI includes 0)'
                        if _ci(d)[0] <= 0 <= _ci(d)[1]
                        else 'differs (CI excludes 0)'),
            'caveat': 'McNemar and DeLong assume independent flows; the '
                      'paired block CI is the primary evidence'}
        print(f'  {a} vs {b}: b={mc["b"]:,} c={mc["c"]:,} OR={mc["odds_ratio"]:.3f} | '
              f'delta macro-F1 CI '
              f'[{st["pairwise"]["delta_macro_f1_block_ci95"][0]:+.4f}, '
              f'{st["pairwise"]["delta_macro_f1_block_ci95"][1]:+.4f}] '
              f'-> {st["pairwise"]["wording"]}')
        self.R['statistics'] = st
        del cms
        _gc()

    # ------------------------------------------------------------------ P6
    def p5b_threshold_transfer(self):
        print('[P5b] Threshold transfer + OOD gate cost (test)')
        sub, th = self._l2f, self.thresholds[self.best]
        p = self._probs[self.best][sub]
        y = self._y[sub]
        conf, pred = p.max(1), p.argmax(1)
        corr = (pred == y)
        hi = conf >= th['l5']
        prec_test = float(corr[hi].mean()) if hi.any() else None
        order = np.argsort(-conf)
        run_prec = np.cumsum(corr[order]) / np.arange(1, len(order) + 1)
        ok = np.flatnonzero(run_prec >= CFG['l5_precision_target'])
        tau_needed = float(conf[order[ok[-1]]]) if len(ok) else None
        bid = self._bid[sub]
        blk = []
        for b in np.unique(bid):
            m = (bid == b) & hi
            if m.sum() > 50:
                blk.append(float(corr[m].mean()))
        blk = np.sort(np.array(blk)) if blk else np.array([])
        e = -logsumexp(self.models[self.best].margin(self._X_test[sub]), axis=1)
        atk = self._atk[sub]
        gate = {'known_attack_rows': int(atk.sum()),
                'dumped_to_l4_by_confidence':
                    float((conf[atk] < th['ood_conf']).mean()) if atk.any() else None,
                'dumped_to_l4_by_energy':
                    float((e[atk] > th['ood_energy']).mean()) if atk.any() else None,
                'target_on_validation': CFG['ood_val_flag_rate'],
                'reading': ('share of KNOWN attacks the gate sends to Layer 4 at '
                            'test time; the gate was fitted to flag %.0f%% of '
                            'known validation flows'
                            % (100 * CFG['ood_val_flag_rate']))}
        self.R['threshold_transfer'] = {
            'tau_l5': {
                'target_top1_precision': CFG['l5_precision_target'],
                'tau_deployed': th['l5'],
                'achieved_precision_on_test': prec_test,
                'rows_above_tau_l5': int(hi.sum()),
                'share_routed_to_l5_only': float(hi.mean()) if len(hi) else None,
                'tau_that_would_hold_target_on_test': tau_needed,
                'tau_movement_required': (None if tau_needed is None
                                          else float(tau_needed - th['l5'])),
                'per_block_precision': {
                    'n_blocks_scored': int(len(blk)),
                    'p50': float(np.percentile(blk, 50)) if len(blk) else None,
                    'p05': float(np.percentile(blk, 5)) if len(blk) else None,
                    'min': float(blk.min()) if len(blk) else None,
                    'n_blocks_below_target':
                        int((blk < CFG['l5_precision_target']).sum())
                        if len(blk) else 0},
                'finding': ('a top-1 precision target fitted on validation is '
                            'a validation-time guarantee; the test value and '
                            'the per-block spread are what the SOAR layer '
                            'inherits')},
            'ood_gate_cost_known_attack_l2_flagged': gate,
            'prior_shift': {
                'calibration_slice_prior_normal': getattr(self, '_cal_prior_normal', None),
                'deployment_prior_normal':
                    (float((self._y[self._l2f] == self.reject_idx).mean())
                     if self.reject_idx >= 0 else None),
                'reading': ('per-class Platt is prior-sensitive; a gap '
                            'between these two numbers is a stated cause of '
                            'the tau movement above')}}
        print(f'  [FINDING] tau_L5 {th["l5"]:.4f}: target '
              f'{CFG["l5_precision_target"]:.2f} -> test '
              f'{"n/a" if prec_test is None else round(prec_test, 4)}'
              + ('' if tau_needed is None
                 else f'; would need {tau_needed - th["l5"]:+.4f}')
              + (f'; {int((blk < CFG["l5_precision_target"]).sum())}/{len(blk)} '
                 'blocks below target' if len(blk) else ''))
        if atk.any():
            print(f'  [FINDING] OOD gate dumps '
                  f'{100 * gate["dumped_to_l4_by_confidence"]:.2f}% '
                  f'(confidence) / {100 * gate["dumped_to_l4_by_energy"]:.2f}% '
                  '(energy) of KNOWN attacks to Layer 4')

    def p6_ood_and_routing(self):
        print('[P6] Zero-day routing coverage + OOD statistic comparison')
        zpath = self.l2 / 'l2_scores_zeroday.parquet'
        if not zpath.exists():
            self.R['routing'] = {'status': 'L2 zero-day score export missing'}
            print('  SKIPPED: l2_scores_zeroday.parquet not found')
            return
        z = pd.read_parquet(zpath)
        flag = z['flag'].to_numpy(bool)
        ytxt = z['Target'].to_numpy()
        Xz = []
        for df in _iter(self.paths['test_zeroday'], self.features):
            Xz.append(df[self.features].to_numpy(np.float32))
        Xz = np.vstack(Xz)
        if len(Xz) != len(z):
            raise RuntimeError(f'row mismatch: L2 export {len(z)} vs L1 {len(Xz)}')
        mdl = self.models[self.best]
        th = self.thresholds[self.best]
        n = len(z)
        n_att = int(flag.sum())
        n_norm = n - n_att
        conf = np.zeros(n)
        energy = np.zeros(n)
        if n_att:
            p = self._cal(mdl.proba(Xz[flag]), self.best)
            conf[flag] = p.max(1)
            energy[flag] = -logsumexp(mdl.margin(Xz[flag]), axis=1)
        ood_conf = flag & (conf < th['ood_conf'])
        ood_en = flag & (energy > th['ood_energy'])
        l5_only_conf = flag & (conf >= th['l5'])
        cov_conf = (n_norm + int(ood_conf.sum())) / max(n, 1)
        cov_en = (n_norm + int(ood_en.sum())) / max(n, 1)
        ka = self._atk & self._l2f
        p_known = self._probs[self.best][ka]
        e_known = -logsumexp(self.models[self.best].margin(self._X_test[ka]), axis=1)
        lab = np.r_[np.zeros(int(ka.sum())), np.ones(int(flag.sum()))]
        auc_conf = (float(roc_auc_score(lab, np.r_[-p_known.max(1), -conf[flag]]))
                    if flag.any() and ka.any() else None)
        auc_en = (float(roc_auc_score(lab, np.r_[e_known, energy[flag]]))
                  if flag.any() and ka.any() else None)
        per = {}
        for c in np.unique(ytxt):
            m = ytxt == c
            per[str(c)] = {
                'n': int(m.sum()),
                'l2_flag_rate': float(flag[m].mean()),
                'l4_coverage_conf': float(((~flag[m]) | ood_conf[m]).mean()),
                'l4_coverage_energy': float(((~flag[m]) | ood_en[m]).mean())}
        self.R['routing'] = {
            'n_zeroday': n,
            'X_l2_flagged_fraction': float(n_att / max(n, 1)),
            'Y_l3_ood_fraction_conf':
                float(ood_conf.sum() / max(n_att, 1)),
            'Y_l3_ood_fraction_energy':
                float(ood_en.sum() / max(n_att, 1)),
            'l4_coverage_conf': float(cov_conf),
            'l4_coverage_energy': float(cov_en),
            'formula': '(1 - X) + X*Y',
            'zero_day_confident_bypassing_l4': int(l5_only_conf.sum()),
            'zero_day_mean_confidence':
                float(conf[flag].mean()) if n_att else None,
            'ood_auroc_max_softmax': auc_conf,
            'ood_auroc_energy': auc_en,
            'ood_auroc_population': ('L2-flagged known attacks vs L2-flagged '
                                     'zero-day: the only flows Layer 3 scores '
                                     'in deployment'),
            'recommended_ood_statistic':
                ('energy' if (auc_en or 0) > (auc_conf or 0) else 'max_softmax'),
            'per_class': per,
            'note': ('max-softmax confidence is reported with its AUROC '
                     'against the energy score so Layer 4 uses the statistic '
                     'the data supports, not the one assumed.')}
        print(f'  zero-day {n:,} | X={n_att / max(n, 1):.4f} | '
              f'L4 coverage conf={cov_conf:.4f} energy={cov_en:.4f}')
        print('  OOD AUROC: max-softmax='
              + (f'{auc_conf:.4f}' if auc_conf else 'n/a')
              + ' energy=' + (f'{auc_en:.4f}' if auc_en else 'n/a')
              + f' -> use {self.R["routing"]["recommended_ood_statistic"]}')

        # ── C4: measured L2∪L4 union on the same rows ─────────────────────
        # L2's own per-row flag for zero-day rows is `flag`. L4's own per-row
        # decision is `ood_conf | ood_en` on the rows L4 actually sees.
        # The union is measured, not a product.
        flag_l2 = flag
        flag_l4 = ood_conf | ood_en
        union = flag_l2 | (flag_l4 & flag_l2)   # L4 only fires on L2-flagged rows
        self.R['combined_zeroday'] = {
            'n_zeroday_rows': n,
            'l2_flag_rate': float(flag_l2.mean()),
            'l4_fires_on_l2_flagged_rate': float(flag_l4.mean()),
            'measured_union_rate': float(union.mean()),
            'measured_union_count': int(union.sum()),
            'formula': 'measured (flag_l2 | flag_l4).mean() on the same rows',
            'reading': ('the union is measured on L2-flagged zero-day rows; '
                        'rows L2 misses entirely are counted as not detected, '
                        'which is what a deployment would see')}
        print(f'  [FINDING] measured L2∪L4 zero-day DR: '
              f'{union.mean():.4f} ({union.sum():,}/{n:,})')
        del Xz
        _gc()

    # ------------------------------------------------------------------ P7
    def p7_latency(self):
        print('[P7] Latency (batch=1, measured)')
        X = self._X_test[:2000]
        lat = {'hardware': _hardware(), 'unit': 'microseconds per flow',
               'batch_size': 1, 'protocol': f'{CFG["lat_warmup"]} warm-up, '
                                            f'{CFG["lat_calls"]} timed calls'}
        for nm, mdl in self.models.items():
            for i in range(CFG['lat_warmup']):
                mdl.proba(X[i % len(X)][None, :])
            t = np.empty(CFG['lat_calls'])
            for i in range(CFG['lat_calls']):
                r = X[i % len(X)][None, :]
                t0 = time.perf_counter_ns()
                mdl.proba(r)
                t[i] = (time.perf_counter_ns() - t0) / 1e3
            lat[nm] = {'p50': float(np.percentile(t, 50)),
                       'p95': float(np.percentile(t, 95)),
                       'p99': float(np.percentile(t, 99)),
                       'mean': float(t.mean())}
            if nm == self.best:
                self._lat_samples = t.copy()
            print(f'  {nm:<10} P50 {lat[nm]["p50"]:.1f} '
                  f'P95 {lat[nm]["p95"]:.1f} P99 {lat[nm]["p99"]:.1f} us')
        l2p95 = None
        try:
            l2p95 = float(self.m2['latency']['model_only_batch1']['p95'])
        except (KeyError, TypeError):
            pass
        lat['l2_plus_l3_p95_sum'] = (l2p95 + lat[self.best]['p95']) if l2p95 else None
        lat['sum_caveat'] = ('a sum of two measured P95 values is an upper '
                             'bound, not a measured P95 of the joint path')
        self.R['latency'] = lat

    # ------------------------------------------------------------------ P8
    def p8_importance_ablation(self):
        print('[P8] Importance + same-sample ablation')
        m = self.models[self.best]
        if m.kind == 'lgb':
            g = m.obj.feature_importance('gain')
            g = g.sum(0) if g.ndim == 2 else g
            fi = dict(zip(self.features, g.tolist()))
        else:
            sc = m.obj.get_fscore()
            fi = {f: float(sc.get(f, 0.0)) for f in self.features}
        self.importance = dict(sorted(fi.items(), key=lambda x: -x[1]))
        rng = np.random.RandomState(SEED + 8)
        n = min(CFG['ablation_n'], len(self.y_tr))
        idx = np.sort(rng.choice(len(self.y_tr), n, replace=False))
        Xa = self._pool_rows(idx)
        ya = self.y_tr[idx]
        cnt = np.bincount(ya, minlength=self.k)
        wa = (len(ya) / (self.k * np.maximum(cnt, 1)))[ya].astype(np.float32)
        te = np.sort(rng.choice(len(self._y), min(n, len(self._y)), replace=False))
        Xt, yt = self._X_test[te], self._y[te]

        pab = np.random.RandomState(SEED + 12).permutation(len(ya))
        k_ab = max(1, int(CFG['es_frac'] * len(ya)))
        es_ab, fit_ab = np.sort(pab[:k_ab]), np.sort(pab[k_ab:])

        def run(cols):
            ci = [self.features.index(f) for f in cols]
            mm, _ = _fit('LightGBM', Xa[np.ix_(fit_ab, ci)], ya[fit_ab],
                         wa[fit_ab], Xa[np.ix_(es_ab, ci)], ya[es_ab],
                         wa[es_ab], self.k, cols)
            return float(f1_score(yt, mm.proba(Xt[:, ci]).argmax(1),
                                  average='macro', zero_division=0))

        base = run(self.features)
        top = list(self.importance)[:max(CFG['ablation_topk'])]
        abl = {'protocol': f'LightGBM retrained on the same {n:,}-row routed '
                           'sample for the baseline and every variant',
               'baseline_macro_f1': base, 'remove_top_k': {},
               'early_stopping': f'disjoint {CFG["es_froc"] if "es_froc" in CFG else CFG["es_frac"]:.0%} '
                                 'hold-out of the ablation sample, identical '
                                 'for every variant'}
        for k in CFG['ablation_topk']:
            cols = [f for f in self.features if f not in set(top[:k])]
            v = run(cols)
            abl['remove_top_k'][f'top{k}'] = {'macro_f1': v, 'drop': base - v}
            print(f'  remove top-{k:<3} macro-F1 {v:.4f} (drop {base - v:+.4f})')
        self.R['importance_top20'] = dict(list(self.importance.items())[:20])
        self.R['ablation'] = abl
        del Xa, Xt
        _gc()

    def _pool_rows(self, idx):
        return self.X_tr[idx]

    # ------------------------------------------------------------------ P9
    def p9_adversarial(self):
        print('[P9] Robustness in RAW space (identity-guarded)')
        sp = _read_json(self.paths['scaler'])
        mu = np.array([sp['mean'][f] for f in self.features])
        sd = np.array([sp['std'][f] for f in self.features])
        rng = np.random.RandomState(SEED + 9)
        n = min(CFG['adv_n'], len(self._y))
        ii = np.sort(rng.choice(len(self._y), n, replace=False))
        Z = self._X_test[ii].astype(np.float64)
        y = self._y[ii]
        R0 = Z * sd + mu
        rt = float(np.max(np.abs((R0 - mu) / sd - Z)))
        if rt > 1e-3:
            raise RuntimeError(f'raw round trip failed ({rt:.2e})')
        fi = {f: i for i, f in enumerate(self.features)}
        mdl = self.models[self.best]

        def score(R):
            return self._cal(
                mdl.proba(((np.maximum(R, 0) - mu) / sd).astype(np.float32)),
                self.best)

        base = self._mm(y, score(R0.copy()))
        disc = {f for f in self.features
                if any(k in f for k in ('Flag', 'Cnt', 'Count', '__present',
                                        'N_Invalid', 'N_Clipped', 'Protocol'))
                or len(np.unique(R0[:min(5000, n), fi[f]])) <= 20}
        cont = [f for f in self.features if f not in disc and f not in DERIVED_ALL]
        ci = [fi[f] for f in cont]
        smp = R0[:min(20000, n)]
        integral = [fi[f] for f in cont
                    if np.all(np.abs(smp[:, fi[f]] - np.round(smp[:, fi[f]]))
                              <= 1e-6 * np.maximum(1.0, np.abs(smp[:, fi[f]])))]
        self._rounded = 0

        def phys(R, b):
            if integral:
                d = R[:, integral] - b[:, integral]
                R[:, integral] = b[:, integral] + np.round(d)
                self._rounded += int(np.count_nonzero(d))
            return R

        corr = {}
        for eps in [0.0] + CFG['adv_levels']:
            R = R0.copy()
            if eps > 0:
                M = rng.uniform(1 - eps, 1 + eps, (n, len(ci)))
                R[:, ci] = np.where(M == 1.0, R0[:, ci],
                                    np.maximum(R0[:, ci] * M, 0.0))
            R = _rederive(phys(R, R0), R0, fi)
            m = self._mm(y, score(R))
            drop = 1 - m['macro_f1'] / max(base['macro_f1'], 1e-9)
            corr[str(eps)] = {'macro_f1': m['macro_f1'],
                              'rel_drop': float(drop),
                              'verdict': ('NOT APPLICABLE (baseline < 0.5)'
                                          if base['macro_f1'] < 0.5 else
                                          'ROBUST' if drop < CFG['robust_drop']
                                          else 'NOT ROBUST')}
            if eps == 0.0 and abs(m['macro_f1'] - base['macro_f1']) > CFG['identity_tol']:
                raise RuntimeError('0% corruption does not reproduce the baseline')
            print(f'  corruption {eps:.0%}: macro-F1 {m["macro_f1"]:.4f} '
                  f'({corr[str(eps)]["verdict"]})')
        atk = y != self.reject_idx if self.reject_idx >= 0 else np.ones(n, bool)
        eva = {}
        for s in CFG['evasion_scales']:
            for mode, (sb, stt) in {'padding': (s, 1.0), 'delay': (1.0, s),
                                    'padding+delay': (s, s)}.items():
                R = R0.copy()
                for f in FWD_BYTES:
                    if f in fi:
                        R[atk, fi[f]] = R0[atk, fi[f]] * sb
                for f in TIME_FEATS:
                    if f in fi:
                        R[atk, fi[f]] = R0[atk, fi[f]] * stt
                for f in PKT_RATES:
                    if f in fi:
                        R[atk, fi[f]] = R0[atk, fi[f]] / stt
                R = _rederive(phys(R, R0), R0, fi)
                m = self._mm(y[atk], score(R)[atk])
                eva[f'{mode}_{s}'] = {'scale_bytes': sb, 'scale_time': stt,
                                      'attack_macro_f1': m['macro_f1']}
        base_atk = self._mm(y[atk], score(R0.copy())[atk])['macro_f1']
        single = {}
        for f in cont:
            R = R0.copy()
            R[atk, fi[f]] = R0[atk, fi[f]] * CFG['single_feature_scale']
            R = _rederive(phys(R, R0), R0, fi)
            single[f] = float(base_atk - self._mm(y[atk], score(R)[atk])['macro_f1'])
        self.R['robustness'] = {
            'space': 'raw (inverse L1 scaler), clipped at 0, re-standardised',
            'round_trip_max_abs_error': rt, 'sample_rows': int(n),
            'baseline_macro_f1': base['macro_f1'],
            'baseline_attack_only_macro_f1': base_atk,
            'corruption': corr, 'constrained_evasion': eva,
            'single_feature_drop_top15': dict(sorted(
                single.items(), key=lambda x: -abs(x[1]))[:15]),
            'cells_delta_quantised': int(self._rounded),
            'identity_guarantee': 'multiplier exactly 1 returns the row '
                                  'untouched; asserted at 1e-12',
            'threat_models': {
                'corruption': 'independent noise on continuous features of '
                              'every routed flow: corruption robustness, not '
                              'evasion',
                'evasion': 'attacker pads payload and/or delays packets on '
                           'ATTACK flows only; rates and derived features '
                           'follow'}}
        w = min(eva.items(), key=lambda x: x[1]['attack_macro_f1'])
        print(f'  worst evasion {w[0]}: attack macro-F1 '
              f'{w[1]["attack_macro_f1"]:.4f} (baseline {base_atk:.4f})')
        del R0, Z
        _gc()

    # ------------------------------------------------------------------ P10
    def p10_specialists(self):
        if not CFG.get('train_specialists'):
            print('[P10] Specialists skipped (--train-specialists to enable)')
            return
        print('[P10] Specialists (exploratory, validation-only thresholds)')
        Xv, yv, _, _ = self._routed(self.paths['val'])
        keep = np.isin(yv, self.classes_)
        Xv, yv = Xv[keep], yv[keep]
        out = {}
        for cls in CFG['specialist_candidates']:
            if cls not in self.classes_:
                continue
            yb_t = (self.le.inverse_transform(self.y_tr) == cls).astype(np.int8)
            if yb_t.sum() < 10:
                continue
            rng = np.random.RandomState(SEED + 11)
            pos = np.flatnonzero(yb_t == 1)
            neg = np.flatnonzero(yb_t == 0)
            sel = np.sort(np.concatenate([
                rng.choice(pos, min(len(pos), CFG['spec_n_per']), replace=False),
                rng.choice(neg, min(len(neg), CFG['spec_n_per']), replace=False)]))
            Xs = self._pool_rows(sel)
            ys = yb_t[sel]
            spw = float((len(ys) - ys.sum()) / max(ys.sum(), 1))
            b = lgb.train({**CFG['spec'], 'scale_pos_weight': spw},
                          lgb.Dataset(Xs, ys, feature_name=list(self.features)),
                          CFG['spec_rounds'], callbacks=[lgb.log_evaluation(-1)])
            pv = b.predict(Xv).astype(np.float32)
            yb_v = (yv == cls).astype(np.int8)
            fa, ta, th = roc_curve(yb_v, pv)
            tau = float(th[int(np.argmax(ta - fa))])
            pr = (pv >= tau).astype(np.int8)
            out[cls] = {'tau_val': tau,
                        'val_f1': float(f1_score(yb_v, pr, zero_division=0)),
                        'val_precision': float(precision_score(yb_v, pr,
                                                                zero_division=0)),
                        'val_recall': float(recall_score(yb_v, pr,
                                                          zero_division=0)),
                        'status': 'exploratory, not deployed'}
            print(f'  {cls:<20} tau={tau:.4f} val F1={out[cls]["val_f1"]:.4f}')
        self.R['specialists'] = out
        del Xv
        _gc()

    def _free_pool(self):
        if hasattr(self, 'X_tr'):
            del self.X_tr
            _gc()

    # ------------------------------------------------------------------ P11
    def p11_figures(self):
        print('[P11] Figures')
        y, p = self._y, self._probs[self.best]
        pred = p.argmax(1)
        cls = self.classes_
        short = [c.replace('BruteForce-', 'BF-').replace('DDoS-', 'DD-')
                 .replace('DoS-', 'D-') for c in cls]

        # Fig A: combined L2+L3 confusion, counts + 3 decimals
        cm = confusion_matrix(y, pred, labels=list(range(self.k)))
        rs = cm.sum(1, keepdims=True)
        cmn = np.where(rs == 0, 0, cm / np.maximum(rs, 1))
        fig, ax = plt.subplots(1, 2, figsize=(7.16, 3.6))
        for a_, M, ttl, dec in ((ax[0], cm, '(a) counts', False),
                                (ax[1], cmn, '(b) row-normalised', True)):
            a_.imshow(M if not dec else cmn, cmap='Blues', vmin=0,
                      vmax=(M.max() if not dec else 1))
            a_.set_xticks(range(self.k), short, rotation=45, ha='right', fontsize=6)
            a_.set_yticks(range(self.k), short, fontsize=6)
            for i in range(self.k):
                for j in range(self.k):
                    v = M[i, j]
                    t = (f'{v:,}' if not dec and v else
                         (f'{v:.3f}' if dec and v >= 0.001 else ''))
                    if t:
                        a_.text(j, i, t, ha='center', va='center', fontsize=4.8,
                                color='white'
                                if v > (M.max() if not dec else 1) / 2
                                else 'black')
            a_.set_xlabel('Predicted')
            a_.set_ylabel('True')
            a_.set_title(ttl)
        fig.suptitle(f'Routed L2+L3 population — {self.best}. The Normal row '
                     'is the L2 false positives Layer 3 recovers.', fontsize=8)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_combined_cm.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig B: per-class metrics with block CI
        bm = self.R['test'][self.best]['deployment_routed']
        ciD = self.R['statistics'][self.best]['per_class_ci95']
        order = np.argsort(bm['per_class_f1'])[::-1]
        x = np.arange(self.k)
        w = 0.27
        fig, a_ = plt.subplots(figsize=(7.16, 3.0))
        a_.bar(x - w, [bm['per_class_f1'][i] for i in order], w,
               color=OK['blue'], label='F1 (per class)')
        a_.bar(x, [bm['per_class_precision'][i] for i in order], w,
               color=OK['green'], label='Precision')
        a_.bar(x + w, [bm['per_class_recall'][i] for i in order], w,
               color=OK['verm'], label='Recall')
        missing_ci = [self.classes_[i] for i in range(self.k)
                      if self.classes_[i] not in ciD]
        if missing_ci:
            print(f'  [WARN] no block CI for {missing_ci} (absent from the '
                  'deployment-routed population); bars drawn without error bars')
            self.R['figures_missing_ci'] = missing_ci
        for xi, i in enumerate(order):
            lo, hi = ciD.get(cls[i], (float('nan'), float('nan')))
            v = bm['per_class_f1'][i]
            a_.errorbar(xi - w, v,
                        yerr=[[max(0, v - lo)], [max(0, hi - v)]],
                        fmt='none', ecolor='black', capsize=2.5, lw=0.8)
        a_.set_xticks(x, [short[i] for i in order], rotation=40, ha='right',
                      fontsize=6.5)
        a_.set_ylim(0, 1.08)
        a_.set_ylabel('Score (all bars are per-class rates)')
        a_.legend(frameon=False, ncol=3)
        a_.grid(axis='y', alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_per_class.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig C: typed summary with CIs
        st = self.R['statistics'][self.best]
        rt = self.R.get('routing', {})
        dep = self.R['test'][self.best]['deployment_routed']['macro_f1']
        items = [('L3 macro-F1\n(deployment-routed)', dep,
                  st['macro_f1_block_ci95'], 'macro-F1'),
                 ('L3 macro-F1\n(twin-free)',
                  self.R['test'][self.best]['deployment_routed_twin_free']['macro_f1'],
                  None, 'macro-F1'),
                 ('Zero-day L4\ncoverage', rt.get('l4_coverage_conf', np.nan),
                  None, 'coverage rate'),
                 ('OOD AUROC\n(energy)', rt.get('ood_auroc_energy') or np.nan,
                  None, 'AUROC')]
        fig, a_ = plt.subplots(figsize=(3.5, 2.8))
        for i, (lab, v, ci, typ) in enumerate(items):
            a_.bar(i, v, 0.55,
                   color=[OK['blue'], OK['sky'], OK['verm'], OK['green']][i])
            if ci:
                a_.errorbar(i, v,
                            yerr=[[max(0, v - ci[0])], [max(0, ci[1] - v)]],
                            fmt='none', ecolor='black', capsize=3, lw=0.9)
            a_.text(i, min(v + 0.03, 1.02), f'{v:.4f}', ha='center', fontsize=6.5)
            a_.text(i, 1.09, typ, ha='center', fontsize=5.8, style='italic',
                    color=OK['grey'])
        a_.set_xticks(range(len(items)), [i[0] for i in items], fontsize=6)
        a_.set_ylim(0, 1.18)
        a_.set_ylabel('Value (metric type printed above each bar)')
        a_.grid(axis='y', alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_summary_typed.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig D: OOD statistic comparison
        if rt.get('ood_auroc_energy') is not None:
            fig, a_ = plt.subplots(figsize=(3.5, 2.6))
            a_.bar([0, 1], [rt['ood_auroc_max_softmax'], rt['ood_auroc_energy']],
                   0.5, color=[OK['orange'], OK['green']])
            a_.axhline(0.5, color='black', ls=':', lw=.8)
            a_.set_xticks([0, 1], ['max-softmax\nconfidence',
                                   'energy\n(-logsumexp)'], fontsize=7)
            a_.set_ylim(0, 1.0)
            a_.set_ylabel('AUROC: known attack vs zero-day')
            for i, v in enumerate([rt['ood_auroc_max_softmax'],
                                   rt['ood_auroc_energy']]):
                a_.text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=7)
            a_.grid(axis='y', alpha=.25, lw=.5)
            fig.tight_layout()
            fig.savefig(self.figs / 'L3_fig_ood_statistic.pdf',
                        bbox_inches='tight')
            plt.close(fig)

        # Fig E: robustness
        rb = self.R['robustness']
        lv = [0.0] + CFG['adv_levels']
        fig, a_ = plt.subplots(figsize=(3.5, 2.6))
        a_.plot([v * 100 for v in lv],
                [rb['corruption'][str(v)]['macro_f1'] for v in lv],
                'o-', color=OK['blue'], lw=1.2, ms=3.5,
                label='corruption (routed)')
        for mode, col in (('padding', OK['orange']), ('delay', OK['green']),
                          ('padding+delay', OK['verm'])):
            xs = [100 * (s - 1) for s in CFG['evasion_scales']]
            a_.plot(xs,
                    [rb['constrained_evasion'][f'{mode}_{s}']['attack_macro_f1']
                     for s in CFG['evasion_scales']],
                    's--', color=col, lw=1, ms=3, label=f'evasion: {mode}')
        a_.set_xlabel('Perturbation magnitude (%)')
        a_.set_ylabel('Macro-F1')
        a_.legend(frameon=False, fontsize=6)
        a_.grid(alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_robustness.pdf', bbox_inches='tight')
        plt.close(fig)

        # Fig F: importance + ablation
        top = list(self.importance.items())[:15]
        fig, ax = plt.subplots(1, 2, figsize=(7.16, 3.0))
        mx = max(v for _, v in top) or 1
        ax[0].barh(range(len(top)), [v / mx * 100 for _, v in top][::-1],
                   color=OK['blue'], height=.7)
        ax[0].set_yticks(range(len(top)), [f for f, _ in top][::-1], fontsize=6.5)
        ax[0].set_xlabel('Normalised gain (%)')
        ax[0].set_title('(a) Importance, top 15')
        ab = self.R['ablation']['remove_top_k']
        ks = list(ab)
        ax[1].bar(range(len(ks)), [ab[k]['macro_f1'] for k in ks],
                  color=OK['verm'])
        ax[1].axhline(self.R['ablation']['baseline_macro_f1'], color='black',
                      ls='--', lw=1, label='baseline (same sample)')
        ax[1].set_xticks(range(len(ks)), ks, fontsize=7)
        ax[1].set_ylabel('Macro-F1')
        ax[1].set_title('(b) Ablation')
        ax[1].legend(frameon=False)
        ax[1].grid(axis='y', alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_importance_ablation.pdf',
                    bbox_inches='tight')
        plt.close(fig)

        # Fig G: latency
        fig, a_ = plt.subplots(figsize=(3.5, 2.4))
        t = np.sort(self._lat_samples)
        a_.plot(t, np.arange(1, len(t) + 1) / len(t), color=OK['blue'], lw=1.2,
                label=f'{self.best}; P95 {np.percentile(t, 95):.0f} $\\mu$s')
        a_.set_xscale('log')
        a_.set_xlabel('Latency per flow ($\\mu$s, batch = 1, log)')
        a_.set_ylabel('ECDF')
        a_.legend(frameon=False, loc='lower right')
        a_.grid(alpha=.25, lw=.5)
        fig.tight_layout()
        fig.savefig(self.figs / 'L3_fig_latency.pdf', bbox_inches='tight')
        plt.close(fig)
        print(f'  {len(list(self.figs.glob("*.pdf")))} PDFs written')

    # ------------------------------------------------------------------ P12
    def p12_export(self):
        print('[P12] Exports')
        mdl = self.models[self.best]
        _save(mdl.obj, self.out / 'best_model_l3.pkl')
        _save(self.cals[self.best], self.out / 'best_calibrators_l3.pkl')
        _save(self.le, self.out / 'label_encoder.pkl')
        _atomic_json(self.thresholds[self.best], self.out / 'l3_thresholds.json')
        p = self._probs[self.best]
        thr = self.thresholds[self.best]
        conf = p.max(1).astype(np.float32)
        ood = conf < thr['ood_conf']
        route = np.where(ood, 'layer4',
                         np.where(conf >= thr['l5'], 'layer5',
                                  'layer4+provisional_l5'))
        cols = {'family': self.le.inverse_transform(p.argmax(1)),
                'confidence': conf, 'route': route,
                'ood_by_confidence': ood, 'l2_flagged': self._l2f,
                'true_target': self.le.inverse_transform(self._y),
                '_dup_in_train': self._twin}
        if getattr(self, '_ts_test', None) is not None:
            cols['_ts'] = self._ts_test
        pd.DataFrame(cols).to_parquet(self.out / 'l3_scores_test_routed.parquet')
        self.R['l4_l5_contract'] = {
            'model': str(self.out / 'best_model_l3.pkl'),
            'model_kind': mdl.kind,
            'calibrators': str(self.out / 'best_calibrators_l3.pkl'),
            'encoder': str(self.out / 'label_encoder.pkl'),
            'thresholds_json': str(self.out / 'l3_thresholds.json'),
            'features': self.features, 'classes': self.classes_,
            'reject_class': self.reject if self.reject_idx >= 0 else None,
            'reject_class_active': bool(self.reject_idx >= 0),
            'layer4_requirements': [
                'apply calibrators from this contract BEFORE comparing to any '
                'threshold: per-class Platt on logit(p), then renormalise. '
                'Raw softmax is a different scale and the thresholds do not '
                'apply to it.',
                'score EVERY flow: the measured routing shows most zero-day '
                'flows are flagged Attack by Layer 2 and never reach a '
                'fallback branch, so Layer 4 is a parallel signal fused at '
                'Layer 5, not a branch.',
                'select alpha and tau on the attacks the Layer-2 gate MISSES '
                'in validation (its false negatives): that is the population '
                'Layer 4 exists to catch. Never on the zero-day test partition.',
                'report zero-day detection three ways: support-weighted, macro, '
                'and twin-free (_dup_in_train == 0).',
                'combined detection is the measured union on the same rows '
                '(flag_l2 | flag_l4).mean(), never 1-(1-a)(1-b).',
                'if reject_class_active is false, no Normal class exists in '
                'this model: do not assume the label "Normal" is predictable.'],
            'calibration': 'per-class Platt on logit(p), applied then renormalised',
            'routing_rule': (f'conf < tau_OOD -> Layer 4; tau_OOD <= conf < '
                             f'tau_L5 -> Layer 4 and provisional Layer 5; '
                             f'conf >= tau_L5 -> Layer 5. Predicting the '
                             f'reject class returns the flow to Normal and '
                             f'raises no SOAR action.'),
            'ood_statistic':
                self.R.get('routing', {}).get('recommended_ood_statistic'),
            'scores_test_routed':
                str(self.out / 'l3_scores_test_routed.parquet')}
        self.R['best_model'] = self.best
        self.R['paper_map'] = {
            'Table_VI': 'test[best]: deployment_routed / attack_only / '
                        'attack_only_l2_flagged / attack_only_l2_missed',
            'Table_VII': 'statistics[best] block CI + per_class_ci95',
            'Table_VIII': 'specialists (exploratory)',
            'Table_IX': 'statistics.pairwise (McNemar b/c/OR + DeLong + delta CI)',
            'Fig_combined_cm': 'combined L2+L3 confusion, counts + 3 decimals',
            'Fig_per_class': 'per-class metrics + block CI',
            'Fig_summary_typed': 'metric type printed per bar',
            'Fig_ood_statistic': 'confidence vs energy evidence',
            'Fig_robustness': 'raw space, corruption vs evasion',
            'Sec_VI-A': 'routed population and reject class (cascaded FPR)',
            'Sec_VI-E': 'pool construction'}
        _atomic_json(self.R, self.out / 'manifest_l3.json')

    # ------------------------------------------------------------------ run
    def run(self):
        t0 = time.time()
        for step in (self.p0_contract, self.p1_pool, self.p2_train,
                     self.p3_calibrate_threshold, self.p4_evaluate,
                     self.p5_statistics, self.p5b_threshold_transfer,
                     self.p6_ood_and_routing, self.p7_latency,
                     self.p8_importance_ablation, self.p9_adversarial,
                     self.p10_specialists, self._free_pool, self.p11_figures,
                     self.p12_export):
            step()
            _gc()
        b = self.R['test'][self.best]
        st = self.R['statistics'][self.best]['macro_f1_block_ci95']
        rj = b['reject_recovery']
        est = b['deployment_routed']['macro_f1']
        if not (st[0] <= est <= st[1]):
            raise RuntimeError(
                f'[FATAL] point estimate {est:.4f} outside its own block CI '
                f'[{st[0]:.4f}, {st[1]:.4f}]')
        print('=' * 76)
        print(f'  LAYER 3 R8.0 COMPLETE | {(time.time() - t0) / 60:.1f} min | '
              f'{self.best}')
        print(f'  deployment-routed macro-F1 '
              f'{b["deployment_routed"]["macro_f1"]:.4f} '
              f'[block CI {st[0]:.4f}-{st[1]:.4f}] | twin-free '
              f'{b["deployment_routed_twin_free"]["macro_f1"]:.4f}')
        print(f'  attack-only {b["attack_only"]["macro_f1"]:.4f} | '
              f'L2-flagged attacks '
              f'{b["attack_only_l2_flagged"]["macro_f1"]:.4f}'
              + ('' if b['attack_only_l2_missed'] is None else
                 f' | attacks L2 missed '
                 f'{b["attack_only_l2_missed"]["macro_f1"]:.4f}'))
        if rj.get('enabled'):
            print(f'  L2 false positives recovered as Normal: '
                  f'{rj["recovered_as_normal"]:.4f} of '
                  f'{rj["l2_false_positives"]:,}')
        if 'cascaded_fpr' in self.R:
            print(f'  cascaded FPR: {100 * self.R["cascaded_fpr"]["l2_fpr_on_test_manifest"]:.3f}% '
                  f'-> {100 * self.R["cascaded_fpr"]["deployed_fpr_after_layer3"]:.3f}%')
        if 'combined_zeroday' in self.R:
            print(f'  measured L2∪L4 zero-day DR: '
                  f'{self.R["combined_zeroday"]["measured_union_rate"]:.4f}')
        print('=' * 76)


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='TITANIUM-6 Layer 3 R8.0')
    ap.add_argument('--l1', default=os.environ.get('T6_L1', '/kaggle/working/l1_R8'))
    ap.add_argument('--l2', default=os.environ.get('T6_L2', '/kaggle/working/l2_R8'))
    ap.add_argument('--out', default=os.environ.get('T6_L3', '/kaggle/working/l3_R8'))
    ap.add_argument('--attack-only', action='store_true',
                    help='ablation: no reject class (train on attack rows only)')
    ap.add_argument('--train-specialists', action='store_true')
    a, _ = ap.parse_known_args()

    for tag, d, f in (('Layer 1', Path(a.l1), 'manifest.json'),
                      ('Layer 2', Path(a.l2), 'manifest_l2.json')):
        if not (d / f).exists():
            root = d.parent if d.parent.exists() else Path('.')
            found = sorted(str(q.parent) for q in root.glob('*/' + f))
            raise SystemExit(
                f'[FATAL] no {f} in {d}\n'
                f'        {tag} outputs found here: {found or "none"}\n'
                f'        pass --l1/--l2 <dir>, or set T6_L1 / T6_L2.')

    CFG['train_reject'] = not a.attack_only
    CFG['train_specialists'] = a.train_specialists
    if a.attack_only:
        print('[ABLATION] --attack-only: reject class disabled; cascaded FPR '
              'will be unavailable')
    Layer3(a.l1, a.l2, a.out).run()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
[P0] Contracts
  L2 test FPR reference: 0.085192
  L1 TITANIUM6_LAYER1_R8.0 | L2 TITANIUM6_LAYER2_R8.0
  gate LightGBM (lgb) tau=0.0793 | 77 features | reject class: on
[P1] Routed training pool (attacks + OUT-OF-SAMPLE L2 false positives)
  [WARN] val FP rate 0.0004 vs L2 test FPR 0.0852 (ratio 0.00): the reject class may not represent the deployment population
  reject rows: 410 from validation (val FP rate 0.0004) — 3,445 in-sample train FPs discarded (rate 0.0004)
  routed pool 1,766,823 rows | 12 classes | L2 false positives 410 (0.00% of train Normal)
[P2] Train (same rows, same weights, same ES split)
  LightGBM   256.0s | best_iter=56
  XGBoost    339.5s | best_iter=148
[P3] Per-class Platt on val-CAL half; thresholds on val-THR half
  LightGBM   tau_L5=0.2967 | tau_OOD(conf)=0.6012 | tau_OOD(energy)=-0.3071
  XGBoost    tau_L5=0.2380 | tau_OOD(conf)=0.5952 | tau_OOD(energy)=-2.8023
[P4] Test evaluation: routed vi

In [5]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 4 | R8.0
#  R8.0 over R6.1:
#   T1 AE INPUT TRANSFORM. R6.1 fed StandardScaled, heavy-tailed CICFlowMeter
#      features straight into an MSE autoencoder. Measured consequence: the
#      Attack/Normal per-feature error ratio reached 612,841 for
#      Ratio__FwdBwd_Packets, i.e. one or two extreme features dominated the
#      anomaly score. R8.0 applies the signed log  z' = sign(z)*log1p(|z|)
#      inside the model (training, scoring, attribution and serving all use
#      the same method), and records 'input_transform' in
#      ae_architecture.json. Downstream layers refuse an AE without it.
#   T2 Isolation Forest is still trained and MEASURED, and is disabled by the
#      existing if_min_auc floor when it carries no signal (R6.1: val AUC
#      0.5044; original v1 run: known-test AUC 0.4542). No change in rule.
#   T3 Paths and version strings follow the single R8 run configuration.
#
#
#  R6.0 is written against the MEASURED R5.5 run, not against expectations:
#    IF AUC 0.4795 (below random) | AE 0.7562 | ensemble 0.6053
#    cascaded FPR 11.84% vs L2 9.33%  -> L4 ADDED 2.51 pp of FPR
#    zero-day DR 3.43% weighted / 1.40% twin-free
#    measured L2 union L4 = 0.7666 = L2 alone  -> L4 added 0.00 pp
#  Those numbers say the deployed configuration was worse than its own best
#  component. R6.0 fixes that, and measures the rest honestly.
#
#   G1 ISOLATION FOREST FAIL-LOUD. IF capacity is raised (200 trees, 8192
#      samples). Its validation AUC is then MEASURED: below if_min_auc the
#      forest is disabled (alpha forced to 0) and the collapse is recorded as
#      a negative result instead of being blended into the deployed score.
#   G2 ALPHA CANNOT LOSE TO ITS OWN COMPONENT. R5.5 chose alpha by an
#      objective that picked 0.1 while alpha = 0 had the higher validation
#      AUC. R6.0 records the validation AUC of every alpha and refuses any
#      selection whose AUC is below the best single component; alpha = 0
#      (AE-only) is the fallback, with the override recorded.
#   G3 AE CAPACITY. [64, 32, 16] for 77 inputs; [32, 16, 8] compressed 77
#      features into 8 dimensions and lost the tail the layer exists to see.
#   G4 FPR-MATCHED OPERATING POINT. Beside the F1-driven tau4, R6.0 reports
#      the tau that makes Layer 4 add (almost) no false positives to the
#      cascade, and the zero-day DR at that point. A layer that raises the
#      alarm rate without raising detection must show both operating points.
#   G5 L4 ADDED COVERAGE, EXPLICIT. union(L2, L4) - L2 alone, in points, on
#      the same rows. R5.5 reported the union and left the contribution to be
#      inferred; the measured contribution was zero.
#   G6 TIME-BLOCK BOOTSTRAP. L1 writes _ts into the zero-day file, so the
#      60-minute block bootstrap that Layers 2 and 3 use is computed here too,
#      beside the class-stratified one, instead of being declared impossible.
#   G7 Paths default to L1 R5.4 / L2 R6.5 / L3 R7.12.
#   G8 DEPLOYED-PATH LATENCY. The first R6.0 run reported 'Ensemble' P95 at
#      14,497 us while the deployed configuration was AE-only (alpha = 0), so
#      the headline latency described a path that is not deployed. The three
#      components are still timed as evidence, and a fourth entry, DEPLOYED,
#      times exactly what runs in production. | CSE-CIC-IDS2018 | IEEE OJ-COMS
#
#  Reads L1 R5.2 (schema 5), L2 R6.3, L3 R7.8. Bracket access.
#
#  R5.5 = R5.4 with eight integration fixes. Each converts a claimed
#  measurement into a real one, or a claim into a fact.
#
#   F1  L3 CALIBRATORS. R5.4 read L3's raw softmax and compared it against
#       tau_L5/tau_OOD, which L3 defines on the CALIBRATED scale. Every
#       routing decision in R5.4 was made on the wrong scale. R5.5 loads
#       best_calibrators_l3.pkl, applies per-class Platt, renormalises,
#       and only then compares against any threshold.
#   F2  OOD STATISTIC. L3 R7.8 measures max-softmax AUROC = 0.5085 vs energy
#       AUROC = 0.6155, and RECOMMENDS energy. R5.4 hard-coded ood_conf.
#       R5.5 reads ood_statistic from the L3 contract and computes the
#       matching score (energy = -logsumexp over raw margins) so the L3
#       decision surface used at training time is the one used here.
#   F3  DRIFT WINDOWS. R5.4 computed drift on test_known Normal scores in
#       the shuffled row order L1 writes. Consecutive windows were random
#       samples of the same pool, so the KS monitor could not detect drift.
#       R5.5 carries _ts from L1 through P5 and sorts before windowing.
#   F4  TWIN-FREE ZERO-DAY DR. L1 measured twin-in-train = 45% on zero-day.
#       R5.4 reported test_known twin-free AUC but not twin-free zero-day DR.
#       R5.5 reports both, at the support-weighted and macro level, and per
#       class where the twin-free support is sufficient.
#   F5  INPUT POPULATION BY SOURCE. Given L3 R7.8's measured OOD gate collapse
#       (0.34% incremental routing), L4's input is dominated by L2-missed
#       attacks, not by L3 OOD routing. R5.5 records the breakdown.
#   F6  BOOTSTRAP LABEL. R5.4's note called the class-stratified resample a
#       "per-block bootstrap". The blocks are zero-day classes, not time
#       windows. Renamed for honesty.
#   F7  PATH DEFAULTS + GUARD. Defaults updated to L1 R5.2 / L2 R6.3 / L3 R7.8;
#       a wrong path fails with the directories that DO contain a manifest.
#   F8  unseen_macro_dr printed explicitly (was computed but never printed).
#
#  L4-OWNED COMMENTS CLOSED:
#   19  α selection (val half B, sample stated, not contradictory)
#   30  Fig 19 ROC on test_known (not on val)
#   34  Zero-day DR: weighted AND macro AND unseen, + twin-free (F4)
#   53  4-level detection cascade (binary / novelty / attribution / safe)
#   54  Paired class-stratified bootstrap of combined L2|L4 on the same rows
#   55  "Unsupervised representation learning with supervised calibration"
#   57  Cascaded FPR = (L2 FPs + L4 FPs on L2-normal) / all test_known Normal
#   58  Multi-objective α = F1_known + λ · DR_on_L2_false_negatives(val_B)
#   59  Drift via KS effect size + persistence on held-out reference
#
#  Code discipline (identical to L1 R5.2 / L2 R6.3 / L3 R7.8):
#   no global warnings.filterwarnings; local catch_warnings only around
#   sklearn's lbfgs solver and scipy ks_2samp on small samples; pdf.fonttype
#   42 and a Times stack for IEEE PDF eXpress; atomic JSON and pickle
#   writes; XGBoost truncated to best_iteration by Layer 2 so .predict()
#   equals the evaluated model; all constants in CFG.
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os
import gc
import json
import time
import pickle
import platform
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
import scipy
from scipy.stats import ks_2samp
from scipy.special import logsumexp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, roc_auc_score, roc_curve, confusion_matrix, brier_score_loss,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import lightgbm as lgb
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CFG = {
    'version': 'TITANIUM6_LAYER4_R8.0',
    'seed': SEED,
    'required_l2_schema': 5,

    'if_n_estimators': 200,          # G1
    'if_max_samples': 8192,          # G1
    'if_min_auc': 0.55,              # G1: below this the forest is disabled
    'if_train_n': 500_000,
    'drift_ref_n': 50_000,

    'ae_hidden': [64, 32, 16],       # G3: 77 -> 16, not 77 -> 8
    'ae_train_n': 500_000,
    'ae_epochs': 50,
    'ae_batch_size': 4096,
    'ae_lr': 1e-3,
    'ae_weight_decay': 1e-5,
    'ae_patience': 10,

    'eval_chunk': 200_000,
    'val_sample_n': 400_000,

    'bootstrap_n': 1_000,
    'per_class_bootstrap_n': 500,

    'lat_warmup': 50,
    'lat_runs': 1_000,

    'alpha_grid': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'lambda_novel': 0.5,
    'target_fpr_l4': 0.02,
    'l4_fpr_matched_budget': 0.001,  # G4: added-FPR budget for the second point
    'block_minutes': 60,             # G6

    'drift_ks_thresh': 0.20,
    'drift_persist': 3,
    'drift_windows': 20,

    'min_l2_fn_for_novelty': 10,
    'min_class_dr_rows': 5,
}

# =============================================================================
#  COLOUR PALETTE — psychophysics rationale (identical to L1/L2/L3)
#   navy = deterministic baseline; green = learned reconstruction; red =
#   fused attack / zero-day. Threshold lines use signal-red; colours are
#   luminance-separated for deuteranopia safety.
# =============================================================================
COLORS = {
    'if_model': '#002F6C',
    'ae_model': '#007A33',
    'ensemble': '#C00000',
    'zday':     '#C00000',
    'normal':   '#CCCCCC',
    'attack':   '#F5A623',
    'thresh':   '#E74C3C',
    'edge':     '#333333',
}


# =============================================================================
#  UTILITIES
# =============================================================================
def _gc():
    gc.collect()


def _mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)


def _read_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)


def _json_default(o):
    if isinstance(o, tuple):
        return list(o)
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def _atomic_json(payload, dest):
    tmp = str(dest) + '.__tmp__'
    with open(tmp, 'w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, default=_json_default)
    os.replace(tmp, dest)


def _save(obj, dest):
    tmp = str(dest) + '.__tmp__'
    with open(tmp, 'wb') as fh:
        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, dest)


def _hardware():
    cpu = platform.processor() or ''
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    cpu = line.split(':', 1)[1].strip()
                    break
    except OSError:
        pass
    return {
        'cpu_model': cpu,
        'physical_cores': psutil.cpu_count(logical=False),
        'logical_cores': psutil.cpu_count(logical=True),
        'ram_gb': round(psutil.virtual_memory().total / 2**30, 1),
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': scipy.__version__,
        'torch': torch.__version__,
        'lightgbm': lgb.__version__,
        'xgboost': xgb.__version__,
    }


def _plt_style():
    plt.rcParams.update({
        'pdf.fonttype': 42, 'ps.fonttype': 42,
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Nimbus Roman',
                       'STIXGeneral', 'DejaVu Serif'],
        'mathtext.fontset': 'stix', 'font.size': 8,
        'axes.titlesize': 8.5, 'axes.labelsize': 8,
        'xtick.labelsize': 7, 'ytick.labelsize': 7,
        'legend.fontsize': 6.8, 'axes.linewidth': 0.7,
        'savefig.dpi': 600, 'axes.edgecolor': COLORS['edge'],
    })


def _iter(path, cols, chunk=None):
    for b in pq.ParquetFile(path).iter_batches(
            batch_size=chunk or CFG['eval_chunk'], columns=cols):
        df = b.to_pandas()
        yield df
        del df, b


# =============================================================================
#  SPARSE AUTOENCODER (identical architecture to L4 R5.4)
# =============================================================================
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                    nn.ReLU(), nn.Dropout(0.1)]
            prev = h
        self.encoder = nn.Sequential(*enc)

        dec = []
        for h in reversed(hidden_dims[:-1]):
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)

    INPUT_TRANSFORM = 'signed_log1p'

    @staticmethod
    def prep(x):
        """T1: signed log. Monotone, sign-preserving, identity near 0,
        compresses the |z| >> 1 tail that dominated the MSE in R6.1."""
        x = torch.as_tensor(x, dtype=torch.float32)
        return torch.sign(x) * torch.log1p(torch.abs(x))

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x, batch=262_144):
        self.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(x), batch):
                x_t = self.prep(x[i:i + batch])
                x_r = self(x_t)
                out.append(((x_t - x_r) ** 2).mean(dim=1).numpy().astype(np.float32))
        return (np.concatenate(out) if out else np.zeros(0, np.float32))


# =============================================================================
#  PIPELINE
# =============================================================================
class Layer4Pipeline:

    def __init__(self, l1_dir, l2_dir, l3_dir, l4_dir):
        self.l1 = Path(l1_dir).resolve()
        self.l2 = Path(l2_dir).resolve()
        self.l3 = Path(l3_dir).resolve()
        self.l4 = _mkdir(l4_dir).resolve()
        self.figs = _mkdir(self.l4 / 'figures')
        self.manifest_path = self.l4 / 'manifest_l4.json'

        for p, nm in [(self.l1 / 'manifest.json', 'L1'),
                      (self.l2 / 'manifest_l2.json', 'L2'),
                      (self.l3 / 'manifest_l3.json', 'L3')]:
            if not p.exists():
                raise FileNotFoundError(f'{nm} manifest missing: {p}')

        self.m1 = _read_json(self.l1 / 'manifest.json')
        self.m2 = _read_json(self.l2 / 'manifest_l2.json')
        self.m3 = _read_json(self.l3 / 'manifest_l3.json')

        c2 = self.m2.get('l3_l4_contract', {})
        for k in ('features', 'model', 'cal', 'threshold_json', 'model_kind'):
            if k not in c2:
                raise RuntimeError(f'L2 l3_l4_contract missing {k!r}: run L2 R8.0')

        c3 = self.m3.get('l4_l5_contract', {})
        for k in ('model', 'model_kind', 'classes', 'encoder',
                  'thresholds_json', 'reject_class'):
            if k not in c3:
                raise RuntimeError(f'L3 l4_l5_contract missing {k!r}: run L3 R8.0')

        self.features = list(c2['features'])
        self.l3_classes = list(c3['classes'])
        self.l3_reject = c3['reject_class']
        self.l3_thresholds = _read_json(c3['thresholds_json'])

        self.l3_tau_l5 = float(self.l3_thresholds['l5'])

        # F2: use the L3-recommended OOD statistic.
        self.l3_ood_statistic = c3.get('ood_statistic', 'max_softmax')
        if self.l3_ood_statistic == 'energy':
            self.l3_tau_ood = float(self.l3_thresholds.get(
                'ood_energy', self.l3_thresholds['l5']))
        else:
            self.l3_tau_ood = float(self.l3_thresholds.get(
                'ood_conf', self.l3_thresholds['l5']))

        self.paths = self.m1['l2_input_paths']
        for k in ('train', 'val', 'test_known', 'test_zeroday'):
            if not Path(self.paths[k]).exists():
                raise FileNotFoundError(f'L1 file missing: {self.paths[k]}')

        self.l2_best = self.m2['best_model']
        if self.l2_best not in ('LightGBM', 'XGBoost'):
            raise RuntimeError(f'L2 best_model={self.l2_best} unsupported')
        self.l2_kind = c2['model_kind']
        with open(c2['model'], 'rb') as fh:
            self.l2_model = pickle.load(fh)
        with open(c2['cal'], 'rb') as fh:
            self.l2_cal = pickle.load(fh)
        th2 = _read_json(c2['threshold_json'])
        self.l2_threshold = float(th2['deployed_tau'][self.l2_best])

        self.l3_kind = c3['model_kind']
        with open(c3['model'], 'rb') as fh:
            self.l3_model = pickle.load(fh)
        with open(c3['encoder'], 'rb') as fh:
            self.l3_le = pickle.load(fh)

        # F1: load L3 calibrators, if exported (R7.8 exports them).
        self.l3_cals = None
        if 'calibrators' in c3 and Path(c3['calibrators']).exists():
            with open(c3['calibrators'], 'rb') as fh:
                self.l3_cals = pickle.load(fh)
            n_ok = sum(1 for lr in self.l3_cals if lr is not None)
            print(f'  L3 calibrators : loaded ({n_ok}/{len(self.l3_cals)} fitted)')
        else:
            print('  [WARN] L3 calibrators not found in contract; '
                  'tau_L5 will be compared against uncalibrated scores')

        # ------------------------------------------------------------------
        #  Runtime state
        # ------------------------------------------------------------------
        self.if_model = None
        self.ae_model = None
        self.if_cal = None
        self.ae_cal = None
        self.alpha = None
        self.tau4 = None

        self.calibration_half = None
        self.threshold_half = None
        self.detection_cascade = {}
        self.cascaded_fpr = {}
        self.drift = {}
        self.bootstrap = {}
        self.per_class_dr = {}
        self.per_class_dr_ci = {}
        self.per_class_dr_tf = {}
        self.latency = {}
        self.alpha_diag = {}
        self.input_population = {}

        self._y_test = None
        self._twin_test = None
        self._ts_test = None
        self._if_scores_test = None
        self._ae_scores_test = None
        self._scores_test = None
        self._test_normal_scores = None
        self._test_normal_ts = None

        self._zd_features = None
        self._zd_classes = None
        self._zd_twin = None
        self._zd_scores = None
        self._zd_l2_flag = None
        self._zd_reached_l4 = None
        self._zd_caught = None
        self._zd_l3_argmax = None

        self._X_drift_ref = None
        self.X_normal_train = None

        self._if_prob_valB = None
        self._ae_prob_valB = None
        self._yv_valB = None
        self._val_B_X = None

        print('=' * 76)
        print(f'  TITANIUM-6 | LAYER 4 | {CFG["version"]}')
        print('=' * 76)
        print(f'  features  : {len(self.features)}')
        print(f'  L2 gate   : {self.l2_best} (τ={self.l2_threshold:.4f})')
        print(f'  L3 classes: {len(self.l3_classes)}  '
              f'reject="{self.l3_reject}"  τ_L5={self.l3_tau_l5:.4f}')
        print(f'  L3 OOD    : statistic={self.l3_ood_statistic}  '
              f'τ={self.l3_tau_ood:.4f}')
        print(f'  L4 dir    : {self.l4}')

    # ======================================================================
    #  PHASE 1 — Load Normal-only training flows + held-out drift reference
    # ======================================================================
    def _p1_load_normal(self):
        print('\n[PHASE 1] Load Normal-only training flows (drift ref held out)')
        feats = self.features
        Xl = []
        total = 0
        target = CFG['if_train_n'] + CFG['drift_ref_n']

        for b in pq.ParquetFile(self.paths['train']).iter_batches(
                batch_size=500_000, columns=feats + ['Target']):
            df = b.to_pandas()
            df = df[df['Target'] == self.l3_reject]
            if len(df):
                Xl.append(df[feats].values.astype(np.float32))
                total += len(df)
            del df, b
            _gc()
            if total >= target * 4:
                break

        X = np.vstack(Xl)
        del Xl
        _gc()

        rng = np.random.RandomState(SEED)
        rng.shuffle(X)
        self._X_drift_ref = X[:CFG['drift_ref_n']].copy()
        self.X_normal_train = X[CFG['drift_ref_n']:
                                CFG['drift_ref_n'] + CFG['if_train_n']].copy()
        del X
        _gc()

        print(f'  drift reference : {len(self._X_drift_ref):,} rows '
              f'(never seen by IF/AE)')
        print(f'  training set    : {len(self.X_normal_train):,} rows')

    # ======================================================================
    #  PHASE 2 — Train IF + AE on Normal only
    # ======================================================================
    def _p2_train_models(self):
        print('\n[PHASE 2] Train IF + AE on Normal only')
        X = self.X_normal_train

        t0 = time.perf_counter()
        self.if_model = IsolationForest(
            n_estimators=CFG['if_n_estimators'],
            max_samples=CFG['if_max_samples'],
            contamination='auto',
            random_state=SEED,
            n_jobs=-1)
        self.if_model.fit(X)
        self.latency['train_if_s'] = round(time.perf_counter() - t0, 1)
        print(f'  IF done {self.latency["train_if_s"]}s')

        t0 = time.perf_counter()
        self.ae_model = self._train_ae(X[:CFG['ae_train_n']])
        self.latency['train_ae_s'] = round(time.perf_counter() - t0, 1)
        print(f'  AE done {self.latency["train_ae_s"]}s')

    def _train_ae(self, X):
        n = len(X)
        split = int(n * 0.9)
        rng = np.random.RandomState(SEED)
        idx = rng.permutation(n)
        # T1: train in the transformed space the model scores in
        Xt = SparseAutoencoder.prep(X).numpy()
        X_tr, X_vl = Xt[idx[:split]], Xt[idx[split:]]
        del Xt

        model = SparseAutoencoder(X.shape[1], CFG['ae_hidden'])
        opt = optim.Adam(model.parameters(), lr=CFG['ae_lr'],
                         weight_decay=CFG['ae_weight_decay'])
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
        crit = nn.MSELoss()

        ds = TensorDataset(torch.FloatTensor(X_tr))
        dl = DataLoader(ds, batch_size=CFG['ae_batch_size'], shuffle=True)

        best_val = float('inf')
        best_state = None
        patience_ct = 0
        epoch = 0
        for epoch in range(CFG['ae_epochs']):
            model.train()
            for (xb,) in dl:
                opt.zero_grad()
                loss = crit(model(xb), xb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            model.eval()
            with torch.no_grad():
                xv = torch.FloatTensor(X_vl)
                vloss = float(crit(model(xv), xv).item())
            sched.step(vloss)
            if vloss < best_val - 1e-6:
                best_val = vloss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ct = 0
            else:
                patience_ct += 1
            if patience_ct >= CFG['ae_patience']:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()
        print(f'    AE epochs={epoch + 1} best_val_loss={best_val:.6f}')
        return model

    # ======================================================================
    #  PHASE 3 — Calibrate on val half A; reserve half B for α and τ4
    # ======================================================================
    def _p3_calibrate(self):
        print('\n[PHASE 3] Calibrate on val half A; reserve half B')
        feats = self.features
        Xl = []
        yl = []
        need = CFG['val_sample_n']

        for b in pq.ParquetFile(self.paths['val']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas()
            if len(df):
                Xl.append(df[feats].values.astype(np.float32))
                yl.extend((df['Target'] != self.l3_reject)
                          .astype(np.int8).tolist())
            del df, b
            _gc()
            if sum(len(a) for a in Xl) >= need:
                break

        Xv_all = np.vstack(Xl)[:need]
        yv_all = np.array(yl)[:need]
        del Xl, yl
        _gc()

        rng = np.random.RandomState(SEED + 1)
        perm = rng.permutation(len(yv_all))
        hA, hB = perm[:len(perm) // 2], perm[len(perm) // 2:]
        XA, yA = Xv_all[hA], yv_all[hA]
        XB, yB = Xv_all[hB], yv_all[hB]

        self.calibration_half = {
            'sample': 'A',
            'n': int(len(yA)),
            'n_attack': int(yA.sum()),
            'n_normal': int((yA == 0).sum()),
            'ratio_normal_to_attack': float((yA == 0).sum() / max(yA.sum(), 1))
        }
        self.threshold_half = {
            'sample': 'B',
            'n': int(len(yB)),
            'n_attack': int(yB.sum()),
            'n_normal': int((yB == 0).sum()),
            'ratio_normal_to_attack': float((yB == 0).sum() / max(yB.sum(), 1))
        }
        print(f'  val A (calibrate) : n={len(yA):,}  '
              f'atk={int(yA.sum()):,}  nrm={int((yA == 0).sum()):,}')
        print(f'  val B (α/τ)       : n={len(yB):,}  '
              f'atk={int(yB.sum()):,}  nrm={int((yB == 0).sum()):,}')

        if_raw_A = -self.if_model.score_samples(XA).astype(np.float32)
        ae_raw_A = self.ae_model.reconstruction_error(XA)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            lr_if = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
            lr_if.fit(if_raw_A.reshape(-1, 1), yA)
            lr_ae = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
            lr_ae.fit(ae_raw_A.reshape(-1, 1), yA)
        self.if_cal = lr_if
        self.ae_cal = lr_ae

        if_p_A = lr_if.predict_proba(if_raw_A.reshape(-1, 1))[:, 1]
        ae_p_A = lr_ae.predict_proba(ae_raw_A.reshape(-1, 1))[:, 1]
        b_if = float(brier_score_loss(yA, if_p_A))
        b_ae = float(brier_score_loss(yA, ae_p_A))
        print(f'  IF Brier (val A, natural prior) = {b_if:.5f}')
        print(f'  AE Brier (val A, natural prior) = {b_ae:.5f}')

        if_raw_B = -self.if_model.score_samples(XB).astype(np.float32)
        ae_raw_B = self.ae_model.reconstruction_error(XB)
        self._if_prob_valB = lr_if.predict_proba(
            if_raw_B.reshape(-1, 1))[:, 1]
        self._ae_prob_valB = lr_ae.predict_proba(
            ae_raw_B.reshape(-1, 1))[:, 1]
        self._yv_valB = yB
        self._val_B_X = XB

        del Xv_all, yv_all, XA, yA, XB
        del if_raw_A, ae_raw_A, if_p_A, ae_p_A, if_raw_B, ae_raw_B
        _gc()

    # ======================================================================
    #  PHASE 4 — Multi-objective α on val B with L2-FN novelty proxy
    # ======================================================================
    def _p4_alpha_multobjective(self):
        print('\n[PHASE 4] Multi-objective α on val B with L2-FN novelty proxy')
        if_prob = self._if_prob_valB
        ae_prob = self._ae_prob_valB
        y = self._yv_valB
        XB = self._val_B_X

        p_cal = self._l2_score(XB)
        l2_flag = p_cal >= self.l2_threshold
        a_mask = (y == 1)
        l2_fn_mask = a_mask & (~l2_flag)
        n_l2_fn = int(l2_fn_mask.sum())
        print(f'  val B L2 false negatives (novelty proxy) = {n_l2_fn:,}')

        novelty_ok = n_l2_fn >= CFG['min_l2_fn_for_novelty']
        if not novelty_ok:
            print(f'  [WARN] L2-FN novelty proxy < {CFG["min_l2_fn_for_novelty"]} '
                  f'rows; falling back to separation objective')

        auc_if = float(roc_auc_score(y, if_prob)) if len(np.unique(y)) == 2 else float('nan')
        auc_ae = float(roc_auc_score(y, ae_prob)) if len(np.unique(y)) == 2 else float('nan')
        self.if_usable = bool(auc_if >= CFG['if_min_auc'])
        print(f'  component val AUC: IF={auc_if:.4f}  AE={auc_ae:.4f}')
        if not self.if_usable:
            print(f'  [FINDING] Isolation Forest val AUC {auc_if:.4f} < '
                  f'{CFG["if_min_auc"]}: the forest carries no usable signal on these '
                  'features. alpha is forced to 0 (AE-only) and the collapse is '
                  'reported as a measured negative result.')
        results = []
        for alpha in CFG['alpha_grid']:
            ens = alpha * if_prob + (1 - alpha) * ae_prob
            fpr, tpr, thr = roc_curve(y, ens)
            i_tgt = int(np.argmin(np.abs(fpr - CFG['target_fpr_l4'])))
            tau = float(thr[min(i_tgt, len(thr) - 1)])
            pred = (ens >= tau).astype(np.int8)
            f1 = float(f1_score(y, pred, zero_division=0))

            if novelty_ok:
                dr_nov = float((ens[l2_fn_mask] >= tau).mean())
                obj = f1 + CFG['lambda_novel'] * dr_nov
            else:
                m_a = float(ens[a_mask].mean())
                m_n = float(ens[~a_mask].mean())
                s_a = float(ens[a_mask].std())
                s_n = float(ens[~a_mask].std())
                dr_nov = float((m_a - m_n) / max(s_a + s_n, 1e-9))
                obj = f1 + CFG['lambda_novel'] * dr_nov

            auc_a = (float(roc_auc_score(y, ens))
                     if len(np.unique(y)) == 2 else float('nan'))
            results.append({
                'alpha': float(alpha),
                'val_auc': auc_a,
                'f1_known': f1,
                'novelty_proxy_score': dr_nov,
                'objective': obj,
                'tau': tau,
            })

        cand = [r for r in results if (self.if_usable or r['alpha'] == 0.0)]
        best = max(cand, key=lambda r: r['objective'])
        best_component_auc = max(auc_if if self.if_usable else -1.0, auc_ae)
        override = None
        if np.isfinite(best['val_auc']) and best['val_auc'] < best_component_auc - 1e-9:
            alt = max(cand, key=lambda r: (r['val_auc'], -abs(r['alpha'])))
            override = {
                'rejected_alpha': best['alpha'],
                'rejected_val_auc': best['val_auc'],
                'best_single_component_auc': best_component_auc,
                'selected_alpha': alt['alpha'], 'selected_val_auc': alt['val_auc'],
                'rule': ('G2: a blend may not be deployed when its validation AUC is '
                         'below the best single component; the objective is overridden')}
            print(f'  [FINDING] alpha={best["alpha"]} has val AUC {best["val_auc"]:.4f} '
                  f'< best component {best_component_auc:.4f}: overridden to '
                  f'alpha={alt["alpha"]}')
            best = alt
        self.alpha = float(best['alpha'])
        self.tau4 = float(best['tau'])
        self._alpha_override = override
        self._component_auc = {'if_val_auc': auc_if, 'ae_val_auc': auc_ae,
                               'if_usable': self.if_usable,
                               'if_min_auc': CFG['if_min_auc']}

        self.alpha_diag = {
            'grid': results,
            'selected': best,
            'lambda_novel': CFG['lambda_novel'],
            'novelty_proxy': 'L2 false negatives on val B',
            'novelty_proxy_n': n_l2_fn,
            'objective': ('F1_known(val_B) + λ · DR_on_L2_FN(val_B)'
                          if novelty_ok else
                          'F1_known(val_B) + λ · separation(val_B) [fallback]'),
            'sample': 'val half B (disjoint from calibration half A)',
            'component_val_auc': self._component_auc,
            'alpha_override': self._alpha_override,
            'comment': ('comment 58 closure: α optimised for the population L4 '
                        'exists to catch, not for known-attack F1 alone.'),
        }

        print(f'  selected α={self.alpha}  τ4={self.tau4:.4f}')
        print(f'  F1_known={best["f1_known"]:.4f}  '
              f'novelty={best["novelty_proxy_score"]:.4f}  '
              f'obj={best["objective"]:.4f}')
        for r in results:
            mark = '  <-' if r['alpha'] == self.alpha else ''
            print(f'    α={r["alpha"]:.1f}  F1={r["f1_known"]:.4f}  '
                  f'nov={r["novelty_proxy_score"]:.4f}  '
                  f'obj={r["objective"]:.4f}{mark}')

        del XB, p_cal, l2_flag
        _gc()

    # ======================================================================
    #  Model primitives
    # ======================================================================
    def _l2_score(self, X):
        if self.l2_kind == 'lgb':
            p_raw = self.l2_model.predict(X).astype(np.float32)
        elif self.l2_kind == 'xgb':
            p_raw = np.asarray(self.l2_model.predict(
                xgb.DMatrix(X, feature_names=list(self.features)))).ravel()
        else:
            p_raw = self.l2_model.predict_proba(X)[:, 1]
        return self.l2_cal.predict_proba(
            np.asarray(p_raw).reshape(-1, 1))[:, 1]

    def _l3_raw_proba(self, X):
        """Uncalibrated L3 softmax (for reference; use _l3_cal_proba)."""
        if self.l3_kind == 'lgb':
            return self.l3_model.predict(X).astype(np.float32)
        return self.l3_model.predict(
            xgb.DMatrix(X, feature_names=list(self.features))
        ).reshape(-1, len(self.l3_classes)).astype(np.float32)

    def _l3_cal_proba(self, X):
        """F1: per-class Platt, then renormalise — matches L3's own _cal."""
        q = self._l3_raw_proba(X)
        if self.l3_cals is not None:
            for c, lr in enumerate(self.l3_cals):
                if lr is not None and c < q.shape[1]:
                    q[:, c] = lr.predict_proba(
                        q[:, c].reshape(-1, 1))[:, 1].astype(np.float32)
            s = q.sum(axis=1, keepdims=True)
            q = (q / np.where(s == 0, 1, s)).astype(np.float32)
        return q

    def _l3_margin(self, X):
        """Raw pre-softmax margins, for the energy OOD statistic."""
        if self.l3_kind == 'lgb':
            return self.l3_model.predict(X, raw_score=True).astype(
                np.float64).reshape(-1, len(self.l3_classes))
        return self.l3_model.predict(
            xgb.DMatrix(X, feature_names=list(self.features)),
            output_margin=True
        ).reshape(-1, len(self.l3_classes)).astype(np.float64)

    def _l3_energy(self, X):
        """-logsumexp over raw margins (higher = more anomalous)."""
        return -logsumexp(self._l3_margin(X), axis=1)

    # ======================================================================
    #  PHASE 5 — Test evaluation (test_known, cascaded FPR, zero-day)
    # ======================================================================
    def _p5_evaluate(self):
        print('\n[PHASE 5] Test evaluation (test_known, cascaded FPR, zero-day)')
        feats = self.features

        # ------------------------------------------------------------------
        #  Load test_known with _dup_in_train (twin) and _ts (drift windows)
        # ------------------------------------------------------------------
        Xl, yl, twin_l, ts_l = [], [], [], []
        for b in pq.ParquetFile(self.paths['test_known']).iter_batches(
                batch_size=CFG['eval_chunk'],
                columns=feats + ['Target', '_dup_in_train', '_ts']):
            df = b.to_pandas()
            Xl.append(df[feats].values.astype(np.float32))
            yl.extend((df['Target'] != self.l3_reject)
                      .astype(np.int8).tolist())
            twin_l.append(df['_dup_in_train'].values.astype(np.int8))
            ts_l.append(df['_ts'].values.astype(np.int64))
            del df, b
            _gc()

        Xte = np.vstack(Xl); del Xl; _gc()
        y_te = np.array(yl); del yl; _gc()
        twin_te = np.concatenate(twin_l); del twin_l; _gc()
        ts_te = np.concatenate(ts_l); del ts_l; _gc()

        self._y_test = y_te
        self._twin_test = twin_te
        self._ts_test = ts_te

        # ------------------------------------------------------------------
        #  Score everything with L4
        # ------------------------------------------------------------------
        if_raw_te = -self.if_model.score_samples(Xte).astype(np.float32)
        ae_raw_te = self.ae_model.reconstruction_error(Xte)
        if_p_te = self.if_cal.predict_proba(if_raw_te.reshape(-1, 1))[:, 1]
        ae_p_te = self.ae_cal.predict_proba(ae_raw_te.reshape(-1, 1))[:, 1]
        ens_te = (self.alpha * if_p_te + (1 - self.alpha) * ae_p_te
                  ).astype(np.float32)

        self._scores_test = ens_te
        self._if_scores_test = if_p_te
        self._ae_scores_test = ae_p_te
        self._test_normal_scores = ens_te[y_te == 0]
        self._test_normal_ts = ts_te[y_te == 0]  # F3: aligned for P9

        auc_if = float(roc_auc_score(y_te, if_p_te))
        auc_ae = float(roc_auc_score(y_te, ae_p_te))
        auc_ens = float(roc_auc_score(y_te, ens_te))
        print(f'  test_known AUC: IF={auc_if:.4f}  AE={auc_ae:.4f}  '
              f'Ens={auc_ens:.4f}')

        mask_twin = (twin_te == 0)
        auc_dedup = float(roc_auc_score(
            y_te[mask_twin], ens_te[mask_twin]))
        print(f'  twin-free ensemble AUC = {auc_dedup:.4f}')

        # ------------------------------------------------------------------
        #  Cascaded FPR (comment 57)
        # ------------------------------------------------------------------
        p_l2_cal = self._l2_score(Xte)
        l2_attack = p_l2_cal >= self.l2_threshold
        normal_mask = (y_te == 0)
        n_normal = int(normal_mask.sum())
        l2_fp_mask = normal_mask & l2_attack
        n_l2_fp = int(l2_fp_mask.sum())
        l2_normal_mask = normal_mask & (~l2_attack)
        n_l2_normal = int(l2_normal_mask.sum())

        l4_fp_mask = np.zeros(len(Xte), dtype=bool)
        if n_l2_normal > 0:
            l4_fp_mask[l2_normal_mask] = ens_te[l2_normal_mask] >= self.tau4
        n_l4_fp = int(l4_fp_mask.sum())
        n_total_fp = n_l2_fp + n_l4_fp
        cascaded_fpr = n_total_fp / max(n_normal, 1)

        self.cascaded_fpr = {
            'n_normal_total': n_normal,
            'n_l2_fp': n_l2_fp,
            'l2_fpr': float(n_l2_fp / max(n_normal, 1)),
            'n_l2_normal_to_l4': n_l2_normal,
            'n_l4_fp_on_l2_normal': n_l4_fp,
            'l4_fpr_conditional': float(n_l4_fp / max(n_l2_normal, 1)),
            'n_cascaded_fp': n_total_fp,
            'cascaded_fpr': float(cascaded_fpr),
            'note': ('comment 57 closure: deployed FPR = (L2 FPs + L4 FPs on '
                     'L2-normal subset) / all Normal in test_known.'),
        }
        l2_fpr = n_l2_fp / max(n_normal, 1)
        self.cascaded_fpr['delta_vs_l2'] = float(cascaded_fpr - l2_fpr)
        # G4: the tau that keeps Layer 4's ADDED false positives inside a budget
        pool = ens_te[np.flatnonzero(normal_mask & ~(p_l2_cal >= self.l2_threshold))] \
            if n_l2_normal else np.zeros(0, np.float32)
        budget = CFG['l4_fpr_matched_budget']
        if len(pool):
            k = int(np.floor(budget * n_normal))
            srt = np.sort(pool)[::-1]
            tau_m = float(np.nextafter(srt[min(k, len(srt) - 1)], np.inf))
        else:
            tau_m = float('inf')
        self.tau4_fpr_matched = tau_m
        added = float((pool >= tau_m).sum() / max(n_normal, 1)) if len(pool) else 0.0
        self.cascaded_fpr['fpr_matched_point'] = {
            'added_fpr_budget': budget, 'tau4_fpr_matched': tau_m,
            'added_fpr_achieved': added,
            'cascaded_fpr_at_matched_tau': float(l2_fpr + added),
            'reading': ('the operating point at which Layer 4 costs the cascade '
                        'almost nothing; its zero-day DR at this tau is reported '
                        'beside the F1-driven one')}
        print(f'  cascaded FPR = {cascaded_fpr * 100:.3f}%  '
              f'(L2 FPs {n_l2_fp:,} + L4 FPs {n_l4_fp:,} / {n_normal:,})')
        print(f'  [FINDING] Layer 4 changes the alarm rate by '
              f'{100 * (cascaded_fpr - l2_fpr):+.3f} pp vs Layer 2 alone '
              f'({100 * l2_fpr:.3f}%)')
        print(f'  [FINDING] FPR-matched tau4 = {tau_m:.4f} '
              f'(adds {100 * added:.3f} pp instead of '
              f'{100 * (cascaded_fpr - l2_fpr):.3f} pp)')

        # ------------------------------------------------------------------
        #  F5: input population at test_known (recorded for the manifest)
        # ------------------------------------------------------------------
        attack_mask = (y_te == 1)
        self.input_population['test_known'] = {
            'l2_flagged_attacks': int((attack_mask & l2_attack).sum()),
            'l2_missed_attacks': int((attack_mask & (~l2_attack)).sum()),
            'l2_normal_fp': n_l2_fp,
            'l2_normal_pass': n_l2_normal,
            'total_normal': n_normal,
        }

        del if_raw_te, ae_raw_te, p_l2_cal, l2_attack
        del l2_fp_mask, l2_normal_mask, l4_fp_mask, attack_mask
        _gc()

        # ------------------------------------------------------------------
        #  Zero-day evaluation with F4 (twin-free DR)
        # ------------------------------------------------------------------
        print('\n  Zero-day evaluation')
        zcols = feats + ['Target', '_dup_in_train']
        has_ts_z = '_ts' in pq.read_schema(self.paths['test_zeroday']).names
        if has_ts_z:
            zcols = zcols + ['_ts']
        Xz_l, yz_l, twin_z_l, ts_z_l = [], [], [], []
        for b in pq.ParquetFile(self.paths['test_zeroday']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=zcols):
            df = b.to_pandas()
            Xz_l.append(df[feats].values.astype(np.float32))
            yz_l.extend(df['Target'].tolist())
            twin_z_l.append(df['_dup_in_train'].values.astype(np.int8))
            if has_ts_z:
                ts_z_l.append(df['_ts'].values.astype(np.int64))
            del df, b
            _gc()
        self._zd_ts = np.concatenate(ts_z_l) if has_ts_z else None
        if not has_ts_z:
            print('  [WARN] zero-day file has no _ts: the time-block bootstrap '
                  'is unavailable and only the class-stratified one is reported')

        self._zd_features = np.vstack(Xz_l); del Xz_l; _gc()
        self._zd_classes = np.array(yz_l); del yz_l; _gc()
        self._zd_twin = np.concatenate(twin_z_l); del twin_z_l; _gc()

        if_raw_z = -self.if_model.score_samples(self._zd_features).astype(np.float32)
        ae_raw_z = self.ae_model.reconstruction_error(self._zd_features)
        if_p_z = self.if_cal.predict_proba(if_raw_z.reshape(-1, 1))[:, 1]
        ae_p_z = self.ae_cal.predict_proba(ae_raw_z.reshape(-1, 1))[:, 1]
        ens_z = (self.alpha * if_p_z + (1 - self.alpha) * ae_p_z
                 ).astype(np.float32)
        self._zd_scores = ens_z

        per_class = {}
        for cls in np.unique(self._zd_classes):
            m = self._zd_classes == cls
            dr = float((ens_z[m] >= self.tau4).mean())
            per_class[cls] = {
                'n': int(m.sum()),
                'dr': dr,
                'mean_score': float(ens_z[m].mean()),
            }
        self.per_class_dr = per_class
        self.zd_weighted_dr = float((ens_z >= self.tau4).mean())
        self.zd_macro_dr = float(np.mean([v['dr'] for v in per_class.values()]))

        unseen = ['SQL-Injection', 'BruteForce-Web', 'BruteForce-XSS']
        unseen_vals = [per_class[c]['dr'] for c in unseen if c in per_class]
        self.zd_unseen_macro_dr = (float(np.mean(unseen_vals))
                                   if unseen_vals else None)

        # F4: twin-free variants
        tf_mask = (self._zd_twin == 0)
        self.zd_weighted_dr_twin_free = (
            float((ens_z[tf_mask] >= self.tau4).mean())
            if tf_mask.any() else None)

        per_class_tf = {}
        for cls in np.unique(self._zd_classes):
            m = (self._zd_classes == cls) & tf_mask
            if m.sum() >= CFG['min_class_dr_rows']:
                per_class_tf[cls] = {
                    'n': int(m.sum()),
                    'dr': float((ens_z[m] >= self.tau4).mean()),
                }
        self.per_class_dr_tf = per_class_tf
        self.zd_macro_dr_twin_free = (
            float(np.mean([v['dr'] for v in per_class_tf.values()]))
            if per_class_tf else None)

        # ------------------------------------------------------------------
        #  Print (F8: unseen macro DR is now printed)
        # ------------------------------------------------------------------
        wf = (f'{self.zd_weighted_dr_twin_free * 100:.2f}%'
              if self.zd_weighted_dr_twin_free is not None else 'n/a')
        mf = (f'{self.zd_macro_dr_twin_free * 100:.2f}%'
              if self.zd_macro_dr_twin_free is not None else 'n/a')

        print(f'  zero-day weighted DR = {self.zd_weighted_dr * 100:.2f}%  '
              f'| twin-free = {wf}')
        print(f'  zero-day MACRO DR    = {self.zd_macro_dr * 100:.2f}%  '
              f'| twin-free = {mf}')
        if self.zd_unseen_macro_dr is not None:
            print(f'  fully-unseen MACRO DR = '
                  f'{self.zd_unseen_macro_dr * 100:.2f}%')

        for cls, d in sorted(per_class.items(), key=lambda x: -x[1]['n']):
            tf = per_class_tf.get(cls)
            tf_str = f'  | tf={tf["dr"] * 100:>6.2f}%  (n={tf["n"]})' if tf else ''
            print(f'    {cls:<24} n={d["n"]:>7,}  '
                  f'DR={d["dr"] * 100:>6.2f}%{tf_str}')

        del if_raw_z, ae_raw_z
        _gc()

    # ======================================================================
    #  PHASE 6 — Per-flow inference latency
    # ======================================================================
    def _p6_latency(self):
        print('\n[PHASE 6] Per-flow inference latency')
        feats = self.features
        Xl = []
        for b in pq.ParquetFile(self.paths['val']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas()
            df = df[df['Target'] == self.l3_reject]
            if len(df):
                Xl.append(df[feats].values.astype(np.float32))
            del df, b
            _gc()
            if sum(len(a) for a in Xl) >= 5_000:
                break

        Xs = np.vstack(Xl)[:5_000]
        del Xl
        _gc()

        for nm, fn in [
            ('IF', lambda r: -self.if_model.score_samples(r)[0]),
            ('AE', lambda r: self.ae_model.reconstruction_error(r)[0]),
            ('Ensemble', lambda r: self._ensemble_score(r)),
            ('DEPLOYED', lambda r: self._deployed_score(r)),   # G8
        ]:
            for _ in range(CFG['lat_warmup']):
                fn(Xs[0].reshape(1, -1))
            t = np.empty(CFG['lat_runs'])
            for i in range(CFG['lat_runs']):
                r = Xs[i % len(Xs)].reshape(1, -1)
                t0 = time.perf_counter_ns()
                fn(r)
                t[i] = time.perf_counter_ns() - t0
            t /= 1e3
            self.latency[nm] = {
                'mean_us': float(t.mean()),
                'p50_us': float(np.percentile(t, 50)),
                'p95_us': float(np.percentile(t, 95)),
                'p99_us': float(np.percentile(t, 99)),
                'runs': CFG['lat_runs'],
                'batch_size': 1,
            }
            print(f'  {nm:<12} mean={t.mean():>7.1f}us  '
                  f'p95={np.percentile(t, 95):>7.1f}us')
        del Xs
        _gc()

    def _deployed_score(self, r):
        """G8: exactly the path that runs in deployment. With alpha = 0 the
        Isolation Forest is not consulted at all."""
        ae_raw = float(self.ae_model.reconstruction_error(r)[0])
        ae_p = float(self.ae_cal.predict_proba(
            np.array([ae_raw]).reshape(-1, 1))[0, 1])
        if abs(self.alpha) < 1e-12:
            return ae_p
        return self._ensemble_score(r)

    def _ensemble_score(self, r):
        if_raw = float(-self.if_model.score_samples(r)[0])
        if_p = float(self.if_cal.predict_proba(
            np.array([if_raw]).reshape(-1, 1))[0, 1])
        ae_raw = float(self.ae_model.reconstruction_error(r)[0])
        ae_p = float(self.ae_cal.predict_proba(
            np.array([ae_raw]).reshape(-1, 1))[0, 1])
        return self.alpha * if_p + (1 - self.alpha) * ae_p

    # ======================================================================
    #  PHASE 7 — Bootstrap L2|L4 + detection cascade
    # ======================================================================
    def _p7_stats(self):
        print('\n[PHASE 7] Bootstrap L2|L4 + detection cascade')
        zd_X = self._zd_features

        # L2 gate on zero-day
        p_cal_z = self._l2_score(zd_X)
        l2_flag = p_cal_z >= self.l2_threshold
        self._zd_l2_flag = l2_flag

        argmax_full = np.full(len(zd_X), -1, dtype=np.int32)
        conf_full = np.zeros(len(zd_X), dtype=np.float32)
        reached_l4 = (~l2_flag).copy()
        n_bypassed = 0
        n_ood_routed = 0

        if l2_flag.sum():
            X_att = zd_X[l2_flag]

            # F1: calibrated probabilities
            p_l3_cal = self._l3_cal_proba(X_att)
            argmax_full[l2_flag] = p_l3_cal.argmax(axis=1)
            conf_full[l2_flag] = p_l3_cal.max(axis=1)

            l3_pred_name = self.l3_le.inverse_transform(
                np.maximum(argmax_full[l2_flag], 0))
            l3_says_reject = (l3_pred_name == self.l3_reject)

            # F2: use the L3-recommended OOD statistic
            if self.l3_ood_statistic == 'energy':
                e_l3 = self._l3_energy(X_att)
                l3_ood = e_l3 > self.l3_tau_ood
            else:
                l3_ood = conf_full[l2_flag] < self.l3_tau_ood

            # Bypass to Layer 5 only if: not the reject class AND
            # high confidence AND not flagged as OOD.
            l3_bypass = (
                (conf_full[l2_flag] >= self.l3_tau_l5)
                & (~l3_says_reject)
                & (~l3_ood)
            )
            reached_l4[l2_flag] = ~l3_bypass
            n_bypassed = int(l3_bypass.sum())
            n_ood_routed = int(l3_ood.sum())

        self._zd_l3_argmax = argmax_full
        self._zd_reached_l4 = reached_l4

        l4_catch = (self._zd_scores >= self.tau4) & reached_l4
        caught = self._zd_l2_flag | l4_catch
        self._zd_caught = caught

        # F5: input population at zero-day
        n_l2_missed = int((~l2_flag).sum())
        self.input_population['zeroday'] = {
            'l2_flagged': int(l2_flag.sum()),
            'l2_missed_reaching_l4': n_l2_missed,
            'l3_ood_routed_from_flagged': n_ood_routed,
            'l3_bypassed_to_l5_only': n_bypassed,
            'reached_l4_total': int(reached_l4.sum()),
            'l4_caught': int(l4_catch.sum()),
            'combined_caught': int(caught.sum()),
            'note': ('L4 receives L2-missed flows plus L3-OOD-routed flows. '
                     'Both paths matter for deployment.'),
        }

        print(f'  L2-alone DR             = {self._zd_l2_flag.mean() * 100:.2f}%')
        print(f'  reached L4 fraction     = {reached_l4.mean() * 100:.2f}% '
              f'({int(reached_l4.sum()):,} rows)')
        print(f'  L4-alone DR (on reached)= {l4_catch.mean() * 100:.2f}%')
        print(f'  L2 OR L4 DR             = {caught.mean() * 100:.2f}%')
        print(f'  input: L2-missed={n_l2_missed:,}  '
              f'L3-OOD-routed={n_ood_routed:,}  '
              f'L3-bypassed-to-L5-only={n_bypassed:,}')

        # ------------------------------------------------------------------
        #  Class-stratified bootstrap (comment 54)
        # ------------------------------------------------------------------
        classes_uniq = np.unique(self._zd_classes)
        cls_idx = {c: np.where(self._zd_classes == c)[0]
                   for c in classes_uniq}
        rng = np.random.RandomState(SEED)
        drs, l2_drs, l4_drs = [], [], []

        for _ in range(CFG['bootstrap_n']):
            sel = []
            for c in classes_uniq:
                idx = cls_idx[c]
                sel.append(rng.choice(idx, len(idx), replace=True))
            idx = np.concatenate(sel)
            drs.append(float(caught[idx].mean()))
            l2_drs.append(float(self._zd_l2_flag[idx].mean()))
            l4_drs.append(float(l4_catch[idx].mean()))

        self.bootstrap = {
            'combined_l2_or_l4_mean': float(np.mean(drs)),
            'combined_l2_or_l4_ci95': [float(np.percentile(drs, 2.5)),
                                        float(np.percentile(drs, 97.5))],
            'l2_alone_mean': float(np.mean(l2_drs)),
            'l2_alone_ci95': [float(np.percentile(l2_drs, 2.5)),
                               float(np.percentile(l2_drs, 97.5))],
            'l4_alone_mean': float(np.mean(l4_drs)),
            'l4_alone_ci95': [float(np.percentile(l4_drs, 2.5)),
                               float(np.percentile(l4_drs, 97.5))],
            'n_bootstrap': CFG['bootstrap_n'],
            # F6: honest naming
            'resample_scheme': ('class-stratified bootstrap over zero-day '
                                'classes (NOT a time-block bootstrap)'),
            'n_classes_resampled': len(classes_uniq),
            'note': ('comment 54 closure: paired class-stratified bootstrap '
                     'on the same rows. A time-block bootstrap would require '
                     'propagating L1 _ts through the zero-day export.'),
        }

        # G5: the contribution, in points, on the same rows
        l2_alone = float(self._zd_l2_flag.mean())
        union = float(caught.mean())
        dr_matched = float(((self._zd_scores >= self.tau4_fpr_matched)
                            & reached_l4).mean()) if np.isfinite(
            getattr(self, 'tau4_fpr_matched', float('inf'))) else 0.0
        union_matched = float((self._zd_l2_flag |
                               ((self._zd_scores >= self.tau4_fpr_matched)
                                & reached_l4)).mean()) if np.isfinite(
            getattr(self, 'tau4_fpr_matched', float('inf'))) else l2_alone
        self.l4_contribution = {
            'l2_alone_dr': l2_alone,
            'union_l2_or_l4_dr': union,
            'l4_added_coverage_pp': float(100 * (union - l2_alone)),
            'l4_alone_dr_at_fpr_matched_tau': dr_matched,
            'union_dr_at_fpr_matched_tau': union_matched,
            'added_coverage_pp_at_fpr_matched_tau': float(100 * (union_matched - l2_alone)),
            'rule': ('measured union on the same rows, never 1-(1-a)(1-b); the '
                     'contribution is the difference, stated in points')}
        print(f'  [FINDING] L4 added coverage = '
              f'{100 * (union - l2_alone):+.2f} pp over L2 alone '
              f'({100 * l2_alone:.2f}% -> {100 * union:.2f}%)')
        print(f'  [FINDING] at the FPR-matched tau: '
              f'{100 * (union_matched - l2_alone):+.2f} pp')

        # G6: time-block bootstrap, if L1 exported _ts on the zero-day file
        if getattr(self, '_zd_ts', None) is not None:
            bid = (self._zd_ts // (CFG['block_minutes'] * 60 * 10**9)).astype(np.int64)
            bid = np.unique(bid, return_inverse=True)[1]
            nb = int(bid.max()) + 1
            idxb = [np.flatnonzero(bid == b) for b in range(nb)]
            rb = np.random.RandomState(SEED + 6)
            tb = []
            for _ in range(CFG['bootstrap_n']):
                sel = np.concatenate([idxb[j] for j in rb.randint(0, nb, nb)])
                tb.append(float(caught[sel].mean()))
            self.bootstrap['combined_time_block_ci95'] = [
                float(np.percentile(tb, 2.5)), float(np.percentile(tb, 97.5))]
            self.bootstrap['n_time_blocks'] = nb
            print(f'  time-block CI (60 min, {nb} blocks): '
                  f'[{np.percentile(tb, 2.5) * 100:.2f}%, '
                  f'{np.percentile(tb, 97.5) * 100:.2f}%]')
        else:
            self.bootstrap['combined_time_block_ci95'] = None

        bc = self.bootstrap['combined_l2_or_l4_ci95']
        print(f'  combined DR = '
              f'{self.bootstrap["combined_l2_or_l4_mean"] * 100:.2f}%  '
              f'CI [{bc[0] * 100:.2f}%, {bc[1] * 100:.2f}%]')

        # Per-class CIs
        per_class_ci = {}
        for c in classes_uniq:
            idx = cls_idx[c]
            if len(idx) < CFG['min_class_dr_rows']:
                continue
            bs = []
            for _ in range(CFG['per_class_bootstrap_n']):
                sub = rng.choice(idx, len(idx), replace=True)
                bs.append(float(caught[sub].mean()))
            per_class_ci[c] = {
                'n': int(len(idx)),
                'dr_mean': float(np.mean(bs)),
                'dr_ci95': [float(np.percentile(bs, 2.5)),
                             float(np.percentile(bs, 97.5))],
            }
        self.per_class_dr_ci = per_class_ci

        del p_cal_z, l2_flag
        _gc()

    # ======================================================================
    #  PHASE 8 — Detection cascade (comment 53)
    # ======================================================================
    def _p8_detection_cascade(self):
        print('\n[PHASE 8] Detection cascade (comment 53)')
        zd_cls = self._zd_classes
        l2_flag = self._zd_l2_flag
        caught = self._zd_caught

        binary_dr = float(l2_flag.mean())
        novel_all = (self._zd_scores >= self.tau4)
        novelty_dr = float(novel_all.mean())

        argmax = self._zd_l3_argmax
        pred_l3 = np.where(
            argmax >= 0,
            self.l3_le.inverse_transform(np.maximum(argmax, 0)),
            'unknown')

        known_classes = set(c for c in self.l3_classes
                            if c != self.l3_reject)
        att_correct = 0
        att_total = 0
        for cls in np.unique(zd_cls):
            mask = (zd_cls == cls)
            if cls in known_classes:
                att_total += int(mask.sum())
                att_correct += int((pred_l3[mask] == cls).sum())
        att_rate = (att_correct / att_total) if att_total > 0 else None

        self.detection_cascade = {
            'binary_l2': {
                'rate': binary_dr,
                'definition': 'L2 flags the flow as attack',
                'numerator': int(l2_flag.sum()),
                'denominator': int(len(zd_cls)),
            },
            'novelty_l4': {
                'rate': novelty_dr,
                'definition': 'L4 ensemble >= tau4 (independent of routing)',
                'numerator': int(novel_all.sum()),
                'denominator': int(len(zd_cls)),
            },
            'attribution_l3': {
                'rate': att_rate,
                'definition': ('L3 assigns the CORRECT known family '
                               '(reject class excluded); denominator = ALL '
                               'rows of that known class in zero-day, '
                               'including rows L2 did not flag'),
                'numerator': att_correct,
                'denominator': att_total,
                'classes_covered': sorted(known_classes),
                'reject_class_excluded': self.l3_reject,
            },
            'safe_response': {
                'rate': float(caught.mean()),
                'definition': 'flow reaches at least ALERT_SOC',
                'numerator': int(caught.sum()),
                'denominator': int(len(zd_cls)),
            },
            'note': ('comment 53 closure: 4 distinct measurements. Reject '
                     'class excluded from attribution count.'),
        }

        print(f'  binary (L2)      : {binary_dr * 100:>5.2f}%')
        print(f'  novelty (L4)     : {novelty_dr * 100:>5.2f}%')
        if att_rate is not None:
            print(f'  attribution (L3) : {att_rate * 100:>5.2f}%  '
                  f'(known classes only, n={att_total:,})')
        print(f'  safe response    : {caught.mean() * 100:>5.2f}%')

    # ======================================================================
    #  PHASE 9 — Drift monitoring (comment 59, F3 applies the sort)
    # ======================================================================
    def _p9_drift(self):
        print('\n[PHASE 9] Drift monitoring (comment 59)')

        if_raw = -self.if_model.score_samples(
            self._X_drift_ref).astype(np.float32)
        ae_raw = self.ae_model.reconstruction_error(self._X_drift_ref)
        if_p_ref = self.if_cal.predict_proba(if_raw.reshape(-1, 1))[:, 1]
        ae_p_ref = self.ae_cal.predict_proba(ae_raw.reshape(-1, 1))[:, 1]
        ens_ref = self.alpha * if_p_ref + (1 - self.alpha) * ae_p_ref
        del if_raw, ae_raw, if_p_ref, ae_p_ref
        _gc()
        n_ref = len(ens_ref)

        # F3: sort test Normal scores by L1 _ts before windowing
        sorted_by_ts = self._test_normal_ts is not None
        if sorted_by_ts:
            order = np.argsort(self._test_normal_ts, kind='stable')
            scores = self._test_normal_scores[order]
        else:
            print('  [WARN] _ts missing; drift windows are shuffled '
                  '(F3 not applied)')
            scores = self._test_normal_scores

        n = len(scores)
        n_win = CFG['drift_windows']
        win_len = max(100, n // n_win)
        ks_stats = []

        for w in range(n_win):
            seg = scores[w * win_len:(w + 1) * win_len]
            if len(seg) < 100:
                continue
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                st, _ = ks_2samp(ens_ref, seg)
            ks_stats.append(float(st))

        flags = [1 if s > CFG['drift_ks_thresh'] else 0 for s in ks_stats]
        longest = 0
        cur = 0
        for f in flags:
            if f:
                cur += 1
                longest = max(longest, cur)
            else:
                cur = 0
        detected = longest >= CFG['drift_persist']

        self.drift = {
            'reference_size': int(n_ref),
            'reference_source': 'held-out X_drift_ref (never seen by IF/AE)',
            'test_size': int(n),
            'n_windows': int(n_win),
            'window_length': int(win_len),
            'windows_sorted_by_ts': bool(sorted_by_ts),
            'ks_stats': ks_stats,
            'ks_thresh': CFG['drift_ks_thresh'],
            'persistence_required': CFG['drift_persist'],
            'longest_consecutive_above_threshold': int(longest),
            'drift_detected': bool(detected),
            'note': ('comment 59 closure: KS effect size with persistence rule; '
                     'reference held out from training; test windows sorted by '
                     'L1 _ts so consecutive windows are temporally adjacent.'),
        }

        print(f'  reference = {n_ref:,} rows (held out)')
        print(f'  windows   = {len(ks_stats)} x {win_len:,}  '
              f'(sorted by _ts: {sorted_by_ts})')
        if ks_stats:
            print(f'  max KS    = {max(ks_stats):.4f}  '
                  f'({sum(flags)}/{len(flags)} above '
                  f'{CFG["drift_ks_thresh"]})')
        print(f'  longest run = {longest}  |  drift detected = {detected}')

    # ======================================================================
    #  PHASE 10 — Figures (7 PDFs, IEEE-compliant)
    # ======================================================================
    def _p10_figures(self):
        print('\n[PHASE 10] Figures (7 PDFs, IEEE-compliant)')
        _plt_style()
        y = self._y_test
        n_mask = (y == 0)
        a_mask = (y == 1)
        W = 7.16

        # ------------------------------------------------------------------
        #  Fig 18 — score distributions
        # ------------------------------------------------------------------
        fig, axes = plt.subplots(1, 3, figsize=(W, 2.4))
        for ax, arr, name in [
            (axes[0], self._if_scores_test, 'IF'),
            (axes[1], self._ae_scores_test, 'AE'),
            (axes[2], self._scores_test, 'Ensemble'),
        ]:
            ax.hist(arr[n_mask], bins=60, density=True, alpha=0.65,
                    color=COLORS['normal'], edgecolor=COLORS['edge'],
                    lw=0.3, label='Normal')
            ax.hist(arr[a_mask], bins=60, density=True, alpha=0.65,
                    color=COLORS['attack'], edgecolor=COLORS['edge'],
                    lw=0.3, label='Attack')
            ax.hist(self._zd_scores, bins=60, density=True, alpha=0.45,
                    color=COLORS['zday'], edgecolor='none', label='Zero-day')
            ax.axvline(self.tau4, color=COLORS['thresh'], ls='--', lw=1.2,
                       label=f'τ4={self.tau4:.3f}')
            ax.set_xlabel('Calibrated score')
            ax.set_xlim(0, 1)
            ax.set_title(name)
            ax.legend(fontsize=6)
            ax.grid(alpha=0.3)
        fig.suptitle('Fig. 18.  Layer-4 calibrated score distributions',
                     fontsize=9)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig18_score_distributions.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        # ------------------------------------------------------------------
        #  Fig 19 — ROC on test_known (comment 30)
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(3.6, 3.0))
        for arr, col, name in [
            (self._if_scores_test, COLORS['if_model'], 'IF'),
            (self._ae_scores_test, COLORS['ae_model'], 'AE'),
            (self._scores_test, COLORS['ensemble'], 'Ensemble'),
        ]:
            fpr, tpr, _ = roc_curve(y, arr)
            auc = roc_auc_score(y, arr)
            ax.plot(fpr, tpr, color=col,
                    lw=1.4 if name == 'Ensemble' else 1.0,
                    label=f'{name} AUC={auc:.3f}')
        ax.plot([0, 1], [0, 1], '--', color=COLORS['normal'], lw=0.8)

        pred_tau = (self._scores_test >= self.tau4).astype(np.int8)
        cm = confusion_matrix(y, pred_tau)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            fpr_tau = fp / max(fp + tn, 1)
            tpr_tau = tp / max(tp + fn, 1)
            ax.scatter([fpr_tau], [tpr_tau], s=60, marker='o',
                       color=COLORS['thresh'], edgecolor=COLORS['edge'],
                       lw=1.0, zorder=5,
                       label=f'τ4 (FPR={fpr_tau * 100:.2f}%)')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('Fig. 19.  Layer-4 ROC on test_known', fontsize=8.5)
        ax.legend(fontsize=6, loc='lower right')
        ax.grid(alpha=0.3)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig19_roc.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        # ------------------------------------------------------------------
        #  Fig 20 — AE reconstruction error
        # ------------------------------------------------------------------
        feats = self.features
        Xl, yl = [], []
        for b in pq.ParquetFile(self.paths['test_known']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas()
            Xl.append(df[feats].values.astype(np.float32))
            yl.extend((df['Target'] != self.l3_reject)
                      .astype(np.int8).tolist())
            del df, b
            _gc()
        Xte = np.vstack(Xl)
        yte = np.array(yl)
        del Xl, yl
        _gc()

        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(Xte), min(200_000, len(Xte)), replace=False)
        ae_err = self.ae_model.reconstruction_error(Xte[idx])

        fig, axes = plt.subplots(1, 2, figsize=(W, 2.4))
        for ax, xmax, ttl in [(axes[0], None, 'Full range'),
                              (axes[1], 0.5, 'Zoom [0, 0.5]')]:
            ax.hist(ae_err[yte[idx] == 0], bins=80, density=True,
                    alpha=0.65, color=COLORS['normal'], label='Normal')
            ax.hist(ae_err[yte[idx] == 1], bins=80, density=True,
                    alpha=0.65, color=COLORS['attack'], label='Attack')
            if xmax:
                ax.set_xlim(0, xmax)
            ax.set_xlabel('AE reconstruction error')
            ax.set_ylabel('Density')
            ax.set_title(ttl)
            ax.legend(fontsize=6)
            ax.grid(alpha=0.3)
        fig.suptitle('Fig. 20.  AE reconstruction error', fontsize=9)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig20_ae_reconstruction.{ext}',
                        bbox_inches='tight')
        plt.close(fig)
        del Xte, ae_err
        _gc()

        # ------------------------------------------------------------------
        #  Fig 21 — zero-day DR per class with CI
        # ------------------------------------------------------------------
        per = self.per_class_dr_ci
        classes = sorted(per.keys(), key=lambda c: -per[c]['n'])
        fig, ax = plt.subplots(figsize=(W, 2.8))
        x = np.arange(len(classes))
        vals = [per[c]['dr_mean'] * 100 for c in classes]
        lo = [max(0, per[c]['dr_mean'] * 100 - per[c]['dr_ci95'][0] * 100)
              for c in classes]
        hi = [max(0, per[c]['dr_ci95'][1] * 100 - per[c]['dr_mean'] * 100)
              for c in classes]
        ax.bar(x, vals, color=COLORS['zday'], edgecolor=COLORS['edge'],
               alpha=0.85, yerr=[lo, hi], capsize=3,
               error_kw={'ecolor': COLORS['edge'], 'lw': 1.0})
        ax.axhline(self.zd_macro_dr * 100, color=COLORS['if_model'], ls='--',
                   lw=1.2, label=f'Macro {self.zd_macro_dr * 100:.1f}%')
        ax.axhline(self.zd_weighted_dr * 100, color=COLORS['ae_model'],
                   ls=':', lw=1.2,
                   label=f'Weighted {self.zd_weighted_dr * 100:.1f}%')
        for xi, c in enumerate(classes):
            ax.text(xi, vals[xi] + 2, f'{vals[xi]:.0f}',
                    ha='center', fontsize=6.5)
        ax.set_xticks(x)
        ax.set_xticklabels(
            [c.replace('BruteForce-', 'BF-') for c in classes],
            rotation=35, ha='right', fontsize=6.5)
        ax.set_ylim(-8, 115)
        ax.set_ylabel('DR (%)')
        ax.set_title('Fig. 21.  Zero-day DR per class (95% CI); '
                     'macro vs weighted', fontsize=8.5)
        ax.legend(fontsize=6)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig21_zeroday_dr.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        # ------------------------------------------------------------------
        #  Fig 22 — detection cascade
        # ------------------------------------------------------------------
        dc = self.detection_cascade
        labels = ['Binary\n(L2)', 'Novelty\n(L4)', 'Attribution\n(L3)',
                  'Safe\nresponse']
        vals = [dc['binary_l2']['rate'],
                dc['novelty_l4']['rate'],
                dc['attribution_l3']['rate'] or 0.0,
                dc['safe_response']['rate']]
        cols = [COLORS['if_model'], COLORS['zday'], COLORS['ae_model'],
                COLORS['attack']]
        fig, ax = plt.subplots(figsize=(3.6, 2.8))
        x = np.arange(len(labels))
        ax.bar(x, vals, color=cols, edgecolor=COLORS['edge'], width=0.6)
        for xi, v in enumerate(vals):
            ax.text(xi, v + 0.02, f'{v * 100:.1f}%',
                    ha='center', fontsize=6.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=6.5)
        ax.set_ylim(0, 1.12)
        ax.set_ylabel('Rate')
        ax.set_title('Fig. 22.  Detection cascade (4 levels)', fontsize=8.5)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig22_detection_cascade.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        # ------------------------------------------------------------------
        #  Fig 23 — cascaded FPR
        # ------------------------------------------------------------------
        cf = self.cascaded_fpr
        fig, ax = plt.subplots(figsize=(3.6, 2.6))
        labels = ['L2 FPR', 'L4 FPR\n(cond.)', 'Cascaded\nFPR']
        vals = [cf['l2_fpr'], cf['l4_fpr_conditional'], cf['cascaded_fpr']]
        cols = [COLORS['if_model'], COLORS['ae_model'], COLORS['zday']]
        x = np.arange(len(labels))
        ax.bar(x, vals, color=cols, edgecolor=COLORS['edge'], width=0.55)
        for xi, v in enumerate(vals):
            ax.text(xi, v + 0.002, f'{v * 100:.3f}%',
                    ha='center', fontsize=6.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=6.5)
        ax.set_ylabel('FPR')
        ax.set_title('Fig. 23.  Cascaded FPR', fontsize=8.5)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig23_cascaded_fpr.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        # ------------------------------------------------------------------
        #  Fig 24 — drift monitor
        # ------------------------------------------------------------------
        ks_stats = self.drift['ks_stats']
        fig, ax = plt.subplots(figsize=(W, 2.6))
        x = np.arange(len(ks_stats))
        ax.bar(x, ks_stats, color=COLORS['if_model'],
               edgecolor=COLORS['edge'])
        ax.axhline(CFG['drift_ks_thresh'], color=COLORS['zday'], ls='--',
                   lw=1.2, label=f'thresh={CFG["drift_ks_thresh"]}')
        ax.set_xlabel(f'Window index (len {self.drift["window_length"]:,}, '
                      f'sorted by _ts)')
        ax.set_ylabel('KS vs held-out ref')
        ax.set_title(f'Fig. 24.  Drift (persist ≥ {CFG["drift_persist"]} '
                     f'→ retrain)', fontsize=8.5)
        ax.legend(fontsize=6)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        for ext in ('pdf', 'png'):
            fig.savefig(self.figs / f'L4_fig24_drift_monitor.{ext}',
                        bbox_inches='tight')
        plt.close(fig)

        print(f'  {len(list(self.figs.glob("*.pdf")))} PDFs written')

    # ======================================================================
    #  PHASE 11 — Save models + manifest
    # ======================================================================
    def _p11_save(self):
        print('\n[PHASE 11] Saving')

        _save(self.if_model, self.l4 / 'if_model.pkl')
        _save(self.ae_model, self.l4 / 'ae_model.pkl')
        torch.save(self.ae_model.state_dict(), self.l4 / 'ae_model_state.pt')
        _save(self.if_cal, self.l4 / 'if_calibrator.pkl')
        _save(self.ae_cal, self.l4 / 'ae_calibrator.pkl')

        _atomic_json({
            'input_dim': len(self.features),
            'hidden_dims': CFG['ae_hidden'],
            'input_transform': SparseAutoencoder.INPUT_TRANSFORM,
        }, self.l4 / 'ae_architecture.json')

        _atomic_json({
            'alpha': self.alpha,
            'tau4': self.tau4,
            'features': self.features,
            'target_fpr_l4': CFG['target_fpr_l4'],
            'calibration_half': self.calibration_half,
            'threshold_half': self.threshold_half,
            'l3_thresholds': self.l3_thresholds,
        }, self.l4 / 'layer4_config.json')

        manifest = {
            'version': CFG['version'],
            'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ',
                                        time.gmtime()),
            'seed': SEED,
            'environment': _hardware(),
            'l1_manifest_version': self.m1.get('version'),
            'l2_manifest_version': self.m2.get('version'),
            'l3_manifest_version': self.m3.get('version'),
            'features': self.features,
            'feature_count': len(self.features),
            'calibration_half': self.calibration_half,
            'threshold_half': self.threshold_half,
            'model': {
                'IF': (f'{CFG["if_n_estimators"]} trees, '
                       f'max_samples={CFG["if_max_samples"]}'),
                'AE': (f'{len(self.features)} → {CFG["ae_hidden"]} → '
                       f'{len(self.features)}'),
                'alpha': self.alpha,
                'tau4': self.tau4,
                'selection': ('multi-objective α on val half B; '
                              'novelty proxy = L2 false negatives'),
                'terminology': ('unsupervised representation learning '
                                'with supervised calibration (comment 55)'),
            },
            'alpha_selection': self.alpha_diag,
            'test_known': {
                'auc_if': float(roc_auc_score(
                    self._y_test, self._if_scores_test)),
                'auc_ae': float(roc_auc_score(
                    self._y_test, self._ae_scores_test)),
                'auc_ensemble': float(roc_auc_score(
                    self._y_test, self._scores_test)),
                'auc_ensemble_twin_free': float(roc_auc_score(
                    self._y_test[self._twin_test == 0],
                    self._scores_test[self._twin_test == 0])),
                'note': 'comment 30: ROC on test_known, not on val',
            },
            'zero_day': {
                'weighted_dr': self.zd_weighted_dr,
                'macro_dr': self.zd_macro_dr,
                'unseen_macro_dr': self.zd_unseen_macro_dr,
                'weighted_dr_twin_free': self.zd_weighted_dr_twin_free,
                'macro_dr_twin_free': self.zd_macro_dr_twin_free,
                'per_class': self.per_class_dr,
                'per_class_ci': self.per_class_dr_ci,
                'per_class_twin_free': self.per_class_dr_tf,
                'note': ('comment 34 closure + L1 twin-in-train 45% on '
                         'zero-day: full and twin-free reported side by side.'),
            },
            'detection_cascade': self.detection_cascade,
            'cascaded_fpr': self.cascaded_fpr,
            'input_population': self.input_population,
            'drift': self.drift,
            'bootstrap': self.bootstrap,
            'l4_contribution': self.l4_contribution,
            'latency': dict(self.latency, deployed_path=(
                'AE only (alpha = 0; the Isolation Forest is not consulted)'
                if abs(self.alpha) < 1e-12 else
                f'IF + AE blend at alpha = {self.alpha}'),
                note=('IF and Ensemble rows are evidence for the negative '
                      'result; DEPLOYED is the path that runs in production')),
            'saved_files': {
                'if_model': str(self.l4 / 'if_model.pkl'),
                'ae_model': str(self.l4 / 'ae_model.pkl'),
                'if_cal': str(self.l4 / 'if_calibrator.pkl'),
                'ae_cal': str(self.l4 / 'ae_calibrator.pkl'),
                'config': str(self.l4 / 'layer4_config.json'),
            },
            'l5_contract': {
                'if_model': str(self.l4 / 'if_model.pkl'),
                'ae_model': str(self.l4 / 'ae_model.pkl'),
                'if_cal': str(self.l4 / 'if_calibrator.pkl'),
                'ae_cal': str(self.l4 / 'ae_calibrator.pkl'),
                'config': str(self.l4 / 'layer4_config.json'),
                'alpha': self.alpha,
                'tau4': self.tau4,
                'features': self.features,
                'l3_classes': self.l3_classes,
                'l3_reject': self.l3_reject,
                'l3_ood_statistic': self.l3_ood_statistic,
            },
            'figures': {
                'Fig_18': 'L4_fig18_score_distributions.pdf',
                'Fig_19': 'L4_fig19_roc.pdf (comment 30)',
                'Fig_20': 'L4_fig20_ae_reconstruction.pdf',
                'Fig_21': 'L4_fig21_zeroday_dr.pdf (comment 34)',
                'Fig_22': 'L4_fig22_detection_cascade.pdf (comment 53)',
                'Fig_23': 'L4_fig23_cascaded_fpr.pdf (comment 57)',
                'Fig_24': 'L4_fig24_drift_monitor.pdf (comment 59)',
            },
            'paper_map': {
                'Table_X':  'detection_cascade (4-level, comment 53)',
                'Table_XI': ('bootstrap (class-stratified paired CI, '
                             'comment 54) + per_class_ci'),
                'Table_XII': 'cascaded_fpr (comment 57)',
                'Fig_18':    'score distributions',
                'Fig_19':    'ROC on test_known (comment 30)',
                'Fig_20':    'AE reconstruction',
                'Fig_21':    'zero-day DR macro vs weighted (comment 34)',
                'Fig_22':    'detection cascade (comment 53)',
                'Fig_23':    'cascaded FPR (comment 57)',
                'Fig_24':    'drift monitor (comment 59)',
                'Sec_VII-A': 'architecture + multi-objective α (comment 58)',
                'Sec_VII-B': 'zero-day DR (comment 34)',
                'Sec_VII-C': 'calibration on natural prior (comment 56)',
                'Sec_VII-D': 'cascaded FPR (comment 57)',
                'Sec_VII-E': 'detection cascade (comment 53)',
                'Sec_VII-F': 'drift monitoring (comment 59)',
                'Sec_VII-G': ('terminology: unsupervised representation with '
                              'supervised calibration (comment 55)'),
            },
        }
        _atomic_json(manifest, self.manifest_path)
        print(f'  manifest_l4.json written')

    # ======================================================================
    #  EXECUTE
    # ======================================================================
    def execute(self):
        t0 = time.time()
        self._p1_load_normal()
        self._p2_train_models()
        self._p3_calibrate()
        self._p4_alpha_multobjective()
        self._p5_evaluate()
        self._p6_latency()
        self._p7_stats()
        self._p8_detection_cascade()
        self._p9_drift()
        self._p10_figures()
        self._p11_save()

        print('\n' + '=' * 76)
        print(f'  LAYER 4 R8.0 COMPLETE | {(time.time() - t0) / 60:.1f} min')
        print(f'  α = {self.alpha}   τ4 = {self.tau4:.4f}')
        wf = (f'{self.zd_weighted_dr_twin_free * 100:.2f}%'
              if self.zd_weighted_dr_twin_free is not None else 'n/a')
        print(f'  weighted DR = {self.zd_weighted_dr * 100:.2f}%  '
              f'(twin-free {wf})')
        print(f'  macro DR    = {self.zd_macro_dr * 100:.2f}%')
        b = self.bootstrap
        print(f'  combined L2|L4 DR = '
              f'{b["combined_l2_or_l4_mean"] * 100:.2f}%  '
              f'CI [{b["combined_l2_or_l4_ci95"][0] * 100:.2f}%, '
              f'{b["combined_l2_or_l4_ci95"][1] * 100:.2f}%]')
        print(f'  cascaded FPR = '
              f'{self.cascaded_fpr["cascaded_fpr"] * 100:.3f}%')
        c = self.l4_contribution
        print(f'  L4 added coverage = {c["l4_added_coverage_pp"]:+.2f} pp '
              f'(FPR-matched: {c["added_coverage_pp_at_fpr_matched_tau"]:+.2f} pp)')
        print(f'  alarm-rate change vs L2 alone = '
              f'{100 * self.cascaded_fpr["delta_vs_l2"]:+.3f} pp')
        if not getattr(self, 'if_usable', True):
            print('  [FINDING] Isolation Forest disabled (val AUC below floor): '
                  'deployed configuration is AE-only')
        dep = self.latency.get('DEPLOYED', {})
        if dep:
            print(f'  DEPLOYED path latency P95 = {dep["p95_us"]:.1f} us '
                  f'({"AE only" if abs(self.alpha) < 1e-12 else "blend"})')
        print(f'  drift detected = {self.drift["drift_detected"]}')
        print(f'  7 figures written (IEEE-compliant, 600 dpi)')
        print('=' * 76)


# =============================================================================
#  ENTRY POINT
# =============================================================================
if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='TITANIUM-6 Layer 4 R8.0')
    ap.add_argument('--l1dir',
                    default=os.environ.get('T6_L1', '/kaggle/working/l1_R8'))
    ap.add_argument('--l2dir',
                    default=os.environ.get('T6_L2', '/kaggle/working/l2_R8'))
    ap.add_argument('--l3dir',
                    default=os.environ.get('T6_L3', '/kaggle/working/l3_R8'))
    ap.add_argument('--l4dir',
                    default=os.environ.get('T6_L4', '/kaggle/working/l4_R8'))
    a, _ = ap.parse_known_args()

    # F7: fail-loud path guard with "found here" fallback
    for tag, d, f in (
        ('Layer 1', Path(a.l1dir), 'manifest.json'),
        ('Layer 2', Path(a.l2dir), 'manifest_l2.json'),
        ('Layer 3', Path(a.l3dir), 'manifest_l3.json'),
    ):
        if not (d / f).exists():
            root = d.parent if d.parent.exists() else Path('.')
            found = sorted(str(q.parent) for q in root.glob('*/' + f))
            raise SystemExit(
                f'[FATAL] no {f} in {d}\n'
                f'        {tag} outputs found here: {found or "none"}\n'
                f'        pass --l1dir/--l2dir/--l3dir, or set T6_L1/T6_L2/T6_L3.')

    Layer4Pipeline(a.l1dir, a.l2dir, a.l3dir, a.l4dir).execute()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  L3 calibrators : loaded (12/12 fitted)
  TITANIUM-6 | LAYER 4 | TITANIUM6_LAYER4_R8.0
  features  : 77
  L2 gate   : LightGBM (τ=0.0793)
  L3 classes: 12  reject="Normal"  τ_L5=0.2967
  L3 OOD    : statistic=energy  τ=-0.3071
  L4 dir    : /kaggle/working/l4_R8

[PHASE 1] Load Normal-only training flows (drift ref held out)
  drift reference : 50,000 rows (never seen by IF/AE)
  training set    : 500,000 rows

[PHASE 2] Train IF + AE on Normal only
  IF done 3.1s
    AE epochs=50 best_val_loss=0.004787
  AE done 196.1s

[PHASE 3] Calibrate on val half A; reserve half B
  val A (calibrate) : n=200,000  atk=31,980  nrm=168,020
  val B (α/τ)       : n=200,000  atk=31,845  nrm=168,155
  IF Brier (val A, natural prior) = 0.13410
  AE Brier (val A, natural prior) = 0.12556

[PHASE 4] Multi-objective α on val B with L2-FN novelty proxy
  val B L2 false negatives (novelty proxy) = 757
  component val AUC: IF=0.5044  AE=0.7270
 

In [6]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 5 | R8.0
#  R8.0 over R6.2:
#   T1 AUTO-ACTION PRECISION GATE. The R6.2 run ISOLATED 32,091 true-Normal
#      flows (7.6% of Normal seen), 32,076 of them predicted as Infiltration.
#      Automation is now allowed per family only where its automated
#      decisions are measured precise on VALIDATION (Wilson-95% lower bound
#      >= 0.99, >= 100 cases); all other families are downgraded to
#      ALERT_SOC. The gate table is in manifest_l5.json -> auto_action_gate.
#   T2 Refuses an L4 autoencoder without the R8 input transform.
#   T3 Paths and version strings follow the single R8 run configuration.
#   T4 AE loaded from ae_architecture.json + state_dict (no class pickle), so
#      Layer 5 runs in a fresh kernel.
#
#  R6.2 fixes the two defects the first full R6.1 run exposed:
#   Y1 PLAYBOOK entries are 6-tuples (severity, action, mitre, description,
#      reversal, sla_seconds). The manifest writer read v[5] as the reversal
#      and v[6] as the SLA, so _p8_save raised IndexError AFTER 724,954
#      decisions and a verified hash chain. Correct indices are v[4] and v[5].
#   Y2 The playbook-coverage pass added the REJECT class to the playbook with
#      an ALERT_SOC default. ResponseEngine.decide() short-circuits the reject
#      class to NO_ACTION, so the entry was unreachable and misleading in the
#      printed table and in Table XII. The reject class is now skipped.
#  R6.1 header follows.
#  TITANIUM-6 | LAYER 5 | R6.1 (v3.1, superseded)
#  R6.1 fixes four defects found by audit of the R6.0 draft:
#   X1 logsumexp was used by _l3_energy but never imported: with L3's
#      recommended statistic (energy) the run raised NameError in Phase 3.
#   X2 self.l3_thresholds['ood'] was read in three places (startup print,
#      figure label, manifest) but no such key exists; the file carries
#      l5 / ood_conf / ood_energy. All three now use _tau_ood().
#   X3 a global warnings.filterwarnings('ignore') suppressed everything;
#      Layers 1-4 suppress nothing globally, and L5 must not either.
#   X4 _l4_batch recomputed the autoencoder reconstruction twice when the
#      forest is enabled.
#   X5 verify_chain now recomputes each record's hash from its own content
#      (proved by a tamper test); the previous version followed prev_hash
#      links only and reported a tampered file as verified.
#  R6.0 header follows.
#  TITANIUM-6 | LAYER 5 | R6.0 (v3.0, superseded)
#  Reads L1 R5.4, L2 R6.5, L3 R7.12, L4 R6.1. Contract-based, bracket access.
#
#   H1 OOD STATISTIC. L3 R7.12 measured max-softmax AUROC 0.5713 vs energy
#      0.6050 and exports ood_statistic='energy' with tau=-2.8190. Earlier
#      revisions compared max-softmax confidence against a confidence tau.
#      R6.0 reads the statistic from the L3 contract and computes the
#      matching score (energy = -logsumexp over raw margins).
#   H2 L3 CALIBRATORS. tau_L5 and tau_OOD are defined on CALIBRATED
#      probabilities. R6.0 loads best_calibrators_l3.pkl, applies per-class
#      Platt on the logit, renormalises, and refuses to run without them.
#   H3 ISOLATION FOREST SKIPPED WHEN DISABLED. L4 R6.1 measured IF val AUC
#      0.5044 / test 0.3887 and set alpha = 0. R6.0 reads alpha from the L4
#      contract; at alpha = 0 the forest is never loaded or scored (it cost
#      ~12 ms per flow for a below-chance signal).
#   H4 REJECT CLASS AND FEATURES FROM THE CONTRACT, never hard-coded.
#   H5 PATH DEFAULTS track L1 R5.4 / L2 R6.5 / L3 R7.12 / L4 R6.1, and a
#      missing manifest fails with the directories that DO contain one.
#   H6 CAPTURE RATE PROVENANCE. capture_rate_dps is a planning assumption,
#      not a measurement on this dataset; the manifest says so, so the paper
#      cannot present the per-hour figure as measured.
#   H7 GENESIS BINDS TO INPUTS. The audit chain's genesis commits to the
#      sha256 of all four upstream manifests, so tampering with any of them
#      invalidates the chain.
#   H8 THREE-WAY ROUTE RECORDED. L3 emits a middle band; R6.0 keeps
#      l3_confident / l3_middle / l3_ood / l4_zeroday / normal_pass distinct
#      in the audit record and the route histogram.
#   H9 ATTRIBUTION CROSS-CHECK. L4 R6.1 reported 0.00% family attribution on
#      224,026 partial-holdout zero-day rows. R6.0 counts, independently,
#      how often each true zero-day family is predicted, so that finding is
#      confirmed or refuted by a second implementation.
#  R5.1 header follows.
#  TITANIUM-6 | LAYER 5 | R5.1 (v2.1, superseded) | CSE-CIC-IDS2018 | IEEE OJ-COMS
#
#  SOAR response engine. Reads L2/L3/L4 manifests. No ground-truth labels
#  in the decision path.
#
#  Comments closed (Layer-5-owned):
#   60  zero-day routing is model-derived, not oracle
#   61  FP action rate on Normal flows measured (with family breakdown)
#   62  audit trail hash-chained, verified
#   46  decision latency measured with hardware
#
#  R5.1 fixes over R5.0:
#   1. Batched audit writes (10k/flush) — ~100x fewer syscalls
#   2. Removed dead 1M-row count loop
#   3. Fixed fragile l2_threshold expression (file-handle leak)
#   4. Confidence for L4-derived zero-day flows = L4 score, not 0
#   5. Real verify_chain() result in manifest
#   6. decision_samples size exposed in manifest
#   7. n_attack_seen reported
#   8. FP audit family breakdown
#   9. Audit file size in manifest
#  10. finalize() before verify
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os, gc, json, time, pickle, warnings, argparse, uuid, hashlib, platform
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.special import logsumexp
import psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import lightgbm as lgb
import xgboost as xgb
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)

CFG = {
    'version': 'TITANIUM6_LAYER5_R8.0',
    # T1 auto-action precision gate (decided on VALIDATION only)
    'gate_val_n'         : 1_000_000,
    'gate_min_support'   : 100,
    'gate_min_precision' : 0.99,      # Wilson 95% LOWER bound must reach this
    'seed'            : SEED,
    'conf_auto'       : 0.90,
    'conf_alert'      : 0.70,
    'eval_chunk'      : 100_000,
    'known_sample_n'  : 500_000,
    'audit_batch'     : 10_000,
    'decision_samples_cap': 100_000,
    'decisions_warmup': 50,
    'decisions_runs'  : 1_000,
    'capture_rate_dps': 18.7,
    'capture_rate_source': ('PLANNING ASSUMPTION (mid-size enterprise SOC); '
                            'not measured on CSE-CIC-IDS2018. Any per-hour '
                            'figure derived from it is an estimate.'),
}

COLORS = {
    'lgbm':'#002F6C','xgb':'#007A33','rf':'#C00000',
    'normal':'#CCCCCC','attack':'#F5A623','edge':'#333333',
    'train':'#002F6C','val':'#007A33','test':'#4A90D9','zday':'#C00000',
    'amber':'#F5A623','crimson':'#C00000',
}
ACTION_COLORS = {
    'AUTO_BLOCK'      : '#7B0000',
    'AUTO_RATE_LIMIT' : '#C00000',
    'ISOLATE'         : '#E74C3C',
    'ALERT_SOC'       : '#F5A623',
    'LOG_MONITOR'     : '#4A90D9',
    'NO_ACTION'       : '#CCCCCC',
}
SEVERITY_COLORS = {
    'CRITICAL':'#7B0000','HIGH':'#C00000','MEDIUM':'#F5A623','LOW':'#4A90D9',
}

PLAYBOOK = {
    'DoS-Hulk': ('HIGH','AUTO_RATE_LIMIT','T1498.001',
                 'Rate-limit source IP to 10% bandwidth cap.',
                 'REMOVE_RATE_LIMIT', 300),
    'DoS-GoldenEye': ('HIGH','AUTO_BLOCK','T1498.001',
                      'Block source IP; HTTP keep-alive attack.',
                      'UNBLOCK_IP', 60),
    'DoS-Slowloris': ('HIGH','AUTO_RATE_LIMIT','T1499.002',
                      'Close idle TCP connections exceeding 30s.',
                      'RESTORE_CONNECTIONS', 120),
    'DoS-SlowHTTP': ('HIGH','AUTO_RATE_LIMIT','T1499.002',
                     'Reduce HTTP keep-alive timeout.',
                     'RESTORE_HTTP_TIMEOUT', 120),
    'DDoS-HOIC': ('CRITICAL','AUTO_BLOCK','T1498.001',
                  'Block source /24; alert ISP abuse contact.',
                  'UNBLOCK_SUBNET', 30),
    'DDoS-LOIC-HTTP': ('CRITICAL','AUTO_BLOCK','T1498.001',
                       'Block /24; rate-limit HTTP flood.',
                       'UNBLOCK_SUBNET', 30),
    'DDoS-LOIC-UDP': ('CRITICAL','AUTO_BLOCK','T1498.002',
                      'Drop UDP flood at border firewall.',
                      'UNBLOCK_UDP', 30),
    'BruteForce-FTP': ('HIGH','AUTO_BLOCK','T1110.001',
                       'Block after threshold; alert admin.',
                       'UNBLOCK_IP', 180),
    'BruteForce-SSH': ('HIGH','AUTO_BLOCK','T1110.001',
                       'Block after threshold; recommend key-based auth.',
                       'UNBLOCK_IP', 180),
    'BruteForce-Web': ('HIGH','AUTO_BLOCK','T1110.001',
                       'Block; alert web admin.',
                       'UNBLOCK_IP', 180),
    'BruteForce-XSS': ('HIGH','AUTO_BLOCK','T1110.001',
                       'Block; alert web admin.',
                       'UNBLOCK_IP', 180),
    'SQL-Injection': ('HIGH','AUTO_BLOCK','T1190',
                      'Block; alert web admin; input-validation review.',
                      'UNBLOCK_IP', 180),
    'Botnet': ('CRITICAL','ISOLATE','T1071.001',
               'ISOLATE endpoint; preserve forensics.',
               'RESTORE_NETWORK', 30),
    'Infiltration': ('CRITICAL','ISOLATE','T1190',
                     'ISOLATE segment; IR escalation.',
                     'RESTORE_SEGMENT', 30),
    'Zero-day': ('MEDIUM','ALERT_SOC','T1203',
                 'Novel anomaly (model-derived). Human review; no auto-block.',
                 'CLEAR_ALERT', 600),
}


def _mkdir(p): Path(p).mkdir(parents=True, exist_ok=True); return Path(p)


def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blk in iter(lambda: fh.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()
def _atomic_json(payload, dest):
    tmp = Path(dest).parent/(Path(dest).name+'.__tmp__')
    with open(tmp,'w',encoding='utf-8') as fh: json.dump(payload,fh,indent=2,default=str)
    os.replace(tmp,dest)
def _gc(): gc.collect()
def _hardware():
    cpu = platform.processor() or ''
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    cpu = line.split(':',1)[1].strip(); break
    except OSError: pass
    return {'cpu_model': cpu,
            'physical_cores': psutil.cpu_count(logical=False),
            'logical_cores': psutil.cpu_count(logical=True),
            'ram_gb': round(psutil.virtual_memory().total/2**30, 1)}
def _plt_style():
    plt.rcParams.update({'font.family':'serif','font.size':10,
        'axes.edgecolor':COLORS['edge'],'axes.linewidth':1.0,
        'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':9,
        'figure.dpi':150})
def _now_utc(): return datetime.now(timezone.utc).isoformat()


AUTO_ACTIONS = ('AUTO_BLOCK', 'AUTO_RATE_LIMIT', 'ISOLATE')


class _AE(nn.Module):
    """Same architecture and input transform as Layer 4 R8.0's
    SparseAutoencoder; loaded from state_dict (R8 T4)."""
    INPUT_TRANSFORM = 'signed_log1p'

    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.1)]
            prev = h
        self.encoder = nn.Sequential(*enc)
        dec = []
        for h in reversed(hidden_dims[:-1]):
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def prep(x):
        x = torch.as_tensor(x, dtype=torch.float32)
        return torch.sign(x) * torch.log1p(torch.abs(x))

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x, batch=262_144):
        self.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(x), batch):
                x_t = self.prep(x[i:i + batch])
                out.append(((x_t - self(x_t)) ** 2).mean(dim=1).numpy().astype(np.float32))
        return np.concatenate(out) if out else np.zeros(0, np.float32)


def _wilson_lower(k, n, z=1.959964):
    if n <= 0:
        return 0.0
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float((c - r) / d)


# -------------------------------------------------------------------- SOAR
class ResponseEngine:
    """Stateless SOAR logic. Model-derived only.
    R8.0 T1: an automated action (block / rate-limit / isolate) is taken only
    for families whose automated decisions were measured precise on the
    VALIDATION partition (auto_allowed). Every other family is downgraded to
    ALERT_SOC, i.e. a human decides. auto_allowed=None means 'not gated'."""
    def __init__(self, playbook=None, conf_auto=0.90, conf_alert=0.70,
                 auto_allowed=None):
        self.pb = playbook or PLAYBOOK
        self.conf_auto = conf_auto
        self.conf_alert = conf_alert
        self.auto_allowed = auto_allowed

    def decide(self, family, confidence, is_zeroday=False, flow_meta=None):
        t0 = time.perf_counter()
        incident_id = str(uuid.uuid4())[:8].upper()
        fmeta = flow_meta or {}
        gated = False

        if family == 'Normal':
            action, sev, mitre = 'NO_ACTION', 'LOW', ''
            desc, reversal, sla = 'Normal traffic', 'NONE', 0
        elif is_zeroday:
            rule = self.pb.get('Zero-day')
            sev, action, mitre, desc, reversal, sla = rule
        else:
            rule = self.pb.get(family)
            if rule is None:
                action, sev, mitre = 'ALERT_SOC', 'MEDIUM', 'T0000'
                desc, reversal, sla = f'Unknown family: {family}', 'CLEAR_ALERT', 300
            else:
                sev, primary_action, mitre, desc, reversal, sla = rule
                if confidence >= self.conf_auto and sev in ('CRITICAL','HIGH'):
                    action = primary_action
                elif confidence >= self.conf_alert:
                    action = 'ALERT_SOC'
                else:
                    action = 'LOG_MONITOR'
                allowed = getattr(self, 'auto_allowed', None)
                if (action in AUTO_ACTIONS and allowed is not None
                        and family not in allowed):
                    action, gated = 'ALERT_SOC', True

        lat_us = (time.perf_counter() - t0) * 1e6
        return {
            'gated'       : gated,
            'incident_id' : incident_id,
            'timestamp'   : _now_utc(),
            'family'      : family,
            'confidence'  : round(float(confidence), 6),
            'is_zeroday'  : bool(is_zeroday),
            'severity'    : sev,
            'action'      : action,
            'mitre_id'    : mitre,
            'description' : desc,
            'reversal'    : reversal,
            'sla_seconds' : int(sla),
            'source_ip'   : fmeta.get('src_ip', 'UNKNOWN'),
            'decision_us' : round(lat_us, 2),
            'simulated'   : True,
        }


class AuditLogger:
    """Batched, hash-chained JSONL (comment 62). R5.1: 10k/flush."""
    def __init__(self, path, batch_size=10_000):
        self.path = Path(path)
        if self.path.exists():
            # Fresh chain per run
            backup = self.path.with_suffix(self.path.suffix + '.prev')
            try:
                if backup.exists(): backup.unlink()
                self.path.rename(backup)
            except OSError:
                self.path.unlink()
        self._count = 0
        # Deterministic by default; Layer5Pipeline rebinds it to the sha256 of
        # the four upstream manifests (H7) immediately after construction.
        self._genesis = 'GENESIS_' + hashlib.sha256(
            json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:16]
        self._prev_hash = self._genesis
        self.verify_detail = {}
        self._buffer = []
        self._batch_size = int(batch_size)

    def log(self, decision):
        rec = dict(decision)
        rec['prev_hash'] = self._prev_hash
        canonical = json.dumps({k: rec[k] for k in sorted(rec.keys())},
                                default=str, sort_keys=True)
        rec['this_hash'] = hashlib.sha256(canonical.encode()).hexdigest()
        self._buffer.append(rec)
        self._prev_hash = rec['this_hash']
        self._count += 1
        if len(self._buffer) >= self._batch_size:
            self._flush()

    def _flush(self):
        if not self._buffer: return
        with open(self.path, 'a', encoding='utf-8') as fh:
            for rec in self._buffer:
                fh.write(json.dumps(rec, default=str) + '\n')
        self._buffer = []

    def finalize(self):
        self._flush()

    def verify_chain(self):
        """H10. Tamper-evidence requires RECOMPUTING each record's hash from its
        own content, not merely following prev_hash links. The earlier version
        followed links only, so editing any field of a record left the chain
        'verified'. It now checks three things: the head links to the
        input-bound GENESIS, every record's this_hash equals the SHA-256 of its
        canonical content, and each prev_hash equals the previous this_hash."""
        self._flush()
        if not self.path.exists():
            self.verify_detail = {'ok': False, 'reason': 'audit file missing'}
            return False
        prev = self._genesis
        n = bad_link = bad_hash = 0
        with open(self.path, 'r', encoding='utf-8') as fh:
            for line in fh:
                if not line.strip():
                    continue
                e = json.loads(line)
                n += 1
                if e.get('prev_hash') != prev:
                    bad_link += 1
                body = {k: v for k, v in e.items() if k != 'this_hash'}
                canon = json.dumps({k: body[k] for k in sorted(body)},
                                   default=str, sort_keys=True)
                if hashlib.sha256(canon.encode()).hexdigest() != e.get('this_hash'):
                    bad_hash += 1
                prev = e.get('this_hash')
        ok = (n > 0 and bad_link == 0 and bad_hash == 0)
        self.verify_detail = {
            'ok': bool(ok), 'entries_checked': n,
            'broken_links': bad_link, 'content_hash_mismatches': bad_hash,
            'genesis': self._genesis,
            'method': ('head linked to input-bound GENESIS; every record hash '
                       'recomputed from its canonical content; links checked')}
        return ok

    @property
    def count(self): return self._count

    @property
    def size_bytes(self):
        return int(self.path.stat().st_size) if self.path.exists() else 0


class Layer5Pipeline:
    def __init__(self, l1_dir, l2_dir, l3_dir, l4_dir, l5_dir):
        self.l1 = Path(l1_dir); self.l2 = Path(l2_dir)
        self.l3 = Path(l3_dir); self.l4 = Path(l4_dir)
        self.l5 = _mkdir(l5_dir); self.figs = _mkdir(self.l5/'figures')
        self.manifest_path = self.l5/'manifest_l5.json'

        for p, nm in [(self.l1/'manifest.json','L1'),
                       (self.l2/'manifest_l2.json','L2'),
                       (self.l3/'manifest_l3.json','L3'),
                       (self.l4/'manifest_l4.json','L4')]:
            if not p.exists(): raise FileNotFoundError(f'{nm} manifest missing: {p}')
        with open(self.l1/'manifest.json') as fh: self.m1 = json.load(fh)
        with open(self.l2/'manifest_l2.json') as fh: self.m2 = json.load(fh)
        with open(self.l3/'manifest_l3.json') as fh: self.m3 = json.load(fh)
        with open(self.l4/'manifest_l4.json') as fh: self.m4 = json.load(fh)

        c3 = self.m3.get('l4_l5_contract', {})
        for k in ('model', 'model_kind', 'classes', 'encoder',
                  'thresholds_json', 'reject_class'):
            if k not in c3:
                raise RuntimeError(f'L3 l4_l5_contract missing {k!r}: run L3 R8.0')
        self.features = list(c3.get('features')
                             or self.m2['l3_l4_contract']['features'])   # H4
        self.l3_classes = list(c3['classes'])
        self.l3_reject = c3['reject_class'] or 'Normal'                  # H4
        self.l3_kind = c3['model_kind']
        self.l3_ood_statistic = c3.get('ood_statistic', 'max_softmax')   # H1
        th3 = Path(c3['thresholds_json'])
        if not th3.exists():
            raise FileNotFoundError(f'L3 thresholds missing: {th3}')
        with open(th3) as fh:
            self.l3_thresholds = json.load(fh)
        need_th = ('l5', 'ood_energy') if self.l3_ood_statistic == 'energy' \
            else ('l5', 'ood_conf')
        for k_ in need_th:
            if k_ not in self.l3_thresholds:
                raise KeyError(f'L3 thresholds missing {k_!r} for statistic '
                               f'{self.l3_ood_statistic}: '
                               f'{sorted(self.l3_thresholds)}')
        cal_p = c3.get('calibrators')                                     # H2
        if not (cal_p and Path(cal_p).exists()):
            raise RuntimeError('L3 calibrators not found: thresholds are defined '
                               'on CALIBRATED probabilities, so R6.0 refuses to '
                               'threshold raw softmax. Re-run L3 R8.0.')
        with open(cal_p, 'rb') as fh:
            self.l3_cals = pickle.load(fh)
        print(f'  L3 calibrators: {sum(1 for c in self.l3_cals if c is not None)}'
              f'/{len(self.l3_cals)} fitted')
        # D3 FIX: explicit open with context manager and existence check
        th_path = Path(self.m2['l3_l4_contract']['threshold_json'])
        if not th_path.exists():
            raise FileNotFoundError(f'L2 thresholds missing: {th_path}')
        with open(th_path) as fh:
            th = json.load(fh)
        self.l2_best = self.m2['best_model']
        if self.l2_best not in th['deployed_tau']:
            raise KeyError(f'L2 threshold missing for model {self.l2_best}')
        self.l2_threshold = float(th['deployed_tau'][self.l2_best])
        c4 = self.m4.get('l5_contract', self.m4.get('model', {}))
        self.l4_alpha = float(c4['alpha'])
        self.l4_tau4 = float(c4['tau4'])
        self.l4_if_disabled = bool(abs(self.l4_alpha) < 1e-12)            # H3
        self._genesis_seed = {                                            # H7
            'cfg': CFG,
            'l1': _sha256_file(self.l1 / 'manifest.json'),
            'l2': _sha256_file(self.l2 / 'manifest_l2.json'),
            'l3': _sha256_file(self.l3 / 'manifest_l3.json'),
            'l4': _sha256_file(self.l4 / 'manifest_l4.json')}
        self.paths = self.m1['l2_input_paths']

        self.engine = ResponseEngine(conf_auto=CFG['conf_auto'],
                                       conf_alert=CFG['conf_alert'])
        self.audit = AuditLogger(self.l5/'audit_trail.jsonl',
                                   batch_size=CFG['audit_batch'])
        self.audit._prev_hash = self.audit._genesis = 'GENESIS_' + hashlib.sha256(
            json.dumps(self._genesis_seed, sort_keys=True,
                       default=str).encode()).hexdigest()[:16]            # H7

        self.counts_action = Counter()
        self.counts_severity = Counter()
        self.counts_family_action = defaultdict(Counter)
        self.counts_route = Counter()
        self.zd_pred_by_true = defaultdict(Counter)
        self.fp_actions = Counter()
        self.counts_gated = Counter()
        self.gate = {}
        self.fp_by_family = defaultdict(Counter)
        self.n_normal_seen = 0
        self.n_attack_seen = 0
        self.latencies = []
        self.decision_samples = []

        print('='*76)
        print(f'  TITANIUM-6 | LAYER 5 | {CFG["version"]}')
        print('='*76)
        print(f'  features={len(self.features)} classes={len(self.l3_classes)}')
        print(f'  L2 {self.l2_best} τ={self.l2_threshold:.4f}')
        print(f'  L3 tau_L5={self.l3_thresholds["l5"]:.4f} | '
              f'tau_OOD({self.l3_ood_statistic})={self._tau_ood():.4f} | '
              f'reject="{self.l3_reject}"')
        print(f'  L4 alpha={self.l4_alpha} tau4={self.l4_tau4:.4f} | IF '
              f'{"SKIPPED (alpha=0)" if self.l4_if_disabled else "enabled"}')
        print(f'  L4 α={self.l4_alpha} τ4={self.l4_tau4:.4f}')
        print(f'  L5 dir={self.l5}')

    # ------------------------------------------------------------------ 1
    def _p1_load(self):
        print('\n[PHASE 1] Load L2 / L3 / L4 models')
        with open(self.m2['l3_l4_contract']['model'], 'rb') as fh:
            self.l2_model = pickle.load(fh)
        with open(self.m2['l3_l4_contract']['cal'], 'rb') as fh:
            self.l2_cal = pickle.load(fh)
        with open(self.m3['l4_l5_contract']['model'], 'rb') as fh:
            self.l3_model = pickle.load(fh)
        with open(self.m3['l4_l5_contract']['encoder'], 'rb') as fh:
            self.l3_le = pickle.load(fh)
        self.if_model = self.if_cal = None
        if not self.l4_if_disabled:
            with open(self.m4['l5_contract']['if_model'], 'rb') as fh:
                self.if_model = pickle.load(fh)
        # R8 T4: load the AE from its architecture file + state_dict, not from
        # a pickle, so this layer runs in a fresh kernel (the pickle referenced
        # the class object defined by the Layer-4 cell).
        arch_p, state_p = self.l4 / 'ae_architecture.json', self.l4 / 'ae_model_state.pt'
        if not (arch_p.exists() and state_p.exists()):
            raise FileNotFoundError(f'L4 AE files missing: {arch_p}, {state_p}')
        with open(arch_p) as fh:
            arch = json.load(fh)
        if arch.get('input_transform') != 'signed_log1p':
            raise RuntimeError('L4 autoencoder lacks the R8 signed_log1p input '
                               'transform: re-run Layer 4 R8.0 first')
        ae = _AE(arch['input_dim'], arch['hidden_dims'])
        ae.load_state_dict(torch.load(state_p, map_location='cpu'))
        ae.eval()
        self.ae_model = ae
        if not self.l4_if_disabled:
            with open(self.m4['l5_contract']['if_cal'], 'rb') as fh:
                self.if_cal = pickle.load(fh)
        with open(self.m4['l5_contract']['ae_cal'], 'rb') as fh:
            self.ae_cal = pickle.load(fh)
        print(f'  L2 ok | L3 ok | L4 AE ok | L4 IF '
              f'{"SKIPPED (alpha=0; below-chance signal)" if self.l4_if_disabled else "ok"}')

    def _l2_batch(self, X):
        if isinstance(self.l2_model, lgb.Booster):
            raw = self.l2_model.predict(X).astype(np.float32)
        else:
            raw = np.asarray(self.l2_model.predict(
                xgb.DMatrix(X, feature_names=self.features))).ravel()
        return self.l2_cal.predict_proba(raw.reshape(-1,1))[:,1].astype(np.float32)

    def _l3_batch(self, X):
        if isinstance(self.l3_model, lgb.Booster):
            return self.l3_model.predict(X).astype(np.float32)
        return self.l3_model.predict(
            xgb.DMatrix(X, feature_names=self.features)
        ).reshape(-1, len(self.l3_classes)).astype(np.float32)

    def _l3_cal_proba(self, X):
        """H2: per-class Platt on the logit, then renormalise (L3's own _cal)."""
        q = np.array(self._l3_batch(X), dtype=np.float32, copy=True)
        for c, lr in enumerate(self.l3_cals):
            if lr is not None and c < q.shape[1]:
                with np.errstate(divide='ignore', invalid='ignore'):
                    z = np.log(np.clip(q[:, c], 1e-7, 1 - 1e-7) /
                               (1 - np.clip(q[:, c], 1e-7, 1 - 1e-7)))
                q[:, c] = lr.predict_proba(z.reshape(-1, 1))[:, 1].astype(np.float32)
        t = q.sum(axis=1, keepdims=True)
        return (q / np.where(t == 0, 1, t)).astype(np.float32)

    def _l3_energy(self, X):
        """H1: -logsumexp over raw margins; higher is more anomalous."""
        if isinstance(self.l3_model, lgb.Booster):
            m = self.l3_model.predict(X, raw_score=True).astype(
                np.float64).reshape(-1, len(self.l3_classes))
        else:
            m = self.l3_model.predict(
                xgb.DMatrix(X, feature_names=self.features),
                output_margin=True).reshape(-1, len(self.l3_classes)).astype(np.float64)
        return -logsumexp(m, axis=1)

    def _tau_ood(self):
        return float(self.l3_thresholds['ood_energy']
                     if self.l3_ood_statistic == 'energy'
                     else self.l3_thresholds['ood_conf'])

    def _l4_batch(self, X):
        ae_raw = self.ae_model.reconstruction_error(X)
        ae_p = self.ae_cal.predict_proba(
            ae_raw.reshape(-1, 1))[:, 1].astype(np.float32)
        if self.l4_if_disabled:                                   # H3
            return ae_p
        if_raw = -self.if_model.score_samples(X).astype(np.float32)
        if_p = self.if_cal.predict_proba(
            if_raw.reshape(-1, 1))[:, 1].astype(np.float32)
        return (self.l4_alpha * if_p + (1 - self.l4_alpha) * ae_p).astype(np.float32)

    # ------------------------------------------------------------------ 2
    def _p2_validate(self):
        print('\n[PHASE 2] Playbook validation')
        missing = [c for c in self.l3_classes if c not in PLAYBOOK]
        for c in missing:
            if c == self.l3_reject:        # Y2
                continue
            PLAYBOOK[c] = ('MEDIUM','ALERT_SOC','T0000',
                            f'Default for {c}','CLEAR_ALERT',300)
        print(f'  families: {len(self.l3_classes)} | playbook: {len(PLAYBOOK)}')
        print(f'  coverage: {len([c for c in self.l3_classes if c in PLAYBOOK])}/'
              f'{len(self.l3_classes)}')
        for fam, (sev, act, mitre, _, _, sla) in sorted(PLAYBOOK.items()):
            print(f'    {fam:<22} {sev:<10} {act:<18} {mitre:<12} {sla}s')

    # ------------------------------------------------------------------ 2B
    def _p2b_gate(self):
        """T1: measure, on VALIDATION only, how precise each family's automated
        decisions would be (L2 flag -> L3 family with confidence >= conf_auto),
        and allow automation only where the Wilson 95% lower bound of that
        precision reaches gate_min_precision with >= gate_min_support cases."""
        print('\n[PHASE 2B] Auto-action precision gate (validation partition)')
        feats = self.features
        Xl, yl, n = [], [], 0
        for b in pq.ParquetFile(self.paths['val']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas()
            Xl.append(df[feats].values.astype(np.float32))
            yl.extend(df['Target'].tolist())
            n += len(df)
            del df, b; _gc()
            if n >= CFG['gate_val_n']:
                break
        X = np.vstack(Xl)[:CFG['gate_val_n']]
        y = np.array(yl, dtype=object)[:CFG['gate_val_n']]
        del Xl, yl; _gc()
        att = np.where(self._l2_batch(X) >= self.l2_threshold)[0]
        table, allowed = {}, set()
        if len(att):
            p3 = self._l3_cal_proba(X[att])
            names = self.l3_le.inverse_transform(p3.argmax(axis=1))
            conf = p3.max(axis=1)
            yt = y[att]
            for fam, rule in sorted(PLAYBOOK.items()):
                if rule[1] not in AUTO_ACTIONS or rule[0] not in ('CRITICAL', 'HIGH'):
                    continue
                m = (names == fam) & (conf >= CFG['conf_auto'])
                k_tot = int(m.sum())
                k_ok = int((yt[m] == fam).sum())
                lb = _wilson_lower(k_ok, k_tot)
                ok = (k_tot >= CFG['gate_min_support']
                      and lb >= CFG['gate_min_precision'])
                wrong = Counter(yt[m][yt[m] != fam].tolist()).most_common(3)
                table[fam] = {'n_auto_decisions_val': k_tot, 'n_correct': k_ok,
                              'precision': (k_ok / k_tot) if k_tot else None,
                              'wilson95_lower': lb, 'automation_allowed': bool(ok),
                              'top_wrong_true_labels': wrong}
                if ok:
                    allowed.add(fam)
        self.engine.auto_allowed = allowed
        self.gate = {
            'rule': (f'automate family f only if, on VALIDATION, decisions '
                     f'(L2 flag, L3=f, conf>={CFG["conf_auto"]}) have Wilson-95% '
                     f'lower-bound precision >= {CFG["gate_min_precision"]} with '
                     f'>= {CFG["gate_min_support"]} cases; otherwise ALERT_SOC'),
            'val_rows_used': int(len(y)), 'l2_flagged_val_rows': int(len(att)),
            'families': table, 'auto_allowed': sorted(allowed),
            'test_used': False}
        print(f'  val rows {len(y):,} | L2-flagged {len(att):,}')
        print(f'  {"family":<18} {"n":>9} {"prec":>8} {"wilson_lb":>10}  auto')
        for fam, r in table.items():
            pr = '-' if r['precision'] is None else f'{r["precision"]:.4f}'
            print(f'  {fam:<18} {r["n_auto_decisions_val"]:>9,} {pr:>8} '
                  f'{r["wilson95_lower"]:>10.4f}  '
                  f'{"ALLOWED" if r["automation_allowed"] else "-> ALERT_SOC"}')
        del X, y; _gc()

    # ------------------------------------------------------------------ 3
    def _p3_inference(self):
        """
        Batch inference. No ground-truth in decision path (comment 60).
        D2 FIX: dead 1M-iteration count loop removed.
        D4 FIX: L4-derived zero-day confidence = L4 score, not 0.
        """
        print('\n[PHASE 3] Batch inference on test_known + test_zeroday')
        feats = self.features

        # ---- test_known sample ----
        target_n = CFG['known_sample_n']
        Xl, target_l, twin_l = [], [], []
        collected = 0
        for b in pq.ParquetFile(self.paths['test_known']).iter_batches(
                batch_size=CFG['eval_chunk'],
                columns=feats + ['Target', '_dup_in_train']):
            df = b.to_pandas()
            Xl.append(df[feats].values.astype(np.float32))
            target_l.extend(df['Target'].tolist())
            twin_l.append(df['_dup_in_train'].values.astype(np.int8))
            collected += len(df)
            del df, b; _gc()
            if collected >= target_n: break
        X_known = np.vstack(Xl)[:target_n]
        target_known = np.array(target_l)[:target_n]
        twin_known = np.concatenate(twin_l)[:target_n]
        del Xl, target_l, twin_l; _gc()
        print(f'  test_known sample: {len(X_known):,}')

        # ---- test_zeroday full ----
        Xl, target_l = [], []
        for b in pq.ParquetFile(self.paths['test_zeroday']).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas()
            Xl.append(df[feats].values.astype(np.float32))
            target_l.extend(df['Target'].tolist())
            del df, b; _gc()
        X_zd = np.vstack(Xl); target_zd = np.array(target_l)
        del Xl, target_l; _gc()
        print(f'  test_zeroday:      {len(X_zd):,}')

        def _run(X):
            n = len(X)
            p_l2 = self._l2_batch(X)
            l2_attack = p_l2 >= self.l2_threshold
            family = np.array(['Normal'] * n, dtype=object)
            conf = np.zeros(n, dtype=np.float32)
            is_zeroday = np.zeros(n, dtype=bool)
            l4_score = np.zeros(n, dtype=np.float32)
            route = np.array(['normal_pass'] * n, dtype=object)           # H8
            idx_att = np.where(l2_attack)[0]
            if len(idx_att):
                p_l3 = self._l3_cal_proba(X[idx_att])              # H2
                argmax = p_l3.argmax(axis=1)
                l3_conf_arr = p_l3.max(axis=1)
                names = self.l3_le.inverse_transform(argmax)
                family[idx_att] = names
                conf[idx_att] = l3_conf_arr
                is_reject = (names == self.l3_reject)               # H4
                tau_ood = self._tau_ood()
                if self.l3_ood_statistic == 'energy':               # H1
                    l3_ood = self._l3_energy(X[idx_att]) > tau_ood
                else:
                    l3_ood = l3_conf_arr < tau_ood
                l3_middle = (~l3_ood) & (l3_conf_arr < float(self.l3_thresholds['l5']))
                r = np.full(len(idx_att), 'l3_confident', dtype=object)   # H8
                r[l3_middle & ~is_reject] = 'l3_middle'
                r[l3_ood & ~is_reject] = 'l3_ood'
                r[is_reject] = 'l3_normal'
                route[idx_att] = r
                is_zeroday[idx_att[l3_ood & ~is_reject]] = True
                idx_l4_from_l3 = idx_att[(l3_middle | l3_ood) & ~is_reject]
            else:
                idx_l4_from_l3 = np.array([], dtype=int)
            idx_l4 = np.concatenate([np.where(~l2_attack)[0], idx_l4_from_l3])
            if len(idx_l4):
                l4_score[idx_l4] = self._l4_batch(X[idx_l4])
                l4_anom = l4_score[idx_l4] >= self.l4_tau4
                l2_norm_mask = ~l2_attack
                zd_idx = idx_l4[l4_anom & l2_norm_mask[idx_l4]]
                is_zeroday[zd_idx] = True
                family[zd_idx] = 'Zero-day'
                route[zd_idx] = 'l4_zeroday'
            # D4 FIX: L4-derived zero-day flows get L4 score as confidence
            zeroday_from_l4 = is_zeroday & (~l2_attack)
            conf[zeroday_from_l4] = l4_score[zeroday_from_l4]
            return p_l2, l2_attack, family, conf, is_zeroday, l4_score, route

        print('  running inference on test_known...')
        res_k = _run(X_known)
        print('  running inference on test_zeroday...')
        res_z = _run(X_zd)

        self.X_known = X_known; self.target_known = target_known
        self.twin_known = twin_known
        self.X_zd = X_zd; self.target_zd = target_zd
        self.res_known = res_k; self.res_zd = res_z

    # ------------------------------------------------------------------ 4
    def _p4_decisions(self):
        """Per-flow SOAR decisions (comment 60). D1: batched audit."""
        print('\n[PHASE 4] SOAR decisions from model outputs')
        for tag, X, y, res in [
                ('known', self.X_known, self.target_known, self.res_known),
                ('zd',    self.X_zd,    self.target_zd,    self.res_zd)]:
            p_l2, l2_attack, family, conf, is_zeroday, l4_score, route = res
            n = len(X)
            for i in range(n):
                fam_i = family[i]
                conf_i = float(conf[i])
                z_i = bool(is_zeroday[i])
                dec = self.engine.decide(fam_i, conf_i, is_zeroday=z_i)
                dec['route'] = str(route[i])
                self.counts_route[dec['route']] += 1
                if tag == 'zd':                                            # H9
                    self.zd_pred_by_true[str(y[i])][fam_i] += 1
                self.counts_action[dec['action']] += 1
                self.counts_severity[dec['severity']] += 1
                self.counts_family_action[fam_i][dec['action']] += 1
                self.latencies.append(dec['decision_us'])
                if dec.get('gated'):
                    self.counts_gated[fam_i] += 1
                # FP audit (comment 61)
                if y[i] == 'Normal':
                    self.n_normal_seen += 1
                    if dec['action'] in AUTO_ACTIONS:
                        self.fp_actions[dec['action']] += 1
                        self.fp_by_family[fam_i][dec['action']] += 1
                else:
                    self.n_attack_seen += 1
                # Audit — batched internally
                self.audit.log(dec)
                # Sample for diagnostics
                if len(self.decision_samples) < CFG['decision_samples_cap']:
                    self.decision_samples.append({
                        'family'   : fam_i,
                        'action'   : dec['action'],
                        'severity' : dec['severity'],
                        'confidence': conf_i,
                        'is_zeroday': z_i,
                        'is_normal_gt': (y[i] == 'Normal') if tag == 'known' else False,
                    })
            print(f'  [{tag}] processed {n:,}')
        # Finalize audit file before verification
        self.audit.finalize()
        print(f'  decisions = {sum(self.counts_action.values()):,}')
        print(f'  audit entries = {self.audit.count:,}')
        print(f'  audit file size = {self.audit.size_bytes/2**20:.1f} MB')
        for act, c in sorted(self.counts_action.items(), key=lambda x: -x[1]):
            print(f'    {act:<18} {c:>10,}')
        for sev, c in sorted(self.counts_severity.items(), key=lambda x: -x[1]):
            print(f'    {sev:<10} {c:>10,}')
        # D5 FIX: real verify_chain
        ok = self.audit.verify_chain()
        self.audit_chain_verified = bool(ok)
        print(f'  hash chain verified: {ok}')
        if not ok:
            raise RuntimeError('[FATAL] audit hash chain broken')

    # ------------------------------------------------------------------ 5
    def _p5_latency(self):
        print('\n[PHASE 5] Decision engine latency (measured; comment 46)')
        for i in range(CFG['decisions_warmup']):
            self.engine.decide('DoS-Hulk', 0.95)
        t = np.empty(CFG['decisions_runs'])
        for i in range(CFG['decisions_runs']):
            t0 = time.perf_counter_ns()
            self.engine.decide('DoS-Hulk', 0.95)
            t[i] = time.perf_counter_ns() - t0
        t /= 1e3
        self.latency = {
            'mean_us': float(t.mean()),
            'p50_us': float(np.percentile(t, 50)),
            'p95_us': float(np.percentile(t, 95)),
            'p99_us': float(np.percentile(t, 99)),
            'runs': CFG['decisions_runs'],
            'warmup': CFG['decisions_warmup'],
            'batch_size': 1,
            'hardware': _hardware(),
            'python': platform.python_version(),
            'includes': 'ResponseEngine.decide() including playbook lookup, '
                        'severity gate, metric construction',
            'excludes': 'network I/O (none performed; simulated=True)',
        }
        print(f'  mean={self.latency["mean_us"]:.1f}us  '
              f'p95={self.latency["p95_us"]:.1f}us  '
              f'p99={self.latency["p99_us"]:.1f}us')

    # ------------------------------------------------------------------ 6
    def _p6_fp_audit(self):
        """Comment 61 + D7 + D8."""
        print('\n[PHASE 6] FP action audit (comment 61)')
        n_normal = self.n_normal_seen
        n_fp = sum(self.fp_actions.values())
        fp_rate = n_fp / max(n_normal, 1)
        per_hour_normal = CFG['capture_rate_dps'] * 3600
        fp_per_hour = fp_rate * per_hour_normal
        self.fp_audit = {
            'n_normal_seen': int(n_normal),
            'n_attack_seen': int(self.n_attack_seen),
            'n_fp': int(n_fp),
            'fp_rate': float(fp_rate),
            'by_action': dict(self.fp_actions),
            'by_family': {f: dict(c) for f, c in self.fp_by_family.items()},
            'capture_rate_dps': CFG['capture_rate_dps'],
            'capture_rate_source': CFG['capture_rate_source'],
            'estimated_normal_flows_per_hour': float(per_hour_normal),
            'estimated_fp_per_hour': float(fp_per_hour),
            'note': ('comment 61 closure: actions applied to true-Normal flows, '
                     'estimated at deployment capture rate.'),
        }
        print(f'  Normal flows seen: {n_normal:,}')
        print(f'  Attack flows seen: {self.n_attack_seen:,}')
        print(f'  FP actions       : {n_fp:,}  ({fp_rate*100:.4f}%)')
        for act, c in sorted(self.fp_actions.items(), key=lambda x: -x[1]):
            print(f'    {act:<18} {c:,}')
        if self.fp_by_family:
            print('  FP by family:')
            for fam, cnts in sorted(self.fp_by_family.items(),
                                     key=lambda x: -sum(x[1].values())):
                tot = sum(cnts.values())
                print(f'    {fam:<22} {tot:,}')

    # ------------------------------------------------------------------ 7
    def _p7_figures(self):
        print('\n[PHASE 7] Figures (6 PDFs)')
        _plt_style()

        # Fig 25 — SOAR decision flowchart
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')
        def box(x, y, w, h, txt, fc, fs=9):
            ax.add_patch(plt.Rectangle((x-w/2, y-h/2), w, h,
                                        facecolor=fc, edgecolor=COLORS['edge'], lw=1.3))
            ax.text(x, y, txt, ha='center', va='center', fontsize=fs,
                    multialignment='center')
        def arrow(x1, y1, x2, y2):
            ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle='->', color=COLORS['edge'], lw=1.4))
        box(5, 11, 8, 0.7,
            'Model-derived routing (no ground-truth labels; comment 60)',
            '#E8F1F8', 10)
        box(5, 9.5, 7, 0.6,
            f'L2 gate: p ≥ {self.l2_threshold:.3f} → attack',
            COLORS['train']+'22')
        arrow(5, 10.65, 5, 9.8)
        box(2, 8, 3, 0.6, 'L2 Normal → L4', COLORS['val']+'22')
        box(8, 8, 3, 0.6, 'L2 Attack → L3', COLORS['lgbm']+'22')
        arrow(5, 9.2, 2, 8.3); arrow(5, 9.2, 8, 8.3)
        box(2, 6.5, 3, 0.7,
            f'L4 score ≥ {self.l4_tau4:.3f}?\nZero-day candidate',
            COLORS['crimson']+'22')
        arrow(2, 7.7, 2, 6.85)
        box(8, 6.5, 3, 0.7,
            f'L3 {self.l3_ood_statistic} gate at {self._tau_ood():.3f}\n-> Layer 4',
            COLORS['crimson']+'22')
        arrow(8, 7.7, 8, 6.85)
        box(5, 5, 8, 0.6, 'Response engine: severity × confidence gate', '#FFF9E6')
        arrow(2, 6.15, 5, 5.3); arrow(8, 6.15, 5, 5.3)
        box(1.5, 3, 2.6, 0.8, 'AUTO_BLOCK\n/ RATE_LIMIT\n/ ISOLATE',
            ACTION_COLORS['AUTO_BLOCK'], 9)
        box(4.3, 3, 2.6, 0.8, 'ALERT_SOC\n(human review)',
            ACTION_COLORS['ALERT_SOC'], 9)
        box(7.1, 3, 2.6, 0.8, 'LOG_MONITOR', ACTION_COLORS['LOG_MONITOR'], 9)
        arrow(5, 4.7, 1.5, 3.4); arrow(5, 4.7, 4.3, 3.4); arrow(5, 4.7, 7.1, 3.4)
        box(5, 1.5, 8, 0.7, 'Hash-chained audit trail (comment 62)', '#F4F4F4')
        for x in (1.5, 4.3, 7.1):
            arrow(x, 2.6, 5, 1.85)
        ax.set_title('Fig 25.  Layer 5 SOAR response decision flow',
                     fontsize=11, pad=10)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig25_flowchart.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        # Fig 26 — action matrix
        severities = ['CRITICAL','HIGH','MEDIUM','LOW']
        conf_ranges = ['≥0.90','0.70–0.90','0.50–0.70','<0.50']
        matrix = np.array([[5,4,2,1],[2,2,2,1],[1,1,1,1],[1,1,1,1]], dtype=float)
        lbl = [['AUTO\nBLOCK','AUTO\nRATE\nLIMIT','ALERT\nSOC','LOG'],
                ['ALERT\nSOC','ALERT\nSOC','ALERT\nSOC','LOG'],
                ['LOG','LOG','LOG','LOG'],['LOG','LOG','LOG','LOG']]
        import matplotlib.colors as mcolors
        cmap = mcolors.LinearSegmentedColormap.from_list(
            'soar', [COLORS['test'], COLORS['amber'], COLORS['crimson'], '#7B0000'], N=256)
        fig, ax = plt.subplots(figsize=(9, 6.5))
        im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=5, aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.045,
                      label='Action severity (0=passive, 5=block)')
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        ax.set_xticklabels(severities, fontweight='bold')
        ax.set_yticklabels(conf_ranges)
        ax.set_xlabel('Attack severity'); ax.set_ylabel('Detection confidence')
        for i in range(4):
            for j in range(4):
                v = matrix[i,j]
                ax.text(j, i, lbl[i][j], ha='center', va='center', fontsize=9,
                        color='white' if v >= 3 else COLORS['edge'],
                        fontweight='bold')
        ax.set_title('Fig 26.  Confidence × severity → action tier', fontsize=11)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig26_action_matrix.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        # Fig 27 — action distribution per family
        fams = sorted(self.counts_family_action.keys(),
                       key=lambda f: -sum(self.counts_family_action[f].values()))
        actions = ['AUTO_BLOCK','AUTO_RATE_LIMIT','ISOLATE','ALERT_SOC',
                    'LOG_MONITOR','NO_ACTION']
        data = np.zeros((len(fams), len(actions)))
        for i, f in enumerate(fams):
            tot = sum(self.counts_family_action[f].values())
            for j, a in enumerate(actions):
                data[i, j] = 100*self.counts_family_action[f].get(a,0)/max(tot,1)
        fig, ax = plt.subplots(figsize=(13, 6))
        x = np.arange(len(fams)); bottom = np.zeros(len(fams))
        for j, a in enumerate(actions):
            v = data[:, j]
            if v.sum() == 0: continue
            ax.bar(x, v, bottom=bottom, color=ACTION_COLORS[a],
                    edgecolor=COLORS['edge'], lw=0.5, label=a.replace('_',' '))
            bottom += v
        ax.set_xticks(x)
        ax.set_xticklabels(fams, rotation=35, ha='right', fontsize=8)
        ax.set_ylim(0, 110); ax.set_ylabel('Action distribution (%)')
        ax.set_title('Fig 27.  Response action distribution per family', fontsize=11)
        ax.legend(loc='upper right', ncol=2, fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig27_action_distribution.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        # Fig 28 — severity distribution (model-derived; comment 60)
        fig, ax = plt.subplots(figsize=(8, 5))
        sevs = ['CRITICAL','HIGH','MEDIUM','LOW']
        counts = [self.counts_severity.get(s, 0) for s in sevs]
        bars = ax.bar(sevs, counts, color=[SEVERITY_COLORS[s] for s in sevs],
                       edgecolor=COLORS['edge'])
        for b, c in zip(bars, counts):
            ax.text(b.get_x()+b.get_width()/2, c+max(counts)*0.01,
                    f'{c:,}', ha='center', fontsize=9)
        ax.set_ylabel('Decision count'); ax.set_yscale('symlog')
        ax.set_title('Fig 28.  Severity distribution (model-derived; comment 60)',
                     fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig28_severity_distribution.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        # Fig 29 — FP audit (comment 61 + D8)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        n_normal = self.fp_audit['n_normal_seen']
        acts_ordered = ['AUTO_BLOCK','AUTO_RATE_LIMIT','ISOLATE']
        rates = [100*self.fp_audit['by_action'].get(a,0)/max(n_normal,1)
                  for a in acts_ordered]
        bars = axes[0].bar(acts_ordered, rates,
                            color=[ACTION_COLORS[a] for a in acts_ordered],
                            edgecolor=COLORS['edge'])
        for b, a, v in zip(bars, acts_ordered, rates):
            ax_label = f'{v:.4f}%\n({self.fp_audit["by_action"].get(a,0):,})'
            axes[0].text(b.get_x()+b.get_width()/2, v+max(rates)*0.02,
                          ax_label, ha='center', fontsize=9)
        axes[0].set_ylabel('FP rate (% of Normal flows)')
        axes[0].set_title(f'FP rate by action (n_normal={n_normal:,})', fontsize=10)
        axes[0].grid(axis='y', alpha=0.3)
        # Family breakdown
        if self.fp_by_family:
            fams_list = sorted(self.fp_by_family.keys(),
                                key=lambda f: -sum(self.fp_by_family[f].values()))[:10]
            tots = [sum(self.fp_by_family[f].values()) for f in fams_list]
            axes[1].barh(range(len(fams_list)), tots, color=COLORS['crimson'],
                          edgecolor=COLORS['edge'])
            axes[1].set_yticks(range(len(fams_list)))
            axes[1].set_yticklabels(fams_list, fontsize=8)
            axes[1].invert_yaxis()
            axes[1].set_xlabel('FP actions')
            axes[1].set_xscale('log')
            axes[1].set_title('FP actions by attributed family', fontsize=10)
            axes[1].grid(axis='x', alpha=0.3)
        fig.suptitle('Fig 29.  False-positive action audit (comment 61)',
                     fontsize=11)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig29_fp_audit.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        # Fig 30 — decision latency
        arr = np.array(self.latencies)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.hist(arr, bins=60, color=COLORS['lgbm'], edgecolor=COLORS['edge'], lw=0.3)
        ax.axvline(self.latency['p50_us'], color=COLORS['val'], ls='--', lw=1.4,
                    label=f'P50={self.latency["p50_us"]:.1f}us')
        ax.axvline(self.latency['p95_us'], color=COLORS['amber'], ls='--', lw=1.4,
                    label=f'P95={self.latency["p95_us"]:.1f}us')
        ax.axvline(self.latency['p99_us'], color=COLORS['crimson'], ls='--', lw=1.4,
                    label=f'P99={self.latency["p99_us"]:.1f}us')
        ax.set_xlabel('Decision latency (us)'); ax.set_ylabel('Count')
        ax.set_yscale('log')
        ax.set_title('Fig 30.  Layer 5 decision engine latency (measured; comment 46)',
                     fontsize=11)
        ax.legend(); ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        fig.savefig(self.figs/'L5_fig30_decision_latency.pdf', dpi=300,
                     bbox_inches='tight'); plt.close(fig)

        print('  6 figures written')

    # ------------------------------------------------------------------ 8
    def _p8_save(self):
        print('\n[PHASE 8] Saving')
        with open(self.l5/'response_engine.pkl', 'wb') as fh:
            pickle.dump(self.engine, fh)
        _atomic_json({
            'conf_auto': CFG['conf_auto'],
            'conf_alert': CFG['conf_alert'],
            'playbook': {k: list(v) for k, v in PLAYBOOK.items()},
            'l3_thresholds': self.l3_thresholds,
            'l2_threshold': self.l2_threshold,
            'l2_model': self.l2_best,
            'l4_alpha': self.l4_alpha, 'l4_tau4': self.l4_tau4,
            'auto_allowed': (None if self.engine.auto_allowed is None
                             else sorted(self.engine.auto_allowed)),
            'audit_chain_verified': self.audit_chain_verified,
        }, self.l5/'layer5_config.json')

        man = {
            'version': CFG['version'],
            'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'seed': SEED,
            'l1_manifest_version': self.m1.get('version'),
            'l2_manifest_version': self.m2.get('version'),
            'l3_manifest_version': self.m3.get('version'),
            'l4_manifest_version': self.m4.get('version'),
            'thresholds_used': {
                'l2_threshold': self.l2_threshold,
                'l2_model': self.l2_best,
                'l3_l5': self.l3_thresholds['l5'],
                'l3_ood_statistic': self.l3_ood_statistic,
                'l3_ood_tau': self._tau_ood(),
                'l4_alpha': self.l4_alpha,
                'l4_tau4': self.l4_tau4,
                'l5_auto': CFG['conf_auto'], 'l5_alert': CFG['conf_alert'],
            },
            'decision_source': ('model outputs only — no ground-truth labels '
                                'in the decision path (comment 60 closure)'),
            'n_decisions': sum(self.counts_action.values()),
            'n_audit_entries': self.audit.count,
            'audit_file_size_bytes': self.audit.size_bytes,
            'action_distribution': dict(self.counts_action),
            'severity_distribution': dict(self.counts_severity),
            'family_action_distribution': {f: dict(c) for f, c in
                                             self.counts_family_action.items()},
            'fp_audit': self.fp_audit,
            'route_distribution': dict(self.counts_route),                 # H8
            'auto_action_gate': self.gate,                                 # R8 T1
            'gated_decisions_by_family': dict(self.counts_gated),
            'zero_day_attribution_cross_check': {                          # H9
                'per_true_family_predicted': {k: dict(v) for k, v in
                                              self.zd_pred_by_true.items()},
                'reading': ('independent count of what Layer 3 predicts for each '
                            'true zero-day family; confirms or refutes the Layer-4 '
                            'attribution figure')},
            'l4_if_disabled': self.l4_if_disabled,
            'l3_ood_statistic': self.l3_ood_statistic,
            'l3_ood_tau': self._tau_ood(),
            'audit_genesis_binds': sorted(self._genesis_seed.keys()),
            'latency': self.latency,
            'playbook': {k: {'severity': v[0], 'action': v[1], 'mitre': v[2],
                              'reversal': v[4], 'sla_s': v[5]}
                          for k, v in PLAYBOOK.items()},
            'audit_trail': {
                'file': str(self.l5/'audit_trail.jsonl'),
                'hash_chain': True,
                'entries': self.audit.count,
                'size_bytes': self.audit.size_bytes,
                'verified': bool(self.audit_chain_verified),
                'verification': getattr(self.audit, 'verify_detail', {}),
                'batch_size': CFG['audit_batch'],
                'method': ('SHA-256 over canonical JSON of each entry; '
                           'prev_hash links to previous entry. Chain verified '
                           'after write.'),
            },
            'decision_samples': {
                'kept': len(self.decision_samples),
                'cap': CFG['decision_samples_cap'],
                'note': 'first N decisions retained for post-run inspection',
            },
            'figures': {
                'Fig_25': 'L5_fig25_flowchart.pdf',
                'Fig_26': 'L5_fig26_action_matrix.pdf',
                'Fig_27': 'L5_fig27_action_distribution.pdf',
                'Fig_28': 'L5_fig28_severity_distribution.pdf',
                'Fig_29': 'L5_fig29_fp_audit.pdf',
                'Fig_30': 'L5_fig30_decision_latency.pdf',
            },
            'l6_contract': {
                'engine': str(self.l5/'response_engine.pkl'),
                'config': str(self.l5/'layer5_config.json'),
                'audit': str(self.l5/'audit_trail.jsonl'),
            },
            'paper_map': {
                'Table_XII'  : 'playbook',
                'Table_XIII' : 'latency + fp_audit',
                'Fig_25'     : 'L5_fig25_flowchart.pdf',
                'Fig_26'     : 'L5_fig26_action_matrix.pdf',
                'Fig_27'     : 'L5_fig27_action_distribution.pdf',
                'Fig_28'     : 'L5_fig28_severity_distribution.pdf (comment 60)',
                'Fig_29'     : 'L5_fig29_fp_audit.pdf (comment 61)',
                'Fig_30'     : 'L5_fig30_decision_latency.pdf (comment 46)',
                'Sec_VIII-A' : 'playbook + MITRE mapping',
                'Sec_VIII-B' : 'action/severity distribution (comment 60)',
                'Sec_VIII-C' : 'hash-chained audit (comment 62)',
                'Sec_VIII-D' : 'FP action cost (comment 61)',
                'Sec_VIII-E' : 'decision latency (comment 46)',
            },
        }
        _atomic_json(man, self.manifest_path)
        print('  manifest_l5.json written')

    def execute(self):
        t0 = time.time()
        self._p1_load()
        self._p2_validate()
        self._p2b_gate()
        self._p3_inference()
        self._p4_decisions()
        self._p5_latency()
        self._p6_fp_audit()
        self._p7_figures()
        self._p8_save()
        print('\n' + '='*76)
        print(f'  LAYER 5 R8.0 COMPLETE | {(time.time()-t0)/60:.1f} min')
        print(f'  decisions = {sum(self.counts_action.values()):,}')
        print(f'  audit = {self.audit.count:,} entries | '
              f'{self.audit.size_bytes/2**20:.1f} MB | '
              f'chain verified = {self.audit_chain_verified}')
        print(f'  actions: {dict(self.counts_action)}')
        print(f'  severities: {dict(self.counts_severity)}')
        print(f'  FP actions on Normal: {sum(self.fp_actions.values()):,} '
              f'({self.fp_audit["fp_rate"]*100:.4f}%)')
        print(f'  automation allowed for: {sorted(self.engine.auto_allowed or []) or "none"}')
        print(f'  gated (downgraded to ALERT_SOC): {dict(self.counts_gated) or "none"}')
        print(f'  decision latency P95 = {self.latency["p95_us"]:.1f} us')
        print('  routes: ' + ', '.join(f'{k}={v:,}' for k, v in
                                       sorted(self.counts_route.items(),
                                              key=lambda x: -x[1])))
        for true_fam, preds in sorted(self.zd_pred_by_true.items()):
            top = preds.most_common(2)
            hit = preds.get(true_fam, 0); tot = sum(preds.values())
            print(f'  [H9] zero-day {true_fam:<16} correct-family '
                  f'{100 * hit / max(tot, 1):.2f}% of {tot:,} | top predicted: '
                  + ', '.join(f'{k} {v:,}' for k, v in top))
        print(f'  6 figures written')
        print('='*76)


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='TITANIUM-6 Layer 5 R8.0')
    ap.add_argument('--l1dir', default=os.environ.get('T6_L1','/kaggle/working/l1_R8'))
    ap.add_argument('--l2dir', default=os.environ.get('T6_L2','/kaggle/working/l2_R8'))
    ap.add_argument('--l3dir', default=os.environ.get('T6_L3','/kaggle/working/l3_R8'))
    ap.add_argument('--l4dir', default=os.environ.get('T6_L4','/kaggle/working/l4_R8'))
    ap.add_argument('--l5dir', default=os.environ.get('T6_L5','/kaggle/working/l5_R8'))
    a, _ = ap.parse_known_args()
    Layer5Pipeline(a.l1dir, a.l2dir, a.l3dir, a.l4dir, a.l5dir).execute()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  L3 calibrators: 12/12 fitted
  TITANIUM-6 | LAYER 5 | TITANIUM6_LAYER5_R8.0
  features=77 classes=12
  L2 LightGBM τ=0.0793
  L3 tau_L5=0.2967 | tau_OOD(energy)=-0.3071 | reject="Normal"
  L4 alpha=0.0 tau4=0.2711 | IF SKIPPED (alpha=0)
  L4 α=0.0 τ4=0.2711
  L5 dir=/kaggle/working/l5_R8

[PHASE 1] Load L2 / L3 / L4 models
  L2 ok | L3 ok | L4 AE ok | L4 IF SKIPPED (alpha=0; below-chance signal)

[PHASE 2] Playbook validation
  families: 12 | playbook: 15
  coverage: 11/12
    Botnet                 CRITICAL   ISOLATE            T1071.001    30s
    BruteForce-FTP         HIGH       AUTO_BLOCK         T1110.001    180s
    BruteForce-SSH         HIGH       AUTO_BLOCK         T1110.001    180s
    BruteForce-Web         HIGH       AUTO_BLOCK         T1110.001    180s
    BruteForce-XSS         HIGH       AUTO_BLOCK         T1110.001    180s
    DDoS-HOIC              CRITICAL   AUTO_BLOCK         T1498.001    30s
    DDo

In [7]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 6 | R8.0
#  R8.0 over R6.1:
#   T1 AE attribution uses the Layer-4 R8 signed_log1p input transform (read
#      from ae_architecture.json). The R6.1 'variance-normalised' ratio divided
#      numerator and denominator by the same variance, so it was identical to
#      the plain ratio; it is replaced by the plain ratio plus each feature's
#      share of total attack reconstruction error.
#   T2 Counterfactuals: 50 RANDOMLY sampled flows per family (R6.1 used the 5
#      highest-confidence flows), Wilson 95% CI on the flip rate, and the
#      number of features actually changed (sparsity) per flow.
#   T3 Paths and version strings follow the single R8 run configuration.
#   T4 SHAP BACKEND. With shap 0.51, TreeExplainer(interventional, raw) on a
#      12-class LightGBM Booster returned ONE output (shape (n, f)) instead of
#      one per class; the old 2-D branch of _normalize_shap silently turned it
#      into a fake 1-class array and the run crashed at class index 3. R8.0
#      computes exact TreeSHAP with the model's OWN native routine
#      (LightGBM predict(pred_contrib=True) / XGBoost pred_contribs=True):
#      per class, no library-version dependence, and every call is checked
#      for additivity (sum of contributions + bias == raw margin). This is
#      path-dependent TreeSHAP (conditional expectation over the training
#      data carried in each tree's node covers); the manifest says so.
#      _normalize_shap now refuses a 2-D array for a multiclass model.
#   T5 COUNTERFACTUAL PHYSICS IN RAW UNITS. The min<=mean<=max projection and
#      the ratio/log re-derivation were applied to STANDARDISED values, where
#      those physical relations do not hold; the first run therefore changed
#      ~17 untouched features per flow (median 22 changed although only 5 are
#      moved) and inflated flip rates. Both now run in raw units (Layer-1
#      scaler inverted) and only on features linked to a moved feature.
#   T6 Natural-language summaries report raw values and the Normal-traffic
#      percentile (standardised z-scores reached 19,868 sigma and were
#      unreadable). AE attribution also prints the ranking by share of error.
#  Reads L1..L4 of the single R8 run configuration.
#
#  R6.1 fixes the three SHAP call sites that R6.0 left on incompatible paths:
#   B1 EXPLAINER CRASH. R6.0's try/except caught (TypeError, ValueError,
#      RuntimeError) but SHAP raises a bare Exception when a pickled XGBoost
#      Booster has lost its objective attribute. The try/except never fired;
#      the run crashed at TreeExplainer construction. R6.1 skips the try
#      entirely and constructs with model_output='raw' (the raw margin
#      surface, which is what SHAP can serve for a pickled Booster and what
#      the manifest honestly claims).
#   B2 SHAPE NORMALISATION IN _p6_counterfactual. R6.0 called shap_values on
#      the counterfactual subset and indexed sv[cls_idx, i_local] without
#      normalizing the SHAP output shape. On XGBoost (n,f,c) that slices the
#      wrong axis and crashes on the next line. R6.1 routes through
#      _normalize_shap, the same helper used in _p3 and _p4.
#   B3 SHAPE NORMALISATION IN _p7_nl_summaries. Same defect as B2, one phase
#      later. Same fix.
#
#  A1..A9 (unchanged from R6.0): contract-driven, calibrated confidences,
#  honest surface claim, shape normalization, physical counterfactual,
#  variance-normalized AE attribution, raw-unit reporting, IEEE figures,
#  fail-loud paths.
#
#  XAI explainability. Reads L1 (paths), L2 (features), L3 (model + LE +
#  calibrators), L4 (AE state_dict for reconstruction-error attribution).
#
#  Comments closed (Layer-6-owned):
#   20  Fig cross-references fixed; manifest maps Fig -> text
#   63  SHAP: exact per-class TreeSHAP (native, additivity-checked),
#       background stated, raw margin surface, predicted-class reduction,
#       failed-case count + reason recorded
#   64  Counterfactual: bounded multi-feature search toward the reject
#       distribution, physical consistency projection, 5 flows per family,
#       flip-rate reported, moves in raw units
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os, gc, json, time, pickle, argparse, platform
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import lightgbm as lgb
import xgboost as xgb

try:
    import shap
    _SHAP_OK = True
except ImportError:
    _SHAP_OK = False

try:
    import torch
    import torch.nn as nn
    _TORCH_OK = True
except ImportError:
    _TORCH_OK = False

SEED = 42
np.random.seed(SEED)

CFG = {
    'version'            : 'TITANIUM6_LAYER6_R8.0',
    'seed'               : SEED,
    'shap_bg_n'          : 500,
    'shap_global_n'      : 2_000,
    'shap_per_family_n'  : 150,
    'shap_max_display'   : 20,
    'ae_sample_n'        : 2_000,
    'cf_flows_per_family': 50,
    'cf_top_k_features'  : 5,
    'cf_max_iter'        : 30,
    'cf_step_frac'       : 0.1,
    'cf_target_conf'     : 0.5,
    'nl_top_k'           : 3,
    'eval_chunk'         : 200_000,
    'reject_class'       : 'Normal',
}

COLORS = {
    'lgbm'   : '#002F6C',
    'xgb'    : '#007A33',
    'attack' : '#C00000',
    'normal' : '#CCCCCC',
    'edge'   : '#333333',
    'thresh' : '#F5A623',
    'zday'   : '#C00000',
    'pos'    : '#C00000',
    'neg'    : '#002F6C',
}

DERIVED_SRC = {
    'Ratio__FwdBwd_Packets': ('Total Fwd Packets', 'Total Backward Packets'),
    'Ratio__FwdBwd_Bytes'  : ('Total Length of Fwd Packets',
                              'Total Length of Bwd Packets'),
    'Ratio__FwdBwd_IAT'    : ('Fwd IAT Tot', 'Bwd IAT Tot'),
    'Ratio__Active_Idle'   : ('Active Mean', 'Idle Mean'),
}
DERIVED_ALL = set(DERIVED_SRC) | {
    'Log_Flow_Duration', 'Zero_Duration_Flag',
    'Init Fwd Win Byts__present', 'Init Bwd Win Byts__present',
    'N_Invalid_Features', 'N_Clipped_Negatives'}
FLAG_PREFIXES = ('Flag', 'Cnt', 'Count', '__present', 'Protocol')


def _is_perturbable(name):
    return (name not in DERIVED_ALL
            and not any(k in name for k in FLAG_PREFIXES))


def _normalize_shap(sv, n_classes):
    """LightGBM returns (n_classes, n, f); XGBoost returns (n, f, n_classes).
    Both are normalised to (n_classes, n, f) or the run is refused."""
    arr = np.array(sv) if isinstance(sv, list) else np.asarray(sv)
    if arr.ndim == 3:
        if arr.shape[0] == n_classes:
            return arr
        if arr.shape[2] == n_classes:
            return np.transpose(arr, (2, 0, 1))
        raise RuntimeError(f'SHAP shape {arr.shape} incompatible with '
                           f'n_classes={n_classes}')
    if arr.ndim == 2 and n_classes <= 2:
        return arr[np.newaxis, :, :]
    raise RuntimeError(f'SHAP returned shape {arr.shape} for a {n_classes}-class '
                       'model: one attribution per class is required')


def _gc(): gc.collect()
def _mkdir(p): Path(p).mkdir(parents=True, exist_ok=True); return Path(p)
def _atomic_json(payload, dest):
    tmp = Path(dest).parent/(Path(dest).name+'.__tmp__')
    with open(tmp,'w',encoding='utf-8') as fh: json.dump(payload,fh,indent=2,default=str)
    os.replace(tmp,dest)
def _save(obj, dest):
    tmp = Path(dest).parent/(Path(dest).name+'.__tmp__')
    with open(tmp,'wb') as fh: pickle.dump(obj,fh,protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,dest)
def _plt_style():
    plt.rcParams.update({
        'pdf.fonttype': 42, 'ps.fonttype': 42,
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Nimbus Roman',
                       'STIXGeneral', 'DejaVu Serif'],
        'mathtext.fontset': 'stix', 'font.size': 8,
        'axes.titlesize': 8.5, 'axes.labelsize': 8,
        'xtick.labelsize': 7, 'ytick.labelsize': 7,
        'legend.fontsize': 6.8, 'axes.linewidth': 0.7,
        'savefig.dpi': 600, 'axes.edgecolor': COLORS['edge']})


if _TORCH_OK:
    class SparseAutoencoder(nn.Module):
        def __init__(self, input_dim, hidden_dims):
            super().__init__()
            enc = []; prev = input_dim
            for h in hidden_dims:
                enc += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                        nn.ReLU(), nn.Dropout(0.1)]
                prev = h
            self.encoder = nn.Sequential(*enc)
            dec = []
            for h in reversed(hidden_dims[:-1]):
                dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
                prev = h
            dec += [nn.Linear(prev, input_dim)]
            self.decoder = nn.Sequential(*dec)
        INPUT_TRANSFORM = 'signed_log1p'
        @staticmethod
        def prep(x):
            x = torch.as_tensor(x, dtype=torch.float32)
            return torch.sign(x) * torch.log1p(torch.abs(x))
        def forward(self, x):
            return self.decoder(self.encoder(x))
        def reconstruction_error_per_feature(self, x):
            with torch.no_grad():
                x_t = self.prep(x)
                x_r = self(x_t)
                return ((x_t - x_r) ** 2).numpy().astype(np.float32)


def _wilson(k, n, z=1.959964):
    if n <= 0:
        return [0.0, 1.0]
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return [float(max(0.0, (c - r) / d)), float(min(1.0, (c + r) / d))]


class Layer6Pipeline:
    def __init__(self, l1_dir, l2_dir, l3_dir, l4_dir, l6_dir):
        self.l1 = Path(l1_dir); self.l2 = Path(l2_dir)
        self.l3 = Path(l3_dir); self.l4 = Path(l4_dir)
        self.l6 = _mkdir(l6_dir); self.figs = _mkdir(self.l6/'figures')
        self.manifest_path = self.l6/'manifest_l6.json'

        for p, nm in [(self.l1/'manifest.json','L1'),
                       (self.l2/'manifest_l2.json','L2'),
                       (self.l3/'manifest_l3.json','L3'),
                       (self.l4/'manifest_l4.json','L4')]:
            if not p.exists(): raise FileNotFoundError(f'{nm} manifest missing: {p}')
        with open(self.l1/'manifest.json') as fh: self.m1 = json.load(fh)
        with open(self.l2/'manifest_l2.json') as fh: self.m2 = json.load(fh)
        with open(self.l3/'manifest_l3.json') as fh: self.m3 = json.load(fh)
        with open(self.l4/'manifest_l4.json') as fh: self.m4 = json.load(fh)

        c3 = self.m3['l4_l5_contract']
        for k_ in ('model', 'encoder', 'classes'):
            if k_ not in c3:
                raise KeyError(f'L3 l4_l5_contract missing {k_!r}: run L3 R8.0')
        self.reject_class = c3.get('reject_class') or CFG['reject_class']
        CFG['reject_class'] = self.reject_class
        self.features = list(c3.get('features') or self.m2['l3_l4_contract']['features'])
        self.all_classes = list(c3['classes'])
        self.classes = [c for c in self.all_classes if c != self.reject_class]
        self.l3_model_path = c3['model']
        self.l3_le_path = c3['encoder']

        self.l3_cals = None
        _cp = c3.get('calibrators')
        if _cp and Path(_cp).exists():
            with open(_cp, 'rb') as _fh:
                self.l3_cals = pickle.load(_fh)
            print(f'  L3 calibrators: '
                  f'{sum(1 for c in self.l3_cals if c is not None)}'
                  f'/{len(self.l3_cals)} fitted')
        else:
            raise RuntimeError(
                'L3 calibrators missing from l4_l5_contract. Reported '
                'confidences and the counterfactual stop rule must use the '
                'DEPLOYED (calibrated) probabilities. Re-run Layer 3 R8.0.')

        self.paths = self.m1['l2_input_paths']
        for key in ('train','val','test_known'):
            if not Path(self.paths[key]).exists():
                raise FileNotFoundError(f'L1 file missing: {self.paths[key]}')
        if not _SHAP_OK:
            raise ImportError('shap is required: pip install shap')
        if not _TORCH_OK:
            raise ImportError('torch is required for AE attribution')

        self.l3_model = None; self.l3_le = None
        self.ae_model = None; self.ae_arch = None
        self.feat_stats_attack = None
        self.feat_stats_normal = None
        self._shap_output_mode = None
        self.global_shap_diag = {}
        self.family_shap = {}
        self.ae_attr = {}
        self.counterfactual = {}
        self.nl_summaries = {}

        print('='*76)
        print(f'  TITANIUM-6 | LAYER 6 | {CFG["version"]}')
        print('='*76)
        print(f'  features : {len(self.features)}')
        print(f'  classes  : {len(self.classes)} (+ reject "{self.reject_class}")')
        print(f'  L6 dir   : {self.l6}')

    def _raw_proba(self, X):
        if self.model_type == 'LightGBM':
            return self.l3_model.predict(X).astype(np.float32)
        return self.l3_model.predict(
            xgb.DMatrix(X, feature_names=self.features)
        ).reshape(-1, len(self.all_classes)).astype(np.float32)

    def _predict_proba(self, X):
        q = np.array(self._raw_proba(X), dtype=np.float32, copy=True)
        if self.l3_cals is not None:
            for c, lr in enumerate(self.l3_cals):
                if lr is not None and c < q.shape[1]:
                    pc = np.clip(q[:, c], 1e-7, 1 - 1e-7)
                    z = np.log(pc / (1 - pc)).reshape(-1, 1)
                    q[:, c] = lr.predict_proba(z)[:, 1].astype(np.float32)
            t = q.sum(axis=1, keepdims=True)
            q = (q / np.where(t == 0, 1, t)).astype(np.float32)
        return q

    def _shap(self, X):
        """T4: exact per-class TreeSHAP from the model's native routine.
        Returns (n_classes, n, f). Additivity is verified on every call."""
        X = np.ascontiguousarray(X, dtype=np.float32)
        n, f, k = len(X), len(self.features), len(self.all_classes)
        if n == 0:
            return np.zeros((k, 0, f), np.float32)
        if self.model_type == 'LightGBM':
            c = np.asarray(self.l3_model.predict(X, pred_contrib=True))
            raw = np.asarray(self.l3_model.predict(X, raw_score=True)).reshape(n, k)
            c = c.reshape(n, k, f + 1)
        else:
            d = xgb.DMatrix(X, feature_names=self.features)
            c = np.asarray(self.l3_model.predict(d, pred_contribs=True)).reshape(n, k, f + 1)
            raw = np.asarray(self.l3_model.predict(d, output_margin=True)).reshape(n, k)
        err = float(np.max(np.abs(c.sum(axis=2) - raw)))
        self._additivity_max_err = max(getattr(self, '_additivity_max_err', 0.0), err)
        if err > 1e-3:
            raise RuntimeError(f'TreeSHAP additivity violated (max |err| {err:.2e})')
        return np.transpose(c[:, :, :f], (1, 0, 2)).astype(np.float32)

    # ------------------------------------------------------------------ 1
    def _p1_load(self):
        print('\n[PHASE 1] Load L3 model + L4 AE')
        with open(self.l3_model_path, 'rb') as fh: self.l3_model = pickle.load(fh)
        with open(self.l3_le_path, 'rb') as fh: self.l3_le = pickle.load(fh)
        if isinstance(self.l3_model, lgb.Booster):
            self.model_type = 'LightGBM'
        elif isinstance(self.l3_model, xgb.Booster):
            self.model_type = 'XGBoost'
        else:
            raise TypeError(f'unsupported L3 model: {type(self.l3_model)}')
        print(f'  L3 model: {self.model_type} '
              f'({len(self.all_classes)} classes, reject={self.reject_class})')

        ae_arch_path = self.l4/'ae_architecture.json'
        ae_state_path = self.l4/'ae_model_state.pt'
        if ae_arch_path.exists() and ae_state_path.exists():
            with open(ae_arch_path) as fh: self.ae_arch = json.load(fh)
            if self.ae_arch.get('input_transform') != 'signed_log1p':
                raise RuntimeError('L4 ae_architecture.json lacks input_transform='
                                   'signed_log1p: re-run Layer 4 R8.0 first')
            ae = SparseAutoencoder(self.ae_arch['input_dim'],
                                    self.ae_arch['hidden_dims'])
            ae.load_state_dict(torch.load(ae_state_path, map_location='cpu'))
            ae.eval()
            self.ae_model = ae
            print(f'  L4 AE: {self.ae_arch["input_dim"]} -> '
                  f'{self.ae_arch["hidden_dims"]} -> '
                  f'{self.ae_arch["input_dim"]}')
        else:
            print('  L4 AE: state file missing -- AE attribution skipped')

    # ------------------------------------------------------------------ 2
    def _p2_feature_stats(self):
        print('\n[PHASE 2] Feature statistics (attack + normal)')
        feats = self.features

        def _load(path, mode, n_max):
            Xl = []; collected = 0
            for b in pq.ParquetFile(path).iter_batches(
                    batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
                df = b.to_pandas()
                if mode == 'attack':
                    df = df[df['Target'] != 'Normal']
                elif mode == 'normal':
                    df = df[df['Target'] == 'Normal']
                if len(df):
                    Xl.append(df[feats].values.astype(np.float32))
                    collected += len(df)
                del df, b; _gc()
                if collected >= n_max: break
            return np.vstack(Xl)[:n_max] if Xl else np.zeros((0, len(feats)), np.float32)

        self._scaler_mu = self._scaler_sd = None
        _sp = self.paths.get('scaler')
        if _sp and Path(_sp).exists():
            with open(_sp) as _fh: _sc = json.load(_fh)
            self._scaler_mu = np.array([_sc['mean'][f] for f in feats])
            self._scaler_sd = np.array([_sc['std'][f] for f in feats])
            print('  Layer-1 scaler loaded: counterfactual moves reported in '
                  'raw units')
        else:
            print('  [WARN] Layer-1 scaler not found: counterfactual moves '
                  'reported on the standardised scale only')

        Xa = _load(self.paths['train'], 'attack', 500_000)
        Xn = _load(self.paths['train'], 'normal', 200_000)
        print(f'  attack sample: {len(Xa):,}  normal sample: {len(Xn):,}')

        mu_a = Xa.mean(axis=0); sd_a = Xa.std(axis=0)
        p1_a = np.percentile(Xa, 1, axis=0)
        p99_a = np.percentile(Xa, 99, axis=0)
        mu_n = Xn.mean(axis=0); sd_n = Xn.std(axis=0)
        self._normal_q = np.percentile(Xn, np.arange(101), axis=0)   # T6

        self.feat_stats_attack = {
            f: {'mean': float(mu_a[i]), 'std': float(max(sd_a[i], 1e-9)),
                'p1': float(p1_a[i]), 'p99': float(p99_a[i])}
            for i, f in enumerate(feats)
        }
        self.feat_stats_normal = {
            f: {'mean': float(mu_n[i]), 'std': float(max(sd_n[i], 1e-9))}
            for i, f in enumerate(feats)
        }
        del Xa, Xn; _gc()

    def _sample_attack(self, path, n):
        feats = self.features
        rng = np.random.RandomState(SEED)
        Xl = []; collected = 0
        for b in pq.ParquetFile(path).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas(); df = df[df['Target'] != 'Normal']
            if len(df):
                Xl.append(df[feats].values.astype(np.float32))
                collected += len(df)
            del df, b; _gc()
            if collected >= n * 4: break
        X = np.vstack(Xl) if Xl else np.zeros((0, len(feats)), np.float32)
        if len(X) > n:
            idx = np.sort(rng.choice(len(X), n, replace=False))
            X = X[idx]
        return X

    def _sample_attack_per_family(self, path, n_per):
        feats = self.features
        rng = np.random.RandomState(SEED)
        out = defaultdict(list); counts = defaultdict(int)
        for b in pq.ParquetFile(path).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas(); df = df[df['Target'] != 'Normal']
            for cls in df['Target'].unique():
                if counts[cls] >= n_per: continue
                sub = df[df['Target'] == cls]
                need = n_per - counts[cls]
                take = min(need, len(sub))
                idx = rng.choice(len(sub), take, replace=False)
                out[cls].append(sub.iloc[idx][feats].values.astype(np.float32))
                counts[cls] += take
            del df, b; _gc()
        return {c: np.vstack(v)[:n_per] for c, v in out.items()}

    def _load_normal(self, path, n):
        feats = self.features
        rng = np.random.RandomState(SEED)
        Xl = []; collected = 0
        for b in pq.ParquetFile(path).iter_batches(
                batch_size=CFG['eval_chunk'], columns=feats + ['Target']):
            df = b.to_pandas(); df = df[df['Target'] == 'Normal']
            if len(df):
                Xl.append(df[feats].values.astype(np.float32))
                collected += len(df)
            del df, b; _gc()
            if collected >= n * 3: break
        X = np.vstack(Xl) if Xl else np.zeros((0, len(feats)), np.float32)
        if len(X) > n:
            idx = np.sort(rng.choice(len(X), n, replace=False))
            X = X[idx]
        return X

    # ------------------------------------------------------------------ 3
    def _p3_global_shap(self):
        print('\n[PHASE 3] Global TreeSHAP (exact, native, per class)')
        feats = self.features
        X_test = self._sample_attack(self.paths['test_known'], CFG['shap_global_n'])
        print(f'  test attack flows: {len(X_test):,}')
        # Attributions describe the UNCALIBRATED raw margin (a per-class Platt
        # map cannot be pushed inside the trees); confidences are calibrated.
        self._shap_output_mode = 'raw margin (log-odds), uncalibrated'
        t0 = time.perf_counter()
        sv_arr = _normalize_shap(self._shap(X_test), len(self.all_classes))
        elapsed = time.perf_counter() - t0
        n_samples = sv_arr.shape[1]
        row_valid = np.isfinite(sv_arr).all(axis=(0, 2))
        n_ok = int(row_valid.sum())
        n_failed = int(n_samples - n_ok)
        if n_ok == 0:
            raise RuntimeError('[FATAL] no finite SHAP rows -- cannot proceed')
        if n_failed:
            print(f'  [WARN] {n_failed} of {n_samples} flows produced a '
                  'non-finite SHAP value and are excluded from every figure '
                  'and table below. Cause: the predicted class has no support '
                  'in the background sample, so the interventional '
                  'expectation is undefined for that class.')
        self.global_shap_diag = {
            'mode': 'tree_path_dependent (exact TreeSHAP)',
            'backend': f'{self.model_type} native contributions',
            'background': ('the L3 training data, via the node covers stored in '
                           'each tree (no separate background sample)'),
            'model_output': self._shap_output_mode,
            'multiclass_reduction': 'predicted-class',
            'additivity_max_abs_error': float(self._additivity_max_err),
            'n_requested': int(n_samples),
            'n_processed': int(n_ok),
            'n_failed': n_failed,
            'compute_seconds': float(elapsed),
            'note': ('comment 63 closure: exact per-class TreeSHAP, background '
                     'stated, predicted-class reduction, raw margin surface, '
                     'additivity verified.'),
        }
        print(f'  SHAP {elapsed:.1f}s  shape {sv_arr.shape}  failed {n_failed}  '
              f'additivity max|err| {self._additivity_max_err:.1e}')

        probs = self._predict_proba(X_test)
        pred_cls = probs.argmax(axis=1)
        sv_pred = sv_arr[pred_cls, np.arange(n_samples)]
        self._row_valid = row_valid
        mean_abs = np.abs(sv_pred[row_valid]).mean(axis=0)
        order = np.argsort(mean_abs)[::-1]
        top20 = [(feats[i], float(mean_abs[i])) for i in order[:20]]
        self.global_shap_top20 = top20
        self._sv_pred = sv_pred
        self._X_test_shap = X_test
        self._pred_cls_shap = pred_cls

        print('  top-5 features:')
        for f, v in top20[:5]:
            print(f'    {f:<40} {v:.4f}')

    # ------------------------------------------------------------------ 4
    def _p4_family_shap(self):
        print('\n[PHASE 4] Per-family SHAP')
        feats = self.features
        X_fam = self._sample_attack_per_family(self.paths['test_known'],
                                                 CFG['shap_per_family_n'])
        family_shap = {}
        for cls, X in X_fam.items():
            if len(X) == 0: continue
            sv = _normalize_shap(self._shap(X), len(self.all_classes))
            probs = self._predict_proba(X)
            pred = probs.argmax(axis=1)
            sv_pred = sv[pred, np.arange(sv.shape[1])]
            row_ok = np.isfinite(sv_pred).all(axis=1)
            if row_ok.sum() == 0:
                continue
            mean_abs = np.abs(sv_pred[row_ok]).mean(axis=0)
            top5_idx = np.argsort(mean_abs)[::-1][:5]
            family_shap[cls] = {
                'top5': [{'feature': feats[i],
                          'mean_abs_shap': float(mean_abs[i])}
                          for i in top5_idx],
                'n_flows': int(len(X)),
            }
            del sv, probs, sv_pred; _gc()
        self.family_shap = family_shap
        print('  Table XIV -- top-2 features per family:')
        for cls in sorted(family_shap):
            t = family_shap[cls]['top5']
            t1 = t[0]['feature'] if t else 'N/A'
            t2 = t[1]['feature'] if len(t) > 1 else 'N/A'
            print(f'    {cls:<22} | {t1:<30} | {t2}')

    # ------------------------------------------------------------------ 5
    def _p5_ae_attribution(self):
        print('\n[PHASE 5] AE per-feature reconstruction error')
        if self.ae_model is None:
            self.ae_attr = {'note': 'AE not available'}
            print('  skipped (AE missing)')
            return
        feats = self.features
        Xa = self._sample_attack(self.paths['test_known'], CFG['ae_sample_n'])
        Xn = self._load_normal(self.paths['test_known'], CFG['ae_sample_n'])
        if len(Xa) == 0 or len(Xn) == 0:
            self.ae_attr = {'note': 'insufficient samples'}
            print('  skipped (no data)')
            return
        err_atk = self.ae_model.reconstruction_error_per_feature(Xa).mean(axis=0)
        err_nor = self.ae_model.reconstruction_error_per_feature(Xn).mean(axis=0)
        ratio = err_atk / np.maximum(err_nor, 1e-9)
        share = err_atk / max(float(err_atk.sum()), 1e-12)
        order = np.argsort(ratio)[::-1]
        top20 = [{'feature': feats[i],
                   'ratio': float(ratio[i]),
                   'share_of_attack_error': float(share[i]),
                   'err_atk': float(err_atk[i]),
                   'err_nor': float(err_nor[i])}
                  for i in order[:20]]
        self.ae_attr = {
            'top20_anomalous': top20,
            'n_attack': int(len(Xa)),
            'n_normal': int(len(Xn)),
            'method': ('per-feature squared reconstruction error of the Layer-4 '
                       'autoencoder in its signed_log1p input space, averaged per '
                       'group; ranked by attack/normal ratio; share = the '
                       "feature's fraction of total attack reconstruction error"),
            'ranking_statistic': 'ratio',
        }
        print('  top-5 by attack/normal ratio (inflated when Normal error ~0):')
        for d in top20[:5]:
            print(f'    {d["feature"]:<32} ratio={d["ratio"]:>10.3f}  '
                  f'share={100 * d["share_of_attack_error"]:5.2f}%  '
                  f'err_atk={d["err_atk"]:.3e}  err_nor={d["err_nor"]:.3e}')
        by_share = np.argsort(share)[::-1][:5]
        self.ae_attr['top5_by_share'] = [{'feature': feats[i], 'share': float(share[i]),
                                          'ratio': float(ratio[i])} for i in by_share]
        print('  top-5 by share of total attack reconstruction error:')
        for i in by_share:
            print(f'    {feats[i]:<32} share={100 * share[i]:5.2f}%  ratio={ratio[i]:.2f}')
        self._ae_err_atk = err_atk
        self._ae_err_nor = err_nor

    # ------------------------------------------------------------------ 6
    def _p6_counterfactual(self):
        print(f'\n[PHASE 6] Bounded multi-feature counterfactual '
              f'(target={self.reject_class})')
        feats = self.features
        phys = [
            ('Fwd Packet Length Min','Fwd Packet Length Mean','<='),
            ('Fwd Packet Length Mean','Fwd Packet Length Max','<='),
            ('Bwd Packet Length Min','Bwd Packet Length Mean','<='),
            ('Bwd Packet Length Mean','Bwd Packet Length Max','<='),
            ('Packet Length Min','Packet Length Mean','<='),
            ('Packet Length Mean','Packet Length Max','<='),
            ('Subflow Fwd Bytes','Total Length of Fwd Packets','<='),
            ('Subflow Bwd Bytes','Total Length of Bwd Packets','<='),
            ('Subflow Fwd Packets','Total Fwd Packets','<='),
            ('Subflow Bwd Packets','Total Backward Packets','<='),
            ('Fwd IAT Min','Fwd IAT Mean','<='),
            ('Fwd IAT Mean','Fwd IAT Max','<='),
            ('Bwd IAT Min','Bwd IAT Mean','<='),
            ('Bwd IAT Mean','Bwd IAT Max','<='),
            ('Active Min','Active Mean','<='),
            ('Active Mean','Active Max','<='),
            ('Idle Min','Idle Mean','<='),
            ('Idle Mean','Idle Max','<='),
        ]
        idx_of = {f: i for i, f in enumerate(feats)}
        phys_idx = [(idx_of[t], idx_of[s], op) for t, s, op in phys
                     if t in idx_of and s in idx_of]

        _predict = self._predict_proba
        name_idx = {n: i for i, n in enumerate(feats)}
        mu_s = self._scaler_mu
        sd_s = (None if self._scaler_sd is None
                else np.where(self._scaler_sd == 0, 1.0, self._scaler_sd))

        def _physical(x, moved):
            """T5: enforce physical relations and Layer-1 derivations in RAW
            units, only for features linked to a moved feature."""
            if sd_s is None:
                return x
            r = x[0].astype(np.float64) * sd_s + mu_s
            touched = set(moved)
            for _ in range(2):                              # min<=mean<=max chains
                for jt, js, op in phys_idx:
                    if jt in touched or js in touched:
                        v = min(r[jt], r[js]) if op == '<=' else max(r[jt], r[js])
                        if v != r[jt]:
                            r[jt] = v; touched.add(jt)
            for new, (a, b) in DERIVED_SRC.items():         # same rule as Layer 1
                if new in name_idx and a in name_idx and b in name_idx and \
                        (name_idx[a] in touched or name_idx[b] in touched):
                    den = r[name_idx[b]]
                    den = 1.0 if den == 0 else den
                    r[name_idx[new]] = float(np.clip(r[name_idx[a]] / den, 0.0, 1e6))
                    touched.add(name_idx[new])
            j_d = name_idx.get('Flow Duration')
            if j_d is not None and j_d in touched:
                dur = max(r[j_d], 0.0)
                for nm, val in (('Log_Flow_Duration', np.log1p(dur)),
                                ('Zero_Duration_Flag', float(dur == 0))):
                    if nm in name_idx:
                        r[name_idx[nm]] = val; touched.add(name_idx[nm])
            out = x.copy()
            for j in touched:
                out[0, j] = np.float32((r[j] - mu_s[j]) / sd_s[j])
            return out

        X_fam = self._sample_attack_per_family(self.paths['test_known'],
                                                 CFG['cf_flows_per_family'] * 4)
        cf_results = {}
        for cls in self.classes:
            if cls not in X_fam: continue
            X = X_fam[cls]
            probs = _predict(X)
            cls_idx = int(self.l3_le.transform([cls])[0])
            k = min(CFG['cf_flows_per_family'], len(X))
            # T2: random flows the model assigns to this family, not the
            # most confident ones (representative, not best-case)
            pred_cls = probs.argmax(axis=1) == cls_idx
            pool = np.flatnonzero(pred_cls) if pred_cls.sum() >= k else np.arange(len(X))
            top_k_idx = np.sort(np.random.RandomState(SEED + cls_idx).choice(
                pool, min(k, len(pool)), replace=False))
            k = len(top_k_idx)
            flips = 0
            per_flow = []

            # B2: normalize the SHAP shape before indexing
            sv = _normalize_shap(self._shap(X[top_k_idx]), len(self.all_classes))

            for i_local, i_global in enumerate(top_k_idx):
                x0 = X[i_global:i_global+1].copy()
                orig_prob = float(probs[i_global, cls_idx])
                sv_flow = sv[cls_idx, i_local]
                _ord = np.argsort(np.abs(sv_flow))[::-1]
                top_k_feats = [j for j in _ord
                               if _is_perturbable(feats[j])][:CFG['cf_top_k_features']]
                x = x0.copy()
                found = False; steps = 0
                for step in range(CFG['cf_max_iter']):
                    steps = step + 1
                    for j in top_k_feats:
                        mu_n = self.feat_stats_normal[feats[j]]['mean']
                        sd_n = self.feat_stats_normal[feats[j]]['std']
                        x[0, j] = float(np.clip(
                            x[0, j] + (mu_n - x[0, j]) * CFG['cf_step_frac'],
                            mu_n - 2 * sd_n, mu_n + 2 * sd_n))
                    x = _physical(x, top_k_feats)
                    new_prob = float(_predict(x)[0, cls_idx])
                    if new_prob < CFG['cf_target_conf']:
                        found = True; break
                final_prob = float(_predict(x)[0, cls_idx])
                if found: flips += 1
                raw_moves = None
                if getattr(self, '_scaler_sd', None) is not None:
                    raw_moves = {}
                    for j in top_k_feats:
                        v0 = float(x0[0, j] * self._scaler_sd[j] + self._scaler_mu[j])
                        v1 = float(x[0, j] * self._scaler_sd[j] + self._scaler_mu[j])
                        raw_moves[feats[j]] = {'from_raw': v0, 'to_raw': v1,
                                               'delta_raw': v1 - v0}
                per_flow.append({
                    'original_conf': orig_prob,
                    'final_conf': final_prob,
                    'confidence_scale': 'calibrated (deployed)',
                    'steps': steps,
                    'found': found,
                    'top_k_features_used': [feats[j] for j in top_k_feats],
                    'delta_l2_norm_standardised': float(np.linalg.norm(x[0] - x0[0])),
                    'n_features_changed': int((np.abs(x[0] - x0[0]) > 1e-6).sum()),
                    'physics_space': 'raw units' if sd_s is not None else 'none (no scaler)',
                    'raw_unit_moves': raw_moves,
                    'target_distribution': f'{self.reject_class} (reject class)',
                })
            _nc = [d_['n_features_changed'] for d_ in per_flow if d_['found']]
            cf_results[cls] = {
                'n_flows_tested': k,
                'flips': flips,
                'flip_rate': flips / max(k, 1),
                'flip_rate_wilson95': _wilson(flips, k),
                'median_features_changed_when_found': (float(np.median(_nc))
                                                       if _nc else None),
                'selection': 'random sample of flows L3 assigns to this family',
                'per_flow': per_flow,
            }
            del sv; _gc()
        self.counterfactual = cf_results
        print('  family                    flip_rate   95% CI           n  med#feat')
        for cls in sorted(cf_results):
            d = cf_results[cls]
            lo, hi = d['flip_rate_wilson95']
            mf = d['median_features_changed_when_found']
            print(f'    {cls:<24} {d["flip_rate"]*100:>6.1f}%  '
                  f'[{lo*100:5.1f},{hi*100:5.1f}]  {d["n_flows_tested"]:>4}  '
                  f'{"-" if mf is None else f"{mf:.0f}"}')

    # ------------------------------------------------------------------ 7
    def _p7_nl_summaries(self):
        print('\n[PHASE 7] Natural language summaries')
        feats = self.features
        X_fam = self._sample_attack_per_family(self.paths['test_known'], 5)
        out = {}
        for cls in self.classes:
            if cls not in X_fam: continue
            X = X_fam[cls]
            probs = self._predict_proba(X)
            cls_idx = int(self.l3_le.transform([cls])[0])
            best = int(np.argmax(probs[:, cls_idx]))
            conf = float(probs[best, cls_idx])
            x_row = X[best]

            # B3: normalize the SHAP shape before indexing
            sv = _normalize_shap(self._shap(X[best:best+1]), len(self.all_classes))

            sv_flow = sv[cls_idx, 0]
            top_idx = np.argsort(np.abs(sv_flow))[::-1][:CFG['nl_top_k']]
            top_feats = []
            for j in top_idx:
                feat = feats[j]
                v = float(x_row[j])
                mu_n = self.feat_stats_normal[feat]['mean']
                sd_n = self.feat_stats_normal[feat]['std']
                z = (v - mu_n) / max(sd_n, 1e-9)
                pct = float(np.searchsorted(self._normal_q[:, j], v, side='right') - 1)
                pct = min(max(pct, 0.0), 100.0)
                raw = (float(v * self._scaler_sd[j] + self._scaler_mu[j])
                       if self._scaler_sd is not None else None)
                top_feats.append({'feature': feat, 'value_standardised': v,
                                   'value_raw': raw,
                                   'normal_percentile': pct,
                                   'shap': float(sv_flow[j]),
                                   'z_score_vs_normal': z})
            parts = []
            for t in top_feats:
                val = (f"{t['value_raw']:.4g}" if t['value_raw'] is not None
                       else f"{t['value_standardised']:.3f} (std.)")
                if t['normal_percentile'] >= 99:
                    where = 'above ~99% of Normal flows'
                elif t['normal_percentile'] <= 1:
                    where = 'below ~99% of Normal flows'
                else:
                    where = f"Normal percentile {t['normal_percentile']:.0f}"
                parts.append(f"{t['feature']}={val} ({where})")
            summary = (f"Classified as {cls} with {conf*100:.1f}% confidence. "
                        f"Primary drivers: {', '.join(parts)}.")
            out[cls] = {'nl_summary': summary, 'confidence': conf,
                         'top_features': top_feats}
            print(f'  [{cls}] {summary[:100]}...')
            del sv; _gc()
        self.nl_summaries = out

    # ------------------------------------------------------------------ 8
    def _p8_figures(self):
        print('\n[PHASE 8] Figures (5 PDFs)')
        _plt_style()
        feats = self.features

        # Fig 25: Global SHAP beeswarm
        try:
            fig = plt.figure(figsize=(10, 7))
            _m = self._row_valid
            with warnings.catch_warnings():
                # shap>=0.50 warns that it reads NumPy's global RNG (seeded at
                # the top of this file); the plot is deterministic either way.
                warnings.filterwarnings('ignore', category=FutureWarning,
                                        message='.*global RNG.*')
                shap.summary_plot(
                    self._sv_pred[_m], self._X_test_shap[_m],
                    feature_names=feats,
                    max_display=CFG['shap_max_display'],
                    show=False, plot_size=None)
            plt.title('Fig 25.  Global SHAP beeswarm (predicted-class, '
                       'attack-domain background; comment 63)', fontsize=10)
            plt.tight_layout()
            plt.savefig(self.figs/'L6_fig25_global_beeswarm.pdf',
                         dpi=300, bbox_inches='tight'); plt.close(fig)
            print('  Fig 25 written')
        except Exception as e:
            print(f'  Fig 25 shap.summary_plot failed: {e}; using bar fallback')
            self._fig_bar_importance()

        # Fig 26: Per-family SHAP heatmap
        if self.family_shap:
            top10 = [f for f, _ in self.global_shap_top20[:10]]
            fams = sorted(self.family_shap.keys())
            M = np.zeros((len(fams), len(top10)))
            for i, fam in enumerate(fams):
                d = {t['feature']: t['mean_abs_shap']
                     for t in self.family_shap[fam]['top5']}
                for j, f in enumerate(top10):
                    M[i, j] = d.get(f, 0.0)
            cmax = M.max(axis=0, keepdims=True)
            Mn = np.where(cmax == 0, 0, M / cmax)
            import matplotlib.colors as mcolors
            cmap = mcolors.LinearSegmentedColormap.from_list(
                'shap', [COLORS['normal'], COLORS['thresh'], COLORS['attack']], N=256)
            fig, ax = plt.subplots(figsize=(12, max(6, len(fams) * 0.5)))
            im = ax.imshow(Mn, cmap=cmap, vmin=0, vmax=1, aspect='auto')
            plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02,
                          label='Normalised mean |SHAP|')
            ax.set_xticks(range(len(top10)))
            ax.set_xticklabels(top10, rotation=35, ha='right', fontsize=8)
            ax.set_yticks(range(len(fams)))
            ax.set_yticklabels(fams, fontsize=8)
            for i in range(len(fams)):
                for j in range(len(top10)):
                    v = Mn[i, j]
                    if v > 0.05:
                        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                                 fontsize=6.5,
                                 color='white' if v > 0.6 else COLORS['edge'])
            ax.set_title('Fig 26.  Per-family SHAP heatmap '
                          '(rows=families, cols=top-10 global features)', fontsize=10)
            fig.tight_layout()
            fig.savefig(self.figs/'L6_fig26_family_heatmap.pdf',
                         dpi=300, bbox_inches='tight'); plt.close(fig)
            print('  Fig 26 written')

        # Fig 27: AE per-feature reconstruction
        if self.ae_attr.get('top20_anomalous'):
            top20 = self.ae_attr['top20_anomalous']
            names = [d['feature'] for d in top20]
            ea = [d['err_atk'] for d in top20]
            en = [d['err_nor'] for d in top20]
            x = np.arange(len(names)); w = 0.4
            fig, ax = plt.subplots(figsize=(13, 5.5))
            ax.bar(x - w/2, en, w, color=COLORS['neg'], edgecolor=COLORS['edge'],
                    label='Normal')
            ax.bar(x + w/2, ea, w, color=COLORS['pos'], edgecolor=COLORS['edge'],
                    label='Attack')
            ax.set_xticks(x); ax.set_xticklabels(names, rotation=35,
                                                  ha='right', fontsize=8)
            ax.set_ylabel('Mean squared reconstruction error')
            ax.set_title('Fig 27.  AE per-feature reconstruction error '
                          '(top 20 by attack/normal ratio)', fontsize=10)
            ax.legend(); ax.grid(axis='y', alpha=0.3)
            fig.tight_layout()
            fig.savefig(self.figs/'L6_fig27_ae_attribution.pdf',
                         dpi=300, bbox_inches='tight'); plt.close(fig)
            print('  Fig 27 written')

        # Fig 28: Counterfactual flip rates
        if self.counterfactual:
            fams = sorted(self.counterfactual.keys())
            rates = [self.counterfactual[f]['flip_rate'] * 100 for f in fams]
            fig, ax = plt.subplots(figsize=(11, 5))
            x = np.arange(len(fams))
            ax.bar(x, rates, color=COLORS['lgbm'], edgecolor=COLORS['edge'])
            for xi, r in enumerate(rates):
                ax.text(xi, r + 1.5, f'{r:.1f}%', ha='center', fontsize=8)
            ax.set_xticks(x); ax.set_xticklabels(fams, rotation=35,
                                                  ha='right', fontsize=8)
            ax.set_ylim(0, 105); ax.set_ylabel('Flip rate (%)')
            ax.set_title(f'Fig 28.  Counterfactual flip rate per family '
                          f'({CFG["cf_flows_per_family"]} flows/family, '
                          f'bounded multi-feature search; comment 64)', fontsize=10)
            ax.grid(axis='y', alpha=0.3)
            fig.tight_layout()
            fig.savefig(self.figs/'L6_fig28_counterfactual.pdf',
                         dpi=300, bbox_inches='tight'); plt.close(fig)
            print('  Fig 28 written')

        # Fig 29: NL summary panel
        if self.nl_summaries:
            fams = sorted(self.nl_summaries.keys())
            n = len(fams); cols = 3
            rows = (n + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(16, 3.6 * rows))
            axes = np.array(axes).reshape(-1)
            for i, fam in enumerate(fams):
                ax = axes[i]
                ax.axis('off')
                txt = self.nl_summaries[fam]['nl_summary']
                words = txt.split()
                wrapped = []; line = ''
                for w in words:
                    if len(line) + len(w) + 1 > 60:
                        wrapped.append(line); line = w
                    else:
                        line = (line + ' ' + w).strip()
                if line: wrapped.append(line)
                body = '\n'.join(wrapped)
                ax.text(0.02, 0.98, f'$\\bf{{{fam}}}$', transform=ax.transAxes,
                        fontsize=11, va='top', family='serif')
                ax.text(0.02, 0.82, body, transform=ax.transAxes,
                        fontsize=8, va='top', family='serif')
            for i in range(len(fams), len(axes)): axes[i].axis('off')
            fig.suptitle('Fig 29.  Natural-language summaries per family '
                          '(raw units; reference = Normal-traffic percentiles)', fontsize=11)
            fig.tight_layout()
            fig.savefig(self.figs/'L6_fig29_nl_summary.pdf',
                         dpi=300, bbox_inches='tight'); plt.close(fig)
            print('  Fig 29 written')

        print('  figures done')

    def _fig_bar_importance(self):
        top20 = self.global_shap_top20
        fig, ax = plt.subplots(figsize=(9, 7))
        names = [t[0] for t in top20]
        vals = [t[1] for t in top20]
        mx = max(vals) if vals else 1
        ax.barh(range(len(names)), [v/mx*100 for v in vals],
                color=COLORS['lgbm'], edgecolor=COLORS['edge'], alpha=0.85)
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
        ax.invert_yaxis(); ax.set_xlabel('Normalised mean |SHAP| (%)')
        ax.set_title('Fig 25.  Global SHAP top-20 (bar fallback)')
        ax.grid(axis='x', alpha=0.3)
        fig.tight_layout()
        fig.savefig(self.figs/'L6_fig25_global_beeswarm.pdf',
                     dpi=300, bbox_inches='tight'); plt.close(fig)
        print('  Fig 25 written (bar fallback)')

    # ------------------------------------------------------------------ 9
    def _p9_save(self):
        print('\n[PHASE 9] Saving results')
        _atomic_json({
            'attack_stats': self.feat_stats_attack,
            'normal_stats': self.feat_stats_normal,
        }, self.l6/'feature_stats.json')
        _atomic_json({
            'features': self.features,
            'classes': self.classes,
            'reject_class': self.reject_class,
            'model_type': self.model_type,
            'shap_background_domain': 'L3 training data (tree node covers)',
            'feature_perturbation': 'tree_path_dependent',
            'model_output': self._shap_output_mode,
            'multiclass_reduction': 'predicted-class',
            'shap_bg_n': CFG['shap_bg_n'],
            'shap_global_n': CFG['shap_global_n'],
            'shap_per_family_n': CFG['shap_per_family_n'],
        }, self.l6/'layer6_config.json')

    def _p10_manifest(self):
        print('\n[PHASE 10] Manifest')
        man = {
            'version': CFG['version'],
            'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'seed': SEED,
            'l3_manifest_version': self.m3.get('version'),
            'l4_manifest_version': self.m4.get('version'),
            'model_type': self.model_type,
            'features': self.features,
            'feature_count': len(self.features),
            'classes': self.classes,

            'shap_methodology': self.global_shap_diag,
            'explanation_surface': (
                'Exact per-class TreeSHAP on the UNCALIBRATED raw margin: a '
                'per-class Platt map cannot be pushed inside the trees. SHAP '
                'magnitudes are in log-odds units of the uncalibrated model; '
                'every reported confidence and the counterfactual stop rule '
                'use the DEPLOYED calibrated model.'),
            'confidence_scale': 'calibrated (per-class Platt, renormalised)',
            'reject_class': self.reject_class,
            'counterfactual_target': f'{self.reject_class} (reject distribution)',
            'shap_config': {
                'feature_perturbation': 'tree_path_dependent (exact)',
                'backend': f'{self.model_type} native contributions',
                'model_output': self._shap_output_mode,
                'multiclass_reduction': 'predicted-class',
                'background': 'L3 training data via tree node covers',
                'additivity_max_abs_error': float(self._additivity_max_err),
                'shap_version_for_plots_only': getattr(shap, '__version__', 'unknown'),
            },

            'global_top20': self.global_shap_top20,
            'family_shap': self.family_shap,
            'ae_attribution': self.ae_attr,
            'counterfactual': self.counterfactual,
            'nl_summaries': {k: {'nl': v['nl_summary'],
                                  'confidence': v['confidence'],
                                  'top_features': v['top_features']}
                              for k, v in self.nl_summaries.items()},

            'figures': {
                'Fig_25': 'L6_fig25_global_beeswarm.pdf',
                'Fig_26': 'L6_fig26_family_heatmap.pdf',
                'Fig_27': 'L6_fig27_ae_attribution.pdf',
                'Fig_28': 'L6_fig28_counterfactual.pdf',
                'Fig_29': 'L6_fig29_nl_summary.pdf',
            },
            'paper_map': {
                'Table_XIV'   : 'family_shap (top-5 per family)',
                'Table_XV'    : 'counterfactual (multi-feature bounded)',
                'Fig_25'      : 'L6_fig25_global_beeswarm.pdf  <- global SHAP top-20',
                'Fig_26'      : 'L6_fig26_family_heatmap.pdf   <- per-family heatmap',
                'Fig_27'      : 'L6_fig27_ae_attribution.pdf',
                'Fig_28'      : 'L6_fig28_counterfactual.pdf',
                'Fig_29'      : 'L6_fig29_nl_summary.pdf',
                'Sec_VIII-A'  : 'global SHAP methodology (comment 63)',
                'Sec_VIII-B'  : 'per-family SHAP',
                'Sec_VIII-C'  : 'AE anomaly attribution',
                'Sec_VIII-D'  : 'counterfactual (comment 64)',
                'Sec_VIII-E'  : 'natural-language summaries',
            },
            'cross_references_note': (
                'comment 20 closure: Fig 25 = global beeswarm, Fig 26 = family '
                'heatmap. Manifest maps each to its text reference.'
            ),
            'l7_contract': {
                'feature_stats': str(self.l6/'feature_stats.json'),
                'config': str(self.l6/'layer6_config.json'),
            },
        }
        _atomic_json(man, self.manifest_path)
        print('  manifest_l6.json written')

    def execute(self):
        t0 = time.time()
        self._p1_load()
        self._p2_feature_stats()
        self._p3_global_shap()
        self._p4_family_shap()
        self._p5_ae_attribution()
        self._p6_counterfactual()
        self._p7_nl_summaries()
        self._p8_figures()
        self._p9_save()
        self._p10_manifest()
        print('\n' + '='*76)
        print(f'  LAYER 6 R8.0 COMPLETE | {(time.time()-t0)/60:.1f} min')
        print(f'  global SHAP processed {self.global_shap_diag["n_processed"]:,} '
              f'of {self.global_shap_diag["n_requested"]:,} '
              f'({self.global_shap_diag["n_failed"]} failed)')
        print(f'  families with SHAP: {len(self.family_shap)}')
        print(f'  counterfactual families: {len(self.counterfactual)}')
        print(f'  NL summaries: {len(self.nl_summaries)}')
        print(f'  5 figures written')
        print('='*76)


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='TITANIUM-6 Layer 6 R8.0')
    ap.add_argument('--l1dir', default=os.environ.get('T6_L1', '/kaggle/working/l1_R8'))
    ap.add_argument('--l2dir', default=os.environ.get('T6_L2', '/kaggle/working/l2_R8'))
    ap.add_argument('--l3dir', default=os.environ.get('T6_L3', '/kaggle/working/l3_R8'))
    ap.add_argument('--l4dir', default=os.environ.get('T6_L4', '/kaggle/working/l4_R8'))
    ap.add_argument('--l6dir', default=os.environ.get('T6_L6', '/kaggle/working/l6_R8'))
    a, _ = ap.parse_known_args()
    for _tag, _d, _f in (('Layer 1', Path(a.l1dir), 'manifest.json'),
                         ('Layer 2', Path(a.l2dir), 'manifest_l2.json'),
                         ('Layer 3', Path(a.l3dir), 'manifest_l3.json'),
                         ('Layer 4', Path(a.l4dir), 'manifest_l4.json')):
        if not (_d / _f).exists():
            _root = _d.parent if _d.parent.exists() else Path('.')
            _found = sorted(str(q.parent) for q in _root.glob('*/' + _f))
            raise SystemExit(f'[FATAL] no {_f} in {_d}\n'
                             f'        {_tag} outputs found here: {_found or "none"}\n'
                             f'        pass --l1dir/--l2dir/--l3dir/--l4dir, or set '
                             f'T6_L1/T6_L2/T6_L3/T6_L4.')
    Layer6Pipeline(a.l1dir, a.l2dir, a.l3dir, a.l4dir, a.l6dir).execute()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  L3 calibrators: 12/12 fitted
  TITANIUM-6 | LAYER 6 | TITANIUM6_LAYER6_R8.0
  features : 77
  classes  : 11 (+ reject "Normal")
  L6 dir   : /kaggle/working/l6_R8

[PHASE 1] Load L3 model + L4 AE
  L3 model: LightGBM (12 classes, reject=Normal)
  L4 AE: 77 -> [64, 32, 16] -> 77

[PHASE 2] Feature statistics (attack + normal)
  Layer-1 scaler loaded: counterfactual moves reported in raw units
  attack sample: 500,000  normal sample: 200,000

[PHASE 3] Global TreeSHAP (exact, native, per class)
  test attack flows: 2,000
  SHAP 16.2s  shape (12, 2000, 77)  failed 0  additivity max|err| 8.5e-13
  top-5 features:
    Init Fwd Win Byts                        2.2797
    Min Seg Size Forward                     0.9063
    Init Bwd Win Byts                        0.4632
    Fwd Pkts/s                               0.3242
    Flow IAT Mean                            0.2266

[PHASE 4] Per-family SHAP
  Table XIV -- top-2 features

In [8]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | LAYER 7 | R8.0 (ZERO-TRUST)
#  Reads L1..L5 of the single R8 run configuration.
#
#  R8.0 over R7.2:
#   T1 NO HARD-CODED KEYS. R7.2 shipped three fixed keys and printed them.
#      Keys now come from T6_API_KEYS ("key=scope,scope;key2=scope"); if it is
#      unset, three random keys are generated per start, written to
#      <T6_L7>/api_keys.json (mode 0600) and only masked prefixes are printed.
#   T2 FAIL-CLOSED PATHS. R7.2 silently fell back to a sibling directory
#      (the real run served L5 R6.1 while R6.2 was requested). A missing
#      manifest is now fatal unless T6_ALLOW_MANIFEST_SEARCH=1.
#   T3 L4 autoencoder applies the R8 signed_log1p input transform and refuses
#      an architecture file without it.
#   T4 ResponseEngine stub carries the L5 R8 auto-action precision gate.
#   T6 L5 engine rebuilt from layer5_config.json (no class pickle).
#   T7 _l4_score used raw X instead of the AE's signed_log1p space; it now
#      calls reconstruction_error(), identical to Layers 4, 5 and 6.
#   T8 ISOLATED CONFIG. Every layer defines a global named CFG; when layers
#      run as cells of one kernel, a later cell replaced L7's CFG and the live
#      server crashed (KeyError 'max_body_bytes', HTTP 500). L7's settings are
#      now L7_CFG, a name no other layer uses.
#   T9 ONE SERVER. Re-running this cell stops the previous server (uvicorn
#      Server.should_exit) before binding, instead of silently starting a
#      second server on the next free port. The live URL is written to
#      <T6_L7>/endpoint.json, which 11_api_benchmark.py reads.
#   T5 NOTE: prometheus_client.Counter is imported under the name Counter;
#      run this cell LAST in the kernel (it shadows collections.Counter).
#
#  R7.2 fixes one bug from R7.1:
#   B1 _load_l3 read m['classes'] (manifest root). L3 R7.12 exports classes
#      inside l4_l5_contract, i.e. c['classes']. The KeyError left L3
#      unavailable. One character fix; nothing else changes.
#
#  Zero-trust interface:
#   - Scoped API keys (ingest / read / revert)
#   - Input validation (unknown features, non-finite, out-of-range)
#   - Server-generated trace_id; client id logged separately
#   - /v1/health liveness only; detail behind scope=read
#   - /docs and /redoc disabled by default
#   - Body-size guard (1 MiB)
#   - CORS empty by default (T6_CORS_ORIGINS env to allow origins)
#   - Fail closed if L2 does not load
#   - Path resolution by manifest name, not by string
# =============================================================================

# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
import logging as _t6_logging   # silence matplotlib's PDF font-subsetting INFO spam
_t6_logging.getLogger('fontTools').setLevel(_t6_logging.WARNING)
_t6_logging.getLogger('matplotlib').setLevel(_t6_logging.WARNING)
# -----------------------------------------------------------------------------

import os, json, time, uuid, pickle, asyncio, logging, argparse, threading, platform, socket
import secrets
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from typing import List, Dict, Any, Optional
from contextlib import asynccontextmanager

import numpy as np
import psutil

import torch
import torch.nn as nn
from scipy.special import logsumexp

import lightgbm as lgb
import xgboost as xgb

from fastapi import FastAPI, HTTPException, Header, Depends, Request
from fastapi.responses import PlainTextResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

from prometheus_client import (
    Counter, Histogram, Gauge, Info,
    generate_latest, CONTENT_TYPE_LATEST, CollectorRegistry,
)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(name)s %(message)s')
log = logging.getLogger('titanium6.l7')

VERSION          = 'TITANIUM6_LAYER7_R8.0'
FLOW_BOUND       = 1_000.0
MAX_BODY_BYTES   = 1_048_576
DOCS_ENABLED     = False

_KEY_SCOPES = {'admin': ['ingest', 'read', 'revert'],
               'readonly': ['read'],
               'ingest': ['ingest']}


def _generate_keys():
    """T1: random per-start keys; never hard-coded, never printed in full."""
    return {f't6_{role}_' + secrets.token_urlsafe(24): list(sc)
            for role, sc in _KEY_SCOPES.items()}


def _mask(k):
    return k[:12] + '...' if len(k) > 12 else '***'

L7_CFG = {
    'version'          : VERSION,
    'api_key_header'   : 'X-API-Key',
    'audit_max'        : 10_000,
    'rate_rps'         : 200.0,
    'rate_capacity'    : 200.0,
    'max_batch'        : 256,
    'startup_timeout_s': 20,
    'flow_bound'       : FLOW_BOUND,
    'max_body_bytes'   : MAX_BODY_BYTES,
}


# =============================================================================
#  PICKLE STUBS
# =============================================================================
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                    nn.ReLU(), nn.Dropout(0.1)]
            prev = h
        self.encoder = nn.Sequential(*enc)
        dec = []
        for h in reversed(hidden_dims[:-1]):
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)
    INPUT_TRANSFORM = 'signed_log1p'
    @staticmethod
    def prep(x):
        x = torch.as_tensor(x, dtype=torch.float32)
        return torch.sign(x) * torch.log1p(torch.abs(x))
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def reconstruction_error(self, x):
        self.eval()
        with torch.no_grad():
            x_t = self.prep(x)
            x_r = self(x_t)
            return ((x_t - x_r)**2).mean(dim=1).numpy().astype(np.float32)


AUTO_ACTIONS = ('AUTO_BLOCK', 'AUTO_RATE_LIMIT', 'ISOLATE')


class ResponseEngine:
    """Matches L5 R8.0's class so pickle.load resolves (incl. auto gate)."""
    def __init__(self, playbook=None, conf_auto=0.90, conf_alert=0.70,
                 auto_allowed=None):
        self.pb = playbook or {}
        self.conf_auto = conf_auto
        self.conf_alert = conf_alert
        self.auto_allowed = auto_allowed

    def decide(self, family, confidence, is_zeroday=False, flow_meta=None):
        t0 = time.perf_counter()
        incident_id = str(uuid.uuid4())[:8].upper()
        fmeta = flow_meta or {}
        gated = False

        if family == 'Normal':
            action, sev, mitre = 'NO_ACTION', 'LOW', ''
            desc, reversal, sla = 'Normal traffic', 'NONE', 0
        elif is_zeroday:
            rule = self.pb.get('Zero-day', ('MEDIUM','ALERT_SOC','T1203',
                                             'Novel anomaly','CLEAR_ALERT',600))
            sev, action, mitre, desc, reversal, sla = rule
        else:
            rule = self.pb.get(family)
            if rule is None:
                action, sev, mitre = 'ALERT_SOC', 'MEDIUM', 'T0000'
                desc, reversal, sla = f'Unknown family: {family}', 'CLEAR_ALERT', 300
            else:
                sev, primary_action, mitre, desc, reversal, sla = rule
                if confidence >= self.conf_auto and sev in ('CRITICAL','HIGH'):
                    action = primary_action
                elif confidence >= self.conf_alert:
                    action = 'ALERT_SOC'
                else:
                    action = 'LOG_MONITOR'
                allowed = getattr(self, 'auto_allowed', None)
                if (action in AUTO_ACTIONS and allowed is not None
                        and family not in allowed):
                    action, gated = 'ALERT_SOC', True

        lat_us = (time.perf_counter() - t0) * 1e6
        return {
            'gated'       : gated,
            'incident_id' : incident_id,
            'timestamp'   : datetime.now(timezone.utc).isoformat(),
            'family'      : family,
            'confidence'  : round(float(confidence), 6),
            'is_zeroday'  : bool(is_zeroday),
            'severity'    : sev,
            'action'      : action,
            'mitre_id'    : mitre,
            'description' : desc,
            'reversal'    : reversal,
            'sla_seconds' : int(sla),
            'source_ip'   : fmeta.get('src_ip', 'UNKNOWN'),
            'decision_us' : round(lat_us, 2),
            'simulated'   : True,
        }


def _now_utc():
    return datetime.now(timezone.utc).isoformat()


# =============================================================================
#  PYDANTIC MODELS
# =============================================================================
class ClassifyRequest(BaseModel):
    model_config = {'extra': 'forbid'}
    flow           : Dict[str, Any] = Field(..., description='scaled feature dict')
    src_ip         : str = Field(default='0.0.0.0')
    dst_ip         : str = Field(default='0.0.0.0')
    protocol       : str = Field(default='UNKNOWN')
    client_trace_id: Optional[str] = None


class BatchClassifyRequest(BaseModel):
    model_config = {'extra': 'forbid'}
    flows          : List[Dict[str, Any]]
    client_trace_id: Optional[str] = None


class RevertRequest(BaseModel):
    model_config = {'extra': 'forbid'}
    incident_id: str
    reason     : str = 'Manual revert by SOC analyst'


class DecisionResponse(BaseModel):
    trace_id      : str
    incident_id   : str
    family        : str
    confidence    : float
    anomaly_score : float
    action        : str
    severity      : str
    mitre_id      : str
    description   : str
    reversal      : str
    sla_seconds   : int
    is_attack     : bool
    is_zeroday    : bool
    latency_ms    : float
    degraded      : bool
    layers_used   : List[str]


# =============================================================================
#  PROMETHEUS
# =============================================================================
REG = CollectorRegistry(auto_describe=True)
LATENCY   = Histogram('t6_request_latency_ms', 'Request latency (ms)',
                       ['endpoint', 'layer'], registry=REG,
                       buckets=[0.1, 0.5, 1, 2, 5, 10, 20, 50, 100, 500])
DECISIONS = Counter('t6_decisions_total', 'Decisions by action/family',
                     ['action', 'family'], registry=REG)
ERRORS    = Counter('t6_errors_total', 'Errors by type', ['type'], registry=REG)
FLOWS     = Counter('t6_flows_total', 'Total flows processed', registry=REG)
INFO      = Info('t6_build', 'Build information', registry=REG)
UPTIME    = Gauge('t6_uptime_seconds', 'Service uptime (s)', registry=REG)
_START_TS = time.time()


# =============================================================================
#  RATE LIMITER
# =============================================================================
class RateLimiter:
    def __init__(self, rps, capacity):
        self.rps = float(rps)
        self.capacity = float(capacity)
        self._lock = threading.Lock()
        self._state = {}
    def allow(self, key):
        now = time.monotonic()
        with self._lock:
            tokens, last = self._state.get(key, (self.capacity, now))
            tokens = min(self.capacity, tokens + (now - last) * self.rps)
            if tokens < 1.0:
                self._state[key] = (tokens, now); return False
            self._state[key] = (tokens - 1.0, now); return True


# =============================================================================
#  AUDIT RING
# =============================================================================
class AuditRing:
    def __init__(self, maxlen):
        self._buf = deque(maxlen=maxlen)
        self._lock = threading.Lock()
    def append(self, entry):
        with self._lock:
            self._buf.append(entry)
    def read(self, page, size):
        with self._lock:
            buf = list(self._buf)
        buf.reverse()
        start = page * size
        return buf[start:start+size]
    def size(self):
        with self._lock:
            return len(self._buf)
    def find_incident(self, incident_id):
        with self._lock:
            for e in reversed(self._buf):
                if e.get('incident_id') == incident_id:
                    return e
        return None


# =============================================================================
#  MODEL STORE
# =============================================================================
class ModelStore:
    def __init__(self):
        self.l2_model = None; self.l2_cal = None
        self.l3_model = None; self.l3_le = None; self.l3_cals = None
        self.l4_if = None; self.l4_ae = None
        self.l4_if_cal = None; self.l4_ae_cal = None
        self.l5_engine = None
        self.l2_threshold = None
        self.l3_l5 = None; self.l3_ood = None; self.l3_ood_statistic = 'max_softmax'
        self.l4_alpha = 0.0; self.l4_tau4 = None
        self.l4_if_disabled = False
        self.features = None
        self.l3_classes = None; self.l3_reject = 'Normal'
        self.l2_model_type = None; self.l3_model_type = None
        self.available = {'l2': False, 'l3': False, 'l4': False, 'l5': False}
        self.versions = {}
        self.hardware = self._detect_hardware()
        self.load_errors = {}

    def _detect_hardware(self):
        cpu = platform.processor() or ''
        try:
            with open('/proc/cpuinfo') as fh:
                for line in fh:
                    if line.startswith('model name'):
                        cpu = line.split(':', 1)[1].strip(); break
        except OSError: pass
        return {
            'cpu_model': cpu,
            'physical_cores': psutil.cpu_count(logical=False),
            'logical_cores': psutil.cpu_count(logical=True),
            'ram_gb': round(psutil.virtual_memory().total / 2**30, 1),
            'python': platform.python_version(),
            'platform': platform.platform(),
            'torch': torch.__version__,
            'lightgbm': lgb.__version__,
            'xgboost': xgb.__version__,
            'numpy': np.__version__,
        }

    def load(self, l1_dir, l2_dir, l3_dir, l4_dir, l5_dir):
        t0 = time.perf_counter()
        self._load_l2(Path(l2_dir))
        self._load_l3(Path(l3_dir))
        self._load_l4(Path(l4_dir))
        self._load_l5(Path(l5_dir))
        elapsed_ms = (time.perf_counter() - t0) * 1000
        log.info('Models loaded in %.0f ms | available=%s', elapsed_ms, self.available)
        INFO.info({k: str(v) for k, v in self.versions.items()})
        return elapsed_ms

    def _load_l2(self, d):
        try:
            mp = d/'manifest_l2.json'
            if not mp.exists(): raise FileNotFoundError(f'L2 manifest: {mp}')
            with open(mp) as fh: m = json.load(fh)
            c = m['l3_l4_contract']
            with open(c['model'], 'rb') as fh: self.l2_model = pickle.load(fh)
            with open(c['cal'], 'rb') as fh: self.l2_cal = pickle.load(fh)
            with open(c['threshold_json']) as fh: th = json.load(fh)
            best = m['best_model']
            if best not in th['deployed_tau']:
                raise KeyError(f'L2 threshold missing for {best}')
            self.l2_threshold = float(th['deployed_tau'][best])
            self.features = list(c['features'])
            self.l2_model_type = ('LightGBM' if isinstance(self.l2_model, lgb.Booster)
                                   else 'XGBoost' if isinstance(self.l2_model, xgb.Booster)
                                   else 'sklearn')
            self.versions['l2'] = m.get('version', 'unknown')
            self.available['l2'] = True
            log.info('L2: %s tau=%.4f features=%d',
                      self.l2_model_type, self.l2_threshold, len(self.features))
        except Exception as e:
            self.load_errors['l2'] = f'{type(e).__name__}: {e}'
            log.warning('L2 load failed: %s', e)

    def _load_l3(self, d):
        try:
            mp = d/'manifest_l3.json'
            if not mp.exists(): raise FileNotFoundError(f'L3 manifest: {mp}')
            with open(mp) as fh: m = json.load(fh)
            c = m['l4_l5_contract']
            with open(c['model'], 'rb') as fh: self.l3_model = pickle.load(fh)
            with open(c['encoder'], 'rb') as fh: self.l3_le = pickle.load(fh)
            with open(c['thresholds_json']) as fh: th = json.load(fh)
            self.l3_l5 = float(th['l5'])
            self.l3_ood_statistic = c.get('ood_statistic', 'max_softmax')
            if self.l3_ood_statistic == 'energy':
                self.l3_ood = float(th.get('ood_energy', th['l5']))
            else:
                self.l3_ood = float(th.get('ood_conf', th['l5']))
            # B1: classes live inside l4_l5_contract, not at the manifest root
            self.l3_classes = list(c['classes'])
            self.l3_reject = c.get('reject_class') or 'Normal'
            cal_p = c.get('calibrators')
            if cal_p and Path(cal_p).exists():
                with open(cal_p, 'rb') as fh:
                    self.l3_cals = pickle.load(fh)
                log.info('L3 calibrators: %d/%d fitted',
                          sum(1 for x in self.l3_cals if x is not None),
                          len(self.l3_cals))
            else:
                raise RuntimeError('L3 calibrators missing from l4_l5_contract')
            self.l3_model_type = ('LightGBM' if isinstance(self.l3_model, lgb.Booster)
                                   else 'XGBoost' if isinstance(self.l3_model, xgb.Booster)
                                   else 'sklearn')
            self.versions['l3'] = m.get('version', 'unknown')
            self.available['l3'] = True
            log.info('L3: %s %d classes reject="%s" tau_L5=%.4f tau_OOD(%s)=%.4f',
                      self.l3_model_type, len(self.l3_classes), self.l3_reject,
                      self.l3_l5, self.l3_ood_statistic, self.l3_ood)
        except Exception as e:
            self.load_errors['l3'] = f'{type(e).__name__}: {e}'
            log.warning('L3 load failed: %s', e)

    def _load_l4(self, d):
        try:
            mp = d/'manifest_l4.json'
            if not mp.exists(): raise FileNotFoundError(f'L4 manifest: {mp}')
            with open(mp) as fh: m = json.load(fh)
            c = m['l5_contract']
            self.l4_alpha = float(c['alpha'])
            self.l4_tau4  = float(c['tau4'])
            self.l4_if_disabled = bool(abs(self.l4_alpha) < 1e-12)
            if not self.l4_if_disabled:
                with open(c['if_model'], 'rb') as fh:
                    self.l4_if = pickle.load(fh)
                with open(c['if_cal'], 'rb') as fh:
                    self.l4_if_cal = pickle.load(fh)
            else:
                log.info('L4: Isolation Forest disabled (alpha=0) - not loaded')
            with open(c['ae_cal'], 'rb') as fh: self.l4_ae_cal = pickle.load(fh)
            arch_p = d/'ae_architecture.json'
            state_p = d/'ae_model_state.pt'
            if not arch_p.exists() or not state_p.exists():
                raise FileNotFoundError(f'L4 AE state files missing: {arch_p}, {state_p}')
            with open(arch_p) as fh: arch = json.load(fh)
            if arch.get('input_transform') != 'signed_log1p':
                raise RuntimeError('ae_architecture.json lacks input_transform='
                                   'signed_log1p: re-run Layer 4 R8.0')
            ae = SparseAutoencoder(arch['input_dim'], arch['hidden_dims'])
            ae.load_state_dict(torch.load(state_p, map_location='cpu'))
            ae.eval()
            self.l4_ae = ae
            self.versions['l4'] = m.get('version', 'unknown')
            self.available['l4'] = True
            log.info('L4: alpha=%.2f tau4=%.4f IF=%s',
                      self.l4_alpha, self.l4_tau4,
                      'disabled' if self.l4_if_disabled else 'enabled')
        except Exception as e:
            self.load_errors['l4'] = f'{type(e).__name__}: {e}'
            log.warning('L4 load failed: %s', e)

    def _load_l5(self, d):
        try:
            mp = d/'manifest_l5.json'
            if not mp.exists(): raise FileNotFoundError(f'L5 manifest: {mp}')
            with open(mp) as fh: m = json.load(fh)
            c = m['l6_contract']
            # R8 T6: rebuild the engine from layer5_config.json (plain JSON),
            # not from a pickle of a class object, so L7 runs in a fresh kernel.
            with open(c['config']) as fh:
                cfg5 = json.load(fh)
            for k in ('playbook', 'conf_auto', 'conf_alert', 'auto_allowed'):
                if k not in cfg5:
                    raise KeyError(f'layer5_config.json missing {k!r}: re-run Layer 5 R8.0')
            aa = cfg5['auto_allowed']            # None = gate not run -> not gated
            self.l5_engine = ResponseEngine(
                playbook={fam: tuple(v) for fam, v in cfg5['playbook'].items()},
                conf_auto=float(cfg5['conf_auto']), conf_alert=float(cfg5['conf_alert']),
                auto_allowed=(None if aa is None else set(aa)))
            self.versions['l5'] = m.get('version', 'unknown')
            self.available['l5'] = True
            log.info('L5: ResponseEngine loaded')
        except Exception as e:
            self.load_errors['l5'] = f'{type(e).__name__}: {e}'
            log.warning('L5 load failed: %s', e)

    def _l2_probs(self, X):
        if isinstance(self.l2_model, lgb.Booster):
            raw = self.l2_model.predict(X).astype(np.float32)
        elif isinstance(self.l2_model, xgb.Booster):
            raw = np.asarray(self.l2_model.predict(
                xgb.DMatrix(X, feature_names=self.features))).ravel()
        else:
            raw = self.l2_model.predict_proba(X)[:, 1].astype(np.float32)
        return self.l2_cal.predict_proba(raw.reshape(-1, 1))[:, 1].astype(np.float32)

    def _l3_raw(self, X):
        if isinstance(self.l3_model, lgb.Booster):
            return self.l3_model.predict(X).astype(np.float32)
        return self.l3_model.predict(
            xgb.DMatrix(X, feature_names=self.features)
        ).reshape(-1, len(self.l3_classes)).astype(np.float32)

    def _l3_cal_proba(self, X):
        q = np.array(self._l3_raw(X), dtype=np.float32, copy=True)
        if self.l3_cals is not None:
            for c, lr in enumerate(self.l3_cals):
                if lr is not None and c < q.shape[1]:
                    pc = np.clip(q[:, c], 1e-7, 1 - 1e-7)
                    z = np.log(pc / (1 - pc)).reshape(-1, 1)
                    q[:, c] = lr.predict_proba(z)[:, 1].astype(np.float32)
            s = q.sum(axis=1, keepdims=True)
            q = (q / np.where(s == 0, 1, s)).astype(np.float32)
        return q

    def _l3_energy(self, X):
        if isinstance(self.l3_model, lgb.Booster):
            m = self.l3_model.predict(X, raw_score=True).astype(
                np.float64).reshape(-1, len(self.l3_classes))
        else:
            m = self.l3_model.predict(
                xgb.DMatrix(X, feature_names=self.features),
                output_margin=True).reshape(-1, len(self.l3_classes)).astype(np.float64)
        return -logsumexp(m, axis=1)

    def _l4_score(self, X):
        # R8 T7: same signed_log1p input space the AE was trained in (L4/L5/L6)
        ae_raw = self.l4_ae.reconstruction_error(X)
        ae_p = self.l4_ae_cal.predict_proba(ae_raw.reshape(-1,1))[:,1].astype(np.float32)
        if self.l4_if_disabled:
            return ae_p
        if_raw = -self.l4_if.score_samples(X).astype(np.float32)
        if_p = self.l4_if_cal.predict_proba(if_raw.reshape(-1,1))[:,1].astype(np.float32)
        return (self.l4_alpha * if_p + (1 - self.l4_alpha) * ae_p).astype(np.float32)


STORE = ModelStore()
AUDIT = AuditRing(L7_CFG['audit_max'])
RATE  = RateLimiter(L7_CFG['rate_rps'], L7_CFG['rate_capacity'])

def _load_keys_from_env():
    """Returns (keys, source). Never falls back to a fixed key."""
    env = os.environ.get('T6_API_KEYS')
    out = {}
    if env:
        for entry in env.split(';'):
            if '=' not in entry: continue
            k, scopes = entry.split('=', 1)
            if k.strip():
                out[k.strip()] = [s.strip() for s in scopes.split(',') if s.strip()]
    if out:
        return out, 'env:T6_API_KEYS'
    return _generate_keys(), 'generated'

_API_KEYS, _KEY_SOURCE = _load_keys_from_env()


def _write_key_file(out_dir):
    """Generated keys are handed over through a 0600 file, not stdout."""
    if _KEY_SOURCE != 'generated':
        return None
    d = Path(out_dir); d.mkdir(parents=True, exist_ok=True)
    p = d / 'api_keys.json'
    fd = os.open(str(p), os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, 'w') as fh:
        json.dump(_API_KEYS, fh, indent=2)
    return p


# =============================================================================
#  INPUT VALIDATION
# =============================================================================
def _acceptable_feature_aliases(features):
    out = set()
    for f in features:
        out.add(f)
        out.add(f.replace('_', ' '))
        out.add(f.replace(' ', '_'))
    return out


def _validate_flow(flow, aliases):
    if not isinstance(flow, dict):
        raise HTTPException(400, 'flow must be an object')
    if len(flow) == 0:
        raise HTTPException(400, 'flow is empty')
    unknown = [k for k in flow.keys() if k not in aliases]
    if unknown:
        raise HTTPException(
            400, f'unknown feature(s): {sorted(unknown)[:5]} '
                 f'({len(unknown)} total)')
    bad_type = []; bad_finite = []; out_of_range = []
    for k, v in flow.items():
        try:
            fv = float(v)
        except (TypeError, ValueError):
            bad_type.append(k); continue
        if not np.isfinite(fv):
            bad_finite.append(k); continue
        if abs(fv) > L7_CFG['flow_bound']:
            out_of_range.append((k, fv))
    if bad_type:
        raise HTTPException(400, f'non-numeric value(s): {sorted(bad_type)[:5]}')
    if bad_finite:
        raise HTTPException(400, f'non-finite value(s): {sorted(bad_finite)[:5]}')
    if out_of_range:
        sample = out_of_range[:3]
        raise HTTPException(
            400, f'value out of range (|v| > {L7_CFG["flow_bound"]}): '
                 + ', '.join(f'{k}={v}' for k, v in sample))


def _flow_to_array(flow, features, aliases):
    row = []
    for f in features:
        if f in flow:
            v = flow[f]
        elif f.replace('_', ' ') in flow:
            v = flow[f.replace('_', ' ')]
        else:
            v = 0.0
        try:
            fv = float(v)
        except (TypeError, ValueError):
            fv = 0.0
        if not np.isfinite(fv):
            fv = 0.0
        row.append(fv)
    return np.array(row, dtype=np.float32)


# =============================================================================
#  INFERENCE
# =============================================================================
def _infer_one(flow, src_ip, dst_ip, protocol, client_trace_id):
    t0 = time.perf_counter()
    server_trace_id = str(uuid.uuid4())[:8]
    incident_id = str(uuid.uuid4())[:8].upper()
    layers_used = []
    degraded = False

    if not STORE.available['l2']:
        ERRORS.labels(type='l2_unavailable').inc()
        raise HTTPException(503, 'L2 gate unavailable')

    aliases = _acceptable_feature_aliases(STORE.features or [])
    _validate_flow(flow, aliases)
    X = _flow_to_array(flow, STORE.features, aliases).reshape(1, -1)

    t = time.perf_counter()
    p_l2 = float(STORE._l2_probs(X)[0])
    LATENCY.labels('classify', 'l2').observe((time.perf_counter()-t)*1000)
    layers_used.append('L2')
    is_attack = p_l2 >= STORE.l2_threshold

    if not is_attack:
        dec = _normal_decision(incident_id, p_l2, src_ip)
        lat = (time.perf_counter() - t0) * 1000
        LATENCY.labels('classify', 'total').observe(lat)
        DECISIONS.labels(action=dec['action'], family='Normal').inc()
        FLOWS.inc()
        _audit_log(server_trace_id, client_trace_id, dec, src_ip, dst_ip,
                   protocol, lat, layers_used, degraded=False)
        return _to_response(server_trace_id, dec, l3_conf=1-p_l2, anomaly=0.0,
                             is_attack=False, is_zeroday=False,
                             lat=lat, degraded=False, layers_used=layers_used)

    family = 'Unknown'; l3_conf = p_l2; is_zeroday = False; l3_says_reject = False
    if STORE.available['l3']:
        t = time.perf_counter()
        try:
            p_l3 = STORE._l3_cal_proba(X)
            argmax = int(p_l3.argmax(axis=1)[0])
            l3_conf = float(p_l3[0, argmax])
            family = STORE.l3_le.inverse_transform([argmax])[0]
            l3_says_reject = (family == STORE.l3_reject)
            if l3_says_reject:
                is_zeroday = False
            elif STORE.l3_ood_statistic == 'energy':
                e = float(STORE._l3_energy(X)[0])
                is_zeroday = e > STORE.l3_ood
            else:
                is_zeroday = l3_conf < STORE.l3_ood
            LATENCY.labels('classify', 'l3').observe((time.perf_counter()-t)*1000)
            layers_used.append('L3')
        except Exception as e:
            log.warning('L3 inference failed: %s', e)
            degraded = True
    else:
        degraded = True

    anomaly_score = 0.0
    if STORE.available['l4']:
        t = time.perf_counter()
        try:
            anomaly_score = float(STORE._l4_score(X)[0])
            if anomaly_score >= STORE.l4_tau4 and not l3_says_reject:
                is_zeroday = True
            LATENCY.labels('classify', 'l4').observe((time.perf_counter()-t)*1000)
            layers_used.append('L4')
        except Exception as e:
            log.warning('L4 inference failed: %s', e)
            degraded = True
    else:
        degraded = True

    if l3_says_reject:
        dec = _normal_decision(incident_id, l3_conf, src_ip)
    elif STORE.available['l5']:
        t = time.perf_counter()
        try:
            dec = STORE.l5_engine.decide(
                family=family, confidence=l3_conf,
                is_zeroday=is_zeroday,
                flow_meta={'src_ip': src_ip})
            LATENCY.labels('classify', 'l5').observe((time.perf_counter()-t)*1000)
            layers_used.append('L5')
        except Exception as e:
            log.warning('L5 decision failed: %s', e)
            dec = _fallback_decision(incident_id, family, l3_conf, is_zeroday)
            degraded = True
    else:
        dec = _fallback_decision(incident_id, family, l3_conf, is_zeroday)
        degraded = True

    lat = (time.perf_counter() - t0) * 1000
    LATENCY.labels('classify', 'total').observe(lat)
    DECISIONS.labels(action=dec['action'], family=family).inc()
    FLOWS.inc()
    _audit_log(server_trace_id, client_trace_id, dec, src_ip, dst_ip,
                protocol, lat, layers_used, degraded=degraded)
    return _to_response(server_trace_id, dec, l3_conf=l3_conf,
                         anomaly=anomaly_score, is_attack=True,
                         is_zeroday=is_zeroday, lat=lat,
                         degraded=degraded, layers_used=layers_used)


def _normal_decision(incident_id, conf, src_ip):
    return {
        'incident_id': incident_id, 'family': 'Normal', 'confidence': conf,
        'is_zeroday': False, 'severity': 'LOW', 'action': 'NO_ACTION',
        'mitre_id': '', 'description': 'Normal traffic', 'reversal': 'NONE',
        'sla_seconds': 0, 'source_ip': src_ip, 'simulated': True,
    }


def _fallback_decision(incident_id, family, l3_conf, is_zeroday):
    return {
        'incident_id': incident_id, 'family': family, 'confidence': l3_conf,
        'is_zeroday': is_zeroday, 'severity': 'LOW', 'action': 'LOG_MONITOR',
        'mitre_id': 'T0000', 'description': 'L5 unavailable', 'reversal': 'NONE',
        'sla_seconds': 600, 'source_ip': 'UNKNOWN', 'simulated': True,
    }


def _audit_log(server_trace_id, client_trace_id, dec, src_ip, dst_ip,
                protocol, lat, layers_used, degraded):
    AUDIT.append({
        'server_trace_id': server_trace_id,
        'client_trace_id': client_trace_id,
        'incident_id': dec['incident_id'],
        'timestamp': _now_utc(),
        'family': dec['family'],
        'confidence': round(dec['confidence'], 6),
        'action': dec['action'], 'severity': dec['severity'],
        'src_ip': src_ip, 'dst_ip': dst_ip, 'protocol': protocol,
        'latency_ms': round(lat, 3),
        'reversal': dec.get('reversal', 'NONE'),
        'layers_used': layers_used, 'degraded': degraded,
    })


def _to_response(trace_id, dec, l3_conf, anomaly, is_attack, is_zeroday,
                  lat, degraded, layers_used):
    return DecisionResponse(
        trace_id=trace_id, incident_id=dec['incident_id'],
        family=dec['family'], confidence=round(float(l3_conf), 6),
        anomaly_score=round(float(anomaly), 6),
        action=dec['action'], severity=dec['severity'],
        mitre_id=dec.get('mitre_id', ''),
        description=dec.get('description', ''),
        reversal=dec.get('reversal', 'NONE'),
        sla_seconds=int(dec.get('sla_seconds', 600)),
        is_attack=is_attack, is_zeroday=is_zeroday,
        latency_ms=round(lat, 3), degraded=degraded,
        layers_used=layers_used)


# =============================================================================
#  AUTH + SCOPE
# =============================================================================
def _auth(x_api_key: str = Header(..., alias=L7_CFG['api_key_header'])):
    scopes = _API_KEYS.get(x_api_key)
    if scopes is None:
        ERRORS.labels(type='auth_failure').inc()
        raise HTTPException(401, 'Invalid API key')
    if not RATE.allow(x_api_key):
        ERRORS.labels(type='rate_limit').inc()
        raise HTTPException(429, 'Rate limit exceeded')
    return x_api_key, scopes


def require_scope(scope: str):
    def _dep(auth = Depends(_auth)):
        key, scopes = auth
        if scope not in scopes:
            ERRORS.labels(type=f'scope_denied_{scope}').inc()
            raise HTTPException(403, f'API key lacks scope: {scope}')
        return key
    return _dep


# =============================================================================
#  FASTAPI
# =============================================================================
@asynccontextmanager
async def lifespan(app: FastAPI):
    args = app.state.args
    log.info('Loading TITANIUM-6 models...')
    elapsed = STORE.load(args.l1dir, args.l2dir, args.l3dir, args.l4dir, args.l5dir)
    app.state.load_ms = elapsed
    if not STORE.available['l2']:
        raise RuntimeError('L2 gate unavailable; refusing to serve')
    log.info('TITANIUM-6 ready')
    yield
    log.info('shutting down')

app = FastAPI(
    title='TITANIUM-6 Intrusion Detection API',
    description=('Hierarchical IDS: L2 gate -> L3 family -> L4 anomaly -> L5 SOAR. '
                  'Model-derived decisions only. Zero-trust: scoped keys, input '
                  'validation, server trace IDs, fail-closed on load.'),
    version='1.0.0',
    lifespan=lifespan,
    docs_url='/docs' if DOCS_ENABLED else None,
    redoc_url='/redoc' if DOCS_ENABLED else None,
)

_origins = [o.strip() for o in os.environ.get('T6_CORS_ORIGINS', '').split(',')
            if o.strip()]
app.add_middleware(CORSMiddleware, allow_origins=_origins,
                    allow_methods=['GET', 'POST'],
                    allow_headers=['X-API-Key', 'Content-Type'])


@app.middleware('http')
async def body_size_guard(request: Request, call_next):
    cl = request.headers.get('content-length')
    if cl:
        try:
            if int(cl) > L7_CFG['max_body_bytes']:
                return JSONResponse(
                    {'detail': f'body exceeds {L7_CFG["max_body_bytes"]} bytes'},
                    status_code=413)
        except ValueError:
            return JSONResponse({'detail': 'invalid Content-Length'},
                                 status_code=400)
    return await call_next(request)


@app.post('/v1/classify', response_model=DecisionResponse, tags=['Detection'])
async def classify(req: ClassifyRequest, key: str = Depends(require_scope('ingest'))):
    try:
        return await asyncio.get_event_loop().run_in_executor(
            None, _infer_one,
            req.flow, req.src_ip, req.dst_ip, req.protocol, req.client_trace_id)
    except HTTPException:
        raise
    except Exception as e:
        ERRORS.labels(type='inference_error').inc()
        log.error('inference error: %s', e)
        raise HTTPException(500, f'Inference failed: {str(e)[:200]}')


@app.post('/v1/classify/batch', response_model=List[DecisionResponse],
           tags=['Detection'])
async def classify_batch(req: BatchClassifyRequest,
                          key: str = Depends(require_scope('ingest'))):
    if len(req.flows) > L7_CFG['max_batch']:
        raise HTTPException(400, f'batch > {L7_CFG["max_batch"]}')
    async def _one(flow):
        return await asyncio.get_event_loop().run_in_executor(
            None, _infer_one, flow, '0.0.0.0', '0.0.0.0', 'UNKNOWN',
            req.client_trace_id)
    try:
        return list(await asyncio.gather(*[_one(f) for f in req.flows]))
    except HTTPException:
        raise
    except Exception as e:
        ERRORS.labels(type='batch_error').inc()
        raise HTTPException(500, f'Batch failed: {str(e)[:200]}')


@app.get('/v1/health', tags=['Operations'])
async def health():
    return {'status': 'ok' if STORE.available['l2'] else 'degraded'}


@app.get('/v1/status', tags=['Operations'])
async def status(key: str = Depends(require_scope('read'))):
    return {
        'status'    : 'healthy' if STORE.available['l2'] else 'degraded',
        'version'   : VERSION,
        'layers'    : STORE.available,
        'load_errors': STORE.load_errors,
        'versions'  : STORE.versions,
        'features'  : len(STORE.features) if STORE.features else 0,
        'l2_threshold'      : STORE.l2_threshold,
        'l3_l5_threshold'   : STORE.l3_l5,
        'l3_ood_statistic'  : STORE.l3_ood_statistic,
        'l3_ood_threshold'  : STORE.l3_ood,
        'l4_alpha'          : STORE.l4_alpha,
        'l4_tau4'           : STORE.l4_tau4,
        'l4_if_disabled'    : STORE.l4_if_disabled,
        'audit_ring_size'   : AUDIT.size(),
        'uptime_s'          : round(time.time() - _START_TS, 1),
        'hardware'          : STORE.hardware,
        'timestamp'         : _now_utc(),
    }


@app.get('/v1/version', tags=['Operations'])
async def version(key: str = Depends(require_scope('read'))):
    return {
        'service'   : VERSION,
        'layers'    : STORE.versions,
        'hardware'  : STORE.hardware,
        'python'    : platform.python_version(),
        'numpy'     : np.__version__,
        'lightgbm'  : lgb.__version__,
        'xgboost'   : xgb.__version__,
        'torch'     : torch.__version__,
    }


@app.get('/v1/audit', tags=['Operations'])
async def audit(page: int = 0, size: int = 50,
                 key: str = Depends(require_scope('read'))):
    size = min(size, 200)
    entries = AUDIT.read(page, size)
    return {'page': page, 'size': len(entries), 'entries': entries,
            'ring_capacity': L7_CFG['audit_max'],
            'note': 'Persistent audit is L5 audit_trail.jsonl (hash-chained).'}


@app.post('/v1/action/revert', tags=['Operations'])
async def revert(req: RevertRequest, key: str = Depends(require_scope('revert'))):
    target = AUDIT.find_incident(req.incident_id)
    if target is None:
        raise HTTPException(404, f'incident {req.incident_id} not in ring')
    reversal = target.get('reversal', 'NONE')
    if reversal == 'NONE':
        raise HTTPException(400, f'action {target["action"]} has no reversal')
    rev = {
        'server_trace_id' : str(uuid.uuid4())[:8],
        'incident_id'     : req.incident_id,
        'timestamp'       : _now_utc(),
        'action'          : reversal,
        'original_action' : target['action'],
        'reason'          : req.reason,
        'family'          : target.get('family', ''),
        'src_ip'          : target.get('src_ip', ''),
        'reversal_of'     : 'true',
    }
    AUDIT.append(rev)
    DECISIONS.labels(action=reversal, family='REVERT').inc()
    return {'status': 'reversed', 'incident_id': req.incident_id,
            'reversal_action': reversal, 'original_action': target['action'],
            'timestamp': rev['timestamp']}


@app.get('/v1/metrics', response_class=PlainTextResponse, tags=['Operations'])
async def metrics(key: str = Depends(require_scope('read'))):
    UPTIME.set(time.time() - _START_TS)
    return PlainTextResponse(generate_latest(REG).decode('utf-8'),
                              media_type=CONTENT_TYPE_LATEST)


@app.get('/', include_in_schema=False)
async def root():
    return JSONResponse({'service': 'TITANIUM-6 IDS',
                          'docs': '/docs' if DOCS_ENABLED else None})


# =============================================================================
#  PATH RESOLUTION — by manifest name, not by string
# =============================================================================
def _resolve_manifest_dir(user_path, manifest_name):
    p = Path(user_path)
    if (p / manifest_name).exists():
        return p
    parent = p.parent if p.parent.exists() else Path('.')
    cands = sorted({q.parent for q in parent.glob('*/' + manifest_name)})
    if os.environ.get('T6_ALLOW_MANIFEST_SEARCH') != '1':
        raise SystemExit(
            f'[FATAL] no {manifest_name} in {p} (fail-closed; no fallback).\n'
            f'        candidates present: {[str(c) for c in cands] or "none"}\n'
            f'        run the missing layer, pass the directory explicitly, or set '
            f'T6_ALLOW_MANIFEST_SEARCH=1 to allow a unique-sibling fallback.')
    if len(cands) == 1:
        print(f'[WARN] {user_path} has no {manifest_name}; using {cands[0]} '
              '(T6_ALLOW_MANIFEST_SEARCH=1)')
        return cands[0]
    if not cands:
        raise SystemExit(
            f'[FATAL] no {manifest_name} in {p} and no sibling manifest found.\n'
            f'        searched: {parent}/*/{manifest_name}\n'
            f'        pass the correct directory explicitly.')
    raise SystemExit(
        f'[FATAL] {user_path} has no {manifest_name} and the search is ambiguous:\n'
        + '\n'.join(f'          {c}' for c in cands)
        + '\n        pass the correct directory explicitly.')


# =============================================================================
#  ENTRY POINT
# =============================================================================
def _find_free_port(start):
    for p in range(start, start + 20):
        try:
            # SO_REUSEADDR, as uvicorn uses: a port just released by a stopped
            # server (sockets in TIME_WAIT) counts as free
            s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            s.bind(('', p)); s.close(); return p
        except OSError:
            continue
    raise SystemExit(f'[FATAL] no free port in [{start}, {start + 19}]')


def _preflight(args):
    elapsed = STORE.load(args.l1dir, args.l2dir, args.l3dir,
                          args.l4dir, args.l5dir)
    print('=' * 76)
    print(f'  LAYER 7 R8.0 PREFLIGHT | loaded in {elapsed:.0f} ms')
    print(f'  available: {STORE.available}')
    print(f'  load_errors: {STORE.load_errors}')
    print(f'  versions:  {STORE.versions}')
    print(f'  hardware:  {STORE.hardware["cpu_model"][:50]}')
    print(f'  features:  {len(STORE.features) if STORE.features else 0}')
    print(f'  L2 tau: {STORE.l2_threshold}')
    print(f'  L3 tau_L5: {STORE.l3_l5}  '
          f'tau_OOD({STORE.l3_ood_statistic}): {STORE.l3_ood}')
    print(f'  L3 reject class: "{STORE.l3_reject}"')
    print(f'  L4 alpha: {STORE.l4_alpha}  tau4: {STORE.l4_tau4}  '
          f'IF: {"disabled" if STORE.l4_if_disabled else "enabled"}')
    print(f'  CORS origins: {_origins or "(empty; same-origin only)"}')
    print(f'  docs enabled: {DOCS_ENABLED}')
    print(f'  scoped keys ({_KEY_SOURCE}): '
          + ', '.join(f'{_mask(k)}{v}' for k, v in sorted(_API_KEYS.items())))
    print('=' * 76)
    if not STORE.available['l2']:
        raise SystemExit('[FATAL] L2 unavailable; refusing to serve')
    return elapsed


def _self_test(args):
    import pyarrow.parquet as pq
    if not STORE.available['l2']:
        print('[FATAL] L2 unavailable; self-test aborted'); return 1
    l1 = Path(args.l1dir)
    m1 = json.load(open(l1 / 'manifest.json'))
    tp = Path(m1['l2_input_paths']['test_known'])
    if not tp.exists():
        tp = l1 / 'l2_ready' / 'l2_test_known.parquet'
    if not tp.exists():
        print(f'[FATAL] test_known not found at {tp}'); return 1
    print(f'[SELF-TEST] reading from {tp}')
    df = pq.ParquetFile(tp).read_row_group(
        0, columns=STORE.features + ['Target']).to_pandas()
    cases = []
    for label, name in ((None, 'ATTACK'), ('Normal', 'NORMAL')):
        sub = df[df['Target'] != 'Normal'] if label is None else df[df['Target'] == label]
        if len(sub) == 0:
            print(f'[SELF-TEST] no {name} rows in test partition'); continue
        row = sub.iloc[0]
        flow = {f: float(row[f]) for f in STORE.features}
        cases.append((name, flow, str(row['Target'])))
    if not cases:
        print('[FATAL] no self-test cases available'); return 1
    rc = 0
    for name, flow, truth in cases:
        print(f'\n[SELF-TEST] {name} (ground truth = {truth})')
        try:
            resp = _infer_one(flow, src_ip='127.0.0.1', dst_ip='0.0.0.0',
                               protocol='TCP', client_trace_id=f'SELFTEST_{name}')
        except HTTPException as e:
            print(f'[SELF-TEST] HTTPException {e.status_code}: {e.detail}')
            rc = 1; continue
        except Exception as e:
            import traceback
            print(f'[SELF-TEST] FAILED: {type(e).__name__}: {e}')
            traceback.print_exc(); rc = 1; continue
        for k, v in resp.model_dump().items():
            print(f'    {k:14s} {v}')
    print('\n[SELF-TEST]', 'PASS' if rc == 0 else 'FAIL')
    return rc


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description='TITANIUM-6 Layer 7 R8.0')
    ap.add_argument('--l1dir', default=os.environ.get('T6_L1', '/kaggle/working/l1_R8'))
    ap.add_argument('--l2dir', default=os.environ.get('T6_L2', '/kaggle/working/l2_R8'))
    ap.add_argument('--l3dir', default=os.environ.get('T6_L3', '/kaggle/working/l3_R8'))
    ap.add_argument('--l4dir', default=os.environ.get('T6_L4', '/kaggle/working/l4_R8'))
    ap.add_argument('--l5dir', default=os.environ.get('T6_L5', '/kaggle/working/l5_R8'))
    ap.add_argument('--l7dir', default=os.environ.get('T6_L7', '/kaggle/working/l7_R8'))
    ap.add_argument('--host', default='0.0.0.0')
    ap.add_argument('--port', type=int, default=8000)
    ap.add_argument('--api-key', action='append', dest='extra_keys', default=[],
                     help='format: KEY=scope1,scope2  (repeatable)')
    ap.add_argument('--enable-docs', action='store_true')
    ap.add_argument('--no-serve', action='store_true')
    ap.add_argument('--self-test', action='store_true')
    args, _ = ap.parse_known_args()

    args.l1dir = str(_resolve_manifest_dir(args.l1dir, 'manifest.json'))
    args.l2dir = str(_resolve_manifest_dir(args.l2dir, 'manifest_l2.json'))
    args.l3dir = str(_resolve_manifest_dir(args.l3dir, 'manifest_l3.json'))
    args.l4dir = str(_resolve_manifest_dir(args.l4dir, 'manifest_l4.json'))
    args.l5dir = str(_resolve_manifest_dir(args.l5dir, 'manifest_l5.json'))

    if args.enable_docs:
        DOCS_ENABLED = True

    for entry in args.extra_keys:
        if '=' not in entry:
            raise SystemExit(f'[FATAL] --api-key format must be KEY=scope1,scope2; '
                             f'got {entry!r}')
        k, scopes = entry.split('=', 1)
        _API_KEYS[k.strip()] = [s.strip() for s in scopes.split(',') if s.strip()]

    _kf = _write_key_file(args.l7dir)
    if _kf is not None:
        print(f'[KEYS] generated per-start API keys written to {_kf} (mode 0600); '
              'read them from that file')
    if args.no_serve:
        _preflight(args); raise SystemExit(0)
    if args.self_test:
        _preflight(args); raise SystemExit(_self_test(args))

    import uvicorn, builtins
    app.state.args = args
    # T9: stop a server started by an earlier run of this cell in this kernel
    _prev = getattr(builtins, '_t6_l7_server', None)
    if _prev is not None:
        srv_obj, thr = _prev
        srv_obj.should_exit = True
        thr.join(timeout=10)
        print(f'[L7] previous server stopped: {not thr.is_alive()}')
        builtins._t6_l7_server = None
    _free = _find_free_port(args.port)
    if _free != args.port:
        print(f'[WARN] port {args.port} is held by another process; serving on {_free}. '
              'Restart the kernel to free it. 11_api_benchmark.py follows endpoint.json.')

    print('=' * 76)
    print(f'  TITANIUM-6 | LAYER 7 | {VERSION}')
    print('=' * 76)
    print(f'  L1: {args.l1dir}')
    print(f'  L2: {args.l2dir}  L3: {args.l3dir}')
    print(f'  L4: {args.l4dir}  L5: {args.l5dir}')
    print(f'  host: {args.host}:{_free}')
    print(f'  docs: {"enabled at /docs" if DOCS_ENABLED else "disabled"}')
    print(f'  zero-trust: scoped keys={len(_API_KEYS)}, '
          f'flow_bound=+-{L7_CFG["flow_bound"]}, '
          f'body_limit={L7_CFG["max_body_bytes"]} B, '
          f'CORS origins={_origins or "(none)"}')
    print('=' * 76)

    _server = uvicorn.Server(uvicorn.Config(app, host=args.host, port=_free, workers=1,
                                           log_level='warning', access_log=False))
    _srv = threading.Thread(target=_server.run, daemon=True)
    _srv.start()
    builtins._t6_l7_server = (_server, _srv)
    _ep = Path(args.l7dir); _ep.mkdir(parents=True, exist_ok=True)
    with open(_ep / 'endpoint.json', 'w') as _fh:
        json.dump({'url': f'http://127.0.0.1:{_free}', 'port': _free, 'pid': os.getpid(),
                   'version': VERSION, 'rate_rps': L7_CFG['rate_rps'],
                   'started_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())},
                  _fh, indent=2)

    deadline = time.time() + L7_CFG['startup_timeout_s']
    import urllib.request
    ok = False
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(
                    f'http://127.0.0.1:{_free}/v1/health', timeout=1) as r:
                if r.status == 200:
                    ok = True; break
        except Exception:
            time.sleep(0.4)

    print()
    print('=' * 76)
    print('  LAYER 7 READY' if ok else '  LAYER 7 STARTED (health check timed out)')
    print('=' * 76)
    print(f'  endpoint   : http://127.0.0.1:{_free}')
    print(f'  api keys   : {len(_API_KEYS)} scoped keys ({_KEY_SOURCE}); '
          + ('see ' + str(_kf) if _kf else 'from T6_API_KEYS'))
    print(f'  uvicorn runs as a daemon thread inside this Python process; '
          'it is alive for as long as the Kaggle kernel is alive.')
    print('=' * 76)

[R8] profile=main | config=/kaggle/working/t6_active_config.json
[KEYS] generated per-start API keys written to /kaggle/working/l7_R8/api_keys.json (mode 0600); read them from that file
  TITANIUM-6 | LAYER 7 | TITANIUM6_LAYER7_R8.0
  L1: /kaggle/working/l1_R8
  L2: /kaggle/working/l2_R8  L3: /kaggle/working/l3_R8
  L4: /kaggle/working/l4_R8  L5: /kaggle/working/l5_R8
  host: 0.0.0.0:8000
  docs: disabled
  zero-trust: scoped keys=3, flow_bound=+-1000.0, body_limit=1048576 B, CORS origins=(none)


2026-09-25 12:29:40,005 INFO titanium6.l7 Loading TITANIUM-6 models...
2026-09-25 12:29:40,022 INFO titanium6.l7 L2: LightGBM tau=0.0793 features=77
2026-09-25 12:29:40,041 INFO titanium6.l7 L3 calibrators: 12/12 fitted
2026-09-25 12:29:40,042 INFO titanium6.l7 L3: LightGBM 12 classes reject="Normal" tau_L5=0.2967 tau_OOD(energy)=-0.3071
2026-09-25 12:29:40,043 INFO titanium6.l7 L4: Isolation Forest disabled (alpha=0) - not loaded
2026-09-25 12:29:40,051 INFO titanium6.l7 L4: alpha=0.00 tau4=0.2711 IF=disabled
2026-09-25 12:29:40,053 INFO titanium6.l7 L5: ResponseEngine loaded
2026-09-25 12:29:40,054 INFO titanium6.l7 Models loaded in 48 ms | available={'l2': True, 'l3': True, 'l4': True, 'l5': True}
2026-09-25 12:29:40,054 INFO titanium6.l7 TITANIUM-6 ready



  LAYER 7 READY
  endpoint   : http://127.0.0.1:8000
  api keys   : 3 scoped keys (generated); see /kaggle/working/l7_R8/api_keys.json
  uvicorn runs as a daemon thread inside this Python process; it is alive for as long as the Kaggle kernel is alive.


In [9]:
# =============================================================================
#  TITANIUM-6 | RESULTS SUMMARY | R8.0
#  Run after the layers of the current PROFILE. Prints every headline number
#  from the manifests (never from memory), and writes results_summary.json.
#  Paste the printed block back for the paper-writing step.
# =============================================================================
# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
import logging as _t6_logging   # silence matplotlib's PDF font-subsetting INFO spam
_t6_logging.getLogger('fontTools').setLevel(_t6_logging.WARNING) 
# -----------------------------------------------------------------------------

import os, json
from pathlib import Path

def _load(p):
    p = Path(p)
    if not p.exists():
        return None
    with open(p) as fh:
        return json.load(fh)

def g(d, *path, default='n/a'):
    for k in path:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d

def pct(x):
    return f'{100 * x:.3f}%' if isinstance(x, (int, float)) else str(x)

def num(x, n=4):
    return f'{x:.{n}f}' if isinstance(x, (int, float)) else str(x)

E = os.environ
prof = E.get('T6_PROFILE', '?')
m1 = _load(Path(E['T6_L1']) / 'manifest.json')
m2 = _load(Path(E['T6_L2']) / 'manifest_l2.json')
m3 = _load(Path(E['T6_L3']) / 'manifest_l3.json')
m4 = _load(Path(E['T6_L4']) / 'manifest_l4.json')
m5 = _load(Path(E['T6_L5']) / 'manifest_l5.json')
m6 = _load(Path(E['T6_L6']) / 'manifest_l6.json')
out = {'profile': prof}

print('=' * 76)
print(f'  TITANIUM-6 R8.0 RESULTS SUMMARY | profile = {prof}')
print('=' * 76)
for tag, m in (('L1', m1), ('L2', m2), ('L3', m3), ('L4', m4), ('L5', m5), ('L6', m6)):
    print(f'  {tag}: {g(m, "version") if m else "NOT RUN in this profile"}')

if m2:
    d = g(m2, 'test', 'DEPLOYED')
    print('\n[L2] binary gate')
    print(f'  model {g(m2, "best_model")} | features {g(m2, "feature_selection", "n_final")} '
          f'| excluded {g(m2, "feature_exclusion_profile", "excluded")}')
    print(f'  F1 {num(g(d, "full", "f1"))} | recall {num(g(d, "full", "recall"))} | '
          f'FPR {pct(g(d, "full", "fpr"))} | AUC {num(g(d, "full", "auc_roc"))} | '
          f'twin-free F1 {num(g(d, "dedup", "f1"))}')
    print(f'  block CI F1 {g(m2, "statistics", "deployed_block_ci", "f1")}')
    for r, v in (g(d, 'by_rule', default={}) or {}).items():
        print(f'    rule {r:<22} F1 {num(g(v, "full", "f1"))}  FPR {pct(g(v, "full", "fpr"))}')
    tt = g(m2, 'limitations', 'threshold_transfer', default={})
    print(f'  FPR-budget transfer: val {pct(g(tt, "val_fpr_at_budget_tau"))} -> test '
          f'{pct(g(tt, "test_fpr_at_budget_tau"))}')
    out['L2'] = {'test': d, 'threshold_transfer': tt}

if m3:
    b = g(m3, 'selection', 'best')
    t = g(m3, 'test', b, default={})
    print(f'\n[L3] family classifier ({b})')
    print(f'  attack-only macro-F1 {num(g(t, "attack_only", "macro_f1"))} | deployment-routed '
          f'{num(g(t, "deployment_routed", "macro_f1"))} | twin-free '
          f'{num(g(t, "deployment_routed_twin_free", "macro_f1"))}')
    print(f'  cascaded FPR after L3 {pct(g(m3, "cascaded_fpr", "deployed_fpr_after_layer3"))} '
          f'| L2 FPR measured {pct(g(m3, "cascaded_fpr", "l2_fpr_on_test_measured_routed"))}')
    out['L3'] = {'best': b, 'cascaded_fpr': g(m3, 'cascaded_fpr')}

if m4:
    c4 = _load(Path(E['T6_L4']) / 'layer4_config.json') or {}
    a4 = _load(Path(E['T6_L4']) / 'ae_architecture.json') or {}
    print('\n[L4] zero-day anomaly layer')
    print(f'  alpha {g(c4, "alpha")} | tau4 {num(g(c4, "tau4"))} | AE input transform '
          f'{g(a4, "input_transform")}')
    z = g(m4, 'zero_day', default={})
    print(f'  DR weighted {pct(g(z, "weighted_dr"))} | macro {pct(g(z, "macro_dr"))} | '
          f'fully-unseen macro {pct(g(z, "unseen_macro_dr"))} | macro twin-free '
          f'{pct(g(z, "macro_dr_twin_free"))}')
    cf = g(m4, 'cascaded_fpr', default={})
    print(f'  cascaded FPR {pct(g(cf, "cascaded_fpr"))} | delta vs L2 {pct(g(cf, "delta_vs_l2"))}')
    c = g(m4, 'l4_contribution', default={})
    print(f'  L4 added coverage {g(c, "l4_added_coverage_pp")} pp | at FPR-matched tau '
          f'{g(c, "added_coverage_pp_at_fpr_matched_tau")} pp')
    out['L4'] = {'zero_day': z, 'cascaded_fpr': cf, 'contribution': c}

if m5:
    fa = g(m5, 'fp_audit', default={})
    print('\n[L5] SOAR')
    print(f'  auto allowed: {g(m5, "auto_action_gate", "auto_allowed")}')
    print(f'  gated: {g(m5, "gated_decisions_by_family")}')
    print(f'  automated actions on true-Normal: {g(fa, "n_fp")} ({pct(g(fa, "fp_rate"))}) '
          f'by family {g(fa, "by_family")}')
    print(f'  actions {g(m5, "action_distribution")}')
    print(f'  audit chain verified {g(m5, "audit_trail", "verified")}')
    out['L5'] = {'gate': g(m5, 'auto_action_gate'), 'fp_audit': fa}

if m6:
    print('\n[L6] explainability')
    cfr = g(m6, 'counterfactual', default={}) or {}
    for fam, v in sorted(cfr.items()):
        ci = g(v, 'flip_rate_wilson95', default=None)
        ci_s = f'[{100 * ci[0]:.1f}, {100 * ci[1]:.1f}]%' if ci else 'n/a'
        print(f'  CF {fam:<18} flip {100 * g(v, "flip_rate", default=0):5.1f}% CI {ci_s} '
              f'n={g(v, "n_flows_tested")} median features changed '
              f'{g(v, "median_features_changed_when_found")}')

mx = _load(Path(E.get('T6_X2017', '/nonexistent')) / 'manifest_x2017.json')
if mx:
    print('\n[09] cross-dataset CIC-IDS2017 (zero-shot)')
    l2x = g(mx, 'layer2', 'ids2017', default={})
    print(f'  L2 F1 {num(g(l2x, "f1"))} CI {g(l2x, "f1_ci95_flow_level")} | recall '
          f'{num(g(l2x, "recall"))} | FPR {pct(g(l2x, "fpr"))} | AUC {num(g(l2x, "auc_roc"))}')
    print(f'  L3 fine macro-F1 {num(g(mx, "layer3", "fine_family", "macro_f1_support_ge_min"))} | '
          f'super-family macro-F1 {num(g(mx, "layer3", "super_family", "macro_f1"))}')
    print(f'  features imputed (absent in 2017), L2: {g(mx, "data", "absent_in_L2_features")}')
    print(f'  lost separation 2018->2017: {g(mx, "separation", "lost_separation_2018_to_2017")}')
    out['X2017'] = {'layer2': l2x, 'layer3': g(mx, 'layer3'), 'separation_lost':
                    g(mx, 'separation', 'lost_separation_2018_to_2017')}

dest = Path('/kaggle/working') / f'results_summary_{prof}.json'
with open(dest, 'w') as fh:
    json.dump(out, fh, indent=2, default=str)
print(f'\n  written: {dest}')
print('=' * 76)

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  TITANIUM-6 R8.0 RESULTS SUMMARY | profile = main
  L1: TITANIUM6_LAYER1_R8.0
  L2: TITANIUM6_LAYER2_R8.0
  L3: TITANIUM6_LAYER3_R8.0
  L4: TITANIUM6_LAYER4_R8.0
  L5: TITANIUM6_LAYER5_R8.0
  L6: TITANIUM6_LAYER6_R8.0

[L2] binary gate
  model LightGBM | features 77 | excluded []
  F1 0.8038 | recall 0.9778 | FPR 8.519% | AUC 0.9906 | twin-free F1 0.6720
  block CI F1 [0.4911128778823743, 0.9035325300472242]
    rule fpr_budget             F1 0.7900  FPR 9.331%
    rule youden                 F1 0.8038  FPR 8.519%
    rule youden_clamped_legacy  F1 0.8628  FPR 5.265%
    rule min_recall             F1 0.9699  FPR 0.064%
    rule cost_ratio             F1 0.8038  FPR 8.519%
  FPR-budget transfer: val 0.998% -> test 9.331%

[L3] family classifier (LightGBM)
  attack-only macro-F1 0.8810 | deployment-routed 0.7396 | twin-free 0.6818
  cascaded FPR after L3 8.040% | L2 FPR measured 8.519%

[L4] zero-day anomaly layer
  alpha

In [10]:
#!/usr/bin/env python3

# =============================================================================
#  TITANIUM-6 | 09 | CROSS-DATASET TRANSFER (CIC-IDS2017) | R8.0
#  Zero-shot: the R8 Layer-1 transform, Layer-2 gate and Layer-3 classifier
#  trained on CSE-CIC-IDS2018 are applied to CIC-IDS2017 with NO refitting.
#
#  What changed from the v1 notebook's Cell 10, and why:
#   X1 SAME TRANSFORM AS PRODUCTION. Features are produced by Layer 1's own
#      OnlineTransformer (the function Layer 7 serves), not by a re-implemented
#      copy, so train/serve/transfer parity holds by construction.
#   X2 HONEST MISSING-FEATURE ACCOUNTING. A feature absent from CIC-IDS2017 is
#      IMPUTED with the 2018 training median (Layer-1 policy); every imputed
#      feature and every imputed cell is counted and reported. v1's claim that
#      zero-filling "introduces no missingness" is withdrawn.
#   X3 DDoS TAXONOMY. CIC-IDS2017's generic 'DDoS' label is NOT mapped to
#      DDoS-HOIC. It is evaluated only at super-family level (DDoS / DoS /
#      BruteForce / Botnet / Infiltration); the fine-family table covers only
#      labels that exist in both taxonomies.
#   X4 2018-vs-2017 PER-FEATURE SEPARATION on the SAME statistic, both sides
#      (v1 computed it on 2017 only and left the delta as future work).
#   X5 Web attacks (BruteForce-Web/XSS, SQL-Injection) were never trained on
#      (zero-day in 2018): reported as Layer-2 flag rate only.
#   X6 Runs under any profile; under 'no_shortcut' it measures transfer
#      without the TCP-stack features.
#   X8 SEPARATION = univariate ROC-AUC per feature (both years) plus a count
#      of features whose direction FLIPS between years. The median/IQR
#      statistic used before is blind to zero-inflated features and is kept
#      only for comparison with the earlier paper.
#   X9 PER-FAMILY SHIFT. The pooled comparison mixes different attack
#      compositions per year; each shared family is also compared with Normal
#      separately in each year (and flips among the SHAP top-10 are listed).
#   X7 SELF-CONTAINED. Layer 1's transform is embedded verbatim (generated
#      from 01_layer1_preprocessing.py, sha256 recorded). The Layer-1 file is
#      no longer required; when it is present it is re-hashed and must match.
# =============================================================================
# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
import logging as _t6_logging   # silence matplotlib's PDF font-subsetting INFO spam
_t6_logging.getLogger('fontTools').setLevel(_t6_logging.WARNING)
# -----------------------------------------------------------------------------

import os, re, gc, json, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb


# =============================================================================
#  EMBEDDED LAYER-1 TRANSFORM (generated verbatim from 01_layer1_preprocessing.py)
#  sha256 of the embedded source: 4d4f9ab874040c2034dc1f9c7bcbca2d331482d5335baddbb6893c233e13c879
#  09 does not need the Layer-1 file. If the file IS found, its transform source
#  is re-extracted and hashed; a mismatch stops the run (parity is proven, not
#  assumed).
# =============================================================================
L1_TRANSFORM_SHA256 = '4d4f9ab874040c2034dc1f9c7bcbca2d331482d5335baddbb6893c233e13c879'
L1_TRANSFORM_NAMES = ['COL_NORM', 'SENTINEL_MINUS1', 'RATIO_SPECS', 'COUNTER_FEATURES', '_read_json', 'OnlineTransformer']
_L1_SRC = "COL_NORM = {\n    'attack_type':'Attack_Type','dst port':'Dst Port','destination port':'Dst Port',\n    'src port':'Src Port','source port':'Src Port','src ip':'Src IP','source ip':'Src IP',\n    'dst ip':'Dst IP','destination ip':'Dst IP','timestamp':'Timestamp','flow id':'Flow ID',\n    'fwd header len':'Fwd Header Length','bwd header len':'Bwd Header Length',\n    'flow byts/s':'Flow Bytes/s','flow pkts/s':'Flow Packets/s',\n    'tot fwd pkts':'Total Fwd Packets','tot bwd pkts':'Total Backward Packets',\n    'totlen fwd pkts':'Total Length of Fwd Packets',\n    'totlen bwd pkts':'Total Length of Bwd Packets',\n    'fwd pkt len max':'Fwd Packet Length Max','fwd pkt len min':'Fwd Packet Length Min',\n    'fwd pkt len mean':'Fwd Packet Length Mean','fwd pkt len std':'Fwd Packet Length Std',\n    'bwd pkt len max':'Bwd Packet Length Max','bwd pkt len min':'Bwd Packet Length Min',\n    'bwd pkt len mean':'Bwd Packet Length Mean','bwd pkt len std':'Bwd Packet Length Std',\n    'pkt len min':'Packet Length Min','pkt len max':'Packet Length Max',\n    'pkt len mean':'Packet Length Mean','pkt len std':'Packet Length Std',\n    'pkt len var':'Packet Length Variance','pkt size avg':'Packet Size Average',\n    'cwe flag count':'CWR Flag Count','ece flag cnt':'ECE Flag Count',\n    'subflow fwd byts':'Subflow Fwd Bytes','subflow bwd byts':'Subflow Bwd Bytes',\n    'subflow fwd pkts':'Subflow Fwd Packets','subflow bwd pkts':'Subflow Bwd Packets',\n    'init_win_bytes_forward':'Init Fwd Win Byts','init_win_bytes_backward':'Init Bwd Win Byts',\n    'act_data_pkt_fwd':'Act Data Pkt Fwd','min_seg_size_forward':'Min Seg Size Forward',\n    'fwd act data pkts':'Act Data Pkt Fwd','fwd seg size min':'Min Seg Size Forward',\n    'fwd pkts/s':'Fwd Pkts/s','bwd pkts/s':'Bwd Pkts/s',\n}\n\nSENTINEL_MINUS1 = ('Init Fwd Win Byts','Init Bwd Win Byts')\n\nRATIO_SPECS = {\n    'Ratio__FwdBwd_Packets':('Total Fwd Packets','Total Backward Packets'),\n    'Ratio__FwdBwd_Bytes':('Total Length of Fwd Packets','Total Length of Bwd Packets'),\n    'Ratio__FwdBwd_IAT':('Fwd IAT Tot','Bwd IAT Tot'),\n    'Ratio__Active_Idle':('Active Mean','Idle Mean'),\n}\n\nCOUNTER_FEATURES = ('N_Invalid_Features','N_Clipped_Negatives')\n\ndef _read_json(path):\n    with open(path,encoding='utf-8') as fh: return json.load(fh)\n\nclass OnlineTransformer:\n    def __init__(self,features,mean,std,impute_nan,impute_posinf):\n        self.features = list(features)\n        self.mu = np.array([mean[f] for f in self.features],np.float64)\n        self.sd = np.array([std[f] if std[f]>0 else 1.0 for f in self.features])\n        self.inan = np.array([impute_nan.get(f,0.0) for f in self.features])\n        self.iinf = np.array([impute_posinf.get(f,0.0) for f in self.features])\n        self.pos = {f:i for i,f in enumerate(self.features)}\n        self.i_ninv = self.pos.get('N_Invalid_Features')\n        self.i_nclp = self.pos.get('N_Clipped_Negatives')\n        self.norm = {}\n    @classmethod\n    def from_json(cls,path):\n        p = _read_json(path)\n        return cls(p['features'],p['mean'],p['std'],p['impute_nan'],p['impute_posinf'])\n    @staticmethod\n    def _num(v):\n        try: x = float(v)\n        except (TypeError,ValueError): return np.nan\n        x = float(np.float32(x))\n        return np.nan if x == -np.inf else x\n    def __call__(self,raw):\n        g = {}\n        for k,v in raw.items():\n            nk = self.norm.get(k)\n            if nk is None:\n                nk = COL_NORM.get(str(k).strip().lower(),str(k).strip())\n                self.norm[k] = nk\n            g[nk] = v\n        num = self._num; d = {}; sent = set()\n        for col in SENTINEL_MINUS1:\n            v = num(g[col]) if col in g else None\n            if v is None: d[f'{col}__present'] = 0.0\n            else:\n                d[f'{col}__present'] = 1.0 if (v==v and v!=-1.0) else 0.0\n                if v == -1.0: d[col] = np.nan; sent.add(col)\n                else: d[col] = v\n        for new_col,(nc,dc) in RATIO_SPECS.items():\n            n = num(g[nc]) if nc in g else 0.0\n            dd = num(g[dc]) if dc in g else 1.0\n            dd = 1.0 if dd==0.0 else dd\n            with np.errstate(invalid='ignore',divide='ignore'):   # IEEE: inf/inf -> NaN, as pandas\n                r = np.float32(np.float32(n)/np.float32(dd)) if (n==n and dd==dd) else np.nan\n            d[new_col] = float(np.clip(r,0.0,1e6)) if r==r else np.nan\n        if 'Flow Duration' in g:\n            dur = num(g['Flow Duration'])\n            dur = max(dur,0.0) if dur==dur else np.nan\n            d['Log_Flow_Duration'] = (float(np.log1p(np.float32(dur)))\n                                      if dur==dur else np.nan)\n            d['Zero_Duration_Flag'] = 1.0 if dur==0.0 else 0.0\n        else: d['Log_Flow_Duration'] = 0.0; d['Zero_Duration_Flag'] = 0.0\n        x = np.empty(len(self.features)); n_inv = n_clp = 0\n        for i,f in enumerate(self.features):\n            if i == self.i_ninv or i == self.i_nclp: continue\n            v = d[f] if f in d else (num(g[f]) if f in g else np.nan)\n            if f in sent: x[i] = self.inan[i]; continue\n            if v != v: n_inv += 1; v = self.inan[i]\n            elif v == np.inf: n_inv += 1; v = self.iinf[i]\n            elif v < 0.0: n_clp += 1; v = 0.0\n            x[i] = v\n        if self.i_ninv is not None: x[self.i_ninv] = n_inv\n        if self.i_nclp is not None: x[self.i_nclp] = n_clp\n        return ((x-self.mu)/self.sd).astype(np.float32)"

SEED = 42
CFG = {
    'version': 'TITANIUM6_CROSS2017_R8.0',
    'seed': SEED,
    'chunk_rows': 100_000,
    'boot_n': 1000,
    'sep_sample_n': 300_000,
    'min_family_support': 20,
    'dataset_candidates': [
        '/kaggle/input/cicids2017', '/kaggle/input/datasets/dhoogla/cicids2017',
        '/kaggle/input/datasets/cicids2017', '/kaggle/input/network-intrusion-dataset',
        '/kaggle/input/datasets/chethuhn/network-intrusion-dataset'],
}

# CIC-IDS2017 column names -> the canonical CSE-CIC-IDS2018 names Layer 1 uses.
# Names already identical are not listed. Keys are lower-case.
X2017_ALIASES = {
    'fwd packets length total': 'Total Length of Fwd Packets',
    'bwd packets length total': 'Total Length of Bwd Packets',
    'total length of fwd packets': 'Total Length of Fwd Packets',
    'total length of bwd packets': 'Total Length of Bwd Packets',
    'fwd iat total': 'Fwd IAT Tot', 'bwd iat total': 'Bwd IAT Tot',
    'fwd packets/s': 'Fwd Pkts/s', 'bwd packets/s': 'Bwd Pkts/s',
    'fin flag count': 'FIN Flag Cnt', 'syn flag count': 'SYN Flag Cnt',
    'rst flag count': 'RST Flag Cnt', 'psh flag count': 'PSH Flag Cnt',
    'ack flag count': 'ACK Flag Cnt', 'urg flag count': 'URG Flag Cnt',
    'cwe flag count': 'CWR Flag Count',
    'avg packet size': 'Packet Size Average', 'average packet size': 'Packet Size Average',
    'avg fwd segment size': 'Fwd Seg Size Avg', 'avg bwd segment size': 'Bwd Seg Size Avg',
    'fwd avg bytes/bulk': 'Fwd Byts/b Avg', 'fwd avg packets/bulk': 'Fwd Pkts/b Avg',
    'fwd avg bulk rate': 'Fwd Blk Rate Avg', 'bwd avg bytes/bulk': 'Bwd Byts/b Avg',
    'bwd avg packets/bulk': 'Bwd Pkts/b Avg', 'bwd avg bulk rate': 'Bwd Blk Rate Avg',
    'init fwd win bytes': 'Init Fwd Win Byts', 'init bwd win bytes': 'Init Bwd Win Byts',
    'init_win_bytes_forward': 'Init Fwd Win Byts', 'init_win_bytes_backward': 'Init Bwd Win Byts',
    'fwd act data packets': 'Act Data Pkt Fwd', 'act_data_pkt_fwd': 'Act Data Pkt Fwd',
    'fwd seg size min': 'Min Seg Size Forward', 'min_seg_size_forward': 'Min Seg Size Forward',
}

def _norm_label(s):
    s = str(s).strip().lower()
    s = re.sub(r'[\u2010-\u2015\ufffd_\-]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip().replace('slowhttptest', 'slowhttp')

# fine family (Layer-3 taxonomy) or None; DDoS kept generic (X3)
_LABELS = {
    'Normal': ['benign'],
    'BruteForce-FTP': ['ftp patator'], 'BruteForce-SSH': ['ssh patator'],
    'DoS-Slowloris': ['dos slowloris'], 'DoS-SlowHTTP': ['dos slowhttp'],
    'DoS-Hulk': ['dos hulk'], 'DoS-GoldenEye': ['dos goldeneye'],
    'DDoS(generic)': ['ddos'],
    'Botnet': ['bot'], 'Infiltration': ['infiltration'],
    'BruteForce-Web': ['web attack brute force', 'brute force web', 'webattack bruteforce'],
    'BruteForce-XSS': ['web attack xss', 'brute force xss', 'webattack xss'],
    'SQL-Injection': ['web attack sql injection', 'sql injection', 'webattack sqlinjection'],
}
LABEL_MAP = {_norm_label(v): t for t, vs in _LABELS.items() for v in vs}
UNMAPPED_BY_DESIGN = {'portscan', 'heartbleed'}
UNSEEN_FAMILIES = {'BruteForce-Web', 'BruteForce-XSS', 'SQL-Injection'}

def super_family(f):
    if f is None: return None
    if f == 'Normal': return 'Normal'
    if f.startswith('DDoS'): return 'DDoS'
    if f.startswith('DoS'): return 'DoS'
    if f in ('BruteForce-FTP', 'BruteForce-SSH'): return 'BruteForce'
    if f in UNSEEN_FAMILIES: return 'Web(unseen)'
    return f      # Botnet, Infiltration


def _atomic_json(obj, dest):
    tmp = Path(str(dest) + '.tmp')
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=2, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
    os.replace(tmp, dest)


def _pick(path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)


def _bin_metrics(y, s, tau):
    pred = s >= tau
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
    return {'f1': 2 * pr * rc / max(pr + rc, 1e-12), 'precision': pr, 'recall': rc,
            'fpr': fp / max(fp + tn, 1), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'auc_roc': float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else None,
            'auc_pr': float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else None}


def _uni_auc(X, y):
    """Per-feature univariate ROC-AUC of 'value predicts Attack'. Unlike a
    median/IQR statistic it sees zero-inflated features (flags present in 5%
    vs 30% of flows). 0.5 = no signal; <0.5 = attack has LOWER values."""
    out = np.full(X.shape[1], np.nan)
    if len(np.unique(y)) < 2:
        return out
    for j in range(X.shape[1]):
        col = X[:, j]
        out[j] = 0.5 if np.all(col == col[0]) else roc_auc_score(y, col)
    return out


def _separation(X, y):
    """|median(A)-median(N)| / pooled IQR per column (affine-invariant)."""
    out = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        a, n = X[y == 1, j], X[y == 0, j]
        if len(a) == 0 or len(n) == 0:
            out[j] = np.nan; continue
        iqr = 0.5 * ((np.percentile(a, 75) - np.percentile(a, 25)) +
                     (np.percentile(n, 75) - np.percentile(n, 25)))
        d = abs(np.median(a) - np.median(n))
        out[j] = d / iqr if iqr > 1e-12 else (0.0 if d < 1e-12 else np.inf)
    return out


class CrossDataset2017:
    def __init__(self):
        E = os.environ
        self.l1, self.l2, self.l3 = Path(E['T6_L1']), Path(E['T6_L2']), Path(E['T6_L3'])
        self.out = Path(E.get('T6_X2017', '/kaggle/working/x2017_R8'))
        self.out.mkdir(parents=True, exist_ok=True)
        self.R = {'version': CFG['version'], 'profile': E.get('T6_PROFILE'), 'config': CFG,
                  'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}

    @staticmethod
    def _load_l1_transform():
        """Build Layer 1's transform from the embedded verbatim source and, when
        the Layer-1 file is available, prove it is identical to that file."""
        import ast, hashlib, types
        if hashlib.sha256(_L1_SRC.encode()).hexdigest() != L1_TRANSFORM_SHA256:
            raise SystemExit('[FATAL] embedded Layer-1 transform was edited (hash mismatch)')
        ns = {'np': np, 'json': json, 'Path': Path, 'os': os}
        exec(compile(_L1_SRC, 'layer1_transform_embedded', 'exec'), ns)
        parity = 'embedded copy (Layer-1 file not found to compare)'
        name = '01_layer1_preprocessing.py'
        cands = [os.environ.get('T6_L1_CODE', ''), f'/kaggle/working/{name}']
        for root, _d, files in os.walk('/kaggle/input', followlinks=True):
            if name in files:
                cands.append(os.path.join(root, name))
        for c in cands:
            if c and os.path.exists(c):
                src = open(c, encoding='utf-8').read()
                seg = {}
                for n in ast.parse(src).body:
                    k = (n.name if isinstance(n, (ast.FunctionDef, ast.ClassDef)) else
                         n.targets[0].id if isinstance(n, ast.Assign)
                         and isinstance(n.targets[0], ast.Name) else None)
                    if k in L1_TRANSFORM_NAMES:
                        seg[k] = ast.get_source_segment(src, n)
                if set(seg) != set(L1_TRANSFORM_NAMES):
                    continue
                h = hashlib.sha256('\n\n'.join(seg[k] for k in L1_TRANSFORM_NAMES)
                                   .encode()).hexdigest()
                if h != L1_TRANSFORM_SHA256:
                    raise SystemExit(f'[FATAL] {c} has a DIFFERENT Layer-1 transform than '
                                     'the one embedded here: use matching package versions')
                parity = f'VERIFIED identical to {c}'
                break
        return types.SimpleNamespace(**{k: ns[k] for k in L1_TRANSFORM_NAMES}), parity

    # ------------------------------------------------------------ P0
    def p0_contracts(self):
        print('[P0] Contracts (R8 manifests, Layer-1 code)')
        for d, f in ((self.l1, 'manifest.json'), (self.l2, 'manifest_l2.json'),
                     (self.l3, 'manifest_l3.json')):
            if not (d / f).exists():
                raise SystemExit(f'[FATAL] {d / f} missing: run that layer for this profile first')
        with open(self.l1 / 'manifest.json') as fh: self.m1 = json.load(fh)
        with open(self.l2 / 'manifest_l2.json') as fh: self.m2 = json.load(fh)
        with open(self.l3 / 'manifest_l3.json') as fh: self.m3 = json.load(fh)
        self.L1, parity = self._load_l1_transform()
        print(f'  Layer-1 transform: sha256 {L1_TRANSFORM_SHA256[:12]} | {parity}')
        self.R['layer1_transform'] = {'sha256': L1_TRANSFORM_SHA256, 'parity': parity}
        sp = Path(self.m1.get('l2_input_paths', {}).get('scaler') or
                  (self.l1 / 'scaler_params.json'))
        if not sp.exists():
            raise SystemExit(f'[FATAL] {sp} missing')
        self.T = self.L1.OnlineTransformer.from_json(str(sp))
        self.feats_all = list(self.T.features)
        c2 = self.m2['l3_l4_contract']; c3 = self.m3['l4_l5_contract']
        self.f2, self.f3 = list(c2['features']), list(c3['features'])
        for nm, fl in (('L2', self.f2), ('L3', self.f3)):
            miss = [f for f in fl if f not in self.feats_all]
            if miss:
                raise SystemExit(f'[FATAL] {nm} features not produced by Layer 1: {miss}')
        self.i2 = [self.feats_all.index(f) for f in self.f2]
        self.i3 = [self.feats_all.index(f) for f in self.f3]
        self.l2_model = _pick(c2['model']); self.l2_cal = _pick(c2['cal'])
        self.l2_kind = c2['model_kind']
        with open(c2['threshold_json']) as fh: th = json.load(fh)
        self.l2_name = self.m2['best_model']
        self.l2_tau = float(th['deployed_tau'][self.l2_name])
        self.l2_rules = th.get('all_rules_deployed', {})
        self.l3_model = _pick(c3['model']); self.l3_cals = _pick(c3['calibrators'])
        self.l3_le = _pick(c3['encoder']); self.l3_kind = c3['model_kind']
        self.l3_classes = list(c3['classes']); self.reject = c3.get('reject_class') or 'Normal'
        excl = self.m2.get('feature_exclusion_profile', {}).get('excluded', [])
        print(f'  L1 {self.m1["version"]} | L2 {self.m2["version"]} ({self.l2_name}, '
              f'tau={self.l2_tau:.4f}, {len(self.f2)} feats) | L3 {self.m3["version"]} '
              f'({len(self.l3_classes)} classes)')
        print(f'  excluded features in this profile: {excl or "none"}')
        self.R['inputs'] = {'l1': self.m1['version'], 'l2': self.m2['version'],
                            'l3': self.m3['version'], 'l2_model': self.l2_name,
                            'l2_tau': self.l2_tau, 'excluded_features': excl}

    # ------------------------------------------------------------ P1
    def _find_files(self):
        roots = [Path(p) for p in CFG['dataset_candidates'] if Path(p).exists()]
        if os.environ.get('T6_X2017_DATA'):
            roots.insert(0, Path(os.environ['T6_X2017_DATA']))
        for r in roots:
            pqs = sorted(p for p in r.rglob('*.parquet') if p.stat().st_size > 100_000)
            csv = sorted(p for p in r.rglob('*.csv') if p.stat().st_size > 1_000_000)
            if pqs or csv:
                return r, (pqs or csv)
        raise SystemExit('[FATAL] CIC-IDS2017 not found. Attach it or set T6_X2017_DATA.')

    def p1_load_transform(self):
        print('[P1] Load CIC-IDS2017 and transform with the Layer-1 OnlineTransformer')
        root, files = self._find_files()
        print(f'  dataset: {root} | {len(files)} files')
        Xs, fams, raw_cols = [], [], set()
        drop = {'unmapped_by_design': {}, 'unmapped_other': {}, 'negative_duration': 0,
                'no_label': 0}
        n_raw = 0; n_invalid_cells = 0
        i_ninv = self.T.i_ninv
        for fp in files:
            df = pd.read_parquet(fp) if fp.suffix == '.parquet' else pd.read_csv(fp, low_memory=False)
            n_raw += len(df)
            df.columns = [X2017_ALIASES.get(str(c).strip().lower(), str(c).strip())
                          for c in df.columns]
            raw_cols |= set(df.columns)
            lab_col = next((c for c in df.columns if c.lower() == 'label'), None)
            if lab_col is None:
                drop['no_label'] += len(df); continue
            # dhoogla stores Label as a pandas category: cast to str so empty
            # categories are not reported as dropped labels
            nl = df[lab_col].astype(str).map(_norm_label)
            fam = nl.map(LABEL_MAP)
            for v, c in nl[fam.isna()].value_counts().items():
                if c == 0:
                    continue
                key = 'unmapped_by_design' if v in UNMAPPED_BY_DESIGN else 'unmapped_other'
                drop[key][v] = drop[key].get(v, 0) + int(c)
            keep = fam.notna().to_numpy(copy=True)
            if 'Flow Duration' in df.columns:
                dur = pd.to_numeric(df['Flow Duration'], errors='coerce').values
                neg = keep & (dur < 0)
                drop['negative_duration'] += int(neg.sum()); keep &= ~neg
            df = df.loc[keep].drop(columns=[lab_col]); fam = fam[keep].values
            for s in range(0, len(df), CFG['chunk_rows']):
                recs = df.iloc[s:s + CFG['chunk_rows']].to_dict('records')
                X = np.vstack([self.T(r) for r in recs]).astype(np.float32)
                if i_ninv is not None:
                    # undo standardisation of the counter to recover the raw count
                    n_invalid_cells += int(np.rint(X[:, i_ninv] * self.T.sd[i_ninv]
                                                   + self.T.mu[i_ninv]).sum())
                Xs.append(X); del recs
            fams.append(fam)
            print(f'    {fp.name:<45} kept {int(keep.sum()):>9,}')
            del df; gc.collect()
        self.X = np.vstack(Xs); del Xs
        self.fam = np.concatenate(fams).astype(object)
        self.y = (self.fam != 'Normal').astype(np.int8)
        # X2: which features had no source column in CIC-IDS2017
        derived_ok = set()
        for d, (a, b) in self.L1.RATIO_SPECS.items():
            if a in raw_cols and b in raw_cols: derived_ok.add(d)
        if 'Flow Duration' in raw_cols: derived_ok |= {'Log_Flow_Duration', 'Zero_Duration_Flag'}
        for c in self.L1.SENTINEL_MINUS1:
            if c in raw_cols: derived_ok.add(f'{c}__present')
        derived_ok |= set(self.L1.COUNTER_FEATURES)
        absent = [f for f in self.feats_all if f not in raw_cols and f not in derived_ok]
        self.R['data'] = {
            'files': [p.name for p in files], 'raw_rows': n_raw, 'rows_kept': int(len(self.y)),
            'attack_rows': int(self.y.sum()), 'normal_rows': int((self.y == 0).sum()),
            'dropped': drop,
            'family_counts': {k: int(v) for k, v in pd.Series(self.fam).value_counts().items()},
            'features_absent_in_2017_imputed_with_2018_train_median': absent,
            'absent_in_L2_features': [f for f in absent if f in self.f2],
            'absent_in_L3_features': [f for f in absent if f in self.f3],
            'invalid_cells_imputed_total': n_invalid_cells,
            'imputation_policy': 'Layer-1 R8 OnlineTransformer (train-fitted medians; '
                                 'training-quantile for +Inf); no refitting on 2017'}
        print(f'  rows kept {len(self.y):,} (attack {int(self.y.sum()):,}) | dropped: '
              f'{sum(drop["unmapped_by_design"].values()):,} by design '
              f'{drop["unmapped_by_design"]}, other {drop["unmapped_other"] or 0}, '
              f'neg-duration {drop["negative_duration"]}')
        print(f'  features absent in 2017 (imputed): {len(absent)} total | '
              f'{len(self.R["data"]["absent_in_L2_features"])} of L2 | '
              f'{len(self.R["data"]["absent_in_L3_features"])} of L3')
        if self.R['data']['absent_in_L2_features']:
            print(f'    L2 imputed: {self.R["data"]["absent_in_L2_features"]}')
        if drop['unmapped_other']:
            print(f'  [WARN] unexpected unmapped labels: {drop["unmapped_other"]}')

    # ------------------------------------------------------------ helpers
    def _l2_score(self, X):
        if self.l2_kind == 'lgb':
            p = self.l2_model.predict(X)
        elif self.l2_kind == 'xgb':
            p = self.l2_model.predict(xgb.DMatrix(X, feature_names=self.f2))
        else:
            p = self.l2_model.predict_proba(X)[:, 1]
        return self.l2_cal.predict_proba(np.asarray(p, np.float64).reshape(-1, 1))[:, 1]

    def _l3_proba(self, X):
        if self.l3_kind == 'lgb' or isinstance(self.l3_model, lgb.Booster):
            q = self.l3_model.predict(X)
        else:
            q = self.l3_model.predict(xgb.DMatrix(X, feature_names=self.f3))
        q = np.array(q, np.float64).reshape(-1, len(self.l3_classes))
        for c, lr in enumerate(self.l3_cals):
            if lr is not None and c < q.shape[1]:
                pc = np.clip(q[:, c], 1e-7, 1 - 1e-7)
                q[:, c] = lr.predict_proba(np.log(pc / (1 - pc)).reshape(-1, 1))[:, 1]
        t = q.sum(1, keepdims=True)
        return q / np.where(t == 0, 1, t)

    # ------------------------------------------------------------ P2
    def p2_layer2(self):
        print('[P2] Layer 2 zero-shot')
        s = np.concatenate([self._l2_score(self.X[i:i + 500_000][:, self.i2])
                            for i in range(0, len(self.y), 500_000)])
        self.s2 = s
        m = _bin_metrics(self.y, s, self.l2_tau)
        # flow-level bootstrap of F1 from resampled confusion counts (fast, exact)
        rng = np.random.RandomState(SEED)
        cell = (self.y.astype(np.int64) * 2 + (s >= self.l2_tau).astype(np.int64))
        bf = []
        for _ in range(CFG['boot_n']):
            c = np.bincount(cell[rng.randint(0, len(cell), len(cell))], minlength=4)
            fp_, fn_, tp_ = c[1], c[2], c[3]
            bf.append(2 * tp_ / max(2 * tp_ + fp_ + fn_, 1))
        m['f1_ci95_flow_level'] = [float(np.percentile(bf, 2.5)), float(np.percentile(bf, 97.5))]
        ref = self.m2.get('test', {}).get('DEPLOYED', {}).get('full', {})
        by_rule = {r: _bin_metrics(self.y, s, float(t)) for r, t in self.l2_rules.items()}
        flag = {f: float((s[self.fam == f] >= self.l2_tau).mean())
                for f in sorted(set(self.fam)) if f != 'Normal'}
        self.R['layer2'] = {'ids2017': m, 'ids2018_test_reference': ref, 'by_rule_2017': by_rule,
                            'flag_rate_by_2017_label': flag,
                            'ci_note': ('flow-level bootstrap: this CIC-IDS2017 release has no '
                                        'timestamps, so no block bootstrap is possible; the '
                                        'interval understates uncertainty')}
        print(f'  2018 test: F1 {ref.get("f1", float("nan")):.4f} | FPR {100 * ref.get("fpr", float("nan")):.3f}%'
              f' | AUC {ref.get("auc_roc", float("nan")):.4f}')
        print(f'  2017     : F1 {m["f1"]:.4f} {m["f1_ci95_flow_level"]} | FPR {100 * m["fpr"]:.3f}%'
              f' | recall {m["recall"]:.4f} | AUC {m["auc_roc"]:.4f}')
        for f, r in flag.items():
            print(f'    flag rate {f:<16} {100 * r:6.2f}%  (n={int((self.fam == f).sum()):,})')

    # ------------------------------------------------------------ P3
    def p3_layer3(self):
        print('[P3] Layer 3 zero-shot (attack rows; fine and super-family views)')
        atk = self.y == 1
        Xa = self.X[atk][:, self.i3]
        P = np.vstack([self._l3_proba(Xa[i:i + 500_000]) for i in range(0, len(Xa), 500_000)])
        del Xa
        pred = np.array(self.l3_classes, dtype=object)[P.argmax(1)]
        truth = self.fam[atk]
        l2f = self.s2[atk] >= self.l2_tau
        # fine view: labels that exist in the Layer-3 taxonomy
        fine = np.isin(truth, self.l3_classes)
        labs = sorted(set(truth[fine]))
        fine_f1 = f1_score(truth[fine], pred[fine], labels=labs, average=None, zero_division=0)
        sup = {l: int((truth[fine] == l).sum()) for l in labs}
        ok = [l for l in labs if sup[l] >= CFG['min_family_support']]
        ref_pc = {}
        try:
            b = self.m3['selection']['best']; ao = self.m3['test'][b]['attack_only']
            ref_pc = dict(zip(self.l3_classes, ao['per_class_f1']))
            ref_macro = ao['macro_f1']
        except KeyError:
            ref_macro = None
        # super-family view
        st = np.array([super_family(t) for t in truth], dtype=object)
        sp = np.array([super_family(p) for p in pred], dtype=object)
        seen = st != 'Web(unseen)'
        slabs = sorted(set(st[seen]))
        sf1 = f1_score(st[seen], sp[seen], labels=slabs, average=None, zero_division=0)
        self.R['layer3'] = {
            'fine_family': {'labels': labs, 'f1': dict(zip(labs, map(float, fine_f1))),
                            'support': sup,
                            'macro_f1_support_ge_min': float(np.mean([fine_f1[labs.index(l)]
                                                                      for l in ok])) if ok else None,
                            'ids2018_attack_only_per_class_f1': ref_pc,
                            'ids2018_attack_only_macro_f1': ref_macro,
                            'excluded': 'DDoS(generic) is not mapped to any 2018 DDoS tool (X3)'},
            'super_family': {'labels': slabs, 'f1': dict(zip(slabs, map(float, sf1))),
                             'macro_f1': float(np.mean(sf1)) if len(sf1) else None,
                             'support': {l: int((st[seen] == l).sum()) for l in slabs}},
            'predicted_normal_share_of_attacks': float((pred == self.reject).mean()),
            'l2_flagged_attack_share': float(l2f.mean()),
            'deployment_routed_fine_macro_f1': (float(np.mean(f1_score(
                truth[fine & l2f], pred[fine & l2f], labels=ok, average=None,
                zero_division=0))) if ok and (fine & l2f).any() else None)}
        print(f'  fine-family macro-F1 (support>={CFG["min_family_support"]}): '
              f'{self.R["layer3"]["fine_family"]["macro_f1_support_ge_min"]} '
              f'| 2018 attack-only reference {ref_macro}')
        for l in labs:
            print(f'    {l:<16} F1 2017 {fine_f1[labs.index(l)]:.4f} (n={sup[l]:>7,}) | '
                  f'2018 {ref_pc.get(l, float("nan")):.4f}')
        print(f'  super-family macro-F1: {self.R["layer3"]["super_family"]["macro_f1"]}')
        for l in slabs:
            print(f'    {l:<14} F1 {sf1[slabs.index(l)]:.4f}')
        print(f'  attacks predicted as "{self.reject}" by L3: '
              f'{100 * self.R["layer3"]["predicted_normal_share_of_attacks"]:.2f}%')
        self._fine = (labs, fine_f1, ref_pc)

    # ------------------------------------------------------------ P4
    def p4_separation(self):
        print('[P4] Per-feature separation, 2018 test vs 2017 (same statistic, L2 features)')
        rng = np.random.RandomState(SEED)
        n = min(CFG['sep_sample_n'], len(self.y))
        ii = rng.choice(len(self.y), n, replace=False)
        X17s, y17s = self.X[ii][:, self.i2], self.y[ii]
        s17 = _separation(X17s, y17s)
        path = self.m1['l2_input_paths']['test_known']
        if not os.path.exists(path):
            path = str(self.l1 / 'l2_ready' / Path(path).name)
        pf = pq.ParquetFile(path); parts, got = [], 0
        for b in pf.iter_batches(batch_size=200_000, columns=self.f2 + ['Target']):
            parts.append(b.to_pandas()); got += len(parts[-1])
            if got >= 4 * n: break
        d18 = pd.concat(parts, ignore_index=True)
        d18 = d18.sample(min(n, len(d18)), random_state=SEED)
        X18s = d18[self.f2].to_numpy(np.float32)
        y18s = (d18['Target'].values != 'Normal').astype(np.int8)
        s18 = _separation(X18s, y18s)
        a18, a17 = _uni_auc(X18s, y18s), _uni_auc(X17s, y17s)
        inf18, inf17 = np.abs(a18 - 0.5) >= 0.1, np.abs(a17 - 0.5) >= 0.1
        flipped = inf18 & inf17 & (np.sign(a18 - 0.5) != np.sign(a17 - 0.5))
        lost = inf18 & ~inf17
        rows = [{'feature': f, 'auc_2018': float(a), 'auc_2017': float(b),
                 'median_iqr_sep_2018': float(c), 'median_iqr_sep_2017': float(d),
                 'lost': bool(l), 'direction_flipped': bool(fl)}
                for f, a, b, c, d, l, fl in zip(self.f2, a18, a17, s18, s17, lost, flipped)]
        self.R['separation'] = {
            'primary_statistic': ('univariate ROC-AUC of each feature for Attack vs Normal; '
                                  'informative if |AUC-0.5| >= 0.1'),
            'why_not_median_iqr': ('the median/IQR statistic is blind to zero-inflated features '
                                   '(e.g. a flag present in 5% vs 30% of flows scores 0); it is '
                                   'kept for comparison with the earlier paper only'),
            'sample_per_dataset': int(n), 'features': rows,
            'n_informative_2018': int(inf18.sum()), 'n_informative_2017': int(inf17.sum()),
            'lost_2018_to_2017': [r['feature'] for r in rows if r['lost']],
            'direction_flipped': [r['feature'] for r in rows if r['direction_flipped']],
            'median_iqr_below_0.1': {'2018': int(np.sum(s18 < 0.1)),
                                     '2017': int(np.sum(s17 < 0.1))}}
        print(f'  informative features (|AUC-0.5|>=0.1): 2018 {int(inf18.sum())} | 2017 '
              f'{int(inf17.sum())} of {len(self.f2)}')
        print(f'  informative in 2018 but not 2017: {int(lost.sum())} | '
              f'direction FLIPPED (attack high in one year, low in the other): {int(flipped.sum())}')
        top = sorted(rows, key=lambda r: -abs(r['auc_2018'] - 0.5))[:12]
        print(f'  {"feature (top-12 by 2018 signal)":<32} {"AUC18":>6} {"AUC17":>6}')
        for r in top:
            tag = ' FLIPPED' if r['direction_flipped'] else (' lost' if r['lost'] else '')
            print(f'  {r["feature"]:<32} {r["auc_2018"]:>6.3f} {r["auc_2017"]:>6.3f}{tag}')
        print(f'  (median/IQR statistic, for comparison only: below 0.1 in 2018 '
              f'{int(np.sum(s18 < 0.1))}, 2017 {int(np.sum(s17 < 0.1))})')
        self._sep = (a18, a17)

    # ------------------------------------------------------------ P4B
    def p4b_family_shift(self):
        """X9: the pooled Attack-vs-Normal comparison mixes different attack
        compositions (2018: HOIC/LOIC floods; 2017: Hulk/generic DDoS). Here each
        SHARED family is compared with Normal separately in each year, so a flip
        means the SAME attack looks different, not a different attack mix."""
        print('[P4B] Per-family shift: same family vs Normal, 2018 vs 2017')
        cap, n_norm = 50_000, 100_000
        pairs = {'BruteForce-FTP': ['BruteForce-FTP'], 'BruteForce-SSH': ['BruteForce-SSH'],
                 'DoS-GoldenEye': ['DoS-GoldenEye'], 'DoS-Hulk': ['DoS-Hulk'],
                 'DoS-SlowHTTP': ['DoS-SlowHTTP'], 'DoS-Slowloris': ['DoS-Slowloris'],
                 'Botnet': ['Botnet'],
                 'DDoS(generic)': ['DDoS-HOIC', 'DDoS-LOIC-HTTP', 'DDoS-LOIC-UDP']}
        want18 = {t for v in pairs.values() for t in v}
        path = self.m1['l2_input_paths']['test_known']
        if not os.path.exists(path):
            path = str(self.l1 / 'l2_ready' / Path(path).name)
        got = {k: [] for k in want18 | {'Normal'}}; cnt = {k: 0 for k in got}
        for b in pq.ParquetFile(path).iter_batches(batch_size=250_000,
                                                   columns=self.f2 + ['Target']):
            df = b.to_pandas()
            for k in got:
                lim = n_norm if k == 'Normal' else cap
                if cnt[k] >= lim:
                    continue
                sub = df.loc[df['Target'] == k, self.f2].to_numpy(np.float32)[:lim - cnt[k]]
                if len(sub):
                    got[k].append(sub); cnt[k] += len(sub)
            if all(cnt[k] >= (n_norm if k == 'Normal' else cap) for k in got):
                break
        X18 = {k: (np.vstack(v) if v else np.zeros((0, len(self.f2)), np.float32))
               for k, v in got.items()}
        rng = np.random.RandomState(SEED)

        def pick17(lbl, lim):
            idx = np.flatnonzero(self.fam == lbl)
            if len(idx) > lim:
                idx = rng.choice(idx, lim, replace=False)
            return self.X[idx][:, self.i2]
        N17 = pick17('Normal', n_norm)
        top_shap = []
        m6p = Path(os.environ.get('T6_L6', '')) / 'manifest_l6.json'
        if m6p.exists():
            with open(m6p) as fh:
                top_shap = [t[0] for t in json.load(fh).get('global_top20', [])[:10]]
        rows = []
        print(f'  {"family":<15} {"n18":>7} {"n17":>7} {"inf18":>6} {"lost":>5} '
              f'{"flipped":>8} {"flipped in SHAP top-10":<}')
        for fam17, fams18 in pairs.items():
            A18 = np.vstack([X18[t] for t in fams18 if len(X18[t])]) if any(
                len(X18[t]) for t in fams18) else np.zeros((0, len(self.f2)))
            A17 = pick17(fam17, cap)
            if len(A18) < 200 or len(A17) < 200:
                rows.append({'family': fam17, 'n18': int(len(A18)), 'n17': int(len(A17)),
                             'skipped': 'fewer than 200 rows in a year'})
                print(f'  {fam17:<15} {len(A18):>7,} {len(A17):>7,}   skipped (<200 rows)')
                continue
            a18 = _uni_auc(np.vstack([X18['Normal'], A18]),
                           np.r_[np.zeros(len(X18['Normal'])), np.ones(len(A18))].astype(int))
            a17 = _uni_auc(np.vstack([N17, A17]),
                           np.r_[np.zeros(len(N17)), np.ones(len(A17))].astype(int))
            i18, i17 = np.abs(a18 - 0.5) >= 0.1, np.abs(a17 - 0.5) >= 0.1
            fl = i18 & i17 & (np.sign(a18 - 0.5) != np.sign(a17 - 0.5))
            lo = i18 & ~i17
            fl_names = [self.f2[j] for j in np.flatnonzero(fl)]
            fl_top = [f for f in top_shap if f in fl_names]
            lo_top = [f for f in top_shap if f in [self.f2[j] for j in np.flatnonzero(lo)]]
            rows.append({'family': fam17, 'families_2018': fams18,
                         'n18': int(len(A18)), 'n17': int(len(A17)),
                         'n_informative_2018': int(i18.sum()), 'n_lost': int(lo.sum()),
                         'n_flipped': int(fl.sum()), 'flipped': fl_names,
                         'flipped_in_shap_top10': fl_top, 'lost_in_shap_top10': lo_top,
                         'auc_2018': dict(zip(self.f2, map(float, a18))),
                         'auc_2017': dict(zip(self.f2, map(float, a17)))})
            print(f'  {fam17:<15} {len(A18):>7,} {len(A17):>7,} {int(i18.sum()):>6} '
                  f'{int(lo.sum()):>5} {int(fl.sum()):>8}   '
                  f'{", ".join(fl_top) or "-"}' + (f' | lost: {", ".join(lo_top)}' if lo_top else ''))
        done = [r for r in rows if 'skipped' not in r]
        self.R['family_shift'] = {
            'method': ('per shared family: univariate AUC of each L2 feature for Family vs '
                       'Normal, computed separately in 2018 test and 2017; informative if '
                       '|AUC-0.5|>=0.1; flipped = informative in both years with opposite '
                       'direction; lost = informative in 2018 only'),
            'caps': {'per_family': cap, 'normal': n_norm}, 'shap_top10_source': top_shap,
            'families': rows,
            'median_flipped_per_family': (float(np.median([r['n_flipped'] for r in done]))
                                          if done else None)}
        if done:
            print(f'  median flipped features per family: '
                  f'{self.R["family_shift"]["median_flipped_per_family"]:.0f} of {len(self.f2)} '
                  '(same attack family, different year)')

    # ------------------------------------------------------------ P5
    def p5_figures_save(self):
        print('[P5] Figures + manifest')
        plt.rcParams.update({'pdf.fonttype': 42, 'font.family': 'serif', 'font.size': 8})
        labs, f17, ref = self._fine
        if labs:
            x = np.arange(len(labs)); w = 0.4
            fig, ax = plt.subplots(figsize=(7.2, 2.6))
            ax.bar(x - w / 2, [ref.get(l, np.nan) for l in labs], w, label='IDS-2018 test')
            ax.bar(x + w / 2, f17, w, label='IDS-2017 zero-shot')
            ax.set_xticks(x); ax.set_xticklabels(labs, rotation=35, ha='right')
            ax.set_ylabel('F1'); ax.set_ylim(0, 1.05); ax.legend(frameon=False)
            fig.tight_layout(); fig.savefig(self.out / 'X_fig1_family_f1.pdf'); plt.close(fig)
        s18, s17 = self._sep
        fig, ax = plt.subplots(figsize=(3.5, 3.2))
        a = np.nan_to_num(s18, nan=0.5); b = np.nan_to_num(s17, nan=0.5)
        ax.scatter(a, b, s=9)
        ax.plot([0, 1], [0, 1], ls='--', lw=0.7, color='grey')
        ax.axhline(0.5, lw=0.6, color='red'); ax.axvline(0.5, lw=0.6, color='red')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel('univariate AUC, IDS-2018 test'); ax.set_ylabel('univariate AUC, IDS-2017')
        ax.set_title('points in the off-diagonal quadrants = direction flipped', fontsize=7)
        fig.tight_layout(); fig.savefig(self.out / 'X_fig2_separation.pdf'); plt.close(fig)
        _atomic_json(self.R, self.out / 'manifest_x2017.json')
        print(f'  written: {self.out / "manifest_x2017.json"} + 2 figures')

    def run(self):
        t0 = time.time()
        for step in (self.p0_contracts, self.p1_load_transform, self.p2_layer2,
                     self.p3_layer3, self.p4_separation, self.p4b_family_shift,
                     self.p5_figures_save):
            step(); gc.collect()
        print('=' * 76)
        print(f'  CROSS-DATASET R8.0 COMPLETE | {(time.time() - t0) / 60:.1f} min | '
              f'profile={self.R["profile"]}')
        print('=' * 76)


if __name__ == '__main__':
    CrossDataset2017().run()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
[P0] Contracts (R8 manifests, Layer-1 code)
  Layer-1 transform: sha256 4d4f9ab87404 | embedded copy (Layer-1 file not found to compare)
  L1 TITANIUM6_LAYER1_R8.0 | L2 TITANIUM6_LAYER2_R8.0 (LightGBM, tau=0.0793, 77 feats) | L3 TITANIUM6_LAYER3_R8.0 (12 classes)
  excluded features in this profile: none
[P1] Load CIC-IDS2017 and transform with the Layer-1 OnlineTransformer
  dataset: /kaggle/input/datasets/dhoogla/cicids2017 | 8 files
    Benign-Monday-no-metadata.parquet             kept   458,816
    Botnet-Friday-no-metadata.parquet             kept   176,029
    Bruteforce-Tuesday-no-metadata.parquet        kept   389,698
    DDoS-Friday-no-metadata.parquet               kept   221,262
    DoS-Wednesday-no-metadata.parquet             kept   584,959
    Infiltration-Thursday-no-metadata.parquet     kept   207,626
    Portscan-Friday-no-metadata.parquet           kept   117,556
    WebAttacks-Thursday-no-metadata.parq

In [11]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | 10 | WHERE DO THE LAYER-2 FALSE POSITIVES COME FROM? | R8.0
#  Run after Layer 2 (any profile). ~10 min. Reads only saved artifacts.
#
#  Why this exists: at the SAME deployed threshold, Layer 2's false-positive
#  rate on Normal traffic is ~0.04% on validation but 8.5% on test. The paper
#  must say WHY. Two explanations predict different evidence:
#   (A) broad temporal drift      -> FPs spread over many test days/hours
#   (B) a local shift/label issue -> FPs concentrated in a few days/hours
#  This script measures which one the data shows. It does not change any
#  model, threshold or headline number.
# =============================================================================
# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
# -----------------------------------------------------------------------------

import os, json, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

E = os.environ
L1, L2 = Path(E['T6_L1']), Path(E['T6_L2'])
OUT = L2 / 'fp_diagnostic'
OUT.mkdir(parents=True, exist_ok=True)


def _load(p):
    with open(p) as fh:
        return json.load(fh)


def _pick(p):
    with open(p, 'rb') as fh:
        return pickle.load(fh)


def _day(ts_ns):
    return pd.to_datetime(ts_ns, unit='ns').strftime('%Y-%m-%d')


def _hour(ts_ns):
    return pd.to_datetime(ts_ns, unit='ns').strftime('%Y-%m-%d %H:00')


def main():
    t0 = time.time()
    m1, m2 = _load(L1 / 'manifest.json'), _load(L2 / 'manifest_l2.json')
    c = m2['l3_l4_contract']
    th = _load(c['threshold_json'])
    name = m2['best_model']
    tau = float(th['deployed_tau'][name])
    tau_b = float(th.get('all_rules_deployed', {}).get('fpr_budget', np.nan))
    feats, kind = list(c['features']), c['model_kind']
    print('=' * 76)
    print(f'  TITANIUM-6 | 10 | L2 false-positive diagnostic | profile={E.get("T6_PROFILE")}')
    print(f'  model {name} ({kind}) | tau {tau:.4f} | budget tau {tau_b:.4f} | {len(feats)} features')
    print('=' * 76)
    R = {'profile': E.get('T6_PROFILE'), 'tau_deployed': tau, 'tau_budget': tau_b}

    # ---------------------------------------------------------------- TEST
    s = pd.read_parquet(L2 / 'l2_scores_test_known.parquet')
    nrm = s[s['Target'] == 'Normal'].copy()
    fpr_test = float(nrm['flag'].mean())
    ref = float(m2['test']['DEPLOYED']['full']['fpr'])
    if abs(fpr_test - ref) > 1e-6:
        raise SystemExit(f'[FATAL] recomputed test FPR {fpr_test:.6f} != manifest {ref:.6f}')
    print(f'[TEST] Normal {len(nrm):,} | FP {int(nrm.flag.sum()):,} | FPR {100 * fpr_test:.3f}% '
          f'(matches manifest)')
    nrm['day'] = _day(nrm['_ts'].values)
    nrm['hour'] = _hour(nrm['_ts'].values)
    by_day = nrm.groupby('day')['flag'].agg(['size', 'sum', 'mean'])
    by_day.columns = ['normal', 'fp', 'fpr']
    by_day['share_of_all_fp'] = by_day['fp'] / max(by_day['fp'].sum(), 1)
    print('  by day (test Normal):')
    for d, r in by_day.iterrows():
        print(f'    {d}  normal {int(r.normal):>9,}  FP {int(r.fp):>8,}  FPR {100 * r.fpr:6.2f}%  '
              f'share of FPs {100 * r.share_of_all_fp:5.1f}%')
    by_hour = nrm.groupby('hour')['flag'].agg(['size', 'sum'])
    top = by_hour.sort_values('sum', ascending=False)
    tot = max(int(nrm.flag.sum()), 1)
    conc = {k: float(top['sum'].head(k).sum() / tot) for k in (1, 3, 5, 10)}
    print(f'  hours with Normal traffic: {len(by_hour)} | share of FPs in top-1/3/5/10 hours: '
          + ' / '.join(f'{100 * v:.1f}%' for v in conc.values()))
    fp = nrm[nrm.flag]
    twin_fp = float(fp['_dup_in_train'].mean()) if len(fp) else float('nan')
    twin_all = float(nrm['_dup_in_train'].mean())
    conf = {f'p_ge_{t}': float((fp['p_cal'] >= t).mean()) for t in (0.5, 0.9, 0.99)}
    print(f'  FPs with an exact twin in train: {100 * twin_fp:.1f}% (all test Normal: '
          f'{100 * twin_all:.1f}%)')
    print(f'  FP confidence: p>=0.5 {100 * conf["p_ge_0.5"]:.1f}% | p>=0.9 '
          f'{100 * conf["p_ge_0.9"]:.1f}% | p>=0.99 {100 * conf["p_ge_0.99"]:.1f}%')
    worst = by_day['fp'].idxmax()
    rest = nrm[nrm.day != worst]
    print(f'  sensitivity only (NOT a headline): FPR without worst day {worst} = '
          f'{100 * rest.flag.mean():.3f}%')
    R['test'] = {'fpr': fpr_test, 'by_day': by_day.reset_index().to_dict('records'),
                 'n_hours': int(len(by_hour)), 'fp_share_top_k_hours': conc,
                 'fp_twin_in_train_share': twin_fp, 'normal_twin_in_train_share': twin_all,
                 'fp_confidence': conf, 'worst_day': worst,
                 'fpr_without_worst_day_sensitivity_only': float(rest.flag.mean()),
                 'top_hours': [{'hour': h, 'normal': int(r['size']), 'fp': int(r['sum'])}
                               for h, r in top.head(10).iterrows()]}

    # ---------------------------------------------------------------- VAL
    model, cal = _pick(c['model']), _pick(c['cal'])
    if kind == 'xgb':
        import xgboost as xgb

    def score(X):
        if kind == 'lgb':
            p = model.predict(X)
        elif kind == 'xgb':
            p = model.predict(xgb.DMatrix(X, feature_names=feats))
        else:
            p = model.predict_proba(X)[:, 1]
        return cal.predict_proba(np.asarray(p, np.float64).reshape(-1, 1))[:, 1]

    vpath = m1['l2_input_paths']['val']
    rows = []
    for b in pq.ParquetFile(vpath).iter_batches(batch_size=500_000,
                                                columns=feats + ['Target', '_ts']):
        df = b.to_pandas()
        df = df[df['Target'] == 'Normal']
        if len(df):
            p = score(df[feats].to_numpy(np.float32))
            rows.append(pd.DataFrame({'_ts': df['_ts'].values, 'fp': p >= tau,
                                      'fp_budget': p >= tau_b}))
    v = pd.concat(rows, ignore_index=True)
    v['day'] = _day(v['_ts'].values)
    vd = v.groupby('day')[['fp', 'fp_budget']].agg(['size', 'mean'])
    print(f'[VAL] Normal {len(v):,} | FPR at deployed tau {100 * v.fp.mean():.3f}% | '
          f'at budget tau {100 * v.fp_budget.mean():.3f}% (full validation, both halves)')
    for d in vd.index:
        print(f'    {d}  normal {int(vd.loc[d, ("fp", "size")]):>9,}  FPR {100 * vd.loc[d, ("fp", "mean")]:6.3f}%')
    R['val'] = {'fpr_deployed_tau': float(v.fp.mean()), 'fpr_budget_tau': float(v.fp_budget.mean()),
                'by_day': {d: {'normal': int(vd.loc[d, ('fp', 'size')]),
                               'fpr': float(vd.loc[d, ('fp', 'mean')])} for d in vd.index}}

    # ---------------------------------------------------------------- spans
    spans = {}
    for part in ('train', 'val', 'test_known'):
        t = pq.read_table(m1['l2_input_paths'][part], columns=['Target', '_ts']).to_pandas()
        t = t[t['Target'] == 'Normal']['_ts']
        spans[part] = [str(pd.to_datetime(t.min(), unit='ns')), str(pd.to_datetime(t.max(), unit='ns'))]
        del t
    print('[SPANS] Normal traffic time span per partition:')
    for k, (a, b) in spans.items():
        print(f'    {k:<11} {a}  ->  {b}')
    R['normal_time_spans'] = spans

    # ---------------------------------------------------------------- verdict
    # Hour-level concentration against an even spread (the test period has only
    # a few calendar days, so a day-level share is not informative).
    n_h = max(len(by_hour), 1)
    fair10 = min(1.0, 10 / n_h)
    R['verdict_rule'] = ('CONCENTRATED if the 5 worst hours hold >=50% of FPs; SPREAD if '
                         'the 10 worst hours hold <= min(50%, 3x an even share); else MIXED')
    if conc[5] >= 0.5:
        verdict = ('CONCENTRATED: most false positives come from a few test days/hours. '
                   'Report as a localized shift; inspect those windows (labels, hosts, '
                   'capture events) before attributing it to general drift.')
    elif conc[10] <= min(0.5, 3 * fair10):
        verdict = ('SPREAD: false positives are distributed across the test period, '
                   'consistent with broad temporal drift of Normal traffic.')
    else:
        verdict = ('MIXED: partly concentrated, partly spread; report both the '
                   'distribution and the worst-window sensitivity.')
    R['verdict'] = verdict
    print(f'[VERDICT] {verdict}')

    # ---------------------------------------------------------------- figure
    plt.rcParams.update({'pdf.fonttype': 42, 'font.family': 'serif', 'font.size': 8})
    fig, ax = plt.subplots(figsize=(7.2, 2.6))
    vx = list(vd.index); tx = list(by_day.index)
    ax.bar(vx, [100 * vd.loc[d, ('fp', 'mean')] for d in vx], color='#4c72b0', label='validation')
    ax.bar(tx, 100 * by_day['fpr'].values, color='#c44e52', label='test')
    ax.set_ylabel('Normal-flow FPR (%)'); ax.set_title(f'Layer-2 FPR by day at tau={tau:.4f}')
    ax.tick_params(axis='x', rotation=35); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(OUT / 'D_fig1_fpr_by_day.pdf'); plt.close(fig)
    with open(OUT / 'fp_diagnostic.json', 'w') as fh:
        json.dump(R, fh, indent=2, default=str)
    print(f'  written: {OUT}/fp_diagnostic.json + D_fig1_fpr_by_day.pdf | '
          f'{(time.time() - t0) / 60:.1f} min')
    print('=' * 76)


if __name__ == '__main__':
    main()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
  TITANIUM-6 | 10 | L2 false-positive diagnostic | profile=main
  model LightGBM (lgb) | tau 0.0793 | budget tau 0.0462 | 77 features
[TEST] Normal 2,022,705 | FP 172,318 | FPR 8.519% (matches manifest)
  by day (test Normal):
    2018-02-23  normal   478,084  FP      302  FPR   0.06%  share of FPs   0.2%
    2018-02-28  normal   544,200  FP  110,888  FPR  20.38%  share of FPs  64.4%
    2018-03-01  normal   238,037  FP   43,052  FPR  18.09%  share of FPs  25.0%
    2018-03-02  normal   762,384  FP   18,076  FPR   2.37%  share of FPs  10.5%
  hours with Normal traffic: 36 | share of FPs in top-1/3/5/10 hours: 16.6% / 36.3% / 49.9% / 69.6%
  FPs with an exact twin in train: 23.2% (all test Normal: 35.8%)
  FP confidence: p>=0.5 48.1% | p>=0.9 21.2% | p>=0.99 0.0%
  sensitivity only (NOT a headline): FPR without worst day 2018-02-28 = 4.155%
[VAL] Normal 1,984,267 | FPR at deployed tau 0.040% | at budget tau 1.012% (full va

In [12]:
#!/usr/bin/env python3
# =============================================================================
#  TITANIUM-6 | 11 | LAYER-7 API BENCHMARK (end-to-end, over HTTP) | R8.0
#  Run AFTER 07_layer7_api.py is serving, as a SEPARATE process so the client
#  does not share the server's Python interpreter:
#      !python /kaggle/input/titanium6-r8/11_api_benchmark.py
#  Measures what the reviewer asked for: real request->decision latency
#  through the deployed API (not a sum of per-layer P95s), per-flow cost of
#  the batch endpoint, and the decision mix on a natural-prior test sample.
#  Reads keys from <T6_L7>/api_keys.json (never printed).
# =============================================================================
# --- R8 run-config bootstrap -------------------------------------------------
# Loads the ACTIVE profile written by 00_run_config.py, so this layer runs the
# same way in the same kernel, in a fresh kernel, or as a script.
import os as _t6_os, json as _t6_json
_T6_ACTIVE = '/kaggle/working/t6_active_config.json'
if not _t6_os.path.exists(_T6_ACTIVE):
    raise SystemExit(f'[FATAL] {_T6_ACTIVE} not found: run 00_run_config.py first')
with open(_T6_ACTIVE) as _t6_fh:
    _t6_os.environ.update(_t6_json.load(_t6_fh)['env'])
print(f"[R8] profile={_t6_os.environ['T6_PROFILE']} | config={_T6_ACTIVE}")
import logging as _t6_logging   # silence matplotlib's PDF font-subsetting INFO spam
_t6_logging.getLogger('fontTools').setLevel(_t6_logging.WARNING)
_t6_logging.getLogger('matplotlib').setLevel(_t6_logging.WARNING)
# -----------------------------------------------------------------------------

import os, sys, json, time, platform, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq

E = os.environ
N_SINGLE, N_WARM, BATCH, N_BATCH = 1000, 30, 256, 8
SEED = 42
PACE_RPS = 100.0          # below L7's security rate limit (200 req/s per key)
URL = None                # resolved in main(): T6_API_URL > endpoint.json > :8000
RETRIES = {'429': 0}


def _resolve_url(l7):
    if E.get('T6_API_URL'):
        return E['T6_API_URL'], 'T6_API_URL'
    ep = Path(l7) / 'endpoint.json'
    if ep.exists():
        with open(ep) as fh:
            return json.load(fh)['url'], str(ep)
    return 'http://127.0.0.1:8000', 'default'


def _req(path, payload=None, key=None, timeout=30):
    data = None if payload is None else json.dumps(payload).encode()
    r = urllib.request.Request(URL + path, data=data,
                               method='GET' if payload is None else 'POST')
    r.add_header('Content-Type', 'application/json')
    if key:
        r.add_header('X-API-Key', key)
    for attempt in range(6):
        t0 = time.perf_counter()
        try:
            with urllib.request.urlopen(r, timeout=timeout) as resp:
                body = json.loads(resp.read().decode())
            return body, (time.perf_counter() - t0) * 1e3
        except urllib.error.HTTPError as e:
            if e.code == 429 and attempt < 5:      # rate limited: back off, retry
                RETRIES['429'] += 1
                time.sleep(0.25 * (attempt + 1))
                continue
            raise SystemExit(f'[FATAL] {path} -> HTTP {e.code}: '
                             f'{e.read().decode(errors="replace")[:300]}')


def _pct(a):
    a = np.asarray(a, float)
    return {'mean': float(a.mean()), 'p50': float(np.percentile(a, 50)),
            'p95': float(np.percentile(a, 95)), 'p99': float(np.percentile(a, 99)),
            'n': int(len(a))}


def _paced(i, t_start):
    """Keep request i at or after t_start + i/PACE_RPS (latency is measured
    per request; pacing only keeps the client under the server's rate limit)."""
    wait = t_start + i / PACE_RPS - time.perf_counter()
    if wait > 0:
        time.sleep(wait)


def main():
    global URL
    l1, l2, l7 = Path(E['T6_L1']), Path(E['T6_L2']), Path(E['T6_L7'])
    URL, url_src = _resolve_url(l7)
    in_kernel = 'ipykernel' in sys.modules
    if in_kernel:
        print('[WARN] running INSIDE the notebook kernel, i.e. the same Python process '
              'as the API server: client and server share one interpreter, so latency is '
              'inflated. For paper numbers run it as a separate process: '
              '!python /kaggle/working/11_api_benchmark.py')
    kf = l7 / 'api_keys.json'
    if kf.exists():
        with open(kf) as fh:
            keys = json.load(fh)
    elif E.get('T6_API_KEYS'):
        keys = {k.split('=')[0]: k.split('=')[1].split(',')
                for k in E['T6_API_KEYS'].split(';') if '=' in k}
    else:
        raise SystemExit(f'[FATAL] no API keys: {kf} missing and T6_API_KEYS unset')
    key = next((k for k, sc in keys.items() if 'ingest' in sc), None)
    if key is None:
        raise SystemExit('[FATAL] no key with the ingest scope')
    try:
        health, _ = _req('/v1/health')
    except (urllib.error.URLError, ConnectionError) as e:
        raise SystemExit(f'[FATAL] API not reachable at {URL} ({e}); run 07 first')
    with open(l1 / 'manifest.json') as fh:
        m1 = json.load(fh)
    with open(l2 / 'manifest_l2.json') as fh:
        feats = json.load(fh)['l3_l4_contract']['features']
    pf = pq.ParquetFile(m1['l2_input_paths']['test_known'])
    df = pf.read_row_group(0, columns=feats + ['Target']).to_pandas()
    df = df.sample(n=min(N_SINGLE + N_WARM, len(df)), random_state=SEED)
    flows = [{f: float(v) for f, v in zip(feats, row)} for row in df[feats].to_numpy()]
    truth = df['Target'].to_numpy()
    print('=' * 76)
    print(f'  TITANIUM-6 | 11 | API benchmark | {URL} (from {url_src}) | '
          f'profile={E.get("T6_PROFILE")}')
    print(f'  health: {health} | client process: '
          f'{"notebook kernel (inflated)" if in_kernel else "separate"} | paced at '
          f'{PACE_RPS:.0f} req/s')
    print('=' * 76)

    t_start = time.perf_counter()
    for i, fl in enumerate(flows[:N_WARM]):
        _paced(i, t_start)
        _req('/v1/classify', {'flow': fl}, key)
    cli, srv, fam, act, atk, zd, layers = [], [], [], [], [], [], []
    t_start = time.perf_counter()
    for i, fl in enumerate(flows[N_WARM:]):
        _paced(i, t_start)
        out, ms = _req('/v1/classify', {'flow': fl}, key)
        cli.append(ms); srv.append(out['latency_ms'])
        fam.append(out['family']); act.append(out['action'])
        atk.append(out['is_attack']); zd.append(out['is_zeroday'])
        layers.append('+'.join(out['layers_used']))
    y = truth[N_WARM:] != 'Normal'
    a = np.array(atk)
    single_cli, single_srv = _pct(cli), _pct(srv)
    print(f'[SINGLE] {len(cli)} sequential requests (new HTTP connection each)')
    print(f'  client round-trip ms : P50 {single_cli["p50"]:.2f} | P95 {single_cli["p95"]:.2f} | '
          f'P99 {single_cli["p99"]:.2f}')
    print(f'  server-side ms       : P50 {single_srv["p50"]:.2f} | P95 {single_srv["p95"]:.2f} | '
          f'P99 {single_srv["p99"]:.2f}')

    bt = []
    for i in range(N_BATCH):
        chunk = [flows[(N_WARM + i * BATCH + j) % len(flows)] for j in range(BATCH)]
        _, ms = _req('/v1/classify/batch', {'flows': chunk}, key, timeout=120)
        bt.append(ms / BATCH)
    batch = _pct(bt)
    print(f'[BATCH ] {N_BATCH} x {BATCH} flows: per-flow ms mean {batch["mean"]:.3f} | '
          f'P95 of batch means {batch["p95"]:.3f}')

    tp, fp = int((a & y).sum()), int((a & ~y).sum())
    mix = {'actions': dict(zip(*np.unique(act, return_counts=True))),
           'layers_used': dict(zip(*np.unique(layers, return_counts=True)))}
    mix = {k: {str(kk): int(vv) for kk, vv in v.items()} for k, v in mix.items()}
    desc = {'n': int(len(y)), 'attacks': int(y.sum()),
            'is_attack_recall': tp / max(int(y.sum()), 1),
            'is_attack_fpr': fp / max(int((~y).sum()), 1),
            'zero_day_flags': int(np.sum(zd))}
    print(f'[MIX   ] actions {mix["actions"]}')
    print(f'         layers  {mix["layers_used"]}')
    print(f'         sample: n={desc["n"]} attacks={desc["attacks"]} | is_attack recall '
          f'{desc["is_attack_recall"]:.3f} | FPR {100 * desc["is_attack_fpr"]:.2f}% '
          '(descriptive; small sample)')
    print(f'         429 retries: {RETRIES["429"]}')
    R = {'url': URL, 'url_source': url_src, 'profile': E.get('T6_PROFILE'),
         'protocol': ('sequential POST /v1/classify, one new HTTP/1.1 connection per '
                      f'request, JSON body, {N_WARM} warm-up requests excluded, paced at '
                      f'{PACE_RPS:.0f} req/s (below the 200 req/s per-key rate limit)'),
         'client_process': 'notebook kernel (shares the server interpreter; latency '
                           'inflated)' if in_kernel else 'separate process, same host',
         'rate_limit_retries_429': RETRIES['429'],
         'single_client_ms': single_cli, 'single_server_ms': single_srv,
         'batch_per_flow_ms': batch, 'batch_size': BATCH,
         'decision_mix': mix, 'sample_detection_descriptive': desc,
         'hardware': {'platform': platform.platform(), 'python': platform.python_version(),
                      'cpu_count': os.cpu_count(), 'processor': platform.processor()},
         'excludes': 'CICFlowMeter flow extraction (not part of the served pipeline)'}
    with open(l7 / 'api_benchmark.json', 'w') as fh:
        json.dump(R, fh, indent=2)
    print(f'  written: {l7 / "api_benchmark.json"}')
    print('=' * 76)


if __name__ == '__main__':
    main()

[R8] profile=main | config=/kaggle/working/t6_active_config.json
[WARN] running INSIDE the notebook kernel, i.e. the same Python process as the API server: client and server share one interpreter, so latency is inflated. For paper numbers run it as a separate process: !python /kaggle/working/11_api_benchmark.py
  TITANIUM-6 | 11 | API benchmark | http://127.0.0.1:8000 (from /kaggle/working/l7_R8/endpoint.json) | profile=main
  health: {'status': 'ok'} | client process: notebook kernel (inflated) | paced at 100 req/s
[SINGLE] 1000 sequential requests (new HTTP connection each)
  client round-trip ms : P50 3.01 | P95 7.45 | P99 8.38
  server-side ms       : P50 0.93 | P95 5.40 | P99 6.00
[BATCH ] 8 x 256 flows: per-flow ms mean 2.240 | P95 of batch means 2.379
[MIX   ] actions {'ALERT_SOC': 85, 'AUTO_BLOCK': 75, 'AUTO_RATE_LIMIT': 24, 'ISOLATE': 6, 'LOG_MONITOR': 3, 'NO_ACTION': 807}
         layers  {'L2': 805, 'L2+L3+L4': 2, 'L2+L3+L4+L5': 193}
         sample: n=1000 attacks=132 | is_

In [13]:
# ---- TITANIUM-6: package results for download ---------------------------
import os, zipfile, time
from pathlib import Path
W = Path('/kaggle/working')
SKIP_EXT  = {'.parquet', '.npy'}          # multi-GB data splits: re-creatable, not results
SKIP_NAME = ('audit', 'api_keys')         # 380 MB audit log (its hash check is in manifest_l5); secret keys
MAX_MB = 200
RESULT_EXT = {'.json', '.pdf', '.png', '.csv', '.txt', '.md'}
MODEL_EXT  = {'.pkl', '.pt', '.txt', '.json', '.joblib', '.ubj'}

def pack(name, keep):
    out = W / name; n = 0; size = 0
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(W.rglob('*')):
            if not p.is_file() or p.suffix == '.zip' or p.suffix in SKIP_EXT:
                continue
            rel = p.relative_to(W)
            if rel.parts[0].startswith('.') or any(s in p.name.lower() for s in SKIP_NAME):
                continue
            if p.stat().st_size > MAX_MB * 2**20 or not keep(p):
                continue
            z.write(p, rel); n += 1; size += p.stat().st_size
    print(f'  {name:<34} {n:>4} files | {size/2**20:7.1f} MB -> {out.stat().st_size/2**20:7.1f} MB zip')
    return out

stamp = time.strftime('%Y%m%d_%H%M')
print('Packaging /kaggle/working ...')
z1 = pack(f'titanium6_RESULTS_{stamp}.zip', lambda p: p.suffix in RESULT_EXT)
z2 = pack(f'titanium6_MODELS_{stamp}.zip',  lambda p: p.suffix in MODEL_EXT)
    
from IPython.display import FileLink, display
print('\nClick to download:')
for z in (z1, z2):
    display(FileLink(z.name, result_html_prefix='⬇ '))  

Packaging /kaggle/working ...
  titanium6_RESULTS_20260925_1239.zip   87 files |    10.7 MB ->     5.9 MB zip
  titanium6_MODELS_20260925_1239.zip   39 files |    28.0 MB ->    10.1 MB zip

Click to download:


/kaggle/working/titanium6_RESULTS_20260925_1239.zip

/kaggle/working/titanium6_MODELS_20260925_1239.zip

In [14]:
from IPython.display import FileLink
FileLink('l7_R8/api_benchmark.json')

/kaggle/working/l7_R8/api_benchmark.json